In [1]:
import tensorflow as tf
import tensorflow.keras as keras
import tensorflow_probability as tfp
tfb = tfp.bijectors
from tensorflow_probability.python import math as tfp_math
from tensorflow_probability.python.internal import prefer_static
import sonnet as snt

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import AutoMinorLocator, FixedLocator, MultipleLocator

import astropy.io.fits as fits

import scipy

import time
import json
from tqdm.auto import tqdm

from astroquery.gaia import Gaia

import plot_utils

2025-12-29 14:34:45.850470: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-29 14:34:45.875313: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767015285.892243 2429448 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767015285.897442 2429448 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767015285.910189 2429448 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Workaround solutions for the Gaia Archive issues following the infrastructure upgrade: https://www.cosmos.esa.int/web/gaia/news#WorkaroundArchive


In [2]:
physical_devices = tf.config.list_physical_devices('GPU')
try:
    for device in physical_devices:
        tf.config.experimental.set_memory_growth(device, True)
except:
    # Invalid device or cannot modify virtual devices once initialized.
    pass

In [3]:
# Use color cycle generated using tool at:
# https://medialab.github.io/iwanthue/

from cycler import cycler
color_cycle = cycler(color=[
    "#019422",
    "#ff5fd9",
    "#3cdeb0",
    "#da005a",
    "#9f9300",
    "#0147c7",
    "#f65135",
    "#b587ff",
    "#784702",
    "#6d4081"
])
plt.rc('axes', prop_cycle=(color_cycle))

# Bijector

Define a bijector, $b_c : \vec{x} \mapsto \vec{y}$, which moves coordinates along curves defined by local "velocity" fields. The transformation depends on a categorical variable, $c$. Effectively, there is a different bijection for each value of $c$. The transformation is defined by the following differential equation:
$$
  \frac{\mathrm{d}\vec{x}}{\mathrm{d}t} = \vec{v}_{c}\left(t,\vec{x}\right) \, ,
$$
with $\vec{x}\left(0\right) = \vec{x}$ and $\vec{y} = \vec{x}\left(1\right)$.

The inverse, $b^{-1} : \vec{y} \mapsto \vec{x}$, is easy to obtain: integrate the above ODE backwards, from $t = 1$ to $t = 0$.

For the purposes of training, we also implement a function that calculates the path length of the transformation for each input coordinate. During training, we can regularize the transformation by penalizing larger path lengths.

In [435]:
class VectorNetwork(snt.Module):
    """
    Neural network that takes a vector and a categorical variable as input,
    and outputs a vector. Internally, the network learns an embedding for
    the categorical variable, so that each category gets an embedding
    vector e. Then, given input vector x and embedding vector e, a series of
    hidden layers are applied to x, where each hidden layer is modulated by e,
    using the FiLM (Feature-wise Linear Modulation) technique.
    
    Each hidden layer looks like this:

      h = Dense1(x)
      h = Activation(h)
      h = LayerNorm(h)
      f = Dense2(e) # 2x the size of h
      gamma, beta = split(f)
      h = gamma * h + beta

    Finally, a dense output layer is applied to produce the output y.
    """
    
    def __init__(self, n_cat, n_out, n_hidden, hidden_size,
                 embed_dim=None, activation=tf.nn.silu,
                 use_layer_norm=True, use_film=True,
                 use_residual=True,
                 output_scale=1e-5,
                 output_bias=0,
                 name=None):
        """
        Constructor for VectorNetwork.

        Inputs:
          n_cat (int): # of categories.
          n_out (int): Dimensionality of output (m).
          n_hidden (int): # of hidden layers.
          hidden_size (int): # of neurons in each hidden layer.
          activation (Optional[function]): Activation function to apply after
            each hidden layer.
          name (Optional[str]): Name to give to model.
        """
        super(VectorNetwork, self).__init__(name=name)

        # Embedding layer for categorical variable
        if embed_dim is None:
            embed_dim = int(np.ceil(np.log2(n_cat)))
            print(f'Using {embed_dim}-dim embedding for {n_cat} categories.')
        self.embed = snt.Embed(
            vocab_size=n_cat,
            embed_dim=embed_dim,
            name='embedding'
        )

        # Each hidden layer consists of a dense layer, layer norm, and FiLM
        self.hidden = [
            snt.Linear(hidden_size, name=f'hidden_{i}')
            for i in range(n_hidden)
        ]
        if use_layer_norm:
            self.norm = [
                snt.LayerNorm(
                    axis=-1,
                    eps=1e-1,
                    create_scale=True,
                    create_offset=True,
                    name=f'norm_{i}'
                )
                for i in range(n_hidden)
            ]
        if use_film:
            self.film = [
                snt.Linear(2*hidden_size, name=f'film_{i}')
                for i in range(n_hidden)
            ]
            self.film_output = snt.Linear(
                n_out,
                w_init=snt.initializers.TruncatedNormal(stddev=output_scale),
                b_init=tf.zeros_initializer(),
                name='film_output'
            )

        # Output layer
        self.output = snt.Linear(
            n_out,
            w_init=snt.initializers.TruncatedNormal(stddev=output_scale),
            b_init=tf.constant_initializer(value=output_bias),
            name='output'
        )

        self.n_cat = n_cat
        self.n_embed = embed_dim
        self.activation = activation
        self.use_layer_norm = use_layer_norm
        self.use_film = use_film
        self.use_residual = use_residual

        self.n_calls = tf.Variable(0, dtype=tf.int32, trainable=False)

    def __call__(self, x, c):
        """
        Transform x,c -> y.

        Inputs:
          x (tf.Tensor): Float-valued tensor with shape (b,n), where n is the
            dimensionality of the input and b is the batch dimension.
          c (tf.Tensor): Integer-valued tensor with shape (b,), where b is
            the batch dimension. This is the categorical variable. It should
            be in the range [0, n_cat-1].
        """
        self.n_calls.assign_add(1)

        e = self.embed(c) # Shape (b, embed_dim)

        for i,dense in enumerate(self.hidden):
            # Dense layer with activation
            h = dense(x)
            h = self.activation(h)

            # Layer normalization
            if self.use_layer_norm:
                h = self.norm[i](h)
            
            # FiLM modulation based on categorical embedding
            if self.use_film:
                f = self.film[i](e)
                gamma, beta = tf.split(f, 2, axis=-1)
                h = gamma * h + beta
            
            # Residual connection
            if self.use_residual and i > 0:
                x = x + h
            else:
                x = h
        
        # Final dense layer
        y = self.output(x)

        # FiLM bias on output
        if self.use_film:
            y = y + self.film_output(e)

        return y

In [286]:
class CatMLP(snt.Module):
    """
    Categorical multi-layer perceptron. This is very similar to a standard
    feed-forward neural network, except that it also takes an integer-valued
    categorical variable, c, as input. The transformation looks like:

      CatMLP: x, c -> y

    where x in R^n, c in Z, and y in R^m.
    """
    
    def __init__(self, n_cat, n_out, n_hidden, hidden_size,
                 activation=tf.nn.silu, name=None):
        """
        Constructor for CatMLP.

        Inputs:
          n_cat (int): # of categories.
          n_out (int): Dimensionality of output (m).
          n_hidden (int): # of hidden layers.
          hidden_size (int): # of neurons in each hidden layer.
          activation (Optional[function]): Activation function to apply after
            each hidden layer.
          name (Optional[str]): Name to give to model.
        """
        super(CatMLP, self).__init__(name=name)
        self.hidden = [
            snt.Linear(hidden_size, name=f'hidden{i}')
            for i in range(n_hidden)
        ]
        self.output = snt.Linear(n_out, name='output')
        self.n_cat = n_cat
        self.activation = activation

    def __call__(self, x, c):
        """
        Transform x,c -> y.

        Inputs:
          x (tf.Tensor): Float-valued tensor with shape (b,n), where n is the
            dimensionality of the input and b is the batch dimension.
          c (tf.Tensor): Integer-valued tensor with shape (b,), where b is
            the batch dimension. This is the categorical variable. It should
            be in the range [0, n_cat-1].
        """
        c = tf.one_hot(c, self.n_cat)
        y = tf.concat([x, c], axis=1)
        for i,h in enumerate(self.hidden):
            if i != 0:
                y = tf.concat([y,x,c], axis=1)
            y = h(y)
            y = self.activation(y)
        y = self.output(y)
        return y

In [287]:
import tensorflow_probability as tfp
tfb = tfp.bijectors
from tensorflow_probability.python import math as tfp_math
from tensorflow_probability.python.internal import prefer_static

class ODEBijector(tfb.Bijector):
    def __init__(self, n_dim, n_cond, hidden_size, n_hidden,
                 rtol=1e-5, atol=1e-5,
                 vector_network_kwargs={},
                 validate_args=False, name='ode_bijector'):
        super(ODEBijector, self).__init__(
            validate_args=validate_args,
            forward_min_event_ndims=0,
            name=name
        )

        # Velocity field
        # self.mlp_blocks = [
        #     snt.nets.MLP(
        #         [hidden_size]*n_hidden + [n_dim],
        #         activation=tf.math.tanh
        #     )
        #     for b in n_blocks
        #
        # self.v = CatMLP(n_cond, n_dim, n_hidden, hidden_size)
        self.v = VectorNetwork(n_cond, n_dim, n_hidden, hidden_size,
                               **vector_network_kwargs)
        # self.v = snt.nets.MLP(
        #     [hidden_size]*n_hidden + [n_dim],
        #     activation=tf.math.tanh
        # )
        # self.v(tf.zeros([1,1+n_dim+n_cond])) # Initialize the velocity field
        self.v(tf.zeros([1,1+n_dim]), tf.zeros([1], dtype=tf.int32))

        # ODE solver
        self.ode_solver = tfp.math.ode.DormandPrince(rtol=rtol, atol=atol)

    def conditional_velocity(self, t, x, c):
        # txc = tf.concat([tf.broadcast_to(t,[x.shape[0],1]), x, c], 1)
        # The following returns (x.shape[0], 1), even when x.shape = (None,n)
        shape = tf.where([True, False], tf.shape(x), [0, 1])
        # Concatenate t and x
        tx = tf.concat([tf.broadcast_to(t,shape), x], 1)
        # tx = tf.concat([tf.broadcast_to(t,[x.shape[0],1]), x], 1)
        # Return the velocity at time t and position x, given category c
        return self.v(tx, c)
    
    def augmented_ode(self, t, xsJ, c):
        x,s,lndetJ = tf.split(xsJ, [-1,1,1], axis=1)

        # dx_dt = self.conditional_velocity(t, x, c)
        dx_dt, diag_J = tfp_math.diag_jacobian(
            xs=x,
            fn=lambda xx: self.conditional_velocity(t, xx, c),
            sample_shape=[prefer_static.size0(x)]
        )
        if isinstance(dx_dt, list):
            dx_dt = dx_dt[0]
        if isinstance(diag_J, list):
            diag_J = diag_J[0]
        # print(dx_dt)
        # print(diag_J)

        # d(lndetJ)/dt = tr(J)
        # print('diag_J.shape =', diag_J.shape)
        dlndetJ_dt = 2*tf.math.reduce_sum(diag_J, axis=1, keepdims=True)
        
        # ds/dt = |dx/dt|
        ds_dt = tf.math.sqrt(tf.reduce_sum(dx_dt**2, axis=1, keepdims=True))

        # Combine derivatives into one vector
        # print('dx_dt.shape =', dx_dt.shape)
        # print('ds_dt.shape =', ds_dt.shape)
        # print('dlndetJ_dt.shape =', dlndetJ_dt.shape)
        dxsJ_dt = tf.concat([dx_dt, ds_dt, dlndetJ_dt], 1)

        return dxsJ_dt
    
    def ode_with_s(self, t, xs, c):
        x,s = tf.split(xs, [-1,1], axis=1)

        # dx_dt = self.conditional_velocity(t, x, c)
        dx_dt = self.conditional_velocity(t, x, c)
        
        # ds/dt = |dx/dt|
        ds_dt = tf.math.sqrt(tf.reduce_sum(dx_dt**2, axis=1, keepdims=True))

        # Combine derivatives into one vector
        dxs_dt = tf.concat([dx_dt, ds_dt], 1)

        return dxs_dt
    
    def _forward(self, x, c=None):
        dx_dt = lambda tt, xx: self.conditional_velocity(tt, xx, c)
        res = self.ode_solver.solve(dx_dt, 0., x, [1.])
        y = tf.squeeze(res.states, 0)
        return y

    def _inverse(self, y, c=None):
        dx_dt = lambda tt, xx: -self.conditional_velocity(1-tt, xx, c)
        res = self.ode_solver.solve(dx_dt, 0., y, [1.])
        x = tf.squeeze(res.states, 0)
        return x
    
    def forward_with_s(self, x, c=None):
        dxs_dt = lambda t, xs: self.ode_with_s(t, xs, c)
        xs0 = tf.concat([
            x,                        # x_0 = x
            tf.zeros([x.shape[0],1]), # s_0 = 0
        ], 1)
        res = self.ode_solver.solve(dxs_dt, 0., xs0, [1.])
        ys = tf.squeeze(res.states, 0)
        y,s = tf.split(ys, [-1,1], axis=1)
        return y, s
    
    def inverse_with_s(self, y, c=None):
        dxs_dt = lambda t, xs: -self.ode_with_s(1-t, xs, c)
        xs0 = tf.concat([
            y,                        # x_0 = y
            tf.zeros([y.shape[0],1]), # s_0 = 0
        ], 1)
        res = self.ode_solver.solve(dxs_dt, 0., xs0, [1.])
        xs = tf.squeeze(res.states, 0)
        x,s = tf.split(xs, [-1,1], axis=1)
        return x, -s
    
    def augmented_forward(self, x, c=None):
        dxsJ_dt = lambda t, xsJ: self.augmented_ode(t, xsJ, c)
        xsJ0 = tf.concat([
            x,                        # x_0 = x
            tf.zeros([x.shape[0],1]), # s_0 = 0
            tf.zeros([x.shape[0],1])  # lndetJ_0 = 0
        ], 1)
        res = self.ode_solver.solve(dxsJ_dt, 0., xsJ0, [1.])
        ysJ = tf.squeeze(res.states, 0)
        y,s,lndetJ = tf.split(ysJ, [-1,1,1], axis=1)
        return y, s, lndetJ

    def return_path(self, x, c, n_timesteps):
        t_solution = tf.linspace(0., 1., n_timesteps)
        dx_dt = lambda tt, xx: self.conditional_velocity(tt, xx, c)
        res = self.ode_solver.solve(dx_dt, 0., x, t_solution)
        y_t = res.states
        return y_t

    def _inverse_log_det_jacobian(self, y):
        return -self._forward_log_det_jacobian(self._inverse(y))

    def _forward_log_det_jacobian(self, x):
        # Notice that we needn't do any reducing, even when`event_ndims > 0`.
        # The base Bijector class will handle reducing for us; it knows how
        # to do so because we called `super` `__init__` with
        # `forward_min_event_ndims = 0`.
        raise NotImplementedError()

# Example 2: Mapping multiple coordinates onto a common scale

In [17]:
rng = np.random.default_rng(2)

n = 1024*64
n_classes = 3
n_dim = 2

# Determine transformations to apply to each class
a_transf = 0.2*rng.chisquare(2, size=n_classes) #* 0
b_transf = 0.5*rng.normal(size=n_classes) #* 0
c_transf = 0.1*rng.normal(size=(n_classes,2))
d_transf = 0.001*rng.chisquare(2, size=n_classes)

# Draw true coordinates
x_true = rng.normal(size=(n,n_dim))

# Choose two classes for each coordinate to belong to
c_obs = rng.integers(0, n_classes, size=(2,n))
for i in range(10):
    idx = np.where(c_obs[0] == c_obs[1])[0]
    c_obs[1,idx] = rng.integers(0, n_classes, size=len(idx))

# Generate observations of each coordinate
x_obs = np.empty((2,n,n_dim))
for i in range(2):
    for k,(a,b,c,d) in enumerate(zip(a_transf,b_transf,c_transf,d_transf)):
        idx = (c_obs[i] == k)
        x = x_true[idx]
        x_obs[i,idx] = x + a*np.exp(b*x) + c + d*rng.normal(size=x.shape)

fig,ax = plt.subplots(1,1, figsize=(8,8), dpi=100)
kw = dict(edgecolors='none', s=4)

prop_cycle = plt.rcParams['axes.prop_cycle']
colors = prop_cycle.by_key()['color']
class_color = colors[:n_classes]

for k in range(n_classes):
    for i in range(2):
        idx = np.where(c_obs[i,:1024] == k)[0]
        x = x_obs[i][idx]
        ax.scatter(x[:,0], x[:,1], c=class_color[k], **kw)

xylim = np.percentile(x_obs[0], [1., 99.], axis=0)
w = xylim[1] - xylim[0]
xylim[0] -= 0.4*w
xylim[1] += 0.4*w
# xlim = (np.max(xylim[0]), np.min(xylim[1]))
# ylim = xlim
xlim,ylim = xylim.T
ax.set_xlim(xlim)
ax.set_ylim(ylim)

plt.show()

<Figure size 800x800 with 1 Axes>

In [ ]:
batch_size = 256
n_epochs = 4

x_obs = x_obs.astype('f4')
c_obs = c_obs.astype('f4')
x0,x1 = x_obs
c0,c1 = c_obs
c0.shape = c0.shape + (1,)
c1.shape = c1.shape + (1,)

data = tf.data.Dataset.from_tensor_slices((x0,x1,c0,c1))
data = data.shuffle(buffer_size=batch_size*64)
data = data.repeat(n_epochs)
data = data.batch(batch_size, drop_remainder=True)

data

In [ ]:
hidden_size = 32
n_hidden = 3
l1_factor = 1e-3
l2_factor = 1e-2
sigma_x = 0.1

# bij = ODEBijector(n_dim, 1, hidden_size, n_hidden)
opt = keras.optimizers.Adam(learning_rate=1e-4)

variables = bij.trainable_variables

@tf.function
def grad_step(x0, x1, c0, c1):
    x0 = x0 + sigma_x * tf.random.normal(x0.shape)
    with tf.GradientTape() as g:
        x0_pred,s0 = bij.forward_with_s(x0, c=c0)
        x1_pred,s1 = bij.inverse_with_s(x0_pred, c=c1)
        chi2 = tf.reduce_sum((x1_pred-x1)**2, axis=1) / sigma_x**2
        chi2 = 5 * tf.math.asinh(chi2/5)
        chi2 = tf.reduce_mean(chi2)
#         s0 = tf.reduce_sum((x0_pred - x0)**2, axis=1)
#         s1 = tf.reduce_sum((x1_pred - x0_pred)**2, axis=1)
        length2 = s0**2 + s1**2
        penalty = (
            l1_factor * tf.reduce_mean(tf.math.sqrt(length2))
          + l2_factor * tf.reduce_mean(length2)
        )
        loss = chi2 + penalty
    grads = g.gradient(loss, variables)
    grads,norm = tf.clip_by_global_norm(grads, 100.)
    opt.apply_gradients(zip(grads, variables))
    return loss, penalty, norm

iter = tqdm(data)
loss_hist = []
penalty_hist = []
for x0,x1,c0,c1 in iter:
    loss, penalty, norm = grad_step(x0, x1, c0, c1)

    loss_hist.append(float(loss.numpy()))
    penalty_hist.append(float(penalty.numpy()))

    loss_avg = np.mean(loss_hist[-16:])
    penalty_avg = np.mean(penalty_hist[-16:])

    iter.set_postfix({
        'loss': loss_avg,
        'chi2': loss_avg - penalty_avg,
        'penalty': penalty_avg,
        'norm': float(norm.numpy())
    })

In [118]:
plt.plot(loss_hist, label='chi2')
plt.plot(penalty_hist, label='penalty')
plt.legend()
plt.xlabel('step')
plt.grid(True, alpha=0.2)

<Figure size 640x480 with 1 Axes>

In [119]:
xt_pred = []

for x,c in zip(x_obs, c_obs):
    x0 = tf.constant(x[:1024].astype('f4'))
    c0 = tf.expand_dims(tf.constant(c[:1024].astype('f4')), 1)
    xt_pred.append(bij.return_path(x0, c0, 90).numpy())

xt_pred[0].shape

(90, 1024, 2)

In [120]:
! mkdir frames

mkdir: cannot create directory ‘frames’: File exists


In [121]:
n_plot = 128
fname = 'frames/paths_{:03d}.png'

fig,ax = plt.subplots(figsize=(8,8), dpi=100)

kw = dict(edgecolors='none', s=4)
prop_cycle = plt.rcParams['axes.prop_cycle']
colors = prop_cycle.by_key()['color']
class_color = colors[:n_classes]

xylim = np.percentile(x_obs[0], [1., 99.], axis=0)
w = xylim[1] - xylim[0]
xylim[0] -= 0.3*w
xylim[1] += 0.3*w
xlim,ylim = xylim.T

scatterplots = []

for k in range(n_classes):
    scatterplots.append([])
    for i in range(2):
        idx = np.where(c_obs[i,:n_plot] == k)[0]
        x = xt_pred[i][0][idx]
        sc = ax.scatter(x[:,0], x[:,1], c=class_color[k], **kw)
        scatterplots[-1].append(sc)

n_frames = xt_pred[0].shape[0]

for j in tqdm(range(n_frames)):
    for k in range(n_classes):
        for i in range(2):
            idx = np.where(c_obs[i,:n_plot] == k)[0]
            x = xt_pred[i][j][idx]
            sc = scatterplots[k][i]
            sc.set_offsets(np.c_[x[:,0], x[:,1]])
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    fig.savefig(fname.format(j))

plt.show()

  0%|          | 0/90 [00:00<?, ?it/s]

<Figure size 800x800 with 1 Axes>

In [122]:
! ffmpeg -y -f image2 -framerate 30 -i "frames/paths_%03d.png" -pix_fmt yuv420p -c:v libx264 -crf 20 paths.mp4

ffmpeg version 2.8.15 Copyright (c) 2000-2018 the FFmpeg developers
  built with gcc 4.8.5 (GCC) 20150623 (Red Hat 4.8.5-36)
  configuration: --prefix=/usr --bindir=/usr/bin --datadir=/usr/share/ffmpeg --incdir=/usr/include/ffmpeg --libdir=/usr/lib64 --mandir=/usr/share/man --arch=x86_64 --optflags='-O2 -g -pipe -Wall -Wp,-D_FORTIFY_SOURCE=2 -fexceptions -fstack-protector-strong --param=ssp-buffer-size=4 -grecord-gcc-switches -m64 -mtune=generic' --extra-ldflags='-Wl,-z,relro ' --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libvo-amrwbenc --enable-version3 --enable-bzlib --disable-crystalhd --enable-gnutls --enable-ladspa --enable-libass --enable-libcdio --enable-libdc1394 --enable-libfdk-aac --enable-nonfree --disable-indev=jack --enable-libfreetype --enable-libgsm --enable-libmp3lame --enable-openal --enable-libopenjpeg --enable-libopus --enable-libpulse --enable-libschroedinger --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libvorbis --enable-libv4l2 

# Merge real stellar catalogs

## Load stellar catalogs

In [7]:
err_floor_default = [0.01, 0.01, 0.04, 0.05] # T_eff, [Fe/H], logg, [alpha/Fe]

### Utility functions

In [9]:
import astropy
from astropy.table import Table
from astropy import units

import os

This is the directory where all of the catalogs will be downloaded to:

In [10]:
data_dir = 'data/'

In [11]:
def extract_table(t, id_key, param_keys, param_unc_keys, extra_keys,
                  param_transf=None, param_unc_transf=None, param_errs=None):
    # Transformation to apply to each parameter
    if param_transf is None:
        param_transf = [lambda x:x for k in param_keys]

    # Transformation to apply to each parameter uncertainty
    if param_unc_transf is None:
        param_unc_transf = [lambda x,x_err: x_err for k in param_keys]
    
    # Remove rows with invalid IDs
    idx = (t[id_key] > 0) # Assume that id<=0 is invalid
    
    # Use default parameter errors, if none given
    n = np.count_nonzero(idx)
    d = len(param_keys)
    if param_errs is None:
        param_errs = 0.1*np.ones((n,d), dtype='f4')
    elif isinstance(param_errs, tuple) or isinstance(param_errs, list):
        errs = np.empty((n,d), dtype='f4')
        for i,e in enumerate(param_errs):
            errs[:,i] = e
        param_errs = errs
    elif isinstance(param_errs, np.ndarray):
        param_errs = param_errs[idx]

    def remove_masking(xx):
        if isinstance(xx, astropy.table.MaskedColumn):
            return xx.filled(np.nan).data
        else:
            return xx.data
    
    t_out = Table(
        dict(
            id=t[id_key][idx],
            x=np.stack([
                f(remove_masking(t[k][idx]))
                for f,k in zip(param_transf,param_keys)
            ], axis=1),
            sigma_x=np.stack([
                f(remove_masking(t[k0][idx]), remove_masking(t[k1][idx]))
                for f,k0,k1 in zip(param_unc_transf,param_keys,param_unc_keys)
            ], axis=1),
            x_err=param_errs,
            **{k:t[k][idx] for k in extra_keys}
        )
    )
    return t_out

In [12]:
# Transformations to apply to T_eff, log(g), [Fe/H], [alpha/Fe]
param_transf = (
    lambda x: 5*np.log(x/5000),
    lambda x: x,
    lambda x: x,
    lambda x: x
)

# Corresponding transformations of the uncertainties
param_unc_transf = (
    lambda x,x_err: 5*x_err/x,
    lambda x,x_err: x_err,
    lambda x,x_err: x_err,
    lambda x,x_err: x_err
)

In [13]:
def plot_catalog_uncertainties(t):
    xlim = plot_utils.choose_lim(t['x'][:,0], expand=0.4, pct=[0.1,99.9])
    ylim = plot_utils.choose_lim(t['x'][:,2], expand=0.4, pct=[0.1,99.9])
    
    d = t['x'].shape[1]
    fig,ax_arr = plt.subplots(
        1, d+1,
        figsize=(2.5*d+0.5,2.8),
        width_ratios=[1]*d+[0.1],
        layout='constrained'
    )

    norm = LogNorm(vmin=2e-2, vmax=5e-1)
    
    for i,ax in enumerate(ax_arr[:-1]):
        im = plot_utils.hist2d_reduce(
            t['x'][:,0], t['x'][:,2],
            t['sigma_x'][:,i],
            xlim=xlim,
            ylim=ylim,
            hist_kw=dict(bins=100, statistic='median'),
            imshow_kw=dict(norm=norm),
            ax=ax
        )
        ax.invert_xaxis()
        ax.invert_yaxis()
        if i == 0:
            ax.set_ylabel('logg')
        else:
            ax.set_xticklabels([])
        ax.set_title(f'parameter {i}')
        ax.set_xlabel('5 ln(teff/5000)')
    
    fig.colorbar(im, cax=ax_arr[-1], label='$\sigma$')
    plt.show()

    fig,ax = plt.subplots(1,1, figsize=(5,3))
    for i,s in enumerate(t['sigma_x'].T):
        ax.hist(
            np.log10(s),
            histtype='step',
            range=(-3,0), bins=100,
            # log=True,
            label=f'param {i}'
        )
    ax.legend()
    ax.set_xlabel(r'$\log_{10}(\mathrm{uncertainty})$')
    plt.show()

def plot_catalog_params(t):
    fig,ax = plt.subplots(1,1, figsize=(5,4))

    xlim = (np.min(t['x'].flat), np.max(t['x'].flat))
    w = xlim[1] - xlim[0]
    xlim = (xlim[0]-0.05*w, xlim[1]+0.05*w)
    
    kw = dict(log=True, range=xlim, bins=100, histtype='step')
    for i,p in enumerate(t['x'].T):
        ax.hist(p, label=f'param {i}', **kw)

    ax.set_xlabel('parameter value')
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.grid(True, alpha=0.2)
    ax.grid(True, which='minor', alpha=0.05)
    ax.legend()
    
    plt.show()

<>:34: SyntaxWarning: invalid escape sequence '\s'
<>:34: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2429448/61674979.py:34: SyntaxWarning: invalid escape sequence '\s'
  fig.colorbar(im, cax=ax_arr[-1], label='$\sigma$')


In [14]:
def apply_err_floor(f, err_floor):
    def f_wrapped(vals, errs):
        errs_p = f(vals, errs)
        return np.sqrt(errs_p**2 + err_floor**2)
    return f_wrapped

def apply_err_floors(f_list, err_floors):
    return [apply_err_floor(f,ef) for f,ef in zip(f_list,err_floors)]

In [15]:
from getpass import getpass
import requests
import os

def download_with_auth(url, fname, username, password):
    r = requests.get(url, auth=(username, password), stream=True)
    r.raise_for_status()
    
    # Figure out total file size (in Bytes)
    file_size = int(r.headers.get('content-length', 0))
    
    # Open output file and wrap a tqdm around the iteration
    with open(fname, 'wb') as f, tqdm(total=file_size, unit='B',
                                      unit_scale=True, unit_divisor=1024,
                                      desc=fn
                                 ) as bar:
        for chunk in r.iter_content(chunk_size=1024):
            if not chunk:
                continue
            f.write(chunk)
            bar.update(len(chunk))

In [16]:
# gaia_username = getpass('Username: ')

# def gaia_crossmatch(ra, dec):
#     t_upload = Table({
#         'ra': ra,
#         'dec': dec,
#         'index': np.arange(len(ra))
#     })
#     Gaia.upload_table(upload_resource=t_upload, table_name='crossmatch_table')
    
#     query = """
#       SELECT
#         t.index,
#         g.source_id, g.ra, g.dec,
#         DISTANCE(
#           POINT(t.ra, t.dec),
#           POINT(g.ra, g.dec)
#         ) * 3600. AS angsep_arcsec
#       FROM user_{gaia_username}.crossmatch_table AS t
#       JOIN gaiadr3.gaia_source_lite AS g
#       ON DISTANCE(POINT(t.ra, t.dec), POINT(g.ra, g.dec)) < 1. / 3600.
#     """.strip().format(gaia_username=gaia_username)
    
#     job = Gaia.launch_job_async(query, verbose=True)

#     Gaia.delete_user_table(table_name='crossmatch_table')
    
#     job.results.sort(['angsep_arcsec'])
#     t_index = astropy.table.unique(
#         job.results,
#         keys=['index'],
#         keep='first'
#     )
#     print(f'{len(t_index):,}')

#     kw = dict(bins=100, range=(0,1), histtype='step', log=True)
#     plt.hist(job.results['angsep_arcsec'], label='cone search', **kw)
#     plt.hist(t_index['angsep_arcsec'], label='nearest only', **kw)
#     plt.xlabel('separation (arcsec)')
#     plt.legend()
#     plt.show()

#     # idx = (t_index['angsep_arcsec'] < 0.5)
#     # print(f'{np.count_nonzero(idx):,} ({np.mean(idx):.1%})')
#     # t_index = t_index[idx]

#     return Table({
#         'match_idx': t_index['index'],
#         'gdr3_source_id': t_index['source_id'],
#         'ra': t_index['ra'],
#         'dec': t_index['dec'],
#         'angsep_arcsec': t_index['angsep_arcsec']
#     })

In [17]:
import astropy.table
import time
from tqdm.auto import tqdm

def gaia_crossmatch_batched(ra, dec,
                            max_separation=0.5*units.arcsec,
                            max_batch_size=1024*256,
                            # gaia_username=None,
                            verbose=False):
    # Maximum match separation in degrees
    max_sep_deg = max_separation.to('deg').value

    # Split coordinates into fixed-size batches
    batches = [(ra[i:i+max_batch_size], dec[i:i+max_batch_size], i)
               for i in range(0,len(ra),max_batch_size)]

    # Upload all tables and launch jobs
    if verbose:
        print('Launching jobs...')
    
    jobs = []
    table_names = []
    
    for batch_ra, batch_dec, offset in tqdm(batches):
        t_upload = Table({
            'ra': batch_ra,
            'dec': batch_dec,
            'index': np.arange(len(batch_ra)) + offset
        })
        table_name = f'crossmatch_table_{offset}'
        table_names.append(table_name)

        query = f"""
          SELECT
            t.index,
            g.source_id, g.ra, g.dec,
            DISTANCE(
              POINT(t.ra, t.dec),
              POINT(g.ra, g.dec)
            ) * 3600. AS angsep_arcsec
          FROM tap_upload.{table_name} AS t
          JOIN gaiadr3.gaia_source AS g
          ON DISTANCE(POINT(t.ra, t.dec), POINT(g.ra, g.dec)) < {max_sep_deg}
        """.strip()

        job = Gaia.launch_job_async(
            query,
            verbose=verbose,
            background=True,
            upload_resource=t_upload,
            upload_table_name=table_name
        )
        
        jobs.append(job)

    # Wait for jobs to finish and collect results
    if verbose:
        print('Waiting for jobs to complete...')
    
    results = []
    
    for i,job in enumerate(tqdm(jobs)):
        results.append(job.get_results())
        if verbose:
            print(f'Downloaded results for job {i+1} of {len(jobs)}.')
        # while job.get_phase(update=True) != 'COMPLETED':
        #     print(f'Waiting for job {job.jobid} to complete...')
        #     time.sleep(2)
        # results.append(job.results)

    # Delete uploaded tables (being a good citizen)
    # if gaia_username is not None:
    for table_name in table_names:
        try:
            Gaia.delete_user_table(table_name=table_name)
        except Exception as e:
            print(f'Failed to delete {table_name}: {e}')

    if verbose:
        print('Concatenating and filtering results ...')
    
    # Concatenate results
    all_results = astropy.table.vstack(results)

    # Only keep closest match to each uploaded coordinate
    all_results.sort(['angsep_arcsec'])
    t_index = astropy.table.unique(all_results, keys=['index'], keep='first')

    print(f'{len(t_index):,} matches')

    # Optional: plotting
    kw = dict(bins=100, range=(0,1), histtype='step', log=True)
    plt.hist(all_results['angsep_arcsec'], label='cone search', **kw)
    plt.hist(t_index['angsep_arcsec'], label='nearest only', **kw)
    plt.xlabel('separation (arcsec)')
    plt.legend()
    plt.show()

    # Final filtering
    max_sep_arcsec = max_separation.to('arcsec').value
    idx = (t_index['angsep_arcsec'] < max_sep_arcsec)
    print(f'{np.count_nonzero(idx):,} ({np.mean(idx):.1%}) under {max_sep_arcsec:.2g} arcsec')
    t_index = t_index[idx]

    if verbose:
        print('Done.')

    return Table({
        'match_idx': t_index['index'],
        'gdr3_source_id': t_index['source_id'],
        'ra': t_index['ra'],
        'dec': t_index['dec'],
        'angsep_arcsec': t_index['angsep_arcsec']
    })


In [18]:
def apply_crossmatch(crossmatch_info, catalog_table, max_separation=None):
    if max_separation is None:
        idx = slice(None)
    else:
        max_sep_arcsec = max_separation.to('arcsec').value
        idx = (crossmatch_info['angsep_arcsec'] < max_sep_arcsec)

    n_orig = len(catalog_table)
    
    catalog_table = catalog_table[crossmatch_info['match_idx'][idx]]
    catalog_table['gdr3_source_id'] = crossmatch_info['gdr3_source_id'][idx]
    catalog_table['ra_gdr3'] = crossmatch_info['ra'][idx]
    catalog_table['dec_gdr3'] = crossmatch_info['dec'][idx]

    n_crossmatch = len(catalog_table)
    print(f'{n_crossmatch/n_orig:.2%} of sources have crossmatch.')
    
    return catalog_table

In [19]:
def hist2d_teff_logg(teff, logg, ax=None):
    if ax is None:
        ax = plt.gca()
    _,_,_,im = ax.hist2d(5*np.log(teff/5000), logg,
                         bins=100, norm=LogNorm(),
                         rasterized=True)
    ax.invert_xaxis()
    ax.invert_yaxis()
    ax.set_xlabel(r'$T_\mathrm{eff} / 1000\,\mathrm{K}$')
    ax.set_ylabel(r'$\log g$')
    ax.grid(True, alpha=0.2)
    
    teff_ticks = np.array(list(range(2,10,1)) + list(range(10,51,10)))
    ax.xaxis.set_major_locator(FixedLocator(5*np.log(teff_ticks/5)))
    ax.set_xticklabels([f'${tt:.0f}$' for tt in teff_ticks])

    teff_minor_ticks = np.hstack([np.arange(2,10,0.2), np.arange(10,51,1)])
    ax.xaxis.set_minor_locator(FixedLocator(5*np.log(teff_minor_ticks/5)))
    
    return im

def hist2d_teff_feh(teff, feh, ax=None):
    if ax is None:
        ax = plt.gca()
    _,_,_,im = ax.hist2d(5*np.log(teff/5000), feh,
                         bins=100, norm=LogNorm(),
                         rasterized=True)
    ax.invert_xaxis()
    ax.set_xlabel(r'$T_\mathrm{eff} / 1000\,\mathrm{K}$')
    ax.set_ylabel(r'$\left[\mathrm{Fe/H}\right]$')
    ax.grid(True, alpha=0.2)
    teff_ticks = np.array(list(range(2,10,1)) + list(range(10,51,10)))
    ax.xaxis.set_major_locator(FixedLocator(5*np.log(teff_ticks/5)))
    ax.set_xticklabels([f'${tt:.0f}$' for tt in teff_ticks])
    return im

def plot_teff_logg_feh(teff, logg, feh, label_as_mh=False):
    fig,ax_arr = plt.subplots(1,2, figsize=(8,4), layout='tight')
    hist2d_teff_logg(teff, logg, ax=ax_arr[0])
    hist2d_teff_feh(teff, feh, ax=ax_arr[1])
    if label_as_mh:
        ax_arr[1].set_ylabel(r'$\left[\mathrm{M/H}\right]$')
    return fig, ax_arr

def plot_teff_logg_feh_abundance(teff, logg, feh, abundance,
                                 abundance_label=r'$\left[\alpha/\mathrm{Fe}\right]$',
                                 label_as_mh=False):
    fig,ax_arr = plt.subplots(1,2, figsize=(8,4), layout='tight')
    
    hist2d_teff_logg(teff, logg, ax=ax_arr[0])
    
    ax = ax_arr[1]
    idx = np.isfinite(abundance)
    ax.hist2d(feh[idx], abundance[idx],
              bins=100, norm=LogNorm())
    if label_as_mh:
        ax.set_xlabel(r'$\left[\mathrm{M/H}\right]$')
    else:
        ax.set_xlabel(r'$\left[\mathrm{Fe/H}\right]$')
    ax.set_ylabel(abundance_label)
    ax.grid(True, alpha=0.2)
    
    return fig, ax_arr

In [20]:
def calc_alpha_Fe_from_abundances(t, X_Fe_fmt, X_Fe_err_fmt,
                                  flag_fmt=None, err_floor=1e-2):
    """
    Calculates an average [alpha/Fe] from the individual abundances in a
    table t, using an inverse-variance-weighted average of the
    abundances of all available alpha-process elements.

    Inputs:
      - t (astropy.table.Table): Catalog containing stellar abundances
      - X_Fe_fmt (str): Format string defining column names for abundances.
                        E.g., `"[{x}/Fe]"`.
      - X_Fe_err_fmt (str): Format string defining column names for
                            abundance uncertainties.
      - flag_fmt (str): Format string for abundance flag column
                        (assume 0=good).
      - err_floor (Optional[float]): Floor to apply on abundance
                                     uncertainties. Defaults to 0.01.

    Outputs:
      - alpha_Fe (np.ndarray): Average [alpha/Fe] for each star in catalog.
      - alpha_Fe_err (np.ndarray): Uncertainties on [alpha/Fe].
      - abundances (astropy.table.Table): Individual abundances for each star.
    """
    alpha_elements = ['O', 'Ne', 'Mg', 'Si', 'S', 'Ar', 'Ca', 'Ti']
    # Note: APOGEE/FERRE uses O, Mg, S, Si, Ca, Ti (missing Ne, Ar)

    X_Fe = []
    X_Fe_weight = []
    X_Fe_bad = []
    X_names = []

    def render_fmt_str(fmt: str, **kwargs):
        """More flexible format-string rendering."""
        return eval(f"f'''{fmt}'''", {}, kwargs)

    def unmask_column(col, fill=np.nan):
        """Ensures that a column is unmasked."""
        if isinstance(col, np.ma.MaskedArray):
            col = col.filled(fill)
        return col
    
    for s in alpha_elements:
        key = render_fmt_str(X_Fe_fmt, x=s)
        if key in t.colnames:
            X_names.append(s)
            err_key = render_fmt_str(X_Fe_err_fmt, x=s)
            X_Fe.append(unmask_column(t[key])+0.) # Make sure a copy is made
            X_Fe_weight.append(1/(unmask_column(t[err_key])**2+err_floor**2))
            if flag_fmt is not None:
                flag_key = render_fmt_str(flag_fmt, x=s)
                idx_bad = ~(t[flag_key] == 0)
                print(f'{np.mean(idx_bad):.3%} of [{s}/Fe] measurements flagged.')
                X_Fe_weight[-1][idx_bad] = 0.
                X_Fe[-1][idx_bad] = 0.
                X_Fe_bad.append(idx_bad)

    # Detect zero-weight stars
    X_Fe_weight = np.stack(X_Fe_weight, axis=1)
    idx = np.all(X_Fe_weight == 0, axis=1)
    
    X_Fe = np.stack(X_Fe, axis=1)
    X_Fe_weight = np.maximum(X_Fe_weight, 1e-10)
    alpha_Fe, alpha_Fe_err = np.average(
        X_Fe, weights=X_Fe_weight,
        axis=1, returned=True
    )
    alpha_Fe_err = 1 / np.sqrt(alpha_Fe_err)
    
    alpha_Fe[idx] = np.nan
    alpha_Fe_err[idx] = np.nan
    
    abundances = Table({s: X_Fe[:,i] for i,s in enumerate(X_names)})

    if len(X_Fe_bad):
        for s,b in zip(X_names,X_Fe_bad):
            abundances[s][b] = np.nan
    
    return alpha_Fe, alpha_Fe_err, abundances

In [21]:
def nearly_square_grid(n_elements):
    """
    Compute the number of rows and columns for a nearly square grid
    that fits n_elements.

    Returns:
        (rows, cols): Tuple of integers
    """
    if n_elements <= 0:
        return (0, 0)
    
    # Start with the square root
    sqrt_n = np.sqrt(n_elements)
    cols = int(np.ceil(sqrt_n))
    rows = int(np.ceil(n_elements / cols))

    return (rows, cols)

for n in (1, 2, 4, 5, 10, 25, 17):
    print(n, nearly_square_grid(n))

def plot_alpha_abundances(alpha_Fe, abundances):
    n_rows, n_cols = nearly_square_grid(len(abundances.colnames))
    
    fig,ax_arr = plt.subplots(
        n_rows, n_cols,
        figsize=(2.5*n_cols, 2.5*n_rows),
        layout='tight'
    )

    for s,ax in zip(abundances.colnames,ax_arr.flat):
        x,y = alpha_Fe, abundances[s]
        idx = np.isfinite(x) & np.isfinite(y)
        ax.hist2d(x[idx], y[idx], bins=100)
        ax.axline((0,0), slope=1, c='w', alpha=0.2)
        ax.set_xlabel(r'$\left[\mathrm{\alpha/Fe}\right]$')
        ax.set_ylabel(fr'$\left[\mathrm{{{s}/Fe}}\right]$')
    
    return fig, ax_arr

1 (1, 1)
2 (1, 2)
4 (2, 2)
5 (2, 3)
10 (3, 4)
25 (5, 5)
17 (4, 5)


In [22]:
def determine_snr_chisq_relation(snr, chisq):
    # Filter out non-positive values of SNR, chi^2
    idx = (snr > 0) & (chisq > 0)
    snr = snr[idx]
    chisq = chisq[idx]
    
    snr_pct = np.reshape(np.nanpercentile(snr, [1.,2.,98.,99.]), [2,2])
    x,y = [],[]
    for snr_low,snr_high in snr_pct:
        idx = (snr>snr_low) & (snr<snr_high)
        x.append(np.median(snr[idx]))
        y.append(np.median(chisq[idx]))
    b = (y[1] - y[0]) / (y[0]*x[1]**2 - y[1]*x[0]**2)
    a = y[0] / (1 + b*x[0]**2)

    log_snr_lim = np.log10(snr_pct.flat[[0,-1]])
    w = log_snr_lim[1] - log_snr_lim[0]
    log_snr_lim = (log_snr_lim[0]-0.2*w, log_snr_lim[1]+0.2*w)
    snr_bins = np.logspace(
        log_snr_lim[0],
        log_snr_lim[1],
        50
    )
    log_chisq_lim = np.log10(np.nanpercentile(chisq, [1., 99.]))
    w = log_chisq_lim[1]-log_chisq_lim[0]
    log_chisq_lim = (log_chisq_lim[0]-0.2*w, log_chisq_lim[1]+0.4*w)
    chisq_bins = np.logspace(
        log_chisq_lim[0],
        log_chisq_lim[1],
        50
    )
    
    fig,ax = plt.subplots(1,1, figsize=(5,4))
    ax.hist2d(
        snr,
        chisq,
        bins=[snr_bins,chisq_bins],
        norm=LogNorm()
    )

    for i,factor in enumerate([1, 2, 4, 8, 16]):
        ax.plot(snr_bins, factor*a*(1+b*(snr_bins*factor)**2),
                c='k', alpha=0.5/(i+1))
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\mathrm{SNR}$')
    ax.set_ylabel(r'$\chi^2$')

    ax.set_xlim(snr_bins[0], snr_bins[-1])
    ax.set_ylim(chisq_bins[0], chisq_bins[-1])

    f_snr = lambda x: a * (1 + b*x**2)
    return f_snr

### SDSS-V (ASPCAP)

Stellar parameter determinations from the ASPCAP pipeline, applied to SDSS-V APOGEE spectra. This should be similar to how the SDSS-IV APOGEE parameters were determined.

Reference: [Mészáros *et al* (2025)](https://ui.adsabs.harvard.edu/abs/2025AJ....170...96M/abstract)

More information: https://sdss.org/dr19/mwm/astra/pipelines-in-astra/aspcap/

In [ ]:
url_base = 'https://data.sdss5.org/sas/dr19/spectro/astra/0.6.0/summary'
fn = 'astraAllStarASPCAP-0.6.0.fits.gz'

out_fn = os.path.join(data_dir, fn)

! wget -O {out_fn} {url_base}/{fn}
! du -sh {out_fn}

In [142]:
# url_base = 'https://data.sdss5.org/sas/sdsswork/mwm/spectro/astra/0.7.0/summary/'
# fn = 'astraAllStarASPCAP-0.7.fits.gz'
# url_base = 'https://data.sdss5.org/sas/dr19/spectro/astra/0.6.0/summary'
# fn = 'astraAllStarASPCAP-0.6.0.fits.gz'

# os.makedirs(os.path.join(data_dir, 'sdssv'), exist_ok=True)

# username = getpass('Username: ')
# password = getpass('Password: ')

# print(f'Downloading {fn} ...')

# fname = os.path.join(data_dir, 'sdssv', fn)
# url = f'{url_base}/{fn}'

# download_with_auth(url, fname, username, password)

# print(f'Downloaded to {fname}.')

Username:  ········
Password:  ········


astraAllStarASPCAP-0.6.0.fits.gz:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Downloaded to /data/beegfs/astro-storage/groups/green/green/projects/gaia_Bp_Rp_reddening_variation/data/sdssv/astraAllStarASPCAP-0.6.0.fits.gz.


In [154]:
fn = 'astraAllStarASPCAP-0.6.0.fits.gz'
with fits.open(os.path.join(data_dir,'sdssv',fn), mode='readonly') as f:
    f.info()
    t_sdssv_aspcap = Table(f[2].data)

t_sdssv_aspcap

Filename: data/sdssv/astraAllStarASPCAP-0.6.0.fits.gz
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      16   ()      
  1                1 BinTableHDU    871   0R x 384C   [K, 1A, K, K, K, J, 1A, J, K, K, K, K, J, J, 1B, K, K, K, K, K, K, K, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, 1A, 1A, 1A, E, E, E, E, E, E, E, E, E, E, K, K, K, K, E, E, E, K, K, K, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, K, E, E, E, E, E, E, 1A, E, E, K, E, E, E, E, E, E, E, E, E, E, E, E, E, E, J, E, J, E, J, E, J, E, J, E, J, E, J, E, J, E, J, E, J, E, J, J, J, J, J, J, J, K, K, 1A, 1A, 1A, 1A, 1A, J, J, J, J, J, E, E, E, E, E, E, E, E, E, E, E, E, K, K, E, K, K, K, 26A, E, E, 1A, E, K, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, K, E, E, E, K, E, E, E, K, E, E, E, K, E, E, E, K, E, E, E, K, E, E, E, K, E, E, E, K, E, E, E, K, E, E, E, K, E, E, E, K, E, E, E, K, E, E, E, K, E, E, E, K, E, E, E, K, E, E, E, K, E, E, E, K, E, E, E

sdss_id,sdss4_apogee_id,gaia_dr2_source_id,gaia_dr3_source_id,tic_v8_id,healpix,lead,version_id,catalogid,catalogid21,catalogid25,catalogid31,n_associated,n_neighborhood,sdss5_target_flags,sdss4_apogee_target1_flags,sdss4_apogee_target2_flags,sdss4_apogee2_target1_flags,sdss4_apogee2_target2_flags,sdss4_apogee2_target3_flags,sdss4_apogee_member_flags,sdss4_apogee_extra_target_flags,ra,dec,l,b,plx,e_plx,pmra,e_pmra,pmde,e_pmde,gaia_v_rad,gaia_e_v_rad,g_mag,bp_mag,rp_mag,j_mag,e_j_mag,h_mag,e_h_mag,k_mag,e_k_mag,ph_qual,bl_flg,cc_flg,w1_mag,e_w1_mag,w1_flux,w1_dflux,w1_frac,w2_mag,e_w2_mag,w2_flux,w2_dflux,w2_frac,w1uflags,w2uflags,w1aflags,w2aflags,mag4_5,d4_5m,rms_f4_5,sqf_4_5,mf4_5,csf,zgr_teff,zgr_e_teff,zgr_logg,zgr_e_logg,zgr_fe_h,zgr_e_fe_h,zgr_e,zgr_e_e,zgr_plx,zgr_e_plx,zgr_teff_confidence,zgr_logg_confidence,zgr_fe_h_confidence,zgr_ln_prior,zgr_chi2,zgr_quality_flags,r_med_geo,r_lo_geo,r_hi_geo,r_med_photogeo,r_lo_photogeo,r_hi_photogeo,bailer_jones_flags,ebv,e_ebv,ebv_flags,ebv_zhang_2023,e_ebv_zhang_2023,ebv_sfd,e_ebv_sfd,ebv_rjce_glimpse,e_ebv_rjce_glimpse,ebv_rjce_allwise,e_ebv_rjce_allwise,ebv_bayestar_2019,e_ebv_bayestar_2019,ebv_edenhofer_2023,e_ebv_edenhofer_2023,c_star,u_jkc_mag,u_jkc_mag_flag,b_jkc_mag,b_jkc_mag_flag,v_jkc_mag,v_jkc_mag_flag,r_jkc_mag,r_jkc_mag_flag,i_jkc_mag,i_jkc_mag_flag,u_sdss_mag,u_sdss_mag_flag,g_sdss_mag,g_sdss_mag_flag,r_sdss_mag,r_sdss_mag_flag,i_sdss_mag,i_sdss_mag_flag,z_sdss_mag,z_sdss_mag_flag,y_ps1_mag,y_ps1_mag_flag,n_boss_visits,boss_min_mjd,boss_max_mjd,n_apogee_visits,apogee_min_mjd,apogee_max_mjd,source,star_pk,spectrum_pk,release,filetype,apred,apstar,obj,telescope,field,prefix,min_mjd,max_mjd,n_entries,n_visits,n_good_visits,n_good_rvs,snr,mean_fiber,std_fiber,spectrum_flags,v_rad,e_v_rad,std_v_rad,median_e_v_rad,doppler_teff,doppler_e_teff,doppler_logg,doppler_e_logg,doppler_fe_h,doppler_e_fe_h,doppler_rchi2,doppler_flags,xcorr_v_rad,xcorr_v_rel,xcorr_e_v_rel,ccfwhm,autofwhm,n_components,task_pk,source_pk,v_astra,created,t_elapsed,t_overhead,tag,irfm_teff,irfm_teff_flags,teff,e_teff,logg,e_logg,v_micro,e_v_micro,v_sini,e_v_sini,m_h_atm,e_m_h_atm,alpha_m_atm,e_alpha_m_atm,c_m_atm,e_c_m_atm,n_m_atm,e_n_m_atm,al_h,e_al_h,al_h_flags,al_h_rchi2,c_12_13,e_c_12_13,c_12_13_flags,c_12_13_rchi2,ca_h,e_ca_h,ca_h_flags,ca_h_rchi2,ce_h,e_ce_h,ce_h_flags,ce_h_rchi2,c_1_h,e_c_1_h,c_1_h_flags,c_1_h_rchi2,c_h,e_c_h,c_h_flags,c_h_rchi2,co_h,e_co_h,co_h_flags,co_h_rchi2,cr_h,e_cr_h,cr_h_flags,cr_h_rchi2,cu_h,e_cu_h,cu_h_flags,cu_h_rchi2,fe_h,e_fe_h,fe_h_flags,fe_h_rchi2,k_h,e_k_h,k_h_flags,k_h_rchi2,mg_h,e_mg_h,mg_h_flags,mg_h_rchi2,mn_h,e_mn_h,mn_h_flags,mn_h_rchi2,na_h,e_na_h,na_h_flags,na_h_rchi2,nd_h,e_nd_h,nd_h_flags,nd_h_rchi2,ni_h,e_ni_h,ni_h_flags,ni_h_rchi2,n_h,e_n_h,n_h_flags,n_h_rchi2,o_h,e_o_h,o_h_flags,o_h_rchi2,p_h,e_p_h,p_h_flags,p_h_rchi2,si_h,e_si_h,si_h_flags,si_h_rchi2,s_h,e_s_h,s_h_flags,s_h_rchi2,ti_h,e_ti_h,ti_h_flags,ti_h_rchi2,ti_2_h,e_ti_2_h,ti_2_h_flags,ti_2_h_rchi2,v_h,e_v_h,v_h_flags,v_h_rchi2,short_grid_name,continuum_order,continuum_reject,interpolation_order,initial_flags,rchi2,ferre_log_snr_sq,ferre_time_elapsed,result_flags,flag_warn,flag_bad,stellar_parameters_task_pk,al_h_task_pk,c_12_13_task_pk,ca_h_task_pk,ce_h_task_pk,c_1_h_task_pk,c_h_task_pk,co_h_task_pk,cr_h_task_pk,cu_h_task_pk,fe_h_task_pk,k_h_task_pk,mg_h_task_pk,mn_h_task_pk,na_h_task_pk,nd_h_task_pk,ni_h_task_pk,n_h_task_pk,o_h_task_pk,p_h_task_pk,si_h_task_pk,s_h_task_pk,ti_h_task_pk,ti_2_h_task_pk,v_h_task_pk,calibrated_flags,mass,radius,raw_teff,raw_e_teff,raw_logg,raw_e_logg,raw_v_micro,raw_e_v_micro,raw_v_sini,raw_e_v_sini,raw_m_h_atm,raw_e_m_h_atm,raw_alpha_m_atm,raw_e_alpha_m_atm,raw_c_m_atm,raw_e_c_m_atm,raw_n_m_atm,raw_e_n_m_atm,raw_al_h,raw_e_al_h,raw_c_12_13,raw_e_c_12_13,raw_ca_h,raw_e_ca_h,raw_ce_h,raw_e_ce_h,raw_c_1_h,raw_e_c_1_h,raw_c_h,raw_e_c_h,raw_co_h,raw_e_co_h,raw_cr_h,raw_e_cr_h,raw_cu_h,raw_e_cu_h,raw_fe_h,raw_e_fe_h,raw_k_h,raw_e_k_h,raw_mg_h,raw_e_mg_h,raw_mn_h,raw_e_mn

In [155]:
print(t_sdssv_aspcap.colnames)

['sdss_id', 'sdss4_apogee_id', 'gaia_dr2_source_id', 'gaia_dr3_source_id', 'tic_v8_id', 'healpix', 'lead', 'version_id', 'catalogid', 'catalogid21', 'catalogid25', 'catalogid31', 'n_associated', 'n_neighborhood', 'sdss5_target_flags', 'sdss4_apogee_target1_flags', 'sdss4_apogee_target2_flags', 'sdss4_apogee2_target1_flags', 'sdss4_apogee2_target2_flags', 'sdss4_apogee2_target3_flags', 'sdss4_apogee_member_flags', 'sdss4_apogee_extra_target_flags', 'ra', 'dec', 'l', 'b', 'plx', 'e_plx', 'pmra', 'e_pmra', 'pmde', 'e_pmde', 'gaia_v_rad', 'gaia_e_v_rad', 'g_mag', 'bp_mag', 'rp_mag', 'j_mag', 'e_j_mag', 'h_mag', 'e_h_mag', 'k_mag', 'e_k_mag', 'ph_qual', 'bl_flg', 'cc_flg', 'w1_mag', 'e_w1_mag', 'w1_flux', 'w1_dflux', 'w1_frac', 'w2_mag', 'e_w2_mag', 'w2_flux', 'w2_dflux', 'w2_frac', 'w1uflags', 'w2uflags', 'w1aflags', 'w2aflags', 'mag4_5', 'd4_5m', 'rms_f4_5', 'sqf_4_5', 'mf4_5', 'csf', 'zgr_teff', 'zgr_e_teff', 'zgr_logg', 'zgr_e_logg', 'zgr_fe_h', 'zgr_e_fe_h', 'zgr_e', 'zgr_e_e', 'zgr_pl

In [156]:
print([s for s in t_sdssv_aspcap.colnames if 'flag' in s])

['sdss5_target_flags', 'sdss4_apogee_target1_flags', 'sdss4_apogee_target2_flags', 'sdss4_apogee2_target1_flags', 'sdss4_apogee2_target2_flags', 'sdss4_apogee2_target3_flags', 'sdss4_apogee_member_flags', 'sdss4_apogee_extra_target_flags', 'w1uflags', 'w2uflags', 'w1aflags', 'w2aflags', 'zgr_quality_flags', 'bailer_jones_flags', 'ebv_flags', 'u_jkc_mag_flag', 'b_jkc_mag_flag', 'v_jkc_mag_flag', 'r_jkc_mag_flag', 'i_jkc_mag_flag', 'u_sdss_mag_flag', 'g_sdss_mag_flag', 'r_sdss_mag_flag', 'i_sdss_mag_flag', 'z_sdss_mag_flag', 'y_ps1_mag_flag', 'spectrum_flags', 'doppler_flags', 'irfm_teff_flags', 'al_h_flags', 'c_12_13_flags', 'ca_h_flags', 'ce_h_flags', 'c_1_h_flags', 'c_h_flags', 'co_h_flags', 'cr_h_flags', 'cu_h_flags', 'fe_h_flags', 'k_h_flags', 'mg_h_flags', 'mn_h_flags', 'na_h_flags', 'nd_h_flags', 'ni_h_flags', 'n_h_flags', 'o_h_flags', 'p_h_flags', 'si_h_flags', 's_h_flags', 'ti_h_flags', 'ti_2_h_flags', 'v_h_flags', 'initial_flags', 'result_flags', 'flag_warn', 'flag_bad', 'calib

Documentation for flags: https://sdss-wiki.atlassian.net/wiki/spaces/MWM/pages/316145718/Flags

In [157]:
for flag_name in ['result_flags', 'spectrum_flags', 'fe_h_flags']:
    print(flag_name)
    for i in range(32):
        idx_sf = t_sdssv_aspcap[flag_name] & 2**i
        n = np.count_nonzero(idx_sf)
        print(f'{i: >2d}: {n: >8d} ({n/len(t_sdssv_aspcap): >6.2%})')
    print()

result_flags
 0:    54460 ( 4.97%)
 1:    21535 ( 1.97%)
 2:    21299 ( 1.94%)
 3:        0 ( 0.00%)
 4:        0 ( 0.00%)
 5:    84509 ( 7.71%)
 6:    50133 ( 4.58%)
 7:    70261 ( 6.41%)
 8:    43140 ( 3.94%)
 9:   358836 (32.76%)
10:   230128 (21.01%)
11:    16466 ( 1.50%)
12:    14502 ( 1.32%)
13:    67457 ( 6.16%)
14:    61009 ( 5.57%)
15:    55673 ( 5.08%)
16:    52576 ( 4.80%)
17:    51627 ( 4.71%)
18:    50887 ( 4.65%)
19:   276214 (25.21%)
20:    69202 ( 6.32%)
21:     3529 ( 0.32%)
22:   172271 (15.73%)
23:    11961 ( 1.09%)
24:    31165 ( 2.84%)
25:    23209 ( 2.12%)
26:    55260 ( 5.04%)
27:    87470 ( 7.98%)
28:        0 ( 0.00%)
29:        0 ( 0.00%)
30:        0 ( 0.00%)
31:        0 ( 0.00%)

spectrum_flags
 0:        0 ( 0.00%)
 1:        0 ( 0.00%)
 2:   122648 (11.20%)
 3:        0 ( 0.00%)
 4:    21038 ( 1.92%)
 5:        0 ( 0.00%)
 6:        0 ( 0.00%)
 7:        0 ( 0.00%)
 8:        0 ( 0.00%)
 9:    50209 ( 4.58%)
10:    31774 ( 2.90%)
11:   139265 (12.71%)
12:

In [158]:
np.count_nonzero(t_sdssv_aspcap['spectrum_flags'] & (2**0 | 2**2 | 2**19))

122648

In [159]:
print(np.count_nonzero(t_sdssv_aspcap['flag_bad']))

303371


In [160]:
fig,(ax0,ax1) = plt.subplots(1,2, figsize=(8,3))

ax0.hist(t_sdssv_aspcap['snr'], bins=np.logspace(0, 4, 100))
ax0.set_xscale('log')
ax0.set_xlabel('SNR')

ax1.hist(t_sdssv_aspcap['rchi2'], bins=np.logspace(-1, 4, 100))
ax1.set_xscale('log')
ax1.set_xlabel(r'$\chi_{\nu}^2$')

plt.show()

<Figure size 800x300 with 2 Axes>

A simple cut on $\chi^2$ might remove many high-SNR stars, because the errors in those stellar fits are dominated by systematic errors (either in the models or in the observed spectra). As SNR increases, the observed spectrum does not come ever closer to the model spectrum. Since the formal uncertainty on the fit still decreases, $\chi^2$ grows.

Here's a toy model that illustrates this effect, and which yields a functional form for $\chi^2$ as a function of SNR. Let's say there's a physical quantity $x$ that we observe, obtaining a measured value $\hat{x} \sim \mathcal{N}\left(x+\Delta,\sigma\right)$. The offset $\Delta$ between the true value and the mean observation is due to model mismatch. Another way of writing this is that $\hat{x} = x + \Delta + \sigma \hat{\delta}$, where $\hat{\delta} \sim \mathcal{N}\left(0,1\right)$. The average $\chi^2$ of the fit is given by
$$
  \langle\chi^2\rangle
    = \langle \left(\frac{\hat{x}-x}{\sigma}\right)^{\! 2} \rangle
    = \langle \left(\frac{\Delta + \hat{\delta}}{\sigma}\right)^{\! 2} \rangle
    = 1 + \left(\frac{\Delta}{\sigma}\right)^{\! 2}
    = 1 + \left(\frac{\Delta}{x} \frac{x}{\sigma}\right)^{\! 2}
    = 1 + \left(\frac{\Delta}{x}\right)^{\! 2} \mathrm{SNR}^2
  \, .
$$
This is an example with one degree of freedom. For a system with an arbitrary number of degrees of freedom, the above suggests that reduced $\chi^2$ should follow the form
$$
  \langle\chi_{\nu}^2\rangle = 1 + c \, \mathrm{SNR}^2 \, ,
$$
where $c$ is some unknown constant that depends on the fractional model mismatch (*e.g.*, the fractional flux residual introduced by model and observational systematics).

In [161]:
# f_log_rchi2_cut = lambda log_snr: (log_snr/2.5+0.2)**4. + 0.5

plt.hist2d(
    np.log10(t_sdssv_aspcap['snr']),
    np.log10(t_sdssv_aspcap['rchi2']),
    bins=100,
    range=[(0,4),(-1,4)],
    norm=LogNorm()
)

log_snr_range = np.linspace(0., 4.)
# log_rchi2_cut = f_log_rchi2_cut(log_snr_range)
# plt.plot(log_snr_range, log_rchi2_cut, c='gray')
plt.plot(log_snr_range, np.log10(1 + 0.0001*10**(2*log_snr_range)), c='gray')

plt.xlabel(r'$\log_{10} \mathrm{SNR}$')
plt.ylabel(r'$\log_{10} \chi^2_\nu$')
plt.colorbar(label=r'$\mathrm{counts}$')
plt.show()

/tmp/ipykernel_2429448/4278353026.py:5: RuntimeWarning: divide by zero encountered in log10
  np.log10(t_sdssv_aspcap['rchi2']),


<Figure size 640x480 with 2 Axes>

The above value of $c = 0.0001$ suggests that the flux systematics for SDSS-V (ASPCAP) are on the order of 1%.

In [162]:
plt.hist2d(
    np.log10(t_sdssv_aspcap['snr']),
    # np.log10(t_sdssv_aspcap['rchi2']/t_sdssv_aspcap['snr']),
    np.log10(t_sdssv_aspcap['rchi2']/(1 + 0.0001*t_sdssv_aspcap['snr']**2)),
    bins=100,
    range=[(0,4),(-2,4)],
    norm=LogNorm()
)

# log_snr_range = np.linspace(0., 4.)
# log_rchi2_cut = f_log_rchi2_cut(log_snr_range)
# plt.plot(log_snr_range, log_rchi2_cut, c='gray')

plt.xlabel(r'$\log_{10} \mathrm{SNR}$')
plt.ylabel(r'$\log_{10} \left[ \chi^2_\nu / \left(1+10^{-4}\mathrm{SNR}^2\right) \right]$')
plt.colorbar(label=r'$\mathrm{counts}$')
plt.show()

/tmp/ipykernel_2429448/1092730983.py:4: RuntimeWarning: divide by zero encountered in log10
  np.log10(t_sdssv_aspcap['rchi2']/(1 + 0.0001*t_sdssv_aspcap['snr']**2)),


<Figure size 640x480 with 2 Axes>

In [163]:
im = plot_utils.hist2d_reduce(
    5*np.log(t_sdssv_aspcap['teff']/5000.),
    t_sdssv_aspcap['logg'],
    t_sdssv_aspcap['rchi2']/(1 + 0.0001*t_sdssv_aspcap['snr']**2),
    xlim=(-3.5,7.5),
    ylim=(-1,6),
    hist_kw=dict(bins=100),
    imshow_kw=dict(norm=LogNorm(vmin=1e-1,vmax=1e2))
)
plt.gca().invert_xaxis()
plt.gca().invert_yaxis()

# log_snr_range = np.linspace(0., 4.)
# log_rchi2_cut = f_log_rchi2_cut(log_snr_range)
# plt.plot(log_snr_range, log_rchi2_cut, c='gray')

# plt.xlabel(r'$\log_{10} \mathrm{SNR}$')
# plt.ylabel(r'$\log_{10} \left[ \chi^2_\nu / \left(1+10^{-4}\mathrm{SNR}^2\right) \right]$')
plt.colorbar(im, label=r'$\mathrm{relative\ badness}$')
plt.show()

<Figure size 640x480 with 2 Axes>

The flux systematics should actually vary with stellar type, which is probably why the distribution flares at the right end of the plot. Using different values of $c$ for different types of stars might yield a narrower distribution on the right side of the plot, but that's way beyond the scope of this work!

We can also see from the above plots that $\chi_{\nu}^2$ starts to deviate from 1 around $\mathrm{SNR} \sim 10^{1.75}$.

In [164]:
plt.hist2d(
    np.log10(t_sdssv_aspcap['snr']),
    np.log10(t_sdssv_aspcap['e_teff']/t_sdssv_aspcap['teff']),
    bins=100,
    range=[(0,4),(-4,4)],
    norm=LogNorm()
)

plt.xlabel(r'$\log_{10} \mathrm{SNR}$')
plt.ylabel(r'$\log_{10} \left(\sigma_{T_{\mathrm{eff}}}/T_{\mathrm{eff}}\right)$')
plt.colorbar(label=r'$\mathrm{counts}$')
plt.show()

<Figure size 640x480 with 2 Axes>

In [165]:
for log_snr in [1.0, 1.5, 2.0]:
    idx_snr = (np.abs(np.log10(t_sdssv_aspcap['snr'])-log_snr) < 0.1)
    plt.hist(
        np.log10(t_sdssv_aspcap['e_teff']/t_sdssv_aspcap['teff'])[idx_snr],
        range=(-3,0),
        bins=100,
        histtype='step',
        density=True,
        label=rf'$\log_{{10}}\mathrm{{SNR}} \approx {log_snr:.1f}$'
    )
plt.legend()
plt.show()

<Figure size 640x480 with 1 Axes>

In [166]:
log_snr_range = np.linspace(0., 3.5, 30)

med_teff_err = []
med_feh_err = []
med_logg_err = []

teff_err = t_sdssv_aspcap['e_teff']/t_sdssv_aspcap['teff']

for log_snr in tqdm(log_snr_range):
    idx_snr = (np.abs(np.log10(t_sdssv_aspcap['snr'])-log_snr) < 0.1)
    med_teff_err.append(np.nanmedian(teff_err[idx_snr]))
    med_feh_err.append(np.nanmedian(t_sdssv_aspcap['e_fe_h'][idx_snr]))
    med_logg_err.append(np.nanmedian(t_sdssv_aspcap['e_logg'][idx_snr]))

plt.loglog(10**log_snr_range, med_teff_err, label='teff')
plt.loglog(10**log_snr_range, med_feh_err, label='feh')
plt.loglog(10**log_snr_range, med_logg_err, label='logg')
plt.axvline(10**1.75, c='k', ls=':', alpha=0.3)
plt.legend()

plt.xlabel('SNR')
plt.ylabel(r'$\sigma$')

plt.grid(True, alpha=0.1)
plt.show()

  0%|          | 0/30 [00:00<?, ?it/s]

<Figure size 640x480 with 1 Axes>

In [167]:
idx = (
    ~(t_sdssv_aspcap['flag_warn'] | t_sdssv_aspcap['flag_bad'])
  & np.isfinite(t_sdssv_aspcap['teff'])
  & np.isfinite(t_sdssv_aspcap['logg'])
  & np.isfinite(t_sdssv_aspcap['fe_h'])
  & np.isfinite(t_sdssv_aspcap['alpha_m_atm'])
  & (t_sdssv_aspcap['fe_h'] > -2.3)
  & (t_sdssv_aspcap['fe_h'] < 0.9)
  # # & np.isfinite(t_sdssv_aspcap['m_h_atm'])
  # # & (t_sdssv_aspcap['m_h_atm'] > -2.3)
  # # & (t_sdssv_aspcap['m_h_atm'] < 0.9)
  & (t_sdssv_aspcap['snr'] > 20.)
  # & (t_sdssv_aspcap['rchi2'] < 50.)
  # & (np.log10(t_sdssv_aspcap['rchi2']) < f_log_rchi2_cut(np.log10(t_sdssv_aspcap['snr'])))
  # & (t_sdssv_aspcap['rchi2']/t_sdssv_aspcap['snr'] < 10.)
  & (t_sdssv_aspcap['rchi2']/(1 + 0.0001*t_sdssv_aspcap['snr']**2) < 1e2)
  & ~((t_sdssv_aspcap['spectrum_flags'] & (2**0 | 2**2 | 2**19)).astype('bool'))
  # # & ~((t_sdssv_aspcap['fe_h_flags'] & (2**5 | 2**6 | 2**7 | 2**8)).astype('bool'))
  & (t_sdssv_aspcap['e_teff']/t_sdssv_aspcap['teff'] < 0.1)
  & (t_sdssv_aspcap['e_fe_h'] < 0.3)
  # # & (t_sdssv_aspcap['e_m_h_atm'] < 0.5)
  & (t_sdssv_aspcap['e_logg'] < 0.3)
  & (t_sdssv_aspcap['e_alpha_m_atm'] < 0.3)
)
print(fr'{np.mean(idx):.2%} ({np.count_nonzero(idx):,}) of sources pass cuts.')

plot_teff_logg_feh_abundance(
    t_sdssv_aspcap['teff'][idx],
    t_sdssv_aspcap['logg'][idx],
    # t_sdssv_aspcap['m_h_atm'][idx],
    t_sdssv_aspcap['fe_h'][idx],
    t_sdssv_aspcap['alpha_m_atm'][idx],
    label_as_mh=False
)

plt.show()

43.03% (471,418) of sources pass cuts.


<Figure size 800x400 with 2 Axes>

In [169]:
id_col = 'gaia_dr3_source_id'
param_cols = ('teff', 'fe_h', 'logg', 'alpha_m_atm')
param_unc_cols = ('e_teff', 'e_fe_h', 'e_logg', 'e_alpha_m_atm')
extra_cols = ('ra', 'dec')

# Increasing errors for T_eff > 6000 K
teff_errs = np.clip(
    0.1 + 0.4*(t_sdssv_aspcap['teff']-6000.)/1000.,
    0.1, 1.0
)
param_errs = np.stack([
    teff_errs,
    teff_errs,
    np.full_like(teff_errs, 0.1),
    np.full_like(teff_errs, 0.05)
], axis=1)

t_sdssv_asp = extract_table(
    t_sdssv_aspcap[idx],
    id_col, param_cols, param_unc_cols, extra_cols,
    param_transf=param_transf,
    param_unc_transf=apply_err_floors(param_unc_transf,err_floor_default),
    param_errs=param_errs[idx]
)
t_sdssv_asp

id,x,sigma_x,x_err,ra,dec
int64,float32[4],float32[4],float32[4],float32,float32
4184200643505162112,-0.50316447 .. -0.049784794,0.015907563 .. 0.051301874,0.1 .. 0.05,288.51138,-15.901711
4184125807991900928,-0.26131096 .. -0.052145794,0.025975363 .. 0.05158663,0.1 .. 0.05,288.96375,-16.299858
4087762031955829120,-0.25379193 .. -0.0718318,0.03186475 .. 0.0517759,0.1 .. 0.05,289.26428,-17.05394
4184137077986034048,-0.32278988 .. -0.1005408,0.07843741 .. 0.054019444,0.1 .. 0.05,289.29712,-16.052406
4183930438518525184,-0.2807538 .. -0.053944796,0.0281584 .. 0.051644582,0.1 .. 0.05,289.54077,-16.27299
422596679964513792,0.8236462 .. 0.014098205,0.045380414 .. 0.051967233,0.1 .. 0.05,0.0013319574,57.619507
422596679964513792,0.85806614 .. 0.033768207,0.036276605 .. 0.051662367,0.1 .. 0.05,0.0013319574,57.619507
4689447878791422208,-0.5652945 .. -0.058326796,0.049930777 .. 0.055163454,0.1 .. 0.05,0.0014806768,-73.394325
421077597267551104,-0.038942255 .. 0.018768206,0.018047854 .. 0.051305458,0.1 .. 0.05,0.0029132576,57.173077


In [170]:
plot_catalog_params(t_sdssv_asp)
plot_catalog_uncertainties(t_sdssv_asp)

<Figure size 500x400 with 1 Axes>

<Figure size 1050x280 with 5 Axes>

<Figure size 500x300 with 1 Axes>

In [171]:
# Save catalog
fn = 'training_data/sdssv_aspcap.h5'
t_sdssv_asp.write(fn, serialize_metadata=True, overwrite=True)
! du -sh {fn}

/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


29M	training_data/sdssv_aspcap.h5


### SDSS-V (BOSSNet)

Reference: [Sizemore *et al* (2024)](https://ui.adsabs.harvard.edu/abs/2024AJ....167..173S/abstract)

More information: https://sdss.org/dr19/mwm/astra/pipelines-in-astra/bossnet/

In [ ]:
url_base = 'https://data.sdss5.org/sas/dr19/spectro/astra/0.6.0/summary'
fn = 'astraAllStarBossNet-0.6.0.fits.gz'

out_fn = os.path.join(data_dir, fn)

! wget -O {out_fn} {url_base}/{fn}
! du -sh {out_fn}

In [448]:
# url_base = 'https://data.sdss5.org/sas/sdsswork/mwm/spectro/astra/0.6.0/summary/'
url_base = 'https://data.sdss5.org/sas/dr19/spectro/astra/0.6.0/summary/'
fn = 'astraAllStarBossNet-0.6.0.fits.gz'

os.makedirs(os.path.join(data_dir, 'sdssv'), exist_ok=True)

username = getpass('Username: ')
password = getpass('Password: ')

print(f'Downloading {fn} ...')

fname = os.path.join(data_dir, 'sdssv', fn)
url = f'{url_base}/{fn}'

download_with_auth(url, fname, username, password)

print(f'Downloaded to {fname}.')

Username:  ········
Password:  ········


astraAllStarBossNet-0.6.0.fits.gz:   0%|          | 0.00/170M [00:00<?, ?B/s]

Downloaded to /data/beegfs/astro-storage/groups/green/green/projects/gaia_Bp_Rp_reddening_variation/data/sdssv/astraAllStarBossNet-0.6.0.fits.gz.


In [40]:
fn = 'astraAllStarBossNet-0.6.0.fits.gz'
with fits.open(f'{data_dir}/sdssv/{fn}', mode='readonly') as f:
    f.info()
    t_sdssv_bossnet = Table(f[1].data)

t_sdssv_bossnet

Filename: data//sdssv/astraAllStarBossNet-0.6.0.fits.gz
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      16   ()      
  1                1 BinTableHDU    436   392839R x 176C   [K, 18A, K, K, K, J, 26A, J, K, K, K, K, J, J, 57B, K, K, K, K, K, K, K, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, 3A, 3A, 3A, E, E, E, E, E, E, E, E, E, E, K, K, K, K, E, E, E, K, K, K, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, K, E, E, E, E, E, E, 5A, E, E, K, E, E, E, E, E, E, E, E, E, E, E, E, E, E, J, E, J, E, J, E, J, E, J, E, J, E, J, E, J, E, J, E, J, E, J, J, J, J, J, J, J, K, K, 5A, 7A, 5A, 6A, 6A, J, J, J, J, J, E, E, E, E, E, E, E, E, E, E, E, E, K, K, E, K, K, K, 26A, E, E, 1A, E, E, E, E, E, E, K, L, L]   
  2                1 BinTableHDU    463   0R x 188C   [K, 1A, K, K, K, J, 1A, J, K, K, K, K, J, J, 1B, K, K, K, K, K, K, K, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, 1A, 1A, 1A, E, E, E, E, E, E, E, E,

sdss_id,sdss4_apogee_id,gaia_dr2_source_id,gaia_dr3_source_id,tic_v8_id,healpix,lead,version_id,catalogid,catalogid21,catalogid25,catalogid31,n_associated,n_neighborhood,sdss5_target_flags,sdss4_apogee_target1_flags,sdss4_apogee_target2_flags,sdss4_apogee2_target1_flags,sdss4_apogee2_target2_flags,sdss4_apogee2_target3_flags,sdss4_apogee_member_flags,sdss4_apogee_extra_target_flags,ra,dec,l,b,plx,e_plx,pmra,e_pmra,pmde,e_pmde,gaia_v_rad,gaia_e_v_rad,g_mag,bp_mag,rp_mag,j_mag,e_j_mag,h_mag,e_h_mag,k_mag,e_k_mag,ph_qual,bl_flg,cc_flg,w1_mag,e_w1_mag,w1_flux,w1_dflux,w1_frac,w2_mag,e_w2_mag,w2_flux,w2_dflux,w2_frac,w1uflags,w2uflags,w1aflags,w2aflags,mag4_5,d4_5m,rms_f4_5,sqf_4_5,mf4_5,csf,zgr_teff,zgr_e_teff,zgr_logg,zgr_e_logg,zgr_fe_h,zgr_e_fe_h,zgr_e,zgr_e_e,zgr_plx,zgr_e_plx,zgr_teff_confidence,zgr_logg_confidence,zgr_fe_h_confidence,zgr_ln_prior,zgr_chi2,zgr_quality_flags,r_med_geo,r_lo_geo,r_hi_geo,r_med_photogeo,r_lo_photogeo,r_hi_photogeo,bailer_jones_flags,ebv,e_ebv,ebv_flags,ebv_zhang_2023,e_ebv_zhang_2023,ebv_sfd,e_ebv_sfd,ebv_rjce_glimpse,e_ebv_rjce_glimpse,ebv_rjce_allwise,e_ebv_rjce_allwise,ebv_bayestar_2019,e_ebv_bayestar_2019,ebv_edenhofer_2023,e_ebv_edenhofer_2023,c_star,u_jkc_mag,u_jkc_mag_flag,b_jkc_mag,b_jkc_mag_flag,v_jkc_mag,v_jkc_mag_flag,r_jkc_mag,r_jkc_mag_flag,i_jkc_mag,i_jkc_mag_flag,u_sdss_mag,u_sdss_mag_flag,g_sdss_mag,g_sdss_mag_flag,r_sdss_mag,r_sdss_mag_flag,i_sdss_mag,i_sdss_mag_flag,z_sdss_mag,z_sdss_mag_flag,y_ps1_mag,y_ps1_mag_flag,n_boss_visits,boss_min_mjd,boss_max_mjd,n_apogee_visits,apogee_min_mjd,apogee_max_mjd,spectrum_pk,source,release,filetype,v_astra,run2d,telescope,min_mjd,max_mjd,n_visits,n_good_visits,n_good_rvs,v_rad,e_v_rad,std_v_rad,median_e_v_rad,xcsao_teff,xcsao_e_teff,xcsao_logg,xcsao_e_logg,xcsao_fe_h,xcsao_e_fe_h,xcsao_meanrxc,snr,gri_gaia_transform_flags,zwarning_flags,nmf_rchi2,nmf_flags,task_pk,source_pk,created,t_elapsed,t_overhead,tag,teff,e_teff,logg,e_logg,fe_h,e_fe_h,result_flags,flag_warn,flag_bad
int64,str18,int64,int64,int64,int32,str26,int32,int64,int64,int64,int64,int32,int32,uint8[57],int64,int64,int64,int64,int64,int64,int64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,str3,str3,str3,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int64,int64,int64,int64,float32,float32,float32,int64,int64,int64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int64,float32,float32,float32,float32,float32,float32,str5,float32,float32,int64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int32,float32,int32,float32,int32,float32,int32,float32,int32,float32,int32,float32,int32,float32,int32,float32,int32,float32,int32,float32,int32,int32,int32,int32,int32,int32,int32,int64,int64,str5,str7,str5,str6,str6,int32,int32,int32,int32,int32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int64,int64,float32,int64,int64,int64,str26,float32,float32,str1,float32,float32,float32,float32,float32,float32,int64,bool,bool
75287391,,3150324259254974592,3150324259254974592,468929587,80808,tic_v8,25,27021598102058280,4544491080,27021598102058280,63050395121643945,1,0,0 .. 0,0,0,0,0,0,0,0,118.361885,10.209756,210.64897,18.35674,0.6104934,0.062768236,-2.4849606,0.06703523,-0.32957748,0.037861325,nan,nan,16.323618,16.655676,15.82058,15.272,0.046,15.046,0.066,14.786,0.079,AAA,111,000,14.7820015,0.00882613,1217.8652,9.9002285,0.9879529,14.778689,0.03004605,1190.4857,32.944828,0.9873288,0,0,0,0,nan,nan,nan,0,0,0,5593.137,72.36395,4.477306,0.078787334,-0.33245793,0.14435722,0.009142888,0.01930513,0.5718634,0.061009586,0.9526367,0.86376953,0.93310547,2.59375,58.0625,0,1564.9097,1408.2542,1709.7014,1647.1108,1512.6243,1846.9017,10033,0.007579454,0.016003953,2,0.007579454,0.

In [41]:
t_sdssv_bossnet.colnames

['sdss_id',
 'sdss4_apogee_id',
 'gaia_dr2_source_id',
 'gaia_dr3_source_id',
 'tic_v8_id',
 'healpix',
 'lead',
 'version_id',
 'catalogid',
 'catalogid21',
 'catalogid25',
 'catalogid31',
 'n_associated',
 'n_neighborhood',
 'sdss5_target_flags',
 'sdss4_apogee_target1_flags',
 'sdss4_apogee_target2_flags',
 'sdss4_apogee2_target1_flags',
 'sdss4_apogee2_target2_flags',
 'sdss4_apogee2_target3_flags',
 'sdss4_apogee_member_flags',
 'sdss4_apogee_extra_target_flags',
 'ra',
 'dec',
 'l',
 'b',
 'plx',
 'e_plx',
 'pmra',
 'e_pmra',
 'pmde',
 'e_pmde',
 'gaia_v_rad',
 'gaia_e_v_rad',
 'g_mag',
 'bp_mag',
 'rp_mag',
 'j_mag',
 'e_j_mag',
 'h_mag',
 'e_h_mag',
 'k_mag',
 'e_k_mag',
 'ph_qual',
 'bl_flg',
 'cc_flg',
 'w1_mag',
 'e_w1_mag',
 'w1_flux',
 'w1_dflux',
 'w1_frac',
 'w2_mag',
 'e_w2_mag',
 'w2_flux',
 'w2_dflux',
 'w2_frac',
 'w1uflags',
 'w2uflags',
 'w1aflags',
 'w2aflags',
 'mag4_5',
 'd4_5m',
 'rms_f4_5',
 'sqf_4_5',
 'mf4_5',
 'csf',
 'zgr_teff',
 'zgr_e_teff',
 'zgr_logg',

In [42]:
fig,ax = plt.subplots(1,1, figsize=(4,3))

ax.hist(t_sdssv_bossnet['snr'], bins=np.logspace(0, 3, 100))
ax.set_xscale('log')
ax.set_xlabel('SNR')

plt.show()

<Figure size 400x300 with 1 Axes>

In [43]:
idx = (
    ~(t_sdssv_bossnet['flag_warn'] | t_sdssv_bossnet['flag_bad'])
  & (t_sdssv_bossnet['snr'] > 20.)
  & (t_sdssv_bossnet['e_teff']/t_sdssv_bossnet['teff'] < 0.1)
  & (t_sdssv_bossnet['e_fe_h'] < 0.3)
  & (t_sdssv_bossnet['e_logg'] < 0.3)
)
print(fr'{np.mean(idx):.2%} ({np.count_nonzero(idx):,}) '
      'of SDSS-V BOSSNet sources pass cuts.')

plot_teff_logg_feh(
    t_sdssv_bossnet['teff'][idx],
    t_sdssv_bossnet['logg'][idx],
    t_sdssv_bossnet['fe_h'][idx],
    label_as_mh=False
)

plt.show()

68.40% (268,699) of SDSS-V BOSSNet sources pass cuts.


<Figure size 800x400 with 2 Axes>

In [44]:
id_col = 'gaia_dr3_source_id'
param_cols = ('teff', 'fe_h', 'logg')
param_unc_cols = ('e_teff', 'e_fe_h', 'e_logg')
extra_cols = ('ra', 'dec')

t_sdssv_boss = extract_table(
    t_sdssv_bossnet[idx],
    id_col, param_cols, param_unc_cols, extra_cols,
    param_transf=param_transf[:-1],
    param_unc_transf=apply_err_floors(param_unc_transf[:-1],err_floor_default[:-1]),
    param_errs=(0.3,0.3,0.3) # Low confidence overall
)

t_sdssv_boss

id,x,sigma_x,x_err,ra,dec
int64,float32[3],float32[3],float32[3],float32,float32
3150324259254974592,0.55857325 .. 4.434894,0.033146475 .. 0.054870844,0.3 .. 0.3,118.361885,10.209756
3147337882594558976,0.7853263 .. 4.5385995,0.03605557 .. 0.04951979,0.3 .. 0.3,119.34362,9.963192
3147240537157873280,0.706309 .. 4.651678,0.044980556 .. 0.0557021,0.3 .. 0.3,119.216286,9.814338
3147289809023919232,0.6248627 .. 4.5898943,0.05048355 .. 0.0780921,0.3 .. 0.3,119.06122,9.918654
3147307164988425984,-1.0731573 .. 4.5949607,0.03737913 .. 0.05043116,0.3 .. 0.3,118.87514,10.048503
472199051104682112,0.376027 .. 4.4033327,0.055409998 .. 0.06412458,0.3 .. 0.3,65.35965,60.876175
947392029410286208,-0.2040601 .. 2.5664442,0.040993888 .. 0.067599215,0.3 .. 0.3,104.95154,39.29016
4372907693863364608,0.78126156 .. 4.2963624,0.019584825 .. 0.042486582,0.3 .. 0.3,267.36627,1.6943085
1182028708426051456,0.8352974 .. 4.1721907,0.054513063 .. 0.13344151,0.3 .. 0.3,224.49348,13.488542


In [45]:
plot_catalog_params(t_sdssv_boss)
plot_catalog_uncertainties(t_sdssv_boss)

<Figure size 500x400 with 1 Axes>

<Figure size 800x280 with 4 Axes>

<Figure size 500x300 with 1 Axes>

In [46]:
# Save catalog
fn = 'training_data/sdssv_bossnet.h5'
t_sdssv_boss.write(fn, serialize_metadata=True, overwrite=True)
! du -sh {fn}

/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


14M	training_data/sdssv_bossnet.h5


### SDSS-V APOGEE (old)

In [244]:
url_base = 'https://data.sdss5.org/sas/sdsswork/mwm/apogee/spectro/redux/1.4/summary'

sdssv_apogee_fnames = [
    'allStar-1.4-apo25m.fits',
    'allStar-1.4-lco25m.fits',
    # 'astraAllStarBossNet-0.6.0.fits.gz',
]

In [273]:
from getpass import getpass
import requests
import os

os.makedirs(os.path.join(data_dir, 'sdssv'), exist_ok=True)

username = getpass('Username: ')
password = getpass('Password: ')

for fn in sdssv_apogee_fnames:
    print(f'Downloading {fn} ...')
    
    fname = os.path.join(data_dir, 'sdssv', fn)
    url = f'{url_base}/{fn}'
    
    r = requests.get(url, auth=(username, password), stream=True)
    r.raise_for_status()
    
    # Figure out total file size (in Bytes)
    file_size = int(r.headers.get('content-length', 0))
    
    # Open output file and wrap a tqdm around the iteration
    with open(fname, 'wb') as f, tqdm(total=file_size, unit='B',
                                      unit_scale=True, unit_divisor=1024,
                                      desc=fn
                                 ) as bar:
        for chunk in r.iter_content(chunk_size=1024):
            if not chunk:
                continue
            f.write(chunk)
            bar.update(len(chunk))
    
    print(f'Downloaded to {fname}.')

allStar-1.4-lco25m.fits:   0%|          | 0.00/694M [00:00<?, ?B/s]

Downloaded to /data/beegfs/astro-storage/groups/green/green/projects/gaia_Bp_Rp_reddening_variation/data/sdssv/allStar-1.4-lco25m.fits.


In [214]:
t_sdssv_apogee = []

for fn in sdssv_apogee_fnames:
    print(f'Loading {fn} ...')
    with fits.open(f'{data_dir}/sdssv/{fn}', mode='readonly') as f:
        d = Table(f[1].data)
    t_sdssv_apogee.append(d)

print('Stacking SDSS-V APOGEE tables ...')
t_sdssv_apogee = astropy.table.vstack(t_sdssv_apogee)

t_sdssv_apogee

Loading allStar-1.4-apo25m.fits ...
Loading allStar-1.4-lco25m.fits ...
Stacking SDSS-V APOGEE tables ...


pk,apogee_id,file,uri,starver,mjdbeg,mjdend,telescope,apred_vers,healpix,snr,ra,dec,glon,glat,jmag,jerr,hmag,herr,kmag,kerr,src_h,targ_pmra,targ_pmdec,targ_pm_src,apogee_target1,apogee_target2,apogee2_target1,apogee2_target2,apogee2_target3,apogee2_target4,catalogid,gaia_sourceid,gaia_plx,gaia_plx_error,gaia_pmra,gaia_pmra_error,gaia_pmdec,gaia_pmdec_error,gaia_gmag,gaia_gerr,gaia_bpmag,gaia_bperr,gaia_rpmag,gaia_rperr,sdssv_apogee_target0,firstcarton,targflags,nvisits,ngoodvisits,ngoodrvs,starflag,starflags,andflag,andflags,vrad,vscatter,verr,vmederr,chisq,rv_teff,rv_tefferr,rv_logg,rv_loggerr,rv_feh,rv_feherr,rv_ccpfwhm,rv_autofwhm,n_components,meanfib,sigfib,created,v_apred,cadence,program,category,gaia_release,sdss_id,sdss5_target_carton_pks,sdss5_target_cartons,sdss5_target_flagshex,ra_sdss_id,dec_sdss_id,sdss5_target_pks,sdss5_target_catalogids,brightneicount,brightneiflag,brightneifluxfrac
int64,str28,str57,str106,str15,int64,int64,str16,str13,int64,float32,float64,float64,float64,float64,float32,float32,float32,float32,float32,float32,str10,float32,float32,str10,int64,int64,int64,int64,int64,int64,int64,int64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int64,str60,str113,int64,int64,int64,int64,str100,int64,str10,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int64,float32,float32,str39,str50,str10,str10,str10,str13,int64,str10,str10,str10,float64,float64,str11,str10,int64,int64,float32
7141954,2M00000007-0529397,apStar-1.4-apo25m-2M00000007-0529397-60547.fits,apogee/spectro/redux/1.4/stars/apo25m/107/107776/apStar-1.4-apo25m-2M00000007-0529397-60547.fits,60547,60547,60547,apo25m,1.4,107776,134.91522,0.0004348245428307492,-5.494409354452808,91.1316945689355,-65.14038277649257,7.387,0.017,6.921,0.051,6.829,0.021,,22.498114,-11.301015,,0,0,0,0,0,0,63050395065135105,2443095153084654208,1.9638819,0.018856796,22.498114,0.0224626,-11.301015,0.014800338,8.871953,0.00014253429,9.367798,0.00037024976,8.215054,0.00030609893,0,mwm_tess_rgb_apogee,,1,1,1,1048576,RV_SUSPECT,0,,-17.116207,0.0,0.026486306,0.026486306,4.467233,5050.6616,8.978264,2.962437,0.017323287,-0.3144337,0.0059460304,nan,nan,1,97.0,0.0,2024-10-09 07:25:13.580544-06,,,,,dr3,69834792,,,,0.0,0.0,0,,0,0,0.0
7143652,2M00000021+0105203,apStar-1.4-apo25m-2M00000021+0105203-60539.fits,apogee/spectro/redux/1.4/stars/apo25m/96/96512/apStar-1.4-apo25m-2M00000021+0105203-60539.fits,60539,59189,60539,apo25m,1.4,96512,1468.2216,0.0008720457546393024,1.0889790710102785,97.1857878678001,-59.18723337522808,8.219,0.021,8.063,0.043,7.95,0.017,,-0.36014107,-5.052624,,0,0,0,0,0,0,27021599488051181,2738327528519591936,5.415096,0.03313692,-0.36014107,0.057715207,-5.052624,0.021570347,8.998321,0.00021043982,9.220084,0.00035643717,8.618513,0.00021609796,0,ops_std_apogee,,20,20,20,2099200,"PERSIST_LOW,MULTIPLE_SUSPECT",0,,10.447248,0.16719754,0.037386507,0.03689169,2.791146,6377.5513,3.2027228,4.4480147,0.0028086666,-0.5029805,0.0018121187,nan,nan,2,126.65161,63.707886,2024-10-09 07:28:58.755305-06,,,,,dr3,71368882,,,,0.0,0.0,0,,0,0,0.0
7141938,2M00000025+5127529,apStar-1.4-apo25m-2M00000025+5127529-60562.fits,apogee/spectro/redux/1.4/stars/apo25m/21/21424/apStar-1.4-apo25m-2M00000025+5127529-60562.fits,60562,60562,60562,apo25m,1.4,21424,131.36763,0.001008998460932705,51.46466051098828,114.82335721643071,-10.595374271654682,10.448,0.018,9.875,0.019,9.754,0.016,,-5.0783653,-7.4657373,,0,0,0,0,0,0,63050396589445116,395574979120575360,0.5227943,0.010095278,-5.0783653,0.008196555,-7.4657373,0.008570265,12.188036,0.00013844862,12.797315,0.00042207493,11.4409895,0.00021232096,0,mwm_tess_rgb_apogee,,1,1,1,1048576,RV_SUSPECT,0,,-43.56185,0.0,0.026387809,0.026387809,3.7554305,5044.4194,8.460055,2.926359,0.016714765,-0.47018722,0.005602602,nan,nan,1,173.0,0.0,2024-10-09 07:25:11.150359-06,,,,,dr3,117290797,,,,0.0,0.0,0,,0,0,0.0
7141913,2M00000032+5737103,apStar-1.4-apo25m-2M00000032+5737103

In [215]:
t_sdssv_apogee.colnames

['pk',
 'apogee_id',
 'file',
 'uri',
 'starver',
 'mjdbeg',
 'mjdend',
 'telescope',
 'apred_vers',
 'healpix',
 'snr',
 'ra',
 'dec',
 'glon',
 'glat',
 'jmag',
 'jerr',
 'hmag',
 'herr',
 'kmag',
 'kerr',
 'src_h',
 'targ_pmra',
 'targ_pmdec',
 'targ_pm_src',
 'apogee_target1',
 'apogee_target2',
 'apogee2_target1',
 'apogee2_target2',
 'apogee2_target3',
 'apogee2_target4',
 'catalogid',
 'gaia_sourceid',
 'gaia_plx',
 'gaia_plx_error',
 'gaia_pmra',
 'gaia_pmra_error',
 'gaia_pmdec',
 'gaia_pmdec_error',
 'gaia_gmag',
 'gaia_gerr',
 'gaia_bpmag',
 'gaia_bperr',
 'gaia_rpmag',
 'gaia_rperr',
 'sdssv_apogee_target0',
 'firstcarton',
 'targflags',
 'nvisits',
 'ngoodvisits',
 'ngoodrvs',
 'starflag',
 'starflags',
 'andflag',
 'andflags',
 'vrad',
 'vscatter',
 'verr',
 'vmederr',
 'chisq',
 'rv_teff',
 'rv_tefferr',
 'rv_logg',
 'rv_loggerr',
 'rv_feh',
 'rv_feherr',
 'rv_ccpfwhm',
 'rv_autofwhm',
 'n_components',
 'meanfib',
 'sigfib',
 'created',
 'v_apred',
 'cadence',
 'program'

In [216]:
t_sdssv_apogee['starflags']

RV_SUSPECT
"PERSIST_LOW,MULTIPLE_SUSPECT"
RV_SUSPECT
""
RV_SUSPECT
""
RV_SUSPECT
RV_SUSPECT
RV_SUSPECT
"PERSIST_LOW,RV_SUSPECT"
""


In [217]:
plt.hist(np.log10(t_sdssv_apogee['snr']), range=(0,4), bins=100)
plt.xlabel('SNR')
plt.show()

/tmp/ipykernel_2991738/2116793509.py:1: RuntimeWarning: divide by zero encountered in log10
  plt.hist(np.log10(t_sdssv_apogee['snr']), range=(0,4), bins=100)


<Figure size 640x480 with 1 Axes>

In [218]:
plt.hist(t_sdssv_apogee['chisq'], range=(0,40), bins=100)
plt.xlabel(r'$\chi^2$')
plt.show()

<Figure size 640x480 with 1 Axes>

In [229]:
plt.hist(t_sdssv_apogee['rv_teff'], bins=100, log=True)
plt.grid(True, alpha=0.2)
plt.xlabel(r'$T_{\mathrm{eff}}$')
plt.axvline(8300., color='k')
plt.show()

<Figure size 640x480 with 1 Axes>

In [219]:
teff_scaled = 5*np.log(t_sdssv_apogee['rv_teff']/5000)

In [233]:
idx = (
    (t_sdssv_apogee['snr'] > 30)
  & (t_sdssv_apogee['chisq'] < 15)
  # & (t_sdssv_apogee['starflag'] == 0)
  & (t_sdssv_apogee['rv_teff'] > 8.3e3)
  & (t_sdssv_apogee['rv_teff'] < 30e3)
  & (t_sdssv_apogee['rv_logg'] > 2.55)
  & (t_sdssv_apogee['rv_logg'] < 4.95)
  & (t_sdssv_apogee['rv_feh'] > -2.45)
  & (t_sdssv_apogee['rv_tefferr'] < 500.)
  & (t_sdssv_apogee['rv_feherr'] < 0.5)
  & (t_sdssv_apogee['rv_loggerr'] < 0.5)
)
print(f'SDSS-V (APOGEE): {np.count_nonzero(idx):,} ({100*np.mean(idx):.2f}%) good sources.')

fig,ax_arr = plt.subplots(1,2, figsize=(8,4), layout='tight')

ax = ax_arr[0]
ax.hist2d(t_sdssv_apogee['rv_teff'][idx], t_sdssv_apogee['rv_logg'][idx],
          bins=100, norm=LogNorm())
ax.invert_xaxis()
ax.invert_yaxis()
ax.set_xlabel(r'$5\ln\left(T_\mathrm{eff}/5000\,\mathrm{K}\right)$')
ax.set_ylabel(r'$\log g$')
ax.grid(True, alpha=0.2)

ax = ax_arr[1]
ax.hist2d(teff_scaled[idx], t_sdssv_apogee['rv_feh'][idx],
          bins=100, norm=LogNorm())
ax.invert_xaxis()
ax.set_xlabel(r'$5\ln\left(T_\mathrm{eff}/5000\,\mathrm{K}\right)$')
ax.set_ylabel(r'$\left[\mathrm{Fe/H}\right]$')
ax.grid(True, alpha=0.2)

plt.show()

SDSS-V (APOGEE): 22,432 (1.33%) good sources.


<Figure size 800x400 with 2 Axes>

### DESI (EDR; DD-Payne method)

References:
* [Zhang *et al* (2024)](https://ui.adsabs.harvard.edu/abs/2024ApJS..273...19Z/abstract)

In [172]:
url = 'https://cdsarc.cds.unistra.fr/viz-bin/nph-Cat/fits.gz?J/ApJS/273/19/catalog.dat.gz'
fn = os.path.join(data_dir, 'desi_edr', 'desi_edr_zhang_etal_ddpayne.fits.gz')

! mkdir {data_dir}/desi_edr
! wget {url} -O {fn}

mkdir: cannot create directory ‘/data/beegfs/astro-storage/groups/green/green/projects/gaia_Bp_Rp_reddening_variation/data//desi_edr’: File exists
--2025-05-30 16:21:43--  https://cdsarc.cds.unistra.fr/viz-bin/nph-Cat/fits.gz?J/ApJS/273/19/catalog.dat.gz
Resolving cdsarc.cds.unistra.fr (cdsarc.cds.unistra.fr)... 130.79.128.5
Connecting to cdsarc.cds.unistra.fr (cdsarc.cds.unistra.fr)|130.79.128.5|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/fits]
Saving to: ‘/data/beegfs/astro-storage/groups/green/green/projects/gaia_Bp_Rp_reddening_variation/data//desi_edr/desi_edr_zhang_etal_ddpayne.fits.gz’

/data/beegfs/astro-     [          <=>       ] 112.02M  4.70MB/s    in 24s     

2025-05-30 16:22:08 (4.58 MB/s) - ‘/data/beegfs/astro-storage/groups/green/green/projects/gaia_Bp_Rp_reddening_variation/data//desi_edr/desi_edr_zhang_etal_ddpayne.fits.gz’ saved [117465823]



In [252]:
fn = os.path.join(data_dir, 'desi_edr', 'desi_edr_zhang_etal_ddpayne.fits.gz')

with fits.open(fn, mode='readonly') as f:
    # print(f[1].header.tostring(sep='\n'))
    t_desi_ddpayne = Table(f[1].data)

print(t_desi_ddpayne.colnames)

['RAdeg', 'DEdeg', 'ID', 'Survey', 'Prog', 'Prim', 'Hpix', 'snrB', 'snrG', 'snrR', 'RV', 'e_RV', 'Teff', 'e_Teff', 'logg', 'e_logg', '[Fe/H]', 'e_[Fe/H]', '[C/Fe]', 'e_[C/Fe]', '[N/Fe]', 'e_[N/Fe]', '[Mg/Fe]', 'e_[Mg/Fe]', '[O/Fe]', 'e_[O/Fe]', '[Al/Fe]', 'e_[Al/Fe]', '[Si/Fe]', 'e_[Si/Fe]', '[Ca/Fe]', 'e_[Ca/Fe]', '[Ti/Fe]', 'e_[Ti/Fe]', '[Cr/Fe]', 'e_[Cr/Fe]', '[Mn/Fe]', 'e_[Mn/Fe]', '[Ni/Fe]', 'e_[Ni/Fe]', 'Vmic', 'e_Vmic', 'CorTeff-logg', 'CorTeff-feh', 'Corlogg-feh', 'q_Chi2', 'Chi2', 'logDist', 'e_logDist', 'EBV', 'e_EBV', 'X', 'Y', 'Z', 'R', 'PHI', 'VX', 'VY', 'VZ', 'VR', 'VT', 'JR', 'JPHI', 'JZ', 'LX', 'LY', 'LZ', 'RadGuid', 'Rapo', 'Rperi', 'Ecc', 'Zmax', 'Energy', 'omegaPHI', 'omegaR', 'omegaZ', 'ER', 'EZ']


In [192]:
fig,(ax0,ax1) = plt.subplots(1,2, figsize=(8,3))

ax0.hist(t_desi_ddpayne['snrB'], bins=np.logspace(-1, 3, 100))
ax0.set_xscale('log')
ax0.set_xlabel('S/N (B band)')

ax1.hist(t_desi_ddpayne['Chi2'], bins=np.logspace(-1, 1, 100))
ax1.set_xscale('log')
ax1.set_xlabel(r'$\chi^2$')

plt.show()

<Figure size 800x300 with 2 Axes>

In [950]:
def gaia_crossmatch(ra, dec):
    t_upload = Table({
        'ra': ra,
        'dec': dec,
        'index': np.arange(len(ra))
    })
    Gaia.upload_table(upload_resource=t_upload, table_name='crossmatch_table')
    
    query = """
      SELECT
        t.index,
        g.source_id, g.ra, g.dec,
        DISTANCE(
          POINT(t.ra, t.dec),
          POINT(g.ra, g.dec)
        ) * 3600. AS angsep_arcsec
      FROM user_ggreen.crossmatch_table AS t
      JOIN gaiadr3.gaia_source_lite AS g
      ON DISTANCE(POINT(t.ra, t.dec), POINT(g.ra, g.dec)) < 1. / 3600.
    """.strip()
    
    job = Gaia.launch_job_async(query, verbose=True)

    Gaia.delete_user_table(table_name='crossmatch_table')
    
    job.results.sort(['angsep_arcsec'])
    t_index = astropy.table.unique(
        job.results,
        keys=['index'],
        keep='first'
    )
    print(f'{len(t_index):,}')

    kw = dict(bins=100, range=(0,1), histtype='step', log=True)
    plt.hist(job.results['angsep_arcsec'], label='cone search', **kw)
    plt.hist(t_index['angsep_arcsec'], label='nearest only', **kw)
    plt.xlabel('separation (arcsec)')
    plt.legend()
    plt.show()

    idx = (t_index['angsep_arcsec'] < 0.5)
    print(f'{np.count_nonzero(idx):,} ({np.mean(idx):.1%})')
    t_index = t_index[idx]

    return {
        'match_idx': t_index['index'],
        'gdr3_source_id': t_index['source_id'],
        'ra': t_index['ra'],
        'dec': t_index['dec']
    }

In [330]:
Gaia.login()

INFO:astroquery:Login to gaia TAP server


INFO: Login to gaia TAP server [astroquery.gaia.core]


User:  ggreen
Password:  ········


INFO:astroquery:OK
INFO:astroquery:Login to gaia data server


INFO: OK [astroquery.utils.tap.core]
INFO: Login to gaia data server [astroquery.gaia.core]


INFO:astroquery:OK


INFO: OK [astroquery.utils.tap.core]


In [334]:
crossmatch_info = gaia_crossmatch(t_desi_ddpayne['RAdeg'], t_desi_ddpayne['DEdeg'])

INFO:astroquery:Sending pytable.


INFO: Sending pytable. [astroquery.utils.tap.core]


INFO:astroquery:Uploaded table 'crossmatch_table'.


INFO: Uploaded table 'crossmatch_table'. [astroquery.utils.tap.core]
Launched query: 'SELECT
        t.index,
        g.source_id, g.ra, g.dec,
        DISTANCE(
          POINT(t.ra, t.dec),
          POINT(g.ra, g.dec)
        ) * 3600. AS angsep_arcsec
      FROM user_ggreen.crossmatch_table AS t
      JOIN gaiadr3.gaia_source_lite AS g
      ON DISTANCE(POINT(t.ra, t.dec), POINT(g.ra, g.dec)) < 1. / 3600.'
------>https
host = gea.esac.esa.int:443
context = /tap-server/tap/async
Content-type = application/x-www-form-urlencoded
303 303
[('Date', 'Sat, 31 May 2025 19:52:50 GMT'), ('Server', 'Apache/2.4.6 (SLES Expanded Support platform 7) OpenSSL/1.0.2k-fips mod_jk/1.2.43'), ('X-VO-Authenticated', 'ggreen'), ('Location', 'https://gea.esac.esa.int/tap-server/tap/async/1748721170696O'), ('Cache-Control', 'no-cache, no-store, max-age=0, must-revalidate'), ('Pragma', 'no-cache'), ('Expires', '0'), ('X-XSS-Protection', '1; mode=block'), ('X-Frame-Options', 'SAMEORIGIN'), ('X-Content-Type-O

INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]


INFO:astroquery:Table 'crossmatch_table' deleted.


INFO: Table 'crossmatch_table' deleted. [astroquery.utils.tap.core]
503,432


<Figure size 640x480 with 1 Axes>

501,739 (99.7%)


In [336]:
t_desi_ddpayne = t_desi_ddpayne[crossmatch_info['match_idx']]
t_desi_ddpayne['ra'] = crossmatch_info['ra']
t_desi_ddpayne['dec'] = crossmatch_info['dec']
t_desi_ddpayne['gdr3_source_id'] = crossmatch_info['gdr3_source_id']

In [337]:
fn = os.path.join(data_dir, 'DESI_EDR_Zhang_etal_2024_with_GDR3.h5')
t_desi_ddpayne.write(fn, compression='gzip', compression_opts=3, chunks=True)

/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


In [172]:
fn = os.path.join(data_dir, 'DESI_EDR_Zhang_etal_2024_with_GDR3.h5')
t_desi_ddpayne = Table.read(fn)
print(f'{len(t_desi_ddpayne):,} sources in catalog.')

501,739 sources in catalog.


In [173]:
plt.hist2d(
    np.log10(t_desi_ddpayne['snrB']),
    np.log10(t_desi_ddpayne['Chi2']/t_desi_ddpayne['snrB']**0.25),
    range=[(-1,4),(-1,4)],
    bins=100,
    norm=LogNorm()
)
plt.show()

/tmp/ipykernel_2429448/2636800517.py:2: RuntimeWarning: divide by zero encountered in log10
  np.log10(t_desi_ddpayne['snrB']),
/tmp/ipykernel_2429448/2636800517.py:2: RuntimeWarning: invalid value encountered in log10
  np.log10(t_desi_ddpayne['snrB']),
/tmp/ipykernel_2429448/2636800517.py:3: RuntimeWarning: invalid value encountered in power
  np.log10(t_desi_ddpayne['Chi2']/t_desi_ddpayne['snrB']**0.25),
/tmp/ipykernel_2429448/2636800517.py:3: RuntimeWarning: divide by zero encountered in divide
  np.log10(t_desi_ddpayne['Chi2']/t_desi_ddpayne['snrB']**0.25),
/tmp/ipykernel_2429448/2636800517.py:3: RuntimeWarning: divide by zero encountered in log10
  np.log10(t_desi_ddpayne['Chi2']/t_desi_ddpayne['snrB']**0.25),


<Figure size 640x480 with 1 Axes>

In [174]:
alpha_Fe,alpha_Fe_err,abundances = calc_alpha_Fe_from_abundances(
    t_desi_ddpayne,
    '[{x}/Fe]',
    'e_[{x}/Fe]'
)

t_desi_ddpayne['[alpha/Fe]'] = alpha_Fe
t_desi_ddpayne['e_[alpha/Fe]'] = alpha_Fe_err

print(f'Used the abundances of {abundances.colnames}')

Used the abundances of ['O', 'Mg', 'Si', 'Ca', 'Ti']


In [175]:
idx = (
    (t_desi_ddpayne['snrB'] > 20.)
  # & (t_desi_ddpayne['Chi2'] < 3.)
  & (t_desi_ddpayne['Chi2']/t_desi_ddpayne['snrB']**0.25 < 3.)
  & (t_desi_ddpayne['[Fe/H]'] > -3.)
  & (t_desi_ddpayne['Teff'] > 4100.)
  & (t_desi_ddpayne['Teff'] < 8000.)
  & ~(
         (t_desi_ddpayne['Teff'] < 4400.)
       & (t_desi_ddpayne['logg'] > 3.)
    )
  & (t_desi_ddpayne['e_Teff'] < 200.)
  & (t_desi_ddpayne['e_logg'] < 0.2)
  & (t_desi_ddpayne['e_[Fe/H]'] < 0.2)
)
print(f'{np.count_nonzero(idx):,} ({np.mean(idx):.2%}) good sources.')

fig,ax_arr = plt.subplots(1,2, figsize=(8,4), layout='tight')

hist2d_teff_logg(
    t_desi_ddpayne['Teff'][idx],
    t_desi_ddpayne['logg'][idx],
    ax=ax_arr[0]
)
# ax = ax_arr[0]
# ax.hist2d(t_desi_ddpayne['Teff'][idx],
#           t_desi_ddpayne['logg'][idx],
#           bins=100, norm=LogNorm())
# ax.invert_xaxis()
# ax.invert_yaxis()
# ax.set_xlabel(r'$T_\mathrm{eff}$')
# ax.set_ylabel(r'$\log g$')
# ax.grid(True, alpha=0.2)

ax = ax_arr[1]
ax.hist2d(t_desi_ddpayne['[Fe/H]'][idx],
          t_desi_ddpayne['[alpha/Fe]'][idx],
          bins=100, norm=LogNorm())
ax.set_xlabel(r'$\left[\mathrm{Fe/H}\right]$')
ax.set_ylabel(r'$\left[\mathrm{\alpha/Fe}\right]$')
ax.grid(True, alpha=0.2)

plt.show()

/tmp/ipykernel_2429448/1162785741.py:4: RuntimeWarning: invalid value encountered in power
  & (t_desi_ddpayne['Chi2']/t_desi_ddpayne['snrB']**0.25 < 3.)
/tmp/ipykernel_2429448/1162785741.py:4: RuntimeWarning: divide by zero encountered in divide
  & (t_desi_ddpayne['Chi2']/t_desi_ddpayne['snrB']**0.25 < 3.)


141,766 (28.25%) good sources.


<Figure size 800x400 with 2 Axes>

In [176]:
plot_alpha_abundances(alpha_Fe[idx], abundances[idx])
plt.show()

<Figure size 750x500 with 6 Axes>

In [177]:
id_col = 'gdr3_source_id'
param_cols = ('Teff', '[Fe/H]', 'logg', '[alpha/Fe]')
param_unc_cols = ('e_Teff', 'e_[Fe/H]', 'e_logg', 'e_[alpha/Fe]')
extra_cols = ('ra', 'dec')

# Increasing errors for T_eff > 6000 K
teff_errs = np.clip(
    0.1 + 0.4*(t_desi_ddpayne['Teff']-6000.)/1000.,
    0.1, 1.0
)
param_errs = np.stack([
    teff_errs,
    np.full_like(teff_errs, 0.1),
    np.full_like(teff_errs, 0.1),
    np.full_like(teff_errs, 0.1)
], axis=1)

t_desi_ddp = extract_table(
    t_desi_ddpayne[idx],
    id_col, param_cols, param_unc_cols, extra_cols,
    param_transf=param_transf,
    param_unc_transf=apply_err_floors(param_unc_transf,err_floor_default),
    param_errs=param_errs[idx]
)
t_desi_ddp

id,x,sigma_x,x_err,ra,dec
int64,float64[4],float64[4],float64[4],float64,float64
315297397490915328,-0.44415606853307854 .. 0.02904215931148048,0.014423024979406047 .. 0.05079208569959466,0.1 .. 0.1,22.949367953056377,30.61326103670483
303280529969032064,0.2561644628169852 .. 0.09619093292896053,0.020944118760948087 .. 0.05156558656769924,0.1 .. 0.1,23.221633071966938,30.365683286961264
303287917313238528,-0.24388232131638032 .. 0.1220961639997064,0.02381041971204163 .. 0.05266119429076815,0.1 .. 0.1,23.22408711788352,30.5902587898615
303269122536415488,1.0848466498854936 .. -0.11613813297371832,0.03182966214493286 .. 0.05811645823323829,0.18461199999999992 .. 0.1,23.32985512473196,30.389537007612486
315395498839035008,0.8266483782363043 .. 0.09911752638868862,0.017227455253907547 .. 0.05197859999905381,0.1 .. 0.1,23.346034688570143,30.836559952594158
303283381827916928,0.6036755920678036 .. 0.11386026881856336,0.03914260561029213 .. 0.06174436807968439,0.1 .. 0.1,23.356598630134364,30.49908289644165
303262967847222528,0.5013891030991697 .. 0.11765161714931825,0.025290753798550714 .. 0.0527927785475533,0.1 .. 0.1,23.367567826889633,30.192177529213545
303263758120550016,0.36430025483366485 .. 0.05753582439613395,0.028434633727795994 .. 0.05305542108547893,0.1 .. 0.1,23.414093203472568,30.253385235575298
315432362542973440,0.6760099819432177 .. 0.19985275324613033,0.019639526730937575 .. 0.05301829042115056,0.1 .. 0.1,23.452742379108493,31.15148313019978


In [178]:
plot_catalog_params(t_desi_ddp)
plot_catalog_uncertainties(t_desi_ddp)

<Figure size 500x400 with 1 Axes>

<Figure size 1050x280 with 5 Axes>

<Figure size 500x300 with 1 Axes>

In [179]:
# Save catalog
fn = 'training_data/desi_ddp.h5'
t_desi_ddp.write(fn, serialize_metadata=True, overwrite=True)
! du -sh {fn}

/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


17M	training_data/desi_ddp.h5


### DESI (DR1)

References:
* [Milky Way Survey Catalog of Stellar Radial Velocities and Stellar Parameters](https://data.desi.lbl.gov/doc/releases/dr1/vac/mws/)
* [Koposov *et al* (2024)](https://ui.adsabs.harvard.edu/abs/2024MNRAS.533.1012K/abstract)

In [180]:
desi_fnames = [
    'rvpix-main-backup.fits',
    'rvpix-main-dark.fits',
    'rvpix-main-bright.fits',
]

In [55]:
url_base = 'https://data.desi.lbl.gov/public/dr1/vac/dr1/mws/iron/v1.0/rv_output/240520'

! mkdir {data_dir}/desi_dr1

for fn in desi_fnames:
    ! wget {url_base}/{fn} -O {data_dir}/desi_dr1/{fn}

In [181]:
t_desi_dr1 = []

for fn in desi_fnames:
    print(f'Loading {fn} ...')
    with fits.open(f'{data_dir}/desi_dr1/{fn}', mode='readonly') as f:
        d = Table(f['RVTAB'].data)
        d['gdr3_source_id'] = f['GAIA'].data['SOURCE_ID'][:]
    t_desi_dr1.append(d)

print('Stacking DESI tables ...')
t_desi_dr1 = astropy.table.vstack(t_desi_dr1)

Loading rvpix-main-backup.fits ...
Loading rvpix-main-dark.fits ...
Loading rvpix-main-bright.fits ...
Stacking DESI tables ...


In [182]:
t_desi_dr1

VRAD,VRAD_ERR,VRAD_SKEW,VRAD_KURT,LOGG,TEFF,ALPHAFE,FEH,LOGG_ERR,TEFF_ERR,ALPHAFE_ERR,FEH_ERR,VSINI,CHISQ_TOT,CHISQ_C_TOT,CHISQ_B,CHISQ_C_B,CHISQ_R,CHISQ_C_R,CHISQ_Z,CHISQ_C_Z,RVS_WARN,REF_ID,REF_CAT,TARGET_RA,TARGET_DEC,TARGETID,SN_B,SN_R,SN_Z,SUCCESS,RR_Z,RR_SPECTYPE,HEALPIX,gdr3_source_id
float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64,str2,float64,float64,int64,float32,float32,float32,bool,float64,str6,int64,int64
-32.71172558951898,0.2687914613010722,-0.0003338662344074674,2.9998198532587614,4.734469140259755,4627.647389940279,0.17635035707702693,-0.04680613446240793,0.02591961874133764,9.628702138899065,0.008336529894478294,0.01410439449009211,0.011193439544849294,20229.699780461706,90850.27050446541,7255.749482062669,39035.92886860692,5403.906040441049,24885.18279134951,7570.044257957985,26929.158844508995,0,3703694511459476096,G2,191.91984692884603,2.958974323225439,2305843036808031251,29.372038,49.481987,57.37849,True,-0.0001109838975971392,STAR,26316,3703694511459476096
10.079396246219963,2.1616987590975203,-0.028796145290791723,3.0055993573916444,4.44438171777907,4029.7913274743314,0.39974623833869655,-0.4872189473559361,0.15563315146867857,39.04432209983848,0.06800872271753226,0.062339219342431834,0.010710255473422432,7487.751181354748,8624.936263234795,2493.281998933804,2734.1102651350297,2212.6661908260135,2740.6649465070846,2781.80299159493,3150.16105159268,0,3703718082239966720,G2,191.5941173994278,3.2768246220219046,2305843036808030356,2.227575,7.851011,11.380466,True,3.5312728352126336e-05,STAR,26316,3703718082239966720
-81.98411075492638,2.7220108058556916,-0.06255338292004022,3.015094790430767,2.457434346498593,5107.862748585958,0.5975244683211958,-1.4736364792203824,0.4103788760794281,120.80521069326002,0.12814572638440966,0.1591399470695639,0.01,7523.141418230767,8165.421727040975,2506.230938271229,2782.50492526796,2269.168242409689,2375.6183643425848,2747.7422375498486,3007.29843743043,0,3703717699986333696,G2,191.61756539723183,3.2354269055702862,2305843036808029406,6.605752,10.676428,11.549229,True,-0.00027240674315139415,STAR,26316,3703717699986333696
56.243202057576475,0.5112977629983783,5.4086423671008666e-05,3.000359239980067,4.46081956514648,5803.398663561478,0.14227210169308507,-0.26294992070720324,0.045535453491225016,19.392605220817472,0.016042638646251144,0.018831849481055214,0.01,9975.615421607363,35587.808800421895,3722.707486003119,21628.40026192342,2748.278888476968,5519.311719639408,3504.6290471272755,8440.096818859069,0,3703712859559758080,G2,191.6692843248912,3.147379940432535,2305843036808029531,33.904686,41.45769,42.795692,True,0.00018698249383295972,STAR,26316,3703712859559758080
-5.585199649873974,0.15776287108292505,-0.0003314918500343109,3.000026198419408,4.9589746931300205,4218.767917990153,0.08872739221094192,-0.3764568453849487,0.016997673123271915,5.6120010016793405,0.005345381242912866,0.004131010502235957,0.01,42795.764965597504,293505.6311885152,22418.430266269246,121834.39516219264,11454.464212718585,103323.09276823129,8922.870486609674,68348.1432580913,0,3703716948368598912,G2,191.63835683782807,3.2227737168898147,2305843036808031127,56.192535,89.67358,113.05488,True,-2.0060078715679778e-05,STAR,26316,3703716948368598912
31.61187393786791,2.379953977366006,0.010409165736564651,3.007752191388748,3.764281377082712,6006.769001508566,0.316695279103562,-1.0981434162863275,0.2219688691724795,68.0882469460221,0.08503023750431124,0.08602929617486406,0.01,7460.84737269686,9193.58765894861,2640.4403935748373,3922.2239192453053,2198.5404557254496,2396.8698754036377,2621.866523396572,2874.4938642996685,0,3703713306236357632,G2,191.70951061945794,3.15899366661375,2305843036808029503,12.17278,15.292509,15.174087,True,9.686380811520378e-05,STAR,26316,3703713306236357632
7.456794299342765,0.17510189954218247,-0.0007187193276606401,3.0000176159092757,4.644199

In [183]:
kw = dict(histtype='step', range=(-1,3), bins=100)

for b in 'BRZ':
    plt.hist(np.log10(t_desi_dr1[f'SN_{b}']), label=b, **kw)

plt.hist(np.log10(np.sqrt(t_desi_dr1['SN_B']**2+t_desi_dr1['SN_R']**2+t_desi_dr1['SN_Z']**2)),
         label='total', **kw)

plt.legend()
plt.show()

/tmp/ipykernel_2429448/582696337.py:4: RuntimeWarning: divide by zero encountered in log10
  plt.hist(np.log10(t_desi_dr1[f'SN_{b}']), label=b, **kw)
/tmp/ipykernel_2429448/582696337.py:4: RuntimeWarning: invalid value encountered in log10
  plt.hist(np.log10(t_desi_dr1[f'SN_{b}']), label=b, **kw)


<Figure size 640x480 with 1 Axes>

In [184]:
plt.hist(np.log10(t_desi_dr1['CHISQ_TOT']), bins=100)
plt.xlabel(r'$\log_{10}\left(\chi^2\right)$')
plt.show()

<Figure size 640x480 with 1 Axes>

In [185]:
# snr_tot = np.sqrt(t_desi_dr1['SN_B']**2+t_desi_dr1['SN_R']**2+t_desi_dr1['SN_Z']**2)

# plt.hist2d(
#     np.log10(snr_tot),
#     np.log10(t_desi_dr1['CHISQ_TOT']),
#     bins=100,
#     range=[(0.2,2.8),(3.5,6.2)],
#     norm=LogNorm()
# )

# snr_plot = 10**np.linspace(0.2, 2.8, 100)
# chisq_of_snr = lambda snr: 10**3.9 * (1 + 7e-4*snr**2)
# plt.plot(np.log10(snr_plot), np.log10(2*chisq_of_snr(snr_plot)), c='cyan')

# plt.show()

In [186]:
desi_dr1_snr_tot = np.sqrt(
    t_desi_dr1['SN_B']**2
  + t_desi_dr1['SN_R']**2
  + t_desi_dr1['SN_Z']**2
)

desi_dr1_chisq_of_snr = determine_snr_chisq_relation(
    desi_dr1_snr_tot,
    t_desi_dr1['CHISQ_TOT']
)
# ax = plt.gca()
# xlim = ax.get_xlim()
# snr_plot = np.logspace(np.log10(xlim[0]), np.log10(xlim[1]), 128)
# ax.plot(snr_plot, 
plt.show()

<Figure size 500x400 with 1 Axes>

In [187]:
snr_tot = np.sqrt(t_desi_dr1['SN_B']**2+t_desi_dr1['SN_R']**2+t_desi_dr1['SN_Z']**2)
# chisq_of_snr = lambda snr: 10**3.9 * (1 + 7e-4*snr**2)

idx = (
    # t_desi['PRIMARY']
    (t_desi_dr1['RVS_WARN']==0)
  & (t_desi_dr1['RR_SPECTYPE']=='STAR')
  & (snr_tot > 20.)
  & (t_desi_dr1['LOGG'] < 6) & (t_desi_dr1['LOGG'] > -0.2)
  & (t_desi_dr1['ALPHAFE'] >-0.35) & (t_desi_dr1['ALPHAFE'] < 1.15)
  # & (t_desi_dr1['CHISQ_TOT'] < 1e5)
  & (t_desi_dr1['CHISQ_TOT'] < 2*desi_dr1_chisq_of_snr(2*snr_tot))
  # & (t_desi_dr1['CHISQ_TOT'] < 2*chisq_of_snr(snr_tot))
  & (t_desi_dr1['TEFF_ERR'] < 500.)
  & (t_desi_dr1['LOGG_ERR'] < 0.5)
  & (t_desi_dr1['FEH_ERR'] < 0.5)
  & (t_desi_dr1['ALPHAFE_ERR'] < 0.5)
)
print(f'DESI DR1: {np.count_nonzero(idx):,} ({100*np.mean(idx):.2f}%) good sources.')

fig,ax_arr = plt.subplots(1,2, figsize=(8,4), layout='tight')

hist2d_teff_logg(
    t_desi_dr1['TEFF'][idx],
    t_desi_dr1['LOGG'][idx],
    ax=ax_arr[0]
)

ax = ax_arr[1]
ax.hist2d(t_desi_dr1['FEH'][idx], t_desi_dr1['ALPHAFE'][idx],
          bins=100, norm=LogNorm())
ax.set_xlabel(r'$\left[\mathrm{Fe/H}\right]$')
ax.set_ylabel(r'$\left[\alpha/\mathrm{Fe}\right]$')
ax.grid(True, alpha=0.2)

plt.show()

DESI DR1: 3,221,987 (57.07%) good sources.


<Figure size 800x400 with 2 Axes>

In [188]:
t0_desi, g0_desi = 6000., 5.
t1_desi, g1_desi = 5000., 3.

teff_edge = lambda logg: t0_desi + (logg-g0_desi) * (t1_desi-t0_desi)/(g1_desi-g0_desi)

desi_err = teff_edge(t_desi_dr1['LOGG']) - t_desi_dr1['TEFF']
desi_err = 0.1 + 0.0004 * np.clip(desi_err, 0, 1000)

# plt.scatter(
#     t_desi_dr1['TEFF'][::100],
#     t_desi_dr1['LOGG'][::100],
#     c=desi_err[::100]
# )
fig,ax = plt.subplots(1,1, figsize=(5,4))
im = plot_utils.hist2d_reduce(
    t_desi_dr1['TEFF'],
    t_desi_dr1['LOGG'],
    desi_err,
    ax=ax,
    hist_kw=dict(bins=50),
    xlim=(2.0e3,8e3),
    ylim=(-0.7,6.8),
    imshow_kw={}
)
ax.invert_xaxis()
ax.invert_yaxis()
ax.set_xlabel(r'$T_{\mathrm{eff}}$')
ax.set_ylabel(r'$\log g$')
fig.colorbar(im, ax=ax, label='penalty error')
plt.show()

fig,ax = plt.subplots(1,1, figsize=(5,4))
ax.hist2d(t_desi_dr1['TEFF'][idx], t_desi_dr1['LOGG'][idx],
          range=[(2.0e3,8e3),(-0.7,6.8)],
          bins=100, norm=LogNorm())
# ax.set_xlim(2.0e3,8e3)
# ax.set_ylim(-0.7,6.8)
ax.invert_xaxis()
ax.invert_yaxis()
ax.set_xlabel(r'$T_{\mathrm{eff}}$')
ax.set_ylabel(r'$\log g$')
fig.colorbar(im, ax=ax, label='density')
plt.show()

fig,ax = plt.subplots(1,1, figsize=(5,4))
im = ax.scatter(
    t_desi_dr1['TEFF'][idx][::10],
    t_desi_dr1['LOGG'][idx][::10],
    c=desi_err[idx][::10],
    edgecolors='none',
    s=4,
    alpha=0.01,
    rasterized=True
)
ax.set_xlim(2.0e3,8e3)
ax.set_ylim(-0.7,6.8)
ax.invert_xaxis()
ax.invert_yaxis()
ax.set_xlabel(r'$T_{\mathrm{eff}}$')
ax.set_ylabel(r'$\log g$')
fig.colorbar(im, ax=ax, label='penalty error')
plt.show()

<Figure size 500x400 with 2 Axes>

/tmp/ipykernel_2429448/3817378263.py:42: UserWarning: Adding colorbar to a different Figure <Figure size 500x400 with 2 Axes> than <Figure size 500x400 with 2 Axes> which fig.colorbar is called on.
  fig.colorbar(im, ax=ax, label='density')


<Figure size 500x400 with 2 Axes>

<Figure size 500x400 with 2 Axes>

In [189]:
kw = dict(range=(0.,0.50), bins=50, histtype='step', alpha=0.7)
plt.hist(0.001*t_desi_dr1['TEFF_ERR'][idx], label='T_eff/1000', **kw)
plt.hist(t_desi_dr1['FEH_ERR'][idx], label='[Fe/H]', **kw)
plt.hist(t_desi_dr1['LOGG_ERR'][idx], label='log(g)', **kw)
plt.legend()
plt.xlabel('Uncertainties')
plt.show()

<Figure size 640x480 with 1 Axes>

In [190]:
id_col = 'gdr3_source_id'
param_cols = ('TEFF', 'FEH', 'LOGG', 'ALPHAFE')
param_unc_cols = ('TEFF_ERR', 'FEH_ERR', 'LOGG_ERR', 'ALPHAFE_ERR')
extra_cols = ('TARGET_RA', 'TARGET_DEC')

# Increasing errors for T_eff > 6000 K
teff_errs = np.clip(
    0.1*(t_desi_dr1['TEFF']-6000.)/1000.,
    0.0, np.inf
)
desi_errs = teff_edge(t_desi_dr1['LOGG']) - t_desi_dr1['TEFF']
desi_errs = 0.0004 * np.clip(desi_errs, 0, 1000)
teff_errs = 0.1 + teff_errs + desi_errs
param_errs = np.stack([
    teff_errs,
    np.full_like(teff_errs, 0.1),
    desi_errs+0.1,
    np.full_like(teff_errs, 0.4)
], axis=1)

t_desi = extract_table(
    t_desi_dr1[idx],
    id_col, param_cols, param_unc_cols, extra_cols,
    param_transf=param_transf,
    param_unc_transf=apply_err_floors(param_unc_transf,err_floor_default),
    param_errs=param_errs[idx]
)
t_desi

id,x,sigma_x,x_err,TARGET_RA,TARGET_DEC
int64,float64[4],float64[4],float64[4],float64,float64
3703694511459476096,-0.38694648298654444 .. 0.17635035707702693,0.014430240050174643 .. 0.05069021336196496,0.5 .. 0.4,191.91984692884603,2.958974323225439
3703712859559758080,0.7450290498860708 .. 0.14227210169308507,0.01947193749531022 .. 0.052510629921323465,0.1 .. 0.4,191.6692843248912,3.147379940432535
3703716948368598912,-0.8494739478856066 .. 0.08872739221094192,0.01200994961439346 .. 0.050284919216720285,0.5 .. 0.4,191.63835683782807,3.2227737168898147
3703713306236357632,0.9172454390498781 .. 0.316695279103562,0.05755170780343035 .. 0.09864147854751357,0.10067690015085665 .. 0.4,191.70951061945794,3.15899366661375
3703712790840280704,0.6881839941915917 .. 0.2186716146319549,0.011506628941777588 .. 0.05022512013282709,0.13372253625030245 .. 0.4,191.68579840386028,3.1344392636392033
3703713546754526464,0.8244901229742087 .. 0.25784614295840763,0.04967416138622682 .. 0.07223587796431236,0.1 .. 0.4,191.6694248323297,3.1859691689889975
3703672589946297856,0.7988221094006399 .. 0.26386539801356856,0.026547867896137316 .. 0.05526204994617286,0.1 .. 0.4,191.12159467451244,2.7459704800796643
3703669360130892032,-1.194422220687487 .. 0.19061195476200382,0.015018141893266812 .. 0.052705480577348286,0.5 .. 0.4,191.19335067209482,2.731436940494683
3703727943484814976,-1.5161106422536432 .. 0.013965599170156072,0.02443173350658029 .. 0.05394366881116757,0.5 .. 0.4,191.11626974629579,2.9300877367724407


In [191]:
plot_catalog_uncertainties(t_desi)
plot_catalog_params(t_desi)

<Figure size 1050x280 with 5 Axes>

<Figure size 500x300 with 1 Axes>

<Figure size 500x400 with 1 Axes>

In [192]:
# Save catalog
fn = 'training_data/desi_dr1.h5'
t_desi.write(fn, serialize_metadata=True, overwrite=True)
! du -sh {fn}

/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


368M	training_data/desi_dr1.h5


### LAMOST Medium Resolution Survey (MRS)

Reference: [Wang *et al* (2023)](https://ui.adsabs.harvard.edu/abs/2023ApJS..266...40W/abstract)

In [ ]:
url = 'https://cdsarc.cds.unistra.fr/viz-bin/nph-Cat/fits.gz?J/ApJS/266/40/catalog.dat.gz'
fname = os.path.join(data_dir, 'lamost_mrs_cyclestarnet.h5')

! wget -O {fname} {url}

In [193]:
fn = os.path.join(data_dir, 'lamost_mrs_cyclestarnet.h5')
t_lamost_mrs = Table.read(fn)
print(t_lamost_mrs.colnames)

['StarID', 'medID', 'Gaia_source_id', 'RA', 'Dec', 'SN_blue', 'SN_red', 'Teff', 'logg', 'FeH', 'vmic', 'vmac', 'CFe', 'NaFe', 'MgFe', 'SiFe', 'CaFe', 'TiFe', 'CrFe', 'MnFe', 'CoFe', 'NiFe', 'CuFe', 'Err_Teff', 'Err_logg', 'Err_FeH', 'Err_vmic', 'Err_vmac', 'Err_CFe', 'Err_NaFe', 'Err_MgFe', 'Err_SiFe', 'Err_CaFe', 'Err_TiFe', 'Err_CrFe', 'Err_MnFe', 'Err_CoFe', 'Err_NiFe', 'Err_CuFe', 'Flag_Teff', 'Flag_logg', 'Flag_FeH', 'Flag_vmic', 'Flag_vmac', 'Flag_CFe', 'Flag_NaFe', 'Flag_MgFe', 'Flag_SiFe', 'Flag_CaFe', 'Flag_TiFe', 'Flag_CrFe', 'Flag_MnFe', 'Flag_CoFe', 'Flag_NiFe', 'Flag_CuFe', 'Flag_quality']


In [203]:
print(f'{len(t_lamost_mrs):,} sources in catalog.')

1,383,749 sources in catalog.


In [194]:
a_Fe,a_Fe_err,abundances = calc_alpha_Fe_from_abundances(
    t_lamost_mrs,
    '{x}Fe',
    'Err_{x}Fe',
    flag_fmt='Flag_{x}Fe'
)

t_lamost_mrs['alphaFe'] = a_Fe
t_lamost_mrs['Err_alphaFe'] = a_Fe_err

print(f'Used the abundances of {abundances.colnames}')

12.806% of [Mg/Fe] measurements flagged.
12.334% of [Si/Fe] measurements flagged.
18.619% of [Ca/Fe] measurements flagged.
15.746% of [Ti/Fe] measurements flagged.
Used the abundances of ['Mg', 'Si', 'Ca', 'Ti']


In [195]:
idx = (
    (t_lamost_mrs['Flag_Teff'] == 0)
  & (t_lamost_mrs['Flag_logg'] == 0)
  & (t_lamost_mrs['Flag_FeH'] == 0)
  & (t_lamost_mrs['Err_alphaFe'] < 10.)
  & (t_lamost_mrs['Flag_quality'] == 0)
)
# t_lamost_mrs = t_lamost_mrs[idx]
print(f'LAMOST MRS: {np.count_nonzero(idx)} ({100*np.mean(idx):.2f}%) good sources.')

plot_teff_logg_feh_abundance(
    t_lamost_mrs['Teff'][idx],
    t_lamost_mrs['logg'][idx],
    t_lamost_mrs['FeH'][idx],
    t_lamost_mrs['alphaFe'][idx],
    # 0.5*(t_lamost_mrs['MgFe'][idx]+t_lamost_mrs['SiFe'][idx]),
    # abundance_label=r'$\frac{1}{2}\left(\left[\mathrm{Mg/Fe}\right] + \left[\mathrm{Si/Fe}\right]\right)$'
)
plt.show()

LAMOST MRS: 854801 (61.77%) good sources.


<Figure size 800x400 with 2 Axes>

In [196]:
plot_alpha_abundances(a_Fe[idx], abundances[idx])
plt.show()

<Figure size 500x500 with 4 Axes>

In [197]:
kw = dict(range=(0.,0.10), bins=50, histtype='step', alpha=0.7)
plt.hist(0.001*t_lamost_mrs['Err_Teff'][idx], label='T_eff/1000', **kw)
plt.hist(t_lamost_mrs['Err_FeH'][idx], label='[Fe/H]', **kw)
plt.hist(t_lamost_mrs['Err_logg'][idx], label='log(g)', **kw)
plt.legend()
plt.xlabel('Uncertainties')
plt.show()

<Figure size 640x480 with 1 Axes>

In [198]:
id_col = 'Gaia_source_id'
param_cols = ('Teff', 'FeH', 'logg', 'alphaFe')
param_unc_cols = ('Err_Teff', 'Err_FeH', 'Err_logg', 'Err_alphaFe')
extra_cols = ('RA', 'Dec')

# Increasing errors for T_eff > 6000 K
teff_errs = np.clip(
    0.1 + 0.4*(t_lamost_mrs['Teff']-6000.)/1000.,
    0.1, np.inf
)
param_errs = np.stack([
    teff_errs,
    np.full_like(teff_errs, 0.1),
    np.full_like(teff_errs, 0.1),
    np.full_like(teff_errs, 0.1)
], axis=1)

t_lmrs = extract_table(
    t_lamost_mrs[idx],
    id_col, param_cols, param_unc_cols, extra_cols,
    param_transf=param_transf,
    param_unc_transf=apply_err_floors(param_unc_transf,err_floor_default),
    param_errs=param_errs[idx]
)
t_lmrs

id,x,sigma_x,x_err,RA,Dec
int64,float64[4],float64[4],float64[4],float64,float64
2007385426887291520,-0.6313093983876193 .. 0.034460692037579725,0.03999356742567552 .. 0.05358740970063989,0.1 .. 0.1,343.13247,58.13707
2007178031509195776,-1.1518857541305179 .. 0.08693337562320538,0.0327392943679104 .. 0.05269396822329207,0.1 .. 0.1,343.0824,57.829343
2007179710828415232,-0.052060087070042116 .. 0.056551899568283465,0.02958946444794321 .. 0.05298337358940787,0.1 .. 0.1,343.15237,57.874716
2007180367971768192,1.077951485878431 .. 0.060805479306552536,0.024090483633637586 .. 0.05597802455688231,0.18118800000000013 .. 0.1,343.34882,57.947485
2007438409604543104,0.2648025191061075 .. -0.006379212898001088,0.028198692446133686 .. 0.05303743383547645,0.1 .. 0.1,342.03345,58.342943
2007452424071586560,0.10438280113820843 .. 0.011090318032353077,0.03406394308863635 .. 0.054056783405297924,0.1 .. 0.1,342.36096,58.45949
2007426314976716160,-0.02145597008620701 .. -0.03190343986548326,0.05189071093954138 .. 0.056135387617204414,0.1 .. 0.1,342.06166,58.288553
2007462461422035456,-0.07669521961378611 .. -0.0392316789924385,0.042077703509750465 .. 0.054813105755907345,0.1 .. 0.1,342.17886,58.4395
2007441295822380544,1.0954366799213884 .. -0.053415677912071795,0.03418755624874057 .. 0.055651201374236466,0.18987999999999994 .. 0.1,341.68264,58.354774


In [199]:
plot_catalog_uncertainties(t_lmrs)
plot_catalog_params(t_lmrs)

<Figure size 1050x280 with 5 Axes>

<Figure size 500x300 with 1 Axes>

<Figure size 500x400 with 1 Axes>

In [200]:
# Save catalog
fn = 'training_data/lamost_mrs.h5'
t_lmrs.write(fn, serialize_metadata=True, overwrite=True)
! du -sh {fn}

/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


98M	training_data/lamost_mrs.h5


### APOGEE (DR17)

ASPCAP reference: [García Pérez *et al* (2016)](https://ui.adsabs.harvard.edu/abs/2016AJ....151..144G/abstract)

SDSS DR17 reference: [Abdurro'uf *et al* (2022)](https://ui.adsabs.harvard.edu/abs/2022ApJS..259...35A/abstract)

In [ ]:
url_base = 'https://data.sdss.org/sas/dr17/apogee/spectro/aspcap/dr17/synspec_rev1'
fname = 'allStarLite-dr17-synspec_rev1.fits'

! wget -O {data_dir}/{fname} {url_base}/{fname}

In [204]:
fn = os.path.join(data_dir, 'allStarLite-dr17-synspec_rev1.fits')
t_apogee = Table.read(fn)
print(f'{len(t_apogee):,} sources in catalog.')
print(t_apogee.colnames)

733,901 sources in catalog.
['APOGEE_ID', 'TELESCOPE', 'FIELD', 'ALT_ID', 'RA', 'DEC', 'GLON', 'GLAT', 'J', 'J_ERR', 'H', 'H_ERR', 'K', 'K_ERR', 'AK_TARG', 'AK_TARG_METHOD', 'AK_WISE', 'SFD_EBV', 'APOGEE_TARGET1', 'APOGEE_TARGET2', 'APOGEE2_TARGET1', 'APOGEE2_TARGET2', 'APOGEE2_TARGET3', 'APOGEE2_TARGET4', 'TARGFLAGS', 'SURVEY', 'PROGRAMNAME', 'NVISITS', 'SNR', 'SNREV', 'STARFLAG', 'STARFLAGS', 'ANDFLAG', 'ANDFLAGS', 'VHELIO_AVG', 'VSCATTER', 'VERR', 'RV_TEFF', 'RV_LOGG', 'RV_FEH', 'RV_ALPHA', 'RV_CARB', 'RV_CHI2', 'RV_CCFWHM', 'RV_AUTOFWHM', 'RV_FLAG', 'N_COMPONENTS', 'MEANFIB', 'SIGFIB', 'MIN_H', 'MAX_H', 'MIN_JK', 'MAX_JK', 'GAIAEDR3_SOURCE_ID', 'GAIAEDR3_PARALLAX', 'GAIAEDR3_PARALLAX_ERROR', 'GAIAEDR3_PMRA', 'GAIAEDR3_PMRA_ERROR', 'GAIAEDR3_PMDEC', 'GAIAEDR3_PMDEC_ERROR', 'GAIAEDR3_PHOT_G_MEAN_MAG', 'GAIAEDR3_PHOT_BP_MEAN_MAG', 'GAIAEDR3_PHOT_RP_MEAN_MAG', 'GAIAEDR3_DR2_RADIAL_VELOCITY', 'GAIAEDR3_DR2_RADIAL_VELOCITY_ERROR', 'GAIAEDR3_R_MED_GEO', 'GAIAEDR3_R_LO_GEO', 'GAIAEDR3_R_HI

In [205]:
[s for s in t_apogee.colnames if 'SNR' in s]

['SNR', 'SNREV', 'FRAC_LOWSNR']

In [206]:
apogee_chisq_of_snr = determine_snr_chisq_relation(
    t_apogee['SNR'],
    t_apogee['ASPCAP_CHI2']
)
plt.show()

<Figure size 500x400 with 1 Axes>

In [207]:
idx = (
    (t_apogee['ASPCAPFLAG'] & 2**7 == 0) # STAR_WARN
  # & (t_apogee['ASPCAP_CHI2'] < 8*apogee_chisq_of_snr(8*t_apogee['SNR']))
  & np.isfinite(t_apogee['TEFF'])
  & np.isfinite(t_apogee['LOGG'])
  & np.isfinite(t_apogee['M_H'])
  & np.isfinite(t_apogee['ALPHA_M'])
  & np.isfinite(t_apogee['TEFF_ERR'])
  & np.isfinite(t_apogee['LOGG_ERR'])
  & np.isfinite(t_apogee['M_H_ERR'])
  & np.isfinite(t_apogee['ALPHA_M_ERR'])
  & (t_apogee['TEFF_ERR'] < 500.)
  & (t_apogee['LOGG_ERR'] < 0.5)
  & (t_apogee['M_H_ERR'] < 0.5)
  & (t_apogee['ALPHA_M_ERR'] < 0.5)
  & (t_apogee['ALPHA_M'] < 0.7)
  & (t_apogee['ALPHA_M'] > -0.4)
)
# t_apogee = t_apogee[idx]
print(fr'{np.mean(idx):.2%} ({np.count_nonzero(idx):,}) of APOGEE sources pass cuts.')

89.10% (576,103) of APOGEE sources pass cuts.


In [208]:
kw = dict(range=(-3,1), bins=50, histtype='step', alpha=0.7, log=True)
plt.hist(np.log10(0.001*t_apogee['TEFF_ERR'][idx]), label='T_eff/1000', **kw)
plt.hist(np.log10(t_apogee['M_H_ERR'][idx]), label='[Fe/H]', **kw)
plt.hist(np.log10(t_apogee['LOGG_ERR'][idx]), label='log(g)', **kw)
plt.hist(np.log10(t_apogee['ALPHA_M_ERR'][idx]), label=r'$\left[\alpha/\mathrm{M}\right]$', **kw)
plt.legend()
plt.xlabel('log_10(Uncertainties)')
plt.show()

<Figure size 640x480 with 1 Axes>

In [209]:
plot_teff_logg_feh(
    t_apogee['TEFF'][idx],
    t_apogee['LOGG'][idx],
    t_apogee['M_H'][idx],
    label_as_mh=True
)
plt.show()

plot_teff_logg_feh_abundance(
    t_apogee['TEFF'][idx],
    t_apogee['LOGG'][idx],
    t_apogee['M_H'][idx],
    t_apogee['ALPHA_M'][idx],
    label_as_mh=True,
    abundance_label=r'$\left[\alpha/\mathrm{M}\right]$'
)
plt.show()

<Figure size 800x400 with 2 Axes>

<Figure size 800x400 with 2 Axes>

In [210]:
a_Fe,a_Fe_err,abundances = calc_alpha_Fe_from_abundances(
    t_apogee,
    '{x.upper()}_FE',
    '{x.upper()}_FE_ERR',
    flag_fmt='{x.upper()}_FE_FLAG'
)

8.188% of [O/Fe] measurements flagged.
8.617% of [Mg/Fe] measurements flagged.
8.062% of [Si/Fe] measurements flagged.
20.841% of [S/Fe] measurements flagged.
10.792% of [Ca/Fe] measurements flagged.
18.189% of [Ti/Fe] measurements flagged.


In [211]:
plt.hist2d(
    t_apogee['ALPHA_M'],
    a_Fe,
    range=[(-0.7,0.7),(-0.7,0.7)],
    bins=100,
    norm=LogNorm()
)
plt.axline((0,0), slope=1, c='gray', alpha=0.5)
plt.xlabel(r'$\texttt{ALPHA\_M}$')
plt.ylabel(r'$\left[\alpha/\mathrm{Fe}\right]$')
plt.gca().set_aspect('equal')
plt.show()

<Figure size 640x480 with 1 Axes>

In [212]:
plot_alpha_abundances(a_Fe, abundances)
plt.show()

<Figure size 750x500 with 6 Axes>

In [214]:
id_col = 'GAIAEDR3_SOURCE_ID'
param_cols = ('TEFF', 'M_H', 'LOGG', 'ALPHA_M')
param_unc_cols = ('TEFF_ERR', 'M_H_ERR', 'LOGG_ERR', 'ALPHA_M_ERR')
extra_cols = ('RA','DEC')

# Increasing errors for T_eff > 6000 K
teff_errs = np.clip(
    0.1 + 0.4*(t_apogee['TEFF'].data.filled()-6000.)/1000.,
    0.1, np.inf
)
param_errs = np.stack([
    teff_errs,
    np.full_like(teff_errs, 0.1),
    np.full_like(teff_errs, 0.1),
    np.full_like(teff_errs, 0.05)
], axis=1)

t_apo = extract_table(
    t_apogee[idx],
    id_col, param_cols, param_unc_cols, extra_cols,
    param_transf=param_transf,
    param_unc_transf=apply_err_floors(param_unc_transf,err_floor_default),
    param_errs=param_errs[idx]
)
t_apo

id,x,sigma_x,x_err,RA,DEC
int64,float32[4],float32[4],float32[4],float64,float64
538028216707715712,-1.4733173 .. 0.036922,0.011389498 .. 0.050223675,0.1 .. 0.05,0.000103,74.285408
2413929812587459072,0.4781623 .. 0.0909783,0.02528456 .. 0.05054066,0.1 .. 0.05,0.000832,-19.413851
422596679964513792,0.9940746 .. 0.0291593,0.03888883 .. 0.051363315,0.1399123 .. 0.05,0.001335,57.61953
422596679964513792,1.0448419 .. 0.0480663,0.0353249 .. 0.050962072,0.16481212 .. 0.05,0.001335,57.61953
4689447878791422208,-0.46561825 .. 0.008442001,0.025366064 .. 0.05631456,0.1 .. 0.05,0.001467,-73.394287
421077597267551104,0.031166488 .. 0.055704,0.014200249 .. 0.050366227,0.1 .. 0.05,0.00285,57.173164
2772097619417608704,0.8661947 .. 0.0124087995,0.038529098 .. 0.0514968,0.1 .. 0.05,0.004322,15.430942
421086363305436800,-0.02546376 .. 0.10715,0.018352715 .. 0.05112278,0.1 .. 0.05,0.005558,57.354549
431594980053422720,-0.32913595 .. 0.0076940004,0.013698991 .. 0.05036069,0.1 .. 0.05,0.008802,63.463078


In [215]:
plot_catalog_uncertainties(t_apo)
plot_catalog_params(t_apo)

<Figure size 1050x280 with 5 Axes>

<Figure size 500x300 with 1 Axes>

<Figure size 500x400 with 1 Axes>

In [216]:
# Save catalog
fn = 'training_data/apogee_dr17.h5'
t_apo.write(fn, serialize_metadata=True, overwrite=True)
! du -sh {fn}

/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


40M	training_data/apogee_dr17.h5


### LAMOST HotPayne

Reference: [Xiang, MS *et al* (2022)](https://ui.adsabs.harvard.edu/abs/2022A%26A...662A..66X)

In [ ]:
url = 'https://cdsarc.cds.unistra.fr/viz-bin/nph-Cat/fits.gz?J/A+A/662/A66/table5.dat.gz'
fname = os.path.join(data_dir, 'lamost_hot_payne.fits.gz')

! wget -O {fname} {url}

In [332]:
fn = os.path.join(data_dir, 'lamost_hot_payne.fits.gz')
t_lamost_hotpayne = Table.read(fn)
print(t_lamost_hotpayne.colnames)

['SpecID', 'RAdeg', 'DEdeg', 'snrg', 'Teff', 'e_Teff', 'logg', 'e_logg', '[Fe/H]', 'e_[Fe/H]', 'vsini', 'e_vsini', '[Si/H]', 'e_[Si/H]', 'corTeff-logg', 'corTeff-[Fe/H]', 'corlogg-[Fe/H]', 'corTeff-vsini', 'corlogg-vsini', 'corvsini-[Fe/H]', 'corTeff-[Si/H]', 'corlogg-[Si/H]', 'cor[Si/H]-[Fe/H]', 'corvsini-[Si/H]', 'chisqred', 'RV', 'e_RV', 'chi2ratio', 'uniqflag', 'StarID', 'sourceDR6oba', 'sourceZari21', 'sourceLiu19', 'GaiaEDR3', 'Plx', 'e_Plx', 'pmRA', 'e_pmRA', 'pmDE', 'e_pmDE', 'RUWE', 'rgeo', 'e_rgeo', 'E_rgeo', 'Fidelity', 'E(B-V)bayes19', 'E(B-V)pair', 'e_E(B-V)pair', 'Gmag', 'e_Gmag', 'BPmag', 'e_BPmag', 'RPmag', 'e_RPmag', 'Jmag', 'e_Jmag', 'Hmag', 'e_Hmag', 'Kmag', 'e_Kmag', 'Q-2MASS', 'W1mag', 'e_W1mag', 'W2mag', 'e_W2mag', 'W3mag', 'e_W3mag', 'W4mag', 'e_W4mag', 'Q-WISE', 'VAR-WISE']


In [333]:
hotpayne_chisq_of_snr = determine_snr_chisq_relation(
    t_lamost_hotpayne['snrg'],
    t_lamost_hotpayne['chisqred']
)
plt.show()

<Figure size 500x400 with 1 Axes>

In [334]:
plt.hist(t_lamost_hotpayne['chisqred'], bins=10**np.arange(-2.,5.,0.25))
plt.gca().set_xscale('log')
# plt.gca().set_yscale('log')
plt.xlabel('chisqred')
plt.show()

print(np.mean(t_lamost_hotpayne['chisqred'] > 5.))

<Figure size 640x480 with 1 Axes>

0.09853461566375554


In [335]:
plt.hist(t_lamost_hotpayne['snrg'], bins=10**np.arange(-2.,5.,0.25))
plt.gca().set_xscale('log')
# plt.gca().set_yscale('log')
plt.xlabel('snrg')
plt.show()

print(np.mean(t_lamost_hotpayne['snrg'] < 20.))

<Figure size 640x480 with 1 Axes>

0.23631109341907616


In [336]:
# rchi2_trend = lambda snr: 10**-0.55 * (1. + (snr/60)**2)
# rchi2_max = lambda snr: 10*rchi2_trend(snr)

# rchi2_max = lambda snr: np.exp(0.02*np.log(snr)**3 + np.log(2))
idx = (
    # (t_lamost_hotpayne['chisqred'] < rchi2_max(t_lamost_hotpayne['snrg']))
    (
        t_lamost_hotpayne['chisqred']
        < 4*hotpayne_chisq_of_snr(4*t_lamost_hotpayne['snrg'])
    )
#     (t_lamost_hotpayne['chisqred'] < 100.)
  & (t_lamost_hotpayne['snrg'] > 16.)
  & np.isfinite(t_lamost_hotpayne['e_Teff'])
  & np.isfinite(t_lamost_hotpayne['e_logg'])
  & np.isfinite(t_lamost_hotpayne['e_[Fe/H]'])
  & (t_lamost_hotpayne['[Fe/H]'] > -3.95)
  # & (t_lamost_hotpayne['[Fe/H]'] > -8.)
)
print(np.mean(idx))

kw = dict(edgecolors='none', alpha=0.2, s=2)

fig,ax = plt.subplots(1,1, figsize=(5,4))

ax.scatter(
    t_lamost_hotpayne['snrg'][idx],
    t_lamost_hotpayne['chisqred'][idx],
    label='accept',
    **kw
)
ax.scatter(
    t_lamost_hotpayne['snrg'][~idx],
    t_lamost_hotpayne['chisqred'][~idx],
    label='reject',
    **kw
)

snr_plot = np.logspace(np.log10(5.), 3., 100, base=10)
# ax.plot(snr_plot, rchi2_trend(snr_plot), c='k', alpha=0.5)
ax.plot(snr_plot, hotpayne_chisq_of_snr(snr_plot), c='k', alpha=0.5)

ax.set_xscale('log')
ax.set_yscale('log')

ax.set_ylim(bottom=1e-2, top=1e5)

# xlim = plt.gca().get_xlim()
# x_plot = np.exp(np.linspace(np.log(xlim[0]), np.log(xlim[1]), 100))
# plt.plot(x_plot, np.exp(0.02*np.log(x_plot)**3 + np.log(3)), alpha=0.7)

ax.set_xlabel('snrg')
ax.set_ylabel('chisqred')

ax.legend(loc='upper left')

plt.show()

plt.hist(
    np.log10(t_lamost_hotpayne['chisqred'][t_lamost_hotpayne['snrg']<6]),
    range=(-2,2),
    bins=100
)
plt.axvline(-0.55, c='k', alpha=0.6)

plt.show()

0.8041689667533919


<Figure size 500x400 with 1 Axes>

<Figure size 640x480 with 1 Axes>

In [337]:
a_Fe,a_Fe_err,abundances = calc_alpha_Fe_from_abundances(
    t_lamost_hotpayne,
    '[{x}/H]',
    'e_[{x}/H]'
)

t_lamost_hotpayne['[alpha/Fe]'] = a_Fe - t_lamost_hotpayne['[Fe/H]']
t_lamost_hotpayne['e_[alpha/Fe]'] = np.sqrt(a_Fe_err**2 + t_lamost_hotpayne['e_[Fe/H]']**2)

print(f'Used the abundances of {abundances.colnames}')

Used the abundances of ['Si']


In [338]:
plot_teff_logg_feh(
    t_lamost_hotpayne['Teff'][idx],
    t_lamost_hotpayne['logg'][idx],
    t_lamost_hotpayne['[Fe/H]'][idx]
)
plt.show()

<Figure size 800x400 with 2 Axes>

In [339]:
plot_teff_logg_feh_abundance(
    t_lamost_hotpayne['Teff'][idx],
    t_lamost_hotpayne['logg'][idx],
    t_lamost_hotpayne['[Fe/H]'][idx],
    t_lamost_hotpayne['[alpha/Fe]'][idx]
)
plt.show()

<Figure size 800x400 with 2 Axes>

In [340]:
kw = dict(edgecolors='none', alpha=0.2, s=2)
plt.scatter(
    t_lamost_hotpayne['Teff'][idx],
    t_lamost_hotpayne['logg'][idx],
    label='accept',
    **kw
)
plt.scatter(
    t_lamost_hotpayne['Teff'][~idx],
    t_lamost_hotpayne['logg'][~idx],
    label='reject',
    **kw
)
# plt.xlim()
plt.ylim(8., -1.)
plt.gca().invert_xaxis()
plt.gca().set_xscale('log')
plt.xlabel(r'$T_{\rm eff}$')
plt.ylabel(r'$\log g$')
# plt.gca().invert_yaxis()
plt.show()

<Figure size 640x480 with 1 Axes>

In [341]:
teff_bins = 10**np.linspace(3.8, 4.8, 100)
kw = dict(bins=teff_bins, histtype='step', log=True)
plt.hist(t_lamost_hotpayne['Teff'][idx], label='accept', **kw)
plt.hist(t_lamost_hotpayne['Teff'][~idx], label='reject', **kw)
plt.gca().set_xscale('log')
plt.legend()
plt.xlabel('Teff')
plt.show()

logg_bins = np.linspace(-2., 7.5, 100)
kw = dict(bins=logg_bins, histtype='step', log=True)
plt.hist(t_lamost_hotpayne['logg'][idx], label='accept', **kw)
plt.hist(t_lamost_hotpayne['logg'][~idx], label='reject', **kw)
plt.legend()
plt.xlabel('logg')
plt.show()

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

In [342]:
kw = dict(range=(0.,500.), bins=100, histtype='step')
plt.hist(t_lamost_hotpayne['e_Teff'][idx], label='accept', **kw)
plt.hist(t_lamost_hotpayne['e_Teff'][~idx], label='reject', **kw)
plt.gca().xaxis.set_minor_locator(AutoMinorLocator())
plt.gca().grid(True, which='major', alpha=0.2)
plt.gca().grid(True, which='minor', alpha=0.05)
plt.legend()
plt.xlabel('e_Teff')
plt.show()

<Figure size 640x480 with 1 Axes>

In [343]:
kw = dict(range=(-3,1), bins=50, histtype='step', alpha=0.7, log=True)
plt.hist(np.log10(0.001*t_lamost_hotpayne['e_Teff'][idx]), label='T_eff/1000', **kw)
plt.hist(np.log10(t_lamost_hotpayne['e_[Fe/H]'][idx]), label='[Fe/H]', **kw)
plt.hist(np.log10(t_lamost_hotpayne['e_logg'][idx]), label='log(g)', **kw)
plt.hist(np.log10(t_lamost_hotpayne['e_[alpha/Fe]'][idx]), label='[alpha/Fe]', **kw)
plt.legend()
plt.xlabel('log_10(Uncertainties)')
plt.show()

<Figure size 640x480 with 1 Axes>

In [344]:
err_floor_default

[0.01, 0.01, 0.04, 0.05]

In [345]:
id_col = 'GaiaEDR3'
param_cols = ('Teff', '[Fe/H]', 'logg', '[alpha/Fe]')
param_unc_cols = ('e_Teff', 'e_[Fe/H]', 'e_logg', '[alpha/Fe]')
extra_cols = ('RAdeg', 'DEdeg')

t_lhp = extract_table(
    t_lamost_hotpayne[idx],
    id_col, param_cols, param_unc_cols, extra_cols,
    param_transf=param_transf,
    param_unc_transf=apply_err_floors(param_unc_transf,err_floor_default),
    param_errs=(0.05,0.2,0.1,0.5)
)
t_lhp

id,x,sigma_x,x_err,RAdeg,DEdeg
,,,,deg,deg
int64,float64[4],float64[4],float32[4],float64,float64
393923478296659200,1.6829325790269198 .. -0.3724629999999999,0.04161980590999671 .. 0.3758040531566948,0.05 .. 0.5,0.008644,50.380315
420512590034739840,2.4710415897127644 .. -0.11557699999999996,0.04218749372489226 .. 0.12592872162060564,0.05 .. 0.5,0.011984,55.573492
422632272352729984,1.9173924882800586 .. -0.4713819999999999,0.022064575871937105 .. 0.4740263599463641,0.05 .. 0.5,0.012862,58.019821
387298370975979520,2.8319638610722326 .. -0.161858,0.10248574760948798 .. 0.16940487644693114,0.05 .. 0.5,0.014918,47.794382
395601470479535616,1.9381347498914225 .. -0.5457270000000002,0.02150910376587595 .. 0.5480127357361325,0.05 .. 0.5,0.018559,51.662715
420945793315071872,3.7851618165923497 .. 0.022260999999999975,0.12346819164150177 .. 0.05473163729507824,0.05 .. 0.5,0.019518,56.620705
423186464874496128,2.939211094572609 .. 0.511159,0.16380366740075855 .. 0.5135986013230566,0.05 .. 0.5,0.022484,59.352074
422764248104464768,3.8739584080016227 .. -0.1607329999999999,0.31510798168232584 .. 0.1683303219535921,0.05 .. 0.5,0.023678,58.594514


In [346]:
plot_catalog_uncertainties(t_lhp)
plot_catalog_params(t_lhp)

<Figure size 1050x280 with 5 Axes>

<Figure size 500x300 with 1 Axes>

<Figure size 500x400 with 1 Axes>

In [347]:
# Save catalog
fn = 'training_data/lamost_hotpayne.h5'
t_lhp.write(fn, serialize_metadata=True, overwrite=True)
! du -sh {fn}

/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


36M	training_data/lamost_hotpayne.h5


### GALAH (DR4)

Reference: [Buder *et al* (2024)](https://ui.adsabs.harvard.edu/abs/2025PASA...42...51B/abstract)

Documentation: https://www.galah-survey.org/dr4/overview/

In [1009]:
base_url = 'https://cloud.datacentral.org.au/teamdata/GALAH/public/GALAH_DR4/catalogs'
fname = 'galah_dr4_allstar_240705.fits'

! wget -O {data_dir}/{fname} {base_url}/{fname}

--2025-06-18 11:25:07--  https://cloud.datacentral.org.au/teamdata/GALAH/public/GALAH_DR4/catalogs/galah_dr4_allstar_240705.fits
Resolving cloud.datacentral.org.au (cloud.datacentral.org.au)... 192.231.167.120
Connecting to cloud.datacentral.org.au (cloud.datacentral.org.au)|192.231.167.120|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 757998720 (723M)
Saving to: ‘/data/beegfs/astro-storage/groups/green/green/projects/gaia_Bp_Rp_reddening_variation/data//galah_dr4_allstar_240705.fits’

/data/beegfs/astro- 100%[===================>] 722.88M  11.1MB/s    in 69s     

2025-06-18 11:26:18 (10.4 MB/s) - ‘/data/beegfs/astro-storage/groups/green/green/projects/gaia_Bp_Rp_reddening_variation/data//galah_dr4_allstar_240705.fits’ saved [757998720/757998720]



In [217]:
fn = os.path.join(data_dir, 'galah_dr4_allstar_240705.fits')
t_galah_dr4 = Table.read(fn)
print(f'{len(t_galah_dr4):,} sources in catalog.')
t_galah_dr4

917,588 sources in catalog.


sobject_id,tmass_id,gaiadr3_source_id,survey_name,field_id,setup,mjd,ra,dec,flag_sp,flag_sp_fit,opt_loop,flag_red,snr_px_ccd1,snr_px_ccd2,snr_px_ccd3,snr_px_ccd4,chi2_sp,px_used_perc,model_name,closest_model,comp_time,fit_global_rv,rv_comp_1,e_rv_comp_1,rv_comp_2,e_rv_comp_2,rv_gaia_dr3,e_rv_gaia_dr3,v_bary_eff,teff,e_teff,logg,e_logg,fe_h,e_fe_h,flag_fe_h,vmic,e_vmic,vsini,e_vsini,nn_li_fe,nn_e_li_fe,nn_flag_li_fe,c_fe,e_c_fe,flag_c_fe,n_fe,e_n_fe,flag_n_fe,o_fe,e_o_fe,flag_o_fe,na_fe,e_na_fe,flag_na_fe,mg_fe,e_mg_fe,flag_mg_fe,al_fe,e_al_fe,flag_al_fe,si_fe,e_si_fe,flag_si_fe,k_fe,e_k_fe,flag_k_fe,ca_fe,e_ca_fe,flag_ca_fe,sc_fe,e_sc_fe,flag_sc_fe,ti_fe,e_ti_fe,flag_ti_fe,v_fe,e_v_fe,flag_v_fe,cr_fe,e_cr_fe,flag_cr_fe,mn_fe,e_mn_fe,flag_mn_fe,co_fe,e_co_fe,flag_co_fe,ni_fe,e_ni_fe,flag_ni_fe,cu_fe,e_cu_fe,flag_cu_fe,zn_fe,e_zn_fe,flag_zn_fe,rb_fe,e_rb_fe,flag_rb_fe,sr_fe,e_sr_fe,flag_sr_fe,y_fe,e_y_fe,flag_y_fe,zr_fe,e_zr_fe,flag_zr_fe,mo_fe,e_mo_fe,flag_mo_fe,ru_fe,e_ru_fe,flag_ru_fe,ba_fe,e_ba_fe,flag_ba_fe,la_fe,e_la_fe,flag_la_fe,ce_fe,e_ce_fe,flag_ce_fe,nd_fe,e_nd_fe,flag_nd_fe,sm_fe,e_sm_fe,flag_sm_fe,eu_fe,e_eu_fe,flag_eu_fe,mass,age,bc_ks,a_ks,lbol,r_med,r_lo,r_hi,sb2_rv_16,sb2_rv_50,sb2_rv_84,ew_h_beta,ew_h_alpha,res_h_beta,res_h_alpha,ew_k_is,sigma_k_is,rv_k_is,ew_dib5780,sigma_dib5780,rv_dib5780,ew_dib5797,sigma_dib5797,rv_dib5797,ew_dib6613,sigma_dib6613,rv_dib6613,ebv,phot_g_mean_mag,bp_rp,j_m,j_msigcom,h_m,h_msigcom,ks_m,ks_msigcom,W2mag,e_W2mag,ruwe,parallax,parallax_error,ew_li,e_ew_li_low,e_ew_li_upp,a_li,a_li_upp_lim,e_a_li_low,e_a_li_upp,e_a_li_teff,flag_a_li
,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,mag,mag,mag,mag,mag,mag,mag,mag,mag,mag,,mas,mas,,,,,,,,,
int64,bytes17,int64,bytes17,int32,bytes7,float32,float64,float64,int32,int32,int32,int32,float32,float32,float32,float32,float32,int64,bytes16,bytes16,float32,bool,float32,float32,float32,float32,float32,float32,float64,float32,float32,float32,float32,float32,float32,int32,float32,float32,float32,float32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,int32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float64,float64,float32,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,int64
170910004101274,00000011+0522500,2745049530295263232,galah_faint,6882,allstar,58006.613,0.0005000000237487257,5.380555629730225,0,0,2,0,26.964375,45.697865,75.889725,72.90598,1.1613754,94,4500_4.50_-0.50,4500_4.50_-0.50,134.00137,True,10.483474,0.18974757,--,--,11.894795,1.3880302,6.854094982147217,4486.2554,68.00265,4.657445,0.10057206,-0.3963167,0.05511797,0,0.7832511,0.28494248,6.9111066,1.4482353,0.74539346,0.06287381,0,0.05938914,0.052370273,32,-0.3913465,0.13808812,32,-0.19825687,0.051651552,0,-0.16347441,0.018229783,0,0.3087402,0.014740429,0,-0.048628658,0.041410007,0,0.19373856,0.029462606,0,-0.03559644,0.02123725,0,0.12320362,0.013022852,0,0.01370742,0.0261386,0,0.3395407,0.015608791,0,0.32754487,0.021382978,0,0.1857635,0.013516717,0,-0.023918502,0.022693036,0,0.136851

In [218]:
[s for s in t_galah_dr4.colnames]# if 'fe' in s]

['sobject_id',
 'tmass_id',
 'gaiadr3_source_id',
 'survey_name',
 'field_id',
 'setup',
 'mjd',
 'ra',
 'dec',
 'flag_sp',
 'flag_sp_fit',
 'opt_loop',
 'flag_red',
 'snr_px_ccd1',
 'snr_px_ccd2',
 'snr_px_ccd3',
 'snr_px_ccd4',
 'chi2_sp',
 'px_used_perc',
 'model_name',
 'closest_model',
 'comp_time',
 'fit_global_rv',
 'rv_comp_1',
 'e_rv_comp_1',
 'rv_comp_2',
 'e_rv_comp_2',
 'rv_gaia_dr3',
 'e_rv_gaia_dr3',
 'v_bary_eff',
 'teff',
 'e_teff',
 'logg',
 'e_logg',
 'fe_h',
 'e_fe_h',
 'flag_fe_h',
 'vmic',
 'e_vmic',
 'vsini',
 'e_vsini',
 'nn_li_fe',
 'nn_e_li_fe',
 'nn_flag_li_fe',
 'c_fe',
 'e_c_fe',
 'flag_c_fe',
 'n_fe',
 'e_n_fe',
 'flag_n_fe',
 'o_fe',
 'e_o_fe',
 'flag_o_fe',
 'na_fe',
 'e_na_fe',
 'flag_na_fe',
 'mg_fe',
 'e_mg_fe',
 'flag_mg_fe',
 'al_fe',
 'e_al_fe',
 'flag_al_fe',
 'si_fe',
 'e_si_fe',
 'flag_si_fe',
 'k_fe',
 'e_k_fe',
 'flag_k_fe',
 'ca_fe',
 'e_ca_fe',
 'flag_ca_fe',
 'sc_fe',
 'e_sc_fe',
 'flag_sc_fe',
 'ti_fe',
 'e_ti_fe',
 'flag_ti_fe',
 'v_fe',
 

In [219]:
a_Fe,a_Fe_err,abundances = calc_alpha_Fe_from_abundances(
    t_galah_dr4,
    '{x.lower()}_fe',
    'e_{x.lower()}_fe',
    flag_fmt='flag_{x.lower()}_fe'
)
t_galah_dr4['alpha_fe'] = a_Fe
t_galah_dr4['e_alpha_fe'] = a_Fe_err

13.046% of [O/Fe] measurements flagged.
18.139% of [Mg/Fe] measurements flagged.
24.903% of [Si/Fe] measurements flagged.
10.374% of [Ca/Fe] measurements flagged.
27.588% of [Ti/Fe] measurements flagged.


In [220]:
plot_alpha_abundances(a_Fe, abundances)
plt.show()

<Figure size 750x500 with 6 Axes>

In [221]:
idx = (
    (t_galah_dr4['snr_px_ccd3'] > 30)
  & (t_galah_dr4['flag_sp'] == 0)
  & (t_galah_dr4['flag_fe_h'] == 0)
  & (t_galah_dr4['logg'] != 5)
  & (t_galah_dr4['teff'] < 7950.)
  & (t_galah_dr4['alpha_fe'] > -0.4)
  & np.isfinite(t_galah_dr4['e_logg'])
  # & (t_galah_dr4['alpha_fe'] != 0)
)
print(f'{np.count_nonzero(idx):,} sources ({np.mean(idx):.2%}%) pass cuts.')

plot_teff_logg_feh_abundance(
    t_galah_dr4['teff'][idx],
    t_galah_dr4['logg'][idx],
    t_galah_dr4['fe_h'][idx],
    t_galah_dr4['alpha_fe'][idx]
    # 0.5*(t_galah_dr4['si_fe']+t_galah_dr4['mg_fe'])[idx]
)
plt.show()

530,521 sources (62.35%%) pass cuts.


<Figure size 800x400 with 2 Axes>

In [222]:
plt.hist(t_galah_dr4['teff'][idx], bins=100, log=True)
plt.show()

<Figure size 640x480 with 1 Axes>

In [224]:
id_col = 'gaiadr3_source_id'
param_cols = ('teff', 'fe_h', 'logg', 'alpha_fe')
param_unc_cols = ('e_teff', 'e_fe_h', 'e_logg', 'e_alpha_fe')
extra_cols = (
    'ra', 'dec',
    # 'si_fe', 'e_si_fe', 'flag_si_fe',
    # 'mg_fe', 'e_mg_fe', 'flag_mg_fe'
)

# Increasing errors for T_eff > 6000 K
teff_errs = np.clip(
    0.1 + 0.4*(t_galah_dr4['teff'].data.filled()-6000.)/1000.,
    0.1, np.inf
)
param_errs = np.stack([
    teff_errs,
    np.full_like(teff_errs, 0.1),
    np.full_like(teff_errs, 0.1),
    np.full_like(teff_errs, 0.05)
], axis=1)

t_galah = extract_table(
    t_galah_dr4[idx],
    id_col, param_cols, param_unc_cols, extra_cols,
    param_transf=param_transf,
    param_unc_transf=apply_err_floors(param_unc_transf,err_floor_default),
    param_errs=param_errs[idx]
)
t_galah

id,x,sigma_x,x_err,ra,dec
int64,float32[4],float32[4],float32[4],float64,float64
2745049530295263232,-0.5420978 .. 0.22808558,0.07644686 .. 0.05088774,0.1 .. 0.05,0.0005000000237487257,5.380555629730225
4922452332795612416,-0.10049133 .. 0.42085564,0.07429648 .. 0.051894255,0.1 .. 0.05,0.0010000000474974513,-57.16233444213867
4919409709243747968,0.51580226 .. 0.21127568,0.066981025 .. 0.05227842,0.1 .. 0.05,0.0010000000474974513,-57.71352767944336
4685221429109310080,0.5028718 .. -0.007158331,0.06308461 .. 0.051112678,0.1 .. 0.05,0.0010416667209938169,-75.68794250488281
4683766775225601664,-0.022599734 .. 0.28421107,0.0759973 .. 0.052435197,0.1 .. 0.05,0.0015833333600312471,-77.94086456298828
4976477592542752384,0.650234 .. 0.015451031,0.060869075 .. 0.05085305,0.1 .. 0.05,0.0016666667070239782,-51.24397277832031
4972049927935844480,0.790152 .. 0.011464476,0.059520386 .. 0.05130803,0.1 .. 0.05,0.0021250001154839993,-53.789249420166016
4972259865937767040,0.88795966 .. 0.033747643,0.05870724 .. 0.051977366,0.1 .. 0.05,0.003874999936670065,-53.47186279296875
2745049976971860480,0.68241215 .. 0.06402012,0.06582613 .. 0.053346626,0.1 .. 0.05,0.0044999998062849045,5.43186092376709


In [225]:
plot_catalog_uncertainties(t_galah)
plot_catalog_params(t_galah)

<Figure size 1050x280 with 5 Axes>

<Figure size 500x300 with 1 Axes>

<Figure size 500x400 with 1 Axes>

In [226]:
# Save catalog
fn = 'training_data/galah_dr4.h5'
t_galah.write(fn, serialize_metadata=True, overwrite=True)
! du -sh {fn}

/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


37M	training_data/galah_dr4.h5


### GALAH (DR3)

Documentation: https://www.galah-survey.org/dr3/overview/

In [44]:
fn = os.path.join(data_dir, 'GALAH_DR3_main_allstar_v2.fits')
t_galah_dr3 = Table.read(fn)
t_galah_dr3

star_id,sobject_id,dr2_source_id,dr3_source_id,survey_name,field_id,flag_repeat,wg4_field,wg4_pipeline,flag_sp,teff,e_teff,irfm_teff,irfm_ebv,irfm_ebv_ref,logg,e_logg,fe_h,e_fe_h,flag_fe_h,fe_h_atmo,vmic,vbroad,e_vbroad,chi2_sp,alpha_fe,e_alpha_fe,nr_alpha_fe,flag_alpha_fe,flux_A_Fe,chi_A_Fe,Li_fe,e_Li_fe,nr_Li_fe,flag_Li_fe,C_fe,e_C_fe,nr_C_fe,flag_C_fe,O_fe,e_O_fe,nr_O_fe,flag_O_fe,Na_fe,e_Na_fe,nr_Na_fe,flag_Na_fe,Mg_fe,e_Mg_fe,nr_Mg_fe,flag_Mg_fe,Al_fe,e_Al_fe,nr_Al_fe,flag_Al_fe,Si_fe,e_Si_fe,nr_Si_fe,flag_Si_fe,K_fe,e_K_fe,nr_K_fe,flag_K_fe,Ca_fe,e_Ca_fe,nr_Ca_fe,flag_Ca_fe,Sc_fe,e_Sc_fe,nr_Sc_fe,flag_Sc_fe,Sc2_fe,e_Sc2_fe,nr_Sc2_fe,flag_Sc2_fe,Ti_fe,e_Ti_fe,nr_Ti_fe,flag_Ti_fe,Ti2_fe,e_Ti2_fe,nr_Ti2_fe,flag_Ti2_fe,V_fe,e_V_fe,nr_V_fe,flag_V_fe,Cr_fe,e_Cr_fe,nr_Cr_fe,flag_Cr_fe,Cr2_fe,e_Cr2_fe,nr_Cr2_fe,flag_Cr2_fe,Mn_fe,e_Mn_fe,nr_Mn_fe,flag_Mn_fe,Co_fe,e_Co_fe,nr_Co_fe,flag_Co_fe,Ni_fe,e_Ni_fe,nr_Ni_fe,flag_Ni_fe,Cu_fe,e_Cu_fe,nr_Cu_fe,flag_Cu_fe,Zn_fe,e_Zn_fe,nr_Zn_fe,flag_Zn_fe,Rb_fe,e_Rb_fe,nr_Rb_fe,flag_Rb_fe,Sr_fe,e_Sr_fe,nr_Sr_fe,flag_Sr_fe,Y_fe,e_Y_fe,nr_Y_fe,flag_Y_fe,Zr_fe,e_Zr_fe,nr_Zr_fe,flag_Zr_fe,Mo_fe,e_Mo_fe,nr_Mo_fe,flag_Mo_fe,Ru_fe,e_Ru_fe,nr_Ru_fe,flag_Ru_fe,Ba_fe,e_Ba_fe,nr_Ba_fe,flag_Ba_fe,La_fe,e_La_fe,nr_La_fe,flag_La_fe,Ce_fe,e_Ce_fe,nr_Ce_fe,flag_Ce_fe,Nd_fe,e_Nd_fe,nr_Nd_fe,flag_Nd_fe,Sm_fe,e_Sm_fe,nr_Sm_fe,flag_Sm_fe,Eu_fe,e_Eu_fe,nr_Eu_fe,flag_Eu_fe,ra_dr2,dec_dr2,parallax_dr2,parallax_error_dr2,r_est_dr2,r_lo_dr2,r_hi_dr2,r_len_dr2,rv_galah,e_rv_galah,rv_gaia_dr2,e_rv_gaia_dr2,red_flag,ebv,snr_c1_iraf,snr_c2_iraf,snr_c3_iraf,snr_c4_iraf,flag_guess,v_jk,j_m,j_msigcom,h_m,h_msigcom,ks_m,ks_msigcom,ph_qual_tmass,w2mpro,w2mpro_error,ph_qual_wise,a_ks,e_a_ks,bc_ks,ruwe_dr2
,,,,,,,,,,K,K,K,mag,,log(cm.s**-2),log(cm.s**-2),,,,,km / s,km / s,km / s,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,deg,deg,mas,mas,pc,pc,pc,pc,,,km / s,km / s,,mag,,,,,,mag,mag,mag,mag,mag,mag,mag,,mag,mag,,mag,mag,mag,
bytes16,int64,int64,int64,bytes12,int64,int64,bytes6,bytes4,int64,float32,float64,float32,float32,bytes3,float32,float64,float32,float64,int64,float32,float32,float32,float64,float32,float64,float64,float64,int64,float32,float32,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float32,float32,int64,float64,float64,float64,float64,float64,int64,float32,float32,float32,float32,float32,float32,float32,bytes3,float64,float64,bytes4,float64,float64,float32,float64
03325271-6840304,131116000501002,4667368899326729856,4667368899326729856,galah_main,138,0,10k_2,lbol,0,4553.3545,137.06088556228917,4138.0,0.048,sfd,4.7149534,0.20121636742743593,-0.33501863,0.12021913955493135,0,-0.109347336,0.94853675,9.345594,2.5080910007092827,0.5873856,0.35136763119226394,0.07919326531297675,405.0,0,0.73446965,0.80794984,-0.18375580310821538,0.12854836388714738,1,1,--,--,--,32,1.2146472740173344,0.2138376266813869,1,1,0.14526838302612344,0.11489051187565578,1,0,-0.08849821090698207,0.19155067846881743,1

In [45]:
print(t_galah_dr3.colnames)

['star_id', 'sobject_id', 'dr2_source_id', 'dr3_source_id', 'survey_name', 'field_id', 'flag_repeat', 'wg4_field', 'wg4_pipeline', 'flag_sp', 'teff', 'e_teff', 'irfm_teff', 'irfm_ebv', 'irfm_ebv_ref', 'logg', 'e_logg', 'fe_h', 'e_fe_h', 'flag_fe_h', 'fe_h_atmo', 'vmic', 'vbroad', 'e_vbroad', 'chi2_sp', 'alpha_fe', 'e_alpha_fe', 'nr_alpha_fe', 'flag_alpha_fe', 'flux_A_Fe', 'chi_A_Fe', 'Li_fe', 'e_Li_fe', 'nr_Li_fe', 'flag_Li_fe', 'C_fe', 'e_C_fe', 'nr_C_fe', 'flag_C_fe', 'O_fe', 'e_O_fe', 'nr_O_fe', 'flag_O_fe', 'Na_fe', 'e_Na_fe', 'nr_Na_fe', 'flag_Na_fe', 'Mg_fe', 'e_Mg_fe', 'nr_Mg_fe', 'flag_Mg_fe', 'Al_fe', 'e_Al_fe', 'nr_Al_fe', 'flag_Al_fe', 'Si_fe', 'e_Si_fe', 'nr_Si_fe', 'flag_Si_fe', 'K_fe', 'e_K_fe', 'nr_K_fe', 'flag_K_fe', 'Ca_fe', 'e_Ca_fe', 'nr_Ca_fe', 'flag_Ca_fe', 'Sc_fe', 'e_Sc_fe', 'nr_Sc_fe', 'flag_Sc_fe', 'Sc2_fe', 'e_Sc2_fe', 'nr_Sc2_fe', 'flag_Sc2_fe', 'Ti_fe', 'e_Ti_fe', 'nr_Ti_fe', 'flag_Ti_fe', 'Ti2_fe', 'e_Ti2_fe', 'nr_Ti2_fe', 'flag_Ti2_fe', 'V_fe', 'e_V_fe', '

In [1011]:
idx = (
    (t_galah_dr3['snr_c3_iraf'] > 30)
  & (t_galah_dr3['flag_sp'] == 0)
  & (t_galah_dr3['flag_fe_h'] == 0)
)
# t_galah_dr3 = t_galah_dr3[idx]
print(f'GALAH: {np.count_nonzero(idx):,} ({100*np.mean(idx):.2f}%) sources pass cuts.')

GALAH: 325,518 (55.31%) sources pass cuts.


In [47]:
plt.hist2d(
    t_galah_dr3['teff'][idx],
    t_galah_dr3['logg'][idx],
    bins=100, norm=LogNorm()
)
plt.gca().invert_yaxis()
plt.xlim(8100.,3900.)
plt.xlabel('teff')
plt.ylabel('logg')

Text(0, 0.5, 'logg')

<Figure size 640x480 with 1 Axes>

In [1014]:
plot_teff_logg_feh_abundance(
    t_galah_dr3['teff'][idx],
    t_galah_dr3['logg'][idx],
    t_galah_dr3['fe_h'][idx],
    t_galah_dr3['alpha_fe'][idx]
)
plt.show()

<Figure size 1200x600 with 2 Axes>

In [48]:
kw = dict(range=(-3,1), bins=50, histtype='step', alpha=0.7, log=True)
plt.hist(np.log10(0.001*t_galah_dr3['e_teff'][idx]), label='T_eff/1000', **kw)
plt.hist(np.log10(t_galah_dr3['e_fe_h'][idx]), label='[Fe/H]', **kw)
plt.hist(np.log10(t_galah_dr3['e_logg'][idx]), label='log(g)', **kw)
plt.legend()
plt.xlabel('log_10(Uncertainties)')
plt.show()

<Figure size 640x480 with 1 Axes>

In [49]:
id_col = 'dr3_source_id'
param_cols = ('teff', 'fe_h', 'logg')
param_unc_cols = ('e_teff', 'e_fe_h', 'e_logg')
extra_cols = ('alpha_fe', 'flag_alpha_fe')

# Increasing errors for T_eff > 6000 K
teff_errs = np.clip(
    0.1 + 0.4*(t_galah_dr3['teff'].data.filled()-6000.)/1000.,
    0.1, np.inf
)
param_errs = np.stack([
    teff_errs,
    np.full_like(teff_errs, 0.1),
    np.full_like(teff_errs, 0.1)
], axis=1)

t_galah = extract_table(
    t_galah_dr3[idx],
    id_col, param_cols, param_unc_cols, extra_cols,
    param_transf=param_transf,
    param_unc_transf=param_unc_transf,
    param_errs=param_errs[idx]
)
t_galah

id,x,sigma_x,x_err,alpha_fe,flag_alpha_fe
int64,float32[3],float64[3],float32[3],float64,int64
4667368899326729856,-0.4678686 .. 4.7149534,0.1505053975013958 .. 0.20121636742743593,0.1 .. 0.1,0.35136763119226394,0
4667320108498310656,0.6046615 .. 4.2985377,0.10294837567865726 .. 0.187614360454785,0.1 .. 0.1,-0.038428970792295826,0
4667320795693075584,-0.060355484 .. 4.5908637,0.12062379950902924 .. 0.19210115039635556,0.1 .. 0.1,-0.08221199373250411,0
4667319455663284352,0.7332392 .. 4.427538,0.10291417129523993 .. 0.18818886238865848,0.1 .. 0.1,0.10966809570771893,0
4667319043346417792,-0.33093214 .. 1.9448414,0.1447741350676227 .. 0.23643389395363013,0.1 .. 0.1,0.17930676846144403,0
4667314370422016896,1.0518343 .. 4.1462836,0.08458261494934251 .. 0.1841535706717135,0.16826153 .. 0.1,0.06803995892679174,0
4667307498474314624,0.3452707 .. 4.5135064,0.09212241307803816 .. 0.18202574587245743,0.1 .. 0.1,0.29573042265322624,0
4667126968112646400,1.1836247 .. 4.1368785,0.07954686516708974 .. 0.18243332823689828,0.23418516 .. 0.1,-0.028634879882749167,0
4667313167831183744,0.40702328 .. 4.4302807,0.11534629404862773 .. 0.18909107201954273,0.1 .. 0.1,0.05109579926286323,0


In [50]:
plot_catalog_uncertainties(t_galah)
plot_catalog_params(t_galah)

<Figure size 800x280 with 4 Axes>

<Figure size 500x300 with 1 Axes>

<Figure size 500x400 with 1 Axes>

### LAMOST LRS DD-Payne (DR9)

References:
  * [Zhang *et al* (2025)](https://ui.adsabs.harvard.edu/abs/2025arXiv250602763Z/abstract)
  * https://nadc.china-vo.org/res/r101614/
  * https://zenodo.org/records/15254859

In [220]:
fn = 'LMDR9_DDPAYNE_recommend_202505.fits'
full_path = os.path.join(data_dir, fn)

In [627]:
url = fr'https://zenodo.org/records/15254859/files/{fn}?download=1'

! wget -O {full_path} {url}

--2025-06-05 11:09:35--  https://zenodo.org/records/15254859/files/LMDR9_DDPAYNE_recommend_202505.fits?download=1
Resolving zenodo.org (zenodo.org)... 188.185.45.92, 188.185.48.194, 188.185.43.25, ...
Connecting to zenodo.org (zenodo.org)|188.185.45.92|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3402351360 (3.2G) [application/octet-stream]
Saving to: ‘/data/beegfs/astro-storage/groups/green/green/projects/gaia_Bp_Rp_reddening_variation/data/LMDR9_DDPAYNE_recommend_202505.fits’

/data/beegfs/astro- 100%[===================>]   3.17G  8.54MB/s    in 5m 59s  

2025-06-05 11:15:35 (9.03 MB/s) - ‘/data/beegfs/astro-storage/groups/green/green/projects/gaia_Bp_Rp_reddening_variation/data/LMDR9_DDPAYNE_recommend_202505.fits’ saved [3402351360/3402351360]



In [221]:
with fits.open(full_path) as f:
    f.info()
    t_lamost_ddpayne_dr9 = Table(f[1].data)

t_lamost_ddpayne_dr9

Filename: data/LMDR9_DDPAYNE_recommend_202505.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       5   ()      
  1  /Users/zhangmeng/Documents/Download_data/LMDR9_ddp_recommend_202505.    1 BinTableHDU    198   9245463R x 93C   [36A, D, D, I, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, I, E, E, I, E, E, I, E, E, I, E, E, I, E, E, I, E, E, I, E, E, I, E, E, I, E, E, I, E, E, I, E, E, I, E, E, I, E, E, E, I, E, E, E, I, E, E, E, I, E, E, I, E, E, E, I, E, E, E, I, E, E, E, I, E, E, E, I, E, E, E, E, E]   


SPECID,RA,DEC,UNIQFLAG,SNRU,SNRG,SNRR,SNRI,SNRZ,TEFF,TEFF_ERR,LOGG,LOGG_ERR,FEH,FEH_LTE,FEH_ERR,VMIC,VMIC_ERR,C_FE,C_FE_ERR,FLAG_C_FE,N_FE,N_FE_ERR,FLAG_N_FE,O_FE,O_FE_ERR,FLAG_O_FE,NA_FE,NA_FE_ERR,FLAG_NA_FE,MG_FE,MG_FE_ERR,FLAG_MG_FE,AL_FE,AL_FE_ERR,FLAG_AL_FE,SI_FE,SI_FE_ERR,FLAG_SI_FE,CA_FE,CA_FE_ERR,FLAG_CA_FE,TI_FE,TI_FE_ERR,FLAG_TI_FE,CR_FE,CR_FE_ERR,FLAG_CR_FE,MN_FE,MN_FE_ERR,FLAG_MN_FE,NI_FE,NI_FE_ERR,FLAG_NI_FE,SR_FE,SR_FE_ERR,FLAG_SR_FE,CHISQ_SR,Y_FE,Y_FE_ERR,FLAG_Y_FE,CHISQ_Y,ZR_FE,ZR_FE_ERR,FLAG_ZR_FE,CHISQ_ZR,BA_FE,BA_FE_ERR,FLAG_BA_FE,LA_FE,LA_FE_ERR,FLAG_LA_FE,CHISQ_LA,CE_FE,CE_FE_ERR,FLAG_CE_FE,CHISQ_CE,ND_FE,ND_FE_ERR,FLAG_ND_FE,CHISQ_ND,SM_FE,SM_FE_ERR,FLAG_SM_FE,CHISQ_SM,EU_FE,EU_FE_ERR,FLAG_EU_FE,CHISQ_EU,CHISQ_LAM_APO,FLAGCHISQ_LAM_APO,CHISQ_LAM_GAL,FLAGCHISQ_LAM_GAL
str36,float64,float64,int16,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int16,float32,float32,int16,float32,float32,int16,float32,float32,int16,float32,float32,int16,float32,float32,int16,float32,float32,int16,float32,float32,int16,float32,float32,int16,float32,float32,int16,float32,float32,int16,float32,float32,int16,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,float32,float32
20111024-F5902-01-001,332.202274,-2.056767,1,0.7,12.22,27.16,36.14,21.07,5209.269,161.91507,4.679,0.31,-0.498,-0.494,0.14,0.773,0.72,0.116,0.24,0,0.329,0.49,0,-0.03,0.35,0,0.298,0.27,0,0.164,0.14,0,0.227,0.38,0,0.179,0.24,0,0.507,0.15,0,-0.053,0.13,0,-0.086,0.13,0,-0.218,0.37,0,-0.067,0.25,0,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,-2.74,0.8,0,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,1.33,2.74,1.33,2.66
20111024-F5902-01-002,332.471576,-2.085015,1,0.23,3.84,20.02,33.87,21.58,4258.285,123.80238,4.71,0.33,-0.774,-0.784,0.16,0.002,0.72,-0.423,0.52,0,0.668,0.64,0,0.345,0.37,0,0.405,0.2,0,0.304,0.15,0,-0.14,0.49,0,0.559,0.28,0,0.286,0.15,0,0.197,0.12,0,0.152,0.15,0,-0.272,0.55,0,-0.055,0.34,0,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,-0.155,0.76,0,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,1.06,1.82,1.06,1.74
20111024-F5902-01-005,332.535456,-2.116436,1,0.71,8.08,16.76,23.27,12.09,5769.0796,224.89018,4.027,0.57,-0.545,-0.542,0.2,1.654,0.63,0.426,0.24,0,0.919,0.4,0,0.341,0.39,0,0.961,0.43,0,0.31,0.2,0,0.364,0.53,0,0.387,0.24,0,0.687,0.22,0,-0.179,0.17,0,0.042,0.25,0,0.036,0.47,0,0.027,0.32,0,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,-1.582,1.34,0,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,1.12,2.0,1.12,1.95
20111024-F5902-01-008,332.368745,-1.955771,1,1.21,20.64,37.99,48.37,27.99,5635.1284,114.74779,4.707,0.22,-0.118,-0.112,0.09,0.465,0.52,0.137,0.13,0,0.496,0.33,0,0.244,0.34,0,-0.019,0.23,0,0.061,0.11,0,0.449,0.2,0,0.201,0.14,0,0.293,0.1,0,0.072,0.1,0,0.079,0.1,0,0.346,0.21,0,-0.016,0.18,0,0.098,0.44,0,1.33,-1.326,0.76,0,1.31,-0.251,0.21,0,1.32,-1.881,0.56,0,-0.416,0.28,0,1.32,0.205,0.37,0,1.31,-0.144,0.4,0,1.33,-0.49,0.34,0,1.31,1.235,0.39,0,1.31,1.32,2.98,1.32,2.83
20111024-F5902-01-009,332.206665,-1.868653,1,1.23,12.63,25.05,32.8,17.69,5845.679,172.66512,4.201,0.38,-0.268,-0.263,0.14,0.393,0.72,0.115,0.22,0,0.533,0.42,0,-0.12,0.31,0,0.359,0.3,0,0.101,0.19,0,0.259,0.41,0,0.44,0.18,0,0.731,0.16,0,0.36,0.14,0,-0.147,0.17,0,-0.177,0.45,0,-0.135,0.27,0,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,-2.041,0.69,0,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,-99.99,-99.99,0,99.99,1.16,2.15,1.16,2.07
20111024-F5902-01-012,332.37844,-2.0

In [1278]:
Gaia.login()

INFO:astroquery:Login to gaia TAP server


INFO: Login to gaia TAP server [astroquery.gaia.core]


User:  ggreen
Password:  ········


INFO:astroquery:OK
INFO:astroquery:Login to gaia data server


INFO: OK [astroquery.utils.tap.core]
INFO: Login to gaia data server [astroquery.gaia.core]


INFO:astroquery:OK


INFO: OK [astroquery.utils.tap.core]


In [1294]:
crossmatch_info = gaia_crossmatch_batched(
    t_lamost_ddpayne_dr9['RA'],
    t_lamost_ddpayne_dr9['DEC'],
    max_batch_size=1024*1024
)

INFO:astroquery:Sending pytable.


INFO: Sending pytable. [astroquery.utils.tap.core]


INFO:astroquery:Uploaded table 'crossmatch_table_0'.


INFO: Uploaded table 'crossmatch_table_0'. [astroquery.utils.tap.core]


INFO:astroquery:Sending pytable.


INFO: Sending pytable. [astroquery.utils.tap.core]


... (36 outputs left out)

INFO:astroquery:Table 'crossmatch_table_8388608' deleted.


INFO: Table 'crossmatch_table_8388608' deleted. [astroquery.utils.tap.core]
9,047,014 matches


<Figure size 960x720 with 1 Axes>

8,585,601 (94.9%) under 0.5 arcsec


In [1299]:
fn = os.path.join(data_dir, 'lamost_ddpayne_dr9_crossmatch.h5')
crossmatch_info.write(fn)

/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


In [222]:
# Run this if the crossmatch is already on file
fn = os.path.join(data_dir, 'lamost_ddpayne_dr9_crossmatch.h5')
crossmatch_info = Table.read(fn)

In [223]:
t_lamost_ddpayne_dr9 = apply_crossmatch(crossmatch_info, t_lamost_ddpayne_dr9)

In [224]:
print(t_lamost_ddpayne_dr9.colnames)

['SPECID', 'RA', 'DEC', 'UNIQFLAG', 'SNRU', 'SNRG', 'SNRR', 'SNRI', 'SNRZ', 'TEFF', 'TEFF_ERR', 'LOGG', 'LOGG_ERR', 'FEH', 'FEH_LTE', 'FEH_ERR', 'VMIC', 'VMIC_ERR', 'C_FE', 'C_FE_ERR', 'FLAG_C_FE', 'N_FE', 'N_FE_ERR', 'FLAG_N_FE', 'O_FE', 'O_FE_ERR', 'FLAG_O_FE', 'NA_FE', 'NA_FE_ERR', 'FLAG_NA_FE', 'MG_FE', 'MG_FE_ERR', 'FLAG_MG_FE', 'AL_FE', 'AL_FE_ERR', 'FLAG_AL_FE', 'SI_FE', 'SI_FE_ERR', 'FLAG_SI_FE', 'CA_FE', 'CA_FE_ERR', 'FLAG_CA_FE', 'TI_FE', 'TI_FE_ERR', 'FLAG_TI_FE', 'CR_FE', 'CR_FE_ERR', 'FLAG_CR_FE', 'MN_FE', 'MN_FE_ERR', 'FLAG_MN_FE', 'NI_FE', 'NI_FE_ERR', 'FLAG_NI_FE', 'SR_FE', 'SR_FE_ERR', 'FLAG_SR_FE', 'CHISQ_SR', 'Y_FE', 'Y_FE_ERR', 'FLAG_Y_FE', 'CHISQ_Y', 'ZR_FE', 'ZR_FE_ERR', 'FLAG_ZR_FE', 'CHISQ_ZR', 'BA_FE', 'BA_FE_ERR', 'FLAG_BA_FE', 'LA_FE', 'LA_FE_ERR', 'FLAG_LA_FE', 'CHISQ_LA', 'CE_FE', 'CE_FE_ERR', 'FLAG_CE_FE', 'CHISQ_CE', 'ND_FE', 'ND_FE_ERR', 'FLAG_ND_FE', 'CHISQ_ND', 'SM_FE', 'SM_FE_ERR', 'FLAG_SM_FE', 'CHISQ_SM', 'EU_FE', 'EU_FE_ERR', 'FLAG_EU_FE', 'CHISQ_E

In [225]:
[s for s in t_lamost_ddpayne_dr9.colnames if 'A_' in s]

['NA_FE',
 'NA_FE_ERR',
 'FLAG_NA_FE',
 'CA_FE',
 'CA_FE_ERR',
 'FLAG_CA_FE',
 'BA_FE',
 'BA_FE_ERR',
 'FLAG_BA_FE',
 'LA_FE',
 'LA_FE_ERR',
 'FLAG_LA_FE']

In [226]:
lamost_dr9_snr_tot = np.zeros(len(t_lamost_ddpayne_dr9))
for b in 'UGRIZ':
    lamost_dr9_snr_tot += t_lamost_ddpayne_dr9[f'SNR{b}']**2
lamost_dr9_snr_tot = np.sqrt(lamost_dr9_snr_tot)
plt.hist(np.log10(lamost_dr9_snr_tot), range=(0,4), bins=100)
plt.xlabel('SNR_TOT')
plt.axvline(np.log10(50), c='k', ls='--')
plt.show()

/tmp/ipykernel_3594745/1477778138.py:5: RuntimeWarning: divide by zero encountered in log10
  plt.hist(np.log10(lamost_dr9_snr_tot), range=(0,4), bins=100)


<Figure size 640x480 with 1 Axes>

In [227]:
lamost_dr9_chisq_of_snr_apo = determine_snr_chisq_relation(
    lamost_dr9_snr_tot,
    t_lamost_ddpayne_dr9['CHISQ_LAM_APO']
)
plt.show()

<Figure size 500x400 with 1 Axes>

In [228]:
lamost_dr9_chisq_of_snr_gal = determine_snr_chisq_relation(
    lamost_dr9_snr_tot,
    t_lamost_ddpayne_dr9['CHISQ_LAM_GAL']
)
plt.show()

<Figure size 500x400 with 1 Axes>

In [229]:
idx = t_lamost_ddpayne_dr9['FLAG_MG_FE'] == 0
np.count_nonzero(t_lamost_ddpayne_dr9['MG_FE'][idx] < -99)

2632

In [230]:
np.unique(t_lamost_ddpayne_dr9['FLAG_O_FE'], return_counts=True)

(<Column name='FLAG_O_FE' dtype='int16' length=2>
  0
 99,
 array([8460383,  125218]))

In [231]:
# Some crazy abundances ([X/Fe] ~ -99) are not flagged. Make
# sure they are.
for el in ['O', 'MG', 'SI', 'CA', 'TI']:
    idx = (
        (t_lamost_ddpayne_dr9[f'{el}_FE'] < -99)
      & (t_lamost_ddpayne_dr9[f'FLAG_{el}_FE'] == 0)
    )
    print(f'{el}_FE: {np.count_nonzero(idx)}')
    t_lamost_ddpayne_dr9[f'FLAG_{el}_FE'][idx] = 98

O_FE: 0
MG_FE: 2632
SI_FE: 0
CA_FE: 0
TI_FE: 1819


In [232]:
a_Fe,a_Fe_err,abundances = calc_alpha_Fe_from_abundances(
    t_lamost_ddpayne_dr9,
    '{x.upper()}_FE',
    '{x.upper()}_FE_ERR',
    flag_fmt='FLAG_{x.upper()}_FE'
)
t_lamost_ddpayne_dr9['ALPHA_FE'] = a_Fe
t_lamost_ddpayne_dr9['ALPHA_FE_ERR'] = a_Fe_err

plot_alpha_abundances(a_Fe, abundances)
plt.show()

1.458% of [O/Fe] measurements flagged.
0.031% of [Mg/Fe] measurements flagged.
1.458% of [Si/Fe] measurements flagged.
1.458% of [Ca/Fe] measurements flagged.
1.480% of [Ti/Fe] measurements flagged.


<Figure size 750x500 with 6 Axes>

In [233]:
for el in abundances.colnames:
    plt.hist2d(
        t_lamost_ddpayne_dr9['FEH'],
        abundances[el],
        range=[(-4.,1.),(-0.6,0.9)],
        bins=50,
        norm=LogNorm()
    )
    plt.gca().set_title(el)
    plt.show()

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

In [241]:
idx = (
    (t_lamost_ddpayne_dr9['UNIQFLAG'] == 1)
  & (lamost_dr9_snr_tot > 40.)
  & (t_lamost_ddpayne_dr9['CHISQ_LAM_APO'] < 5*lamost_dr9_chisq_of_snr_apo(lamost_dr9_snr_tot))
  & (t_lamost_ddpayne_dr9['CHISQ_LAM_GAL'] < 5*lamost_dr9_chisq_of_snr_gal(lamost_dr9_snr_tot))
  # & (t_lamost_ddpayne_dr9['FLAGCHISQ_LAM_APO'] < 3)
  # & (t_lamost_ddpayne_dr9['FLAGCHISQ_LAM_GAL'] < 3)
  & (t_lamost_ddpayne_dr9['TEFF'] > 3000.) # Avoid edges of grid
  & (t_lamost_ddpayne_dr9['FEH'] > -9.)
  & (t_lamost_ddpayne_dr9['LOGG'] > -2.)
  & (t_lamost_ddpayne_dr9['TEFF_ERR'] < 200.)
  & (t_lamost_ddpayne_dr9['LOGG_ERR'] < 0.3)
  & (t_lamost_ddpayne_dr9['FEH_ERR'] < 0.2)
  & (t_lamost_ddpayne_dr9['ALPHA_FE_ERR'] < 0.2)
  & ~(
         (t_lamost_ddpayne_dr9['TEFF'] < 4350.)
       & (t_lamost_ddpayne_dr9['LOGG'] > 3.6)
     )
  & ~(
         (t_lamost_ddpayne_dr9['TEFF'] < 4200.)
       & (t_lamost_ddpayne_dr9['LOGG'] > 2.6)
     )
)
print(f'{np.count_nonzero(idx):,} stars ({np.mean(idx):.2%}) pass cuts.')

plot_teff_logg_feh_abundance(
    t_lamost_ddpayne_dr9['TEFF'][idx],
    t_lamost_ddpayne_dr9['LOGG'][idx],
    t_lamost_ddpayne_dr9['FEH'][idx],
    t_lamost_ddpayne_dr9['ALPHA_FE'][idx]
    # 0.5*(t_lamost_ddpayne_dr9['MG_FE']+t_lamost_ddpayne_dr9['SI_FE'])[idx],
    # abundance_label=r'$\frac{1}{2}\left(\left[\mathrm{Mg/Fe}\right]+\left[\mathrm{Si/Fe}\right]\right)$'
)
plt.show()

4,115,061 stars (47.93%) pass cuts.


<Figure size 800x400 with 2 Axes>

In [243]:
id_col = 'gdr3_source_id'
param_cols = ('TEFF', 'FEH', 'LOGG', 'ALPHA_FE')
param_unc_cols = ('TEFF_ERR', 'FEH_ERR', 'LOGG_ERR', 'ALPHA_FE_ERR')
extra_cols = ('ra_gdr3', 'dec_gdr3')

t_ddp = extract_table(
    t_lamost_ddpayne_dr9[idx],
    id_col, param_cols, param_unc_cols, extra_cols,
    param_transf=param_transf,
    param_unc_transf=apply_err_floors(param_unc_transf,err_floor_default)
)
t_ddp

id,x,sigma_x,x_err,ra_gdr3,dec_gdr3
int64,float32[4],float32[4],float32[4],float64,float64
2676865000324707584,0.5979103 .. 0.15748942,0.102304615 .. 0.073800065,0.1 .. 0.1,332.3687428654065,-1.9557637296266273
2676107368093944576,-0.32621992 .. 0.05947821,0.045275383 .. 0.056805138,0.1 .. 0.1,332.34874121176233,-2.1360900624608155
2676120459154270336,0.7565265 .. 0.19882417,0.099180356 .. 0.08060611,0.1 .. 0.1,332.44440909644413,-1.9240768953839664
2676876029800730368,0.97066134 .. 0.36448923,0.055411913 .. 0.06762034,0.1 .. 0.1,332.35138458727585,-1.8093831436443053
2676115988093011200,0.47170007 .. 0.025471214,0.12327169 .. 0.08756965,0.1 .. 0.1,332.50636945215433,-2.0169298977284527
2676868844320528384,-0.61512953 .. 0.28574243,0.1109665 .. 0.072267555,0.1 .. 0.1,332.2444227638788,-1.919855676556825
2676925576543545856,0.6444606 .. 0.06320097,0.053058572 .. 0.058062438,0.1 .. 0.1,331.55125526868216,-1.6843748782799735
2676859743284797824,0.73217046 .. 0.3001652,0.06499106 .. 0.068977155,0.1 .. 0.1,331.7683311619556,-1.9145430556538234
2676927844286277120,0.6935762 .. 0.12895904,0.0703689 .. 0.06285169,0.1 .. 0.1,331.6216597219184,-1.6114566093476956


In [244]:
plot_catalog_uncertainties(t_ddp)
plot_catalog_params(t_ddp)

<Figure size 1050x280 with 5 Axes>

<Figure size 500x300 with 1 Axes>

<Figure size 500x400 with 1 Axes>

In [245]:
# Save catalog
fn = 'training_data/lamost_ddp_dr9.h5'
t_ddp.write(fn, serialize_metadata=True, overwrite=True)
! du -sh {fn}

/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


283M	training_data/lamost_ddp_dr9.h5


### LAMOST LRS DD-Payne (DR5)

Reference: [Xiang *et al* (2019)](https://ui.adsabs.harvard.edu/abs/2019ApJS..245...34X/abstract)

In [51]:
fn = os.path.join(data_dir, 'lamost/ddpayne_with_gaia.h5')
t_ddpayne = Table.read(fn)
t_ddpayne

SPECID,RA,DEC,SNR_U,SNR_G,SNR_R,SNR_I,SNR_Z,RV,RV_ERR,TEFF,TEFF_ERR,TEFF_FLAG,TEFF_GRADCORR,LOGG,LOGG_ERR,LOGG_FLAG,LOGG_GRADCORR,VMIC,VMIC_ERR,VMIC_FLAG,VMIC_GRADCORR,FEH,FEH_ERR,FEH_FLAG,FEH_GRADCORR,ALPHA_FE,ALPHA_FE_ERR,C_FE,C_FE_ERR,C_FE_FLAG,C_FE_GRADCORR,N_FE,N_FE_ERR,N_FE_FLAG,N_FE_GRADCORR,O_FE,O_FE_ERR,O_FE_FLAG,O_FE_GRADCORR,NA_FE,NA_FE_ERR,NA_FE_FLAG,NA_FE_GRADCORR,MG_FE,MG_FE_ERR,MG_FE_FLAG,MG_FE_GRADCORR,AL_FE,AL_FE_ERR,AL_FE_FLAG,AL_FE_GRADCORR,SI_FE,SI_FE_ERR,SI_FE_FLAG,SI_FE_GRADCORR,CA_FE,CA_FE_ERR,CA_FE_FLAG,CA_FE_GRADCORR,TI_FE,TI_FE_ERR,TI_FE_FLAG,TI_FE_GRADCORR,CR_FE,CR_FE_ERR,CR_FE_FLAG,CR_FE_GRADCORR,MN_FE,MN_FE_ERR,MN_FE_FLAG,MN_FE_GRADCORR,CO_FE,CO_FE_ERR,CO_FE_FLAG,CO_FE_GRADCORR,NI_FE,NI_FE_ERR,NI_FE_FLAG,NI_FE_GRADCORR,CU_FE,CU_FE_ERR,CU_FE_FLAG,CU_FE_GRADCORR,BA_FE,BA_FE_ERR,BA_FE_FLAG,BA_FE_GRADCORR,CHI2,CHI2RATIO,QFLAG_CHI2,CHI2_NA,CHI2RATIO_NA,QFLAG_CHI2NA,SNR_DPARALLAX,FLAG_SINGLESTAR,UQFLAG,STARID,SUBCLASS,FILENAME,index,source_id,ra_gaia,dec_gaia,separation
,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,deg,deg,deg
bytes36,float64,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,bytes4,float32,float32,bytes4,float32,bytes3,int16,bytes15,bytes10,bytes48,int64,int64,float64,float64,float64
20111024-F5902-01-001,332.202274,-2.056767,0.76,11.94,26.28,35.47,19.85,-38.940044,7.689677,4546.194,108.61385,1,0.7830999,3.4178967,0.25073984,1,0.755402,1.4803419,0.44890782,0,0.2439913,-1.4721066,0.17490259,1,0.8211115,0.30152404,0.06451062,0.121059954,0.0827225,1,0.7768615,-0.2687747,0.2192619,1,0.4954435,1.154818,0.2589388,1,0.6804764,-999.0,999.0,0,0.16305172,-999.0,999.0,0,0.6573531,-999.0,999.0,0,0.5746909,0.0472043,0.2084787,1,0.5879342,0.29398775,0.15651219,1,0.72058046,0.12961528,0.115971796,1,0.56485933,-999.0,999.0,0,0.5474243,-999.0,999.0,0,0.75086904,-0.5480199,0.19378707,1,0.416879,0.49461648,0.15446192,1,0.4731944,-999.0,999.0,0,0.15481378,-0.9690957,0.88108677,1,0.24501485,1.4244671,4.6220574,good,0.0,-1.0,bad,0.0,YES,1,S3321208-020324,G8,spec-55859-F5902_sp01-001.fits.gz,0,2676862492063809792,332.20225862477406,-2.056803662973053,6.92898142144228e-05
20111024-F5902-01-002,332.471576,-2.085015,0.11,3.69,20.06,33.83,22.44,35.22262,18.743025,4099.6357,116.68672,0,0.7144854,4.2065735,0.3755912,0,0.6968592,2.023125,0.33970135,0,0.11263688,-2.3911533,0.23108168,0,0.7281972,0.024181152,0.09537365,-999.0,999.0,0,0.5512851,-999.0,999.0,0,0.36306107,-999.0,999.0,0,0.42165866,-999.0,999.0,0,0.47037145,-999.0,999.0,0,0.63631,-999.0,999.0,0,0.7160149,-999.0,999.0,0,0.5712866,-999.0,999.0,0,0.62887514,-999.0,999.0,0,0.6117069,-999.0,999.0,0,0.7501514,-999.0,999.0,0,0.8570372,-999.0,999.0,0,0.39874685,-999.0,999.0,0,0.39454663,-999.0,999.0,0,0.13832052,-999.0,999.0,0,0.13923912,1.2527953,2.521022,good,0.0,-2.9602513,bad,-4.9224486,YES,1,S3322817-020506,K7,spec-55859-F5902_sp01-002.fits.gz,1,2676109292239283200,332.4715506360606,-2.085064930949917,3.7139823768405063e-05
20111024-F5902-01-004,332.431549,-2.062375,0.16,0.18,4.14,7.37,5.59,-1205.5225,50528.223,5325.3467,658.6197,1,0.60182565,0.105232686,1.1847372,1,0.1829677,2.7064576,2.693876,0,0.18568386,-2.6177557,1.2418091,1,0.5370105,-0.63306767,0.6940811,-0.19187333,0.4360624,1,0.14064318,-999.0,999.0,0,0.1689736,-999.0,999.0,0,0.063630156,-999.0,999.0,0,0.27498809,-999.0,999.0,0,0.04291493,-9

In [52]:
print(t_ddpayne.colnames)

['SPECID', 'RA', 'DEC', 'SNR_U', 'SNR_G', 'SNR_R', 'SNR_I', 'SNR_Z', 'RV', 'RV_ERR', 'TEFF', 'TEFF_ERR', 'TEFF_FLAG', 'TEFF_GRADCORR', 'LOGG', 'LOGG_ERR', 'LOGG_FLAG', 'LOGG_GRADCORR', 'VMIC', 'VMIC_ERR', 'VMIC_FLAG', 'VMIC_GRADCORR', 'FEH', 'FEH_ERR', 'FEH_FLAG', 'FEH_GRADCORR', 'ALPHA_FE', 'ALPHA_FE_ERR', 'C_FE', 'C_FE_ERR', 'C_FE_FLAG', 'C_FE_GRADCORR', 'N_FE', 'N_FE_ERR', 'N_FE_FLAG', 'N_FE_GRADCORR', 'O_FE', 'O_FE_ERR', 'O_FE_FLAG', 'O_FE_GRADCORR', 'NA_FE', 'NA_FE_ERR', 'NA_FE_FLAG', 'NA_FE_GRADCORR', 'MG_FE', 'MG_FE_ERR', 'MG_FE_FLAG', 'MG_FE_GRADCORR', 'AL_FE', 'AL_FE_ERR', 'AL_FE_FLAG', 'AL_FE_GRADCORR', 'SI_FE', 'SI_FE_ERR', 'SI_FE_FLAG', 'SI_FE_GRADCORR', 'CA_FE', 'CA_FE_ERR', 'CA_FE_FLAG', 'CA_FE_GRADCORR', 'TI_FE', 'TI_FE_ERR', 'TI_FE_FLAG', 'TI_FE_GRADCORR', 'CR_FE', 'CR_FE_ERR', 'CR_FE_FLAG', 'CR_FE_GRADCORR', 'MN_FE', 'MN_FE_ERR', 'MN_FE_FLAG', 'MN_FE_GRADCORR', 'CO_FE', 'CO_FE_ERR', 'CO_FE_FLAG', 'CO_FE_GRADCORR', 'NI_FE', 'NI_FE_ERR', 'NI_FE_FLAG', 'NI_FE_GRADCORR', '

In [53]:
ddpayne_snr_tot = np.zeros(len(t_ddpayne))
for b in 'UGRIZ':
    ddpayne_snr_tot += t_ddpayne[f'SNR_{b}']**2
ddpayne_snr_tot = np.sqrt(ddpayne_snr_tot)
plt.hist(np.log10(ddpayne_snr_tot), range=(0,4), bins=100)
plt.xlabel('SNR_TOT')
plt.axvline(np.log10(50), c='k', ls='--')
plt.show()

/tmp/ipykernel_2991738/3192498085.py:5: RuntimeWarning: divide by zero encountered in log10
  plt.hist(np.log10(ddpayne_snr_tot), range=(0,4), bins=100)


<Figure size 640x480 with 1 Axes>

In [1030]:
idx = (
    (t_ddpayne['QFLAG_CHI2'] == b'good')
  # & (t_ddpayne['QFLAG_CHI2NA'] == b'good')
  & (t_ddpayne['FLAG_SINGLESTAR'] == b'YES')
  & (t_ddpayne['UQFLAG'] == 1)
  & (ddpayne_snr_tot > 50)#100)
  # & (
  #       (t_ddpayne['SNR_U'] > 20)
  #     | (t_ddpayne['SNR_G'] > 20)
  #     | (t_ddpayne['SNR_R'] > 20)
  #     | (t_ddpayne['SNR_I'] > 20)
  #     | (t_ddpayne['SNR_Z'] > 20)
  #   )
  & (t_ddpayne['TEFF'] > -9998)
  & (t_ddpayne['FEH'] > -998)
  & (t_ddpayne['LOGG'] > -998)
  & (t_ddpayne['TEFF_FLAG'] == 1)
  & (t_ddpayne['FEH_FLAG'] == 1)
  & (t_ddpayne['LOGG_FLAG'] == 1)
)
print(f'DD-Payne: {np.mean(idx):.2%} of stars ({np.count_nonzero(idx):,}) pass cuts')

DD-Payne: 39.27% of stars (3,022,667) pass cuts


In [1031]:
plot_teff_logg_feh_abundance(
    t_ddpayne['TEFF'][idx],
    t_ddpayne['LOGG'][idx],
    t_ddpayne['FEH'][idx],
    t_ddpayne['ALPHA_FE'][idx]
)
plt.show()

<Figure size 1200x600 with 2 Axes>

In [55]:
plt.hist2d(
    np.log10(ddpayne_snr_tot[idx]),
    t_ddpayne['FEH'][idx],
    bins=100,
    norm=LogNorm()
)
# plt.gca().set_xscale('log')
plt.show()

<Figure size 640x480 with 1 Axes>

In [56]:
flag = 'SNR_TOT'
# plot_idx = slice(None,None,100)
# plot_idx = np.where(ddpayne_snr_tot > 10)[0][::100]
# plt.scatter(
#     t_ddpayne['TEFF'][plot_idx],
#     t_ddpayne['LOGG'][plot_idx],
#     c=ddpayne_snr_tot[plot_idx],
#     # c=t_ddpayne[flag][plot_idx],
#     edgecolors='none',
#     alpha=0.3,
#     norm=LogNorm(vmin=10)
# )
for snr_min in (10, 500):
    plot_idx = np.where(idx & (ddpayne_snr_tot > snr_min))[0][::100]
    plt.scatter(
        t_ddpayne['TEFF'][plot_idx],
        t_ddpayne['LOGG'][plot_idx],
        edgecolors='none',
        alpha=0.3
    )
plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.xlabel(r'$T_{\rm eff}$')
plt.ylabel(r'$\log g$')
cb = plt.colorbar(label=flag)
cb.set_alpha(1)
# cb.draw_all()
plt.show()

<Figure size 640x480 with 2 Axes>

In [57]:
kw = dict(range=(-3,1), bins=50, histtype='step', alpha=0.7, log=True)
plt.hist(np.log10(0.001*t_ddpayne['TEFF_ERR'][idx]), label='T_eff/1000', **kw)
plt.hist(np.log10(t_ddpayne['FEH_ERR'][idx]), label='[Fe/H]', **kw)
plt.hist(np.log10(t_ddpayne['LOGG_ERR'][idx]), label='log(g)', **kw)
plt.legend()
plt.xlabel('log_10(Uncertainties)')
plt.show()

<Figure size 640x480 with 1 Axes>

In [58]:
id_col = 'source_id'
param_cols = ('TEFF', 'FEH', 'LOGG')
param_unc_cols = ('TEFF_ERR', 'FEH_ERR', 'LOGG_ERR')
extra_cols = ('ALPHA_FE', 'ALPHA_FE_ERR')

t_ddp = extract_table(
    t_ddpayne[idx],
    id_col, param_cols, param_unc_cols, extra_cols,
    param_transf=param_transf,
    param_unc_transf=apply_err_floors(param_unc_transf,[0.01,0.01,0.01])
)
t_ddp

id,x,sigma_x,x_err,ALPHA_FE,ALPHA_FE_ERR
int64,float32[3],float32[3],float32[3],float32,float32
2676107368093944576,-0.44593823 .. 4.454826,0.04890754 .. 0.09778897,0.1 .. 0.1,0.16617551,0.019678907
2676913889937397376,-0.31007108 .. 4.1433477,0.060256775 .. 0.16981305,0.1 .. 0.1,0.2216916,0.037424516
2676118500649211904,-0.09466745 .. 1.9184622,0.104227714 .. 0.24781403,0.1 .. 0.1,0.5528383,0.07466538
2676199108595296512,0.10411541 .. 3.6568546,0.108520046 .. 0.2734591,0.1 .. 0.1,0.22216442,0.076996736
2676856135512245504,-0.46495885 .. 3.9274948,0.09350529 .. 0.18657164,0.1 .. 0.1,0.36324826,0.048933707
2676899394422949376,-0.21603404 .. 3.6536865,0.08933713 .. 0.20022747,0.1 .. 0.1,0.23878181,0.04261214
2677134896069602688,-0.37477028 .. 4.6055427,0.046959378 .. 0.07564107,0.1 .. 0.1,0.07655059,0.022982087
2677118884431510144,0.654485 .. 4.277011,0.02537071 .. 0.06049516,0.1 .. 0.1,0.13253073,0.020197783
2676929940230331648,-0.46052516 .. 4.640869,0.04147495 .. 0.06037724,0.1 .. 0.1,0.1325333,0.018736219


In [59]:
plot_catalog_uncertainties(t_ddp)
plot_catalog_params(t_ddp)

<Figure size 800x280 with 4 Axes>

<Figure size 500x300 with 1 Axes>

<Figure size 500x400 with 1 Axes>

### LAMOST M-stars (Ding *et al* 2022)

Reference: [Ding *et al* (2022)](https://ui.adsabs.harvard.edu/abs/2022ApJS..260...45D/abstract)

In [247]:
ding2022_dir = os.path.join(data_dir, 'lamost_ding2022_mstars')

! mkdir {ding2022_dir}

mkdir: cannot create directory ‘data/lamost_ding2022_mstars’: File exists


In [249]:
fname = os.path.join(ding2022_dir, 'tableb1.dat.gz')
readme_fn = os.path.join(ding2022_dir, 'readme.txt')

In [964]:
# url = 'https://cdsarc.cds.unistra.fr/viz-bin/nph-Cat/fits.gz?J/ApJS/260/45/tableb1.dat.gz'
url = 'https://cdsarc.cds.unistra.fr/ftp/J/ApJS/260/45/tableb1.dat.gz'
# fname = os.path.join(data_dir, 'lamost_ding2022_mstars', 'lamost_ding2022_mstars.dat.gz')

! wget -O {fname} {url}
! gunzip {fname}

readme_url = 'https://cdsarc.cds.unistra.fr/ftp/J/ApJS/260/45/ReadMe'
# readme_fn = os.path.join(data_dir, 'lamost_ding2022_mstars_readme.txt')

! wget -O {readme_fn} {readme_url}

--2025-08-06 15:58:49--  https://cdsarc.cds.unistra.fr/ftp/J/ApJS/260/45/tableb1.dat.gz
Resolving cdsarc.cds.unistra.fr (cdsarc.cds.unistra.fr)... 130.79.128.5
Connecting to cdsarc.cds.unistra.fr (cdsarc.cds.unistra.fr)|130.79.128.5|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 35910285 (34M) [application/x-gzip]
Saving to: ‘data/lamost_ding2022_mstars/tableb1.dat.gz’

data/lamost_ding202 100%[===================>]  34.25M  9.73MB/s    in 5.7s    

2025-08-06 15:58:55 (6.02 MB/s) - ‘data/lamost_ding2022_mstars/tableb1.dat.gz’ saved [35910285/35910285]

--2025-08-06 15:58:55--  https://cdsarc.cds.unistra.fr/ftp/J/ApJS/260/45/ReadMe
Resolving cdsarc.cds.unistra.fr (cdsarc.cds.unistra.fr)... 130.79.128.5
Connecting to cdsarc.cds.unistra.fr (cdsarc.cds.unistra.fr)|130.79.128.5|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8790 (8.6K)
Saving to: ‘data/lamost_ding2022_mstars/readme.txt’

data/lamost_ding202 100%[===================>]   

In [250]:
data_fn = fname = os.path.join(ding2022_dir, 'tableb1.dat')
# data_fn = os.path.join(data_dir, 'lamost_ding2022_mstars.dat.gz')
# readme_fn = os.path.join(data_dir, 'lamost_ding2022_mstars_readme.txt')
t_lamost_mstar_ding = Table.read(fname, format='cds', readme=readme_fn)

Fill in uncertainties with typical values cited in paper:

In [251]:
n = len(t_lamost_mstar_ding)
t_lamost_mstar_ding['e_Teff'] = np.ones(n) * 45. * units.Kelvin
t_lamost_mstar_ding['e_log(g)'] = np.ones(n) * 0.25
t_lamost_mstar_ding['e_[Fe/H]'] = np.ones(n) * 0.22

Corrections (compared to APOGEE DR17) from paper:

In [252]:
idx_dwarf = (t_lamost_mstar_ding['log(g)'] > 3.)
t_lamost_mstar_ding['Teff'][idx_dwarf] += 176
t_lamost_mstar_ding['log(g)'][idx_dwarf] -= 0.16
t_lamost_mstar_ding['[Fe/H]'][idx_dwarf] += 0.14

In [253]:
t_lamost_mstar_ding['Teff'][~idx_dwarf] += 140
t_lamost_mstar_ding['log(g)'][~idx_dwarf] -= 0.3
t_lamost_mstar_ding['[Fe/H]'][~idx_dwarf] -= 0.18

In [254]:
t_lamost_mstar_ding

LAMOST,ObsID,Gaia,RAh,RAm,RAs,DE-,DEd,DEm,DEs,SNR,RVel,Teff,log(g),[Fe/H],e_Teff,e_log(g),e_[Fe/H]
,,,h,min,s,,deg,arcmin,arcsec,,km / s,K,dex(cm / s2),dex(Sun),K,,
str19,int64,int64,int64,int64,float64,str1,int64,int64,float64,float64,float64,int64,float64,float64,float64,float64,float64
J222404.16+070154.4,364808049,2710178308565988224,22,24,4.16,+,7,1,54.4,11.55,-42.2,4065,4.43,0.0,45.0,0.25,0.22
J042312.45-022407.4,372709009,3205768026438470016,4,23,12.45,-,2,24,7.5,14.34,24.7,4081,4.5,-0.09,45.0,0.25,0.22
J000153.40+214725.7,385915010,2847144991728471552,0,1,53.41,+,21,47,25.8,6.58,-4.9,3455,4.82,-0.32,45.0,0.25,0.22
J043426.92-061836.4,386707008,3198668784797816064,4,34,26.92,-,6,18,36.5,10.39,25.7,3721,4.74,-0.27999999999999997,45.0,0.25,0.22
J115335.08+325649.4,399009146,4027678050808261632,11,53,35.08,+,32,56,49.4,23.49,90.5,3846,4.6899999999999995,-0.32,45.0,0.25,0.22
J111714.68+064813.1,401116130,3817663564601629184,11,17,14.69,+,6,48,13.2,16.15,8.2,3902,4.6,-0.14999999999999997,45.0,0.25,0.22
J104606.24+042813.7,401711181,3858024261340316928,10,46,6.25,+,4,28,13.7,12.46,-28.4,3813,4.6899999999999995,-0.19,45.0,0.25,0.22
J093822.80+380822.9,402111026,800615477736067072,9,38,22.8,+,38,8,23.0,13.44,-61.6,3838,4.67,-0.32999999999999996,45.0,0.25,0.22


In [255]:
plt.hist2d(
    t_lamost_mstar_ding['Teff'],
    t_lamost_mstar_ding['log(g)'],
    norm=LogNorm(),
    bins=100
)
plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.xlabel(r'$T_{\rm eff}$')
plt.ylabel(r'$\log g$')
plt.show()

<Figure size 640x480 with 1 Axes>

In [256]:
idx = (
    (t_lamost_mstar_ding['Teff'] > 2900.)
  & (t_lamost_mstar_ding['Teff'] < 4200.)
  & (t_lamost_mstar_ding['SNR'] > 20.)
  & (
        (t_lamost_mstar_ding['log(g)'] < 2.)
      | (t_lamost_mstar_ding['log(g)'] > 4.)
    )
  & (t_lamost_mstar_ding['log(g)'] < 5.2)
    # & (t_lamost_mstar_ding['log(g)'] > 3.) # Remove giants
)
print(f'{np.count_nonzero(idx):,} ({np.mean(idx)*100:.2f}%) pass cut.')

# plt.hist2d(
#     t_lamost_mstar_ding['Teff'][idx],
#     t_lamost_mstar_ding['log(g)'][idx],
#     norm=LogNorm(),
#     bins=100
# )
# plt.gca().invert_xaxis()
# plt.gca().invert_yaxis()
# plt.xlabel(r'$T_{\rm eff}$')
# plt.ylabel(r'$\log g$')
# plt.show()

plot_teff_logg_feh(
    t_lamost_mstar_ding['Teff'][idx],
    t_lamost_mstar_ding['log(g)'][idx],
    t_lamost_mstar_ding['[Fe/H]'][idx]
)
plt.show()

311,658 (40.84%) pass cut.


<Figure size 800x400 with 2 Axes>

In [257]:
id_col = 'Gaia'
param_cols = ('Teff', '[Fe/H]', 'log(g)')
param_unc_cols = ('e_Teff', 'e_[Fe/H]', 'e_log(g)')
extra_cols = ()

t_lmd = extract_table(
    t_lamost_mstar_ding[idx],
    id_col, param_cols, param_unc_cols, extra_cols,
    param_transf=param_transf,
    param_unc_transf=param_unc_transf,
    param_errs=[0.2,0.2,0.4]
)
t_lmd

id,x,sigma_x,x_err
int64,float64[3],float64[3],float32[3]
4027678050808261632,-1.312021326337562 .. 4.6899999999999995,0.05850234009360374 .. 0.25,0.2 .. 0.4
1004395733012803840,-1.270591778068202 .. 4.55,0.05801959773078907 .. 0.25,0.2 .. 0.4
1456740863993609472,-1.0425063728866988 .. 4.53,0.05543237250554324 .. 0.25,0.2 .. 0.4
1173578961566610560,-1.0216089853681096 .. 4.54,0.055201177625122666 .. 0.25,0.2 .. 0.4
951643123616651520,-1.192555142226658 .. 4.56,0.05712109672505712 .. 0.25,0.2 .. 0.4
214760596867892096,-1.0932682302067886 .. 4.45,0.05599800895968143 .. 0.25,0.2 .. 0.4
1565834923096348288,-1.0351208471716327 .. 4.47,0.055350553505535055 .. 0.25,0.2 .. 0.4
232568154051112192,-1.2809170269620496 .. 4.58,0.05813953488372093 .. 0.25,0.2 .. 0.4
1240020696888912128,-1.6745672607860982 .. 4.6899999999999995,0.06290187307799833 .. 0.25,0.2 .. 0.4


In [258]:
plot_catalog_uncertainties(t_lmd)
plot_catalog_params(t_lmd)

<Figure size 800x280 with 4 Axes>

<Figure size 500x300 with 1 Axes>

<Figure size 500x400 with 1 Axes>

In [259]:
# Save catalog
fn = 'training_data/lamost_mstar_ding2022.h5'
t_lmd.write(fn, serialize_metadata=True, overwrite=True)
! du -sh {fn}

/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


21M	training_data/lamost_mstar_ding2022.h5


### VMP Stars (Li *et al* 2022)

355 very metal-poor stars, observed with Subaru (with targeting data from LAMOST).

Release paper: [Li *et al* (2022)](https://ui.adsabs.harvard.edu/abs/2022ApJ...931..147L/abstract)

Survey design paper: [Aoki *et al* (2022)](https://ui.adsabs.harvard.edu/abs/2022ApJ...931..146A/abstract)

In [266]:
li2022_vmp_dir = os.path.join(data_dir, 'li2022_vmp')

! mkdir {li2022_vmp_dir}

fn_table2 = os.path.join(li2022_vmp_dir, 'table2.dat')
# fn_table3 = os.path.join(li2022_vmp_dir, 'table3.dat')
fn_readme = os.path.join(li2022_vmp_dir, 'readme.txt')

mkdir: cannot create directory ‘data/li2022_vmp’: File exists


In [1002]:
url_table2 = 'https://cdsarc.cds.unistra.fr/ftp/J/ApJ/931/147/table2.dat'
# url_table3 = 'https://cdsarc.cds.unistra.fr/ftp/J/ApJ/931/147/table3.dat'
url_readme = 'https://cdsarc.cds.unistra.fr/ftp/J/ApJ/931/147/ReadMe'

! wget -O {fn_table2} {url_table2}
# ! wget -O {fn_table3} {url_table3}
! wget -O {fn_readme} {url_readme}

--2025-08-06 18:46:14--  https://cdsarc.cds.unistra.fr/ftp/J/ApJ/931/147/table2.dat
Resolving cdsarc.cds.unistra.fr (cdsarc.cds.unistra.fr)... 130.79.128.5
Connecting to cdsarc.cds.unistra.fr (cdsarc.cds.unistra.fr)|130.79.128.5|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 45045 (44K)
Saving to: ‘data/li2022_vmp/table2.dat’

data/li2022_vmp/tab 100%[===================>]  43.99K  --.-KB/s    in 0.04s   

2025-08-06 18:46:14 (1.15 MB/s) - ‘data/li2022_vmp/table2.dat’ saved [45045/45045]

--2025-08-06 18:46:15--  https://cdsarc.cds.unistra.fr/ftp/J/ApJ/931/147/ReadMe
Resolving cdsarc.cds.unistra.fr (cdsarc.cds.unistra.fr)... 130.79.128.5
Connecting to cdsarc.cds.unistra.fr (cdsarc.cds.unistra.fr)|130.79.128.5|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12829 (13K)
Saving to: ‘data/li2022_vmp/readme.txt’

data/li2022_vmp/rea 100%[===================>]  12.53K  --.-KB/s    in 0s      

2025-08-06 18:46:15 (26.8 MB/s) - ‘data/li2022

In [267]:
t_table2 = Table.read(fn_table2, format='cds', readme=fn_readme)
# t_table3 = Table.read(fn_table3, format='cds', readme=fn_readme)

In [268]:
t_table2

ID,RAdeg,DEdeg,RVel,SNR,Teff,e_Teff,logg,e_logg,FeH,e_FeH,vt,e_vt,logLum,e_logLum,f_logg,Teff-L,logg-L,FeH-L
,deg,deg,km / s,,K,K,dex(cm / s2),dex(cm / s2),dex(---),dex(---),km / s,km / s,dex(Lsun),dex(Lsun),,K,dex(cm / s2),dex(---)
str10,float64,float64,float64,float64,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64,float64,float64
J0002+0343,0.645994,3.727085,-91.4,23.7,6207,110,4.16,0.05,-3.02,0.24,1.19,0.3,0.307,0.06,0,6361,4.3,-3.36
J0003+1556,0.795383,15.933828,-185.5,32.2,5302,45,2.31,0.08,-2.23,0.11,2.11,0.12,1.883,0.081,0,5289,1.09,-2.62
J0006+0123,1.657593,1.395354,-65.9,44.1,5675,53,3.23,0.04,-2.27,0.11,1.52,0.08,1.081,0.043,0,5500,2.52,-2.58
J0006+1057,1.571676,10.961624,-312.4,20.9,4691,38,1.63,0.14,-3.15,0.14,2.51,0.11,2.35,0.14,1,5012,1.72,-3.15
J0013+2350,3.463742,23.846972,-259.2,41.4,6093,120,3.98,0.04,-2.26,0.09,1.49,0.1,0.455,0.054,0,6238,3.91,-2.2
J0022+4254,5.510848,42.9165,-183.3,23.2,6566,133,3.98,0.06,-2.99,0.25,0.96,0.3,0.585,0.072,0,6250,4.04,-3.36
J0023+2023,5.795711,20.392651,-126.0,37.3,5197,68,2.79,0.05,-3.29,0.09,1.53,0.11,1.368,0.054,0,5250,2.55,-3.09
J0023+3558,5.916771,35.967957,-303.0,38.0,5145,82,3.2,0.06,-2.56,0.09,1.35,0.08,0.941,0.065,0,4960,3.36,-3.07


In [1009]:
Gaia.login()

INFO: Login to gaia TAP server [astroquery.gaia.core]


User:  ggreen
Password:  ········


INFO: OK [astroquery.utils.tap.core]
INFO: Login to gaia data server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]


In [1025]:
crossmatch_info = gaia_crossmatch(t_table2['RAdeg'], t_table2['DEdeg'])

INFO: Sending pytable. [astroquery.utils.tap.core]
INFO: Uploaded table 'crossmatch_table'. [astroquery.utils.tap.core]
Launched query: 'SELECT
        t.index,
        g.source_id, g.ra, g.dec,
        DISTANCE(
          POINT(t.ra, t.dec),
          POINT(g.ra, g.dec)
        ) * 3600. AS angsep_arcsec
      FROM user_ggreen.crossmatch_table AS t
      JOIN gaiadr3.gaia_source_lite AS g
      ON DISTANCE(POINT(t.ra, t.dec), POINT(g.ra, g.dec)) < 1. / 3600.'
------>https
host = gea.esac.esa.int:443
context = /tap-server/tap/async
Content-type = application/x-www-form-urlencoded
303 303
[('Date', 'Wed, 06 Aug 2025 17:14:27 GMT'), ('Server', 'Apache/2.4.6 (SLES Expanded Support platform 7) OpenSSL/1.0.2k-fips mod_jk/1.2.43'), ('X-VO-Authenticated', 'ggreen'), ('Location', 'https://gea.esac.esa.int/tap-server/tap/async/1754500467822O'), ('Cache-Control', 'no-cache, no-store, max-age=0, must-revalidate'), ('Pragma', 'no-cache'), ('Expires', '0'), ('X-XSS-Protection', '1; mode=block'), ('

<Figure size 640x480 with 1 Axes>

In [1026]:
d_li2022_vmp = apply_crossmatch(crossmatch_info, t_table2, max_separation=1.*units.arcsec)

In [1027]:
d_li2022_vmp

ID,RAdeg,DEdeg,RVel,SNR,Teff,e_Teff,logg,e_logg,FeH,e_FeH,vt,e_vt,logLum,e_logLum,f_logg,Teff-L,logg-L,FeH-L,gdr3_source_id,ra_gdr3,dec_gdr3
,deg,deg,km / s,,K,K,dex(cm / s2),dex(cm / s2),dex(---),dex(---),km / s,km / s,dex(Lsun),dex(Lsun),,K,dex(cm / s2),dex(---),,deg,deg
str10,float64,float64,float64,float64,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64,float64,float64,int64,float64,float64
J0002+0343,0.645994,3.727085,-91.4,23.7,6207,110,4.16,0.05,-3.02,0.24,1.19,0.3,0.307,0.06,0,6361,4.3,-3.36,2740128975242655232,0.6460112878887468,3.726998612336613
J0003+1556,0.795383,15.933828,-185.5,32.2,5302,45,2.31,0.08,-2.23,0.11,2.11,0.12,1.883,0.081,0,5289,1.09,-2.62,2772216224937941760,0.79543263790485,15.933848966913922
J0006+0123,1.657593,1.395354,-65.9,44.1,5675,53,3.23,0.04,-2.27,0.11,1.52,0.08,1.081,0.043,0,5500,2.52,-2.58,2546607403970841472,1.6576733131731816,1.3952667138855257
J0006+1057,1.571676,10.961624,-312.4,20.9,4691,38,1.63,0.14,-3.15,0.14,2.51,0.11,2.35,0.14,1,5012,1.72,-3.15,2765776046033609856,1.571690650335079,10.961594362906721
J0022+4254,5.510848,42.9165,-183.3,23.2,6566,133,3.98,0.06,-2.99,0.25,0.96,0.3,0.585,0.072,0,6250,4.04,-3.36,385205965989661568,5.510893983005021,42.916507942182456
J0023+2023,5.795711,20.392651,-126.0,37.3,5197,68,2.79,0.05,-3.29,0.09,1.53,0.11,1.368,0.054,0,5250,2.55,-3.09,2799103583500062464,5.795697792317216,20.392571588115533
J0023+3558,5.916771,35.967957,-303.0,38.0,5145,82,3.2,0.06,-2.56,0.09,1.35,0.08,0.941,0.065,0,4960,3.36,-3.07,366303780558169344,5.916803257475361,35.967842331798394
J0032+4107,8.214463,41.130272,-396.8,43.6,5074,73,2.16,0.09,-2.69,0.11,1.83,0.08,1.957,0.093,0,5198,2.14,-2.69,381063654713703040,8.214435542737068,41.13027021216643


In [1029]:
fn = os.path.join(data_dir, 'Li_etal2022_VMP_GaiaXMatch.fits')
d_li2022_vmp.write(fn, overwrite=True)

In [269]:
fn = os.path.join(data_dir, 'Li_etal2022_VMP_GaiaXMatch.fits')
d_li2022_vmp = Table.read(fn)
d_li2022_vmp

ID,RAdeg,DEdeg,RVel,SNR,Teff,e_Teff,logg,e_logg,FeH,e_FeH,vt,e_vt,logLum,e_logLum,f_logg,Teff-L,logg-L,FeH-L,gdr3_source_id,ra_gdr3,dec_gdr3
,deg,deg,km / s,,K,K,dex(cm / s2),dex(cm / s2),dex(---),dex(---),km / s,km / s,dex(Lsun),dex(Lsun),,K,dex(cm / s2),dex(---),,deg,deg
bytes10,float64,float64,float64,float64,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64,float64,float64,int64,float64,float64
J0002+0343,0.645994,3.727085,-91.4,23.7,6207,110,4.16,0.05,-3.02,0.24,1.19,0.3,0.307,0.06,0,6361,4.3,-3.36,2740128975242655232,0.6460112878887468,3.726998612336613
J0003+1556,0.795383,15.933828,-185.5,32.2,5302,45,2.31,0.08,-2.23,0.11,2.11,0.12,1.883,0.081,0,5289,1.09,-2.62,2772216224937941760,0.79543263790485,15.933848966913922
J0006+0123,1.657593,1.395354,-65.9,44.1,5675,53,3.23,0.04,-2.27,0.11,1.52,0.08,1.081,0.043,0,5500,2.52,-2.58,2546607403970841472,1.6576733131731816,1.3952667138855257
J0006+1057,1.571676,10.961624,-312.4,20.9,4691,38,1.63,0.14,-3.15,0.14,2.51,0.11,2.35,0.14,1,5012,1.72,-3.15,2765776046033609856,1.571690650335079,10.961594362906721
J0022+4254,5.510848,42.9165,-183.3,23.2,6566,133,3.98,0.06,-2.99,0.25,0.96,0.3,0.585,0.072,0,6250,4.04,-3.36,385205965989661568,5.510893983005021,42.916507942182456
J0023+2023,5.795711,20.392651,-126.0,37.3,5197,68,2.79,0.05,-3.29,0.09,1.53,0.11,1.368,0.054,0,5250,2.55,-3.09,2799103583500062464,5.795697792317216,20.392571588115533
J0023+3558,5.916771,35.967957,-303.0,38.0,5145,82,3.2,0.06,-2.56,0.09,1.35,0.08,0.941,0.065,0,4960,3.36,-3.07,366303780558169344,5.916803257475361,35.967842331798394
J0032+4107,8.214463,41.130272,-396.8,43.6,5074,73,2.16,0.09,-2.69,0.11,1.83,0.08,1.957,0.093,0,5198,2.14,-2.69,381063654713703040,8.214435542737068,41.13027021216643


In [270]:
plt.hist(d_li2022_vmp['FeH'])
plt.xlabel(r'$\left[\mathrm{Fe/H}\right]$')
plt.show()

<Figure size 640x480 with 1 Axes>

In [271]:
plt.hist(d_li2022_vmp['SNR'], bins=np.arange(0.,160.001,5.))
plt.xlabel('SNR')
plt.show()

<Figure size 640x480 with 1 Axes>

In [272]:
d_li2022_vmp.colnames

['ID',
 'RAdeg',
 'DEdeg',
 'RVel',
 'SNR',
 'Teff',
 'e_Teff',
 'logg',
 'e_logg',
 'FeH',
 'e_FeH',
 'vt',
 'e_vt',
 'logLum',
 'e_logLum',
 'f_logg',
 'Teff-L',
 'logg-L',
 'FeH-L',
 'gdr3_source_id',
 'ra_gdr3',
 'dec_gdr3']

In [273]:
fig,ax_arr = plt.subplots(1,3, figsize=(9,3), layout='constrained')
for ax,p in zip(ax_arr,('Teff','FeH','logg')):
    ax.scatter(
        d_li2022_vmp[p],
        d_li2022_vmp[f'{p}-L'],
        s=9,
        edgecolors='none',
        alpha=0.7
    )
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    xlim = [min(xlim[0], ylim[0]), max(xlim[1], ylim[1])]
    ax.plot(xlim, xlim, c='k', alpha=0.5)
    ax.set_xlim(xlim)
    ax.set_ylim(xlim)
    ax.set_xlabel(f'{p} (Li+2022)')
    ax.set_ylabel(f'{p} (LAMOST)')

<Figure size 900x300 with 3 Axes>

In [274]:
plot_teff_logg_feh(
    d_li2022_vmp['Teff'],
    d_li2022_vmp['logg'],
    d_li2022_vmp['FeH']
)
plt.show()

<Figure size 800x400 with 2 Axes>

In [275]:
id_col = 'gdr3_source_id'
param_cols = ('Teff', 'FeH', 'logg')
param_unc_cols = ('e_Teff', 'e_FeH', 'e_logg')
extra_cols = ('ra_gdr3', 'dec_gdr3')

t_vmp = extract_table(
    d_li2022_vmp,
    id_col, param_cols, param_unc_cols, extra_cols,
    param_transf=param_transf,
    param_unc_transf=apply_err_floors(param_unc_transf,err_floor_default[:-1]),
    param_errs=[0.1,0.05,0.1] # Trust [Fe/H] more than for other surveys
)
t_vmp

id,x,sigma_x,x_err,ra_gdr3,dec_gdr3
,,,,deg,deg
int64,float64[3],float64[3],float32[3],float64,float64
2740128975242655232,1.0811988749870767 .. 4.16,0.08917212169687189 .. 0.06403124237432849,0.1 .. 0.1,0.6460112878887468,3.726998612336613
2772216224937941760,0.29323097716366714 .. 2.31,0.04359912129066558 .. 0.08944271909999159,0.1 .. 0.1,0.79543263790485,15.933848966913922
2546607403970841472,0.63316325466683 .. 3.23,0.04775478727153184 .. 0.0565685424949238,0.1 .. 0.1,1.6576733131731816,1.3952667138855257
2765776046033609856,-0.31896066543889684 .. 1.63,0.0417193046755243 .. 0.14560219778561037,0.1 .. 0.1,1.571690650335079,10.961594362906721
385205965989661568,1.3623345332265024 .. 3.98,0.10177180450997254 .. 0.07211102550927978,0.1 .. 0.1,5.510893983005021,42.916507942182456
2799103583500062464,0.19321811796102265 .. 2.79,0.0661822110849955 .. 0.06403124237432849,0.1 .. 0.1,5.795697792317216,20.392571588115533
366303780558169344,0.14293728425956237 .. 3.2,0.08031400664790664 .. 0.07211102550927978,0.1 .. 0.1,5.916803257475361,35.967842331798394
381063654713703040,0.07345774371494841 .. 2.16,0.07262709925716966 .. 0.09848857801796104,0.1 .. 0.1,8.214435542737068,41.13027021216643


In [276]:
plot_catalog_params(t_vmp)
plot_catalog_uncertainties(t_vmp)

<Figure size 500x400 with 1 Axes>

<Figure size 800x280 with 4 Axes>

<Figure size 500x300 with 1 Axes>

In [277]:
# Save catalog
fn = 'training_data/vmp_stars_li2022.h5'
t_vmp.write(fn, serialize_metadata=True, overwrite=True)
! du -sh {fn}

/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


27K	training_data/vmp_stars_li2022.h5


### SAGA VMP Stars

In [820]:
url = 'http://sagadatabase.jp/Data/txt_table_all.tsv'
fn = os.path.join(data_dir, 'SAGA_VMP_stars.tsv')

! wget -O {fn} {url}

--2025-06-08 10:36:49--  http://sagadatabase.jp/Data/txt_table_all.tsv
153.126.128.103atabase.jp (sagadatabase.jp)... 
Connecting to sagadatabase.jp (sagadatabase.jp)|153.126.128.103|:80... connected.
200 OKequest sent, awaiting response... 
Length: 16706564 (16M) [text/tab-separated-values]
Saving to: ‘/data/beegfs/astro-storage/groups/green/green/projects/gaia_Bp_Rp_reddening_variation/data/SAGA_VMP_stars.tsv’

/data/beegfs/astro- 100%[===================>]  15.93M  29.3MB/s    in 0.5s    

2025-06-08 10:36:55 (29.3 MB/s) - ‘/data/beegfs/astro-storage/groups/green/green/projects/gaia_Bp_Rp_reddening_variation/data/SAGA_VMP_stars.tsv’ saved [16706564/16706564]



In [836]:
t_saga_vmp = Table.read(fn, format='csv', delimiter='\t',
                        header_start=0, data_start=1,
                        encoding='cp1252',
                        guess=False, fast_reader=False)
t_saga_vmp

F Author,Reference,Object,RA (2000),Decl (2000),Teff,log g,[M/H],xsi,Binarity,Period,Vrad,Facility,Resolution,SN,Solar Abundance,A(Li),d(Li),f(Li),A(Li I),d(Li I),f(Li I),[Be/H],d(Be),f(Be),[Be I/H],d(Be I),f(Be I),[Be II/H],d(Be II),f(Be II),[B I/H],d(B I),f(B I),[C/H],d(C),f(C),[C2/H],d(C2),f(C2),[CH/H],d(CH),f(CH),[C I/H],d(C I),f(C I),[N/H],d(N),f(N),[CN/H],d(CN),f(CN),[NH/H],d(NH),f(NH),[O/H],d(O),f(O),[O I/H],d(O I),f(O I),[[O I]/H],d([O I]),f([O I]),[OH/H],d(OH),f(OH),[HF/H],d(HF),f(HF),[Na/H],d(Na),f(Na),[Na I/H],d(Na I),f(Na I),[Mg/H],d(Mg),f(Mg),[Mg I/H],d(Mg I),f(Mg I),[Mg II/H],d(Mg II),f(Mg II),[Al/H],d(Al),f(Al),[Al I/H],d(Al I),f(Al I),[Si/H],d(Si),f(Si),[Si I/H],d(Si I),f(Si I),[Si II/H],d(Si II),f(Si II),[P I/H],d(P I),f(P I),[S/H],d(S),f(S),[S I/H],d(S I),f(S I),[[S I]/H],d([S I]),f([S I]),[K/H],d(K),f(K),[K I/H],d(K I),f(K I),[Ca/H],d(Ca),f(Ca),[Ca I/H],d(Ca I),f(Ca I),[Ca II/H],d(Ca II),f(Ca II),[Sc/H],d(Sc),f(Sc),[Sc I/H],d(Sc I),f(Sc I),[Sc II/H],d(Sc II),f(Sc II),[Ti/H],d(Ti),f(Ti),[Ti I/H],d(Ti I),f(Ti I),[Ti II/H],d(Ti II),f(Ti II),[V/H],d(V),f(V),[V I/H],d(V I),f(V I),[V II/H],d(V II),f(V II),[Cr/H],d(Cr),f(Cr),[Cr I/H],d(Cr I),f(Cr I),[Cr II/H],d(Cr II),f(Cr II),[Mn/H],d(Mn),f(Mn),[Mn I/H],d(Mn I),f(Mn I),[Mn II/H],d(Mn II),f(Mn II),[Fe/H],d(Fe),f(Fe),[Fe I/H],d(Fe I),f(Fe I),[Fe II/H],d(Fe II),f(Fe II),[Co/H],d(Co),f(Co),[Co I/H],d(Co I),f(Co I),[Co II/H],d(Co II),f(Co II),[Ni/H],d(Ni),f(Ni),[Ni I/H],d(Ni I),f(Ni I),[Ni II/H],d(Ni II),f(Ni II),[Cu/H],d(Cu),f(Cu),[Cu I/H],d(Cu I),f(Cu I),[Cu II/H],d(Cu II),f(Cu II),[Zn/H],d(Zn),f(Zn),[Zn I/H],d(Zn I),f(Zn I),[Zn II/H],d(Zn II),f(Zn II),[Ga/H],d(Ga),f(Ga),[Ga I/H],d(Ga I),f(Ga I),[Ge/H],d(Ge),f(Ge),[Ge I/H],d(Ge I),f(Ge I),[As I/H],d(As I),f(As I),[Se I/H],d(Se I),f(Se I),[Rb I/H],d(Rb I),f(Rb I),[Sr/H],d(Sr),f(Sr),[Sr I/H],d(Sr I),f(Sr I),[Sr II/H],d(Sr II),f(Sr II),[Y/H],d(Y),f(Y),[Y II/H],d(Y II),f(Y II),[Zr/H],d(Zr),f(Zr),[Zr I/H],d(Zr I),f(Zr I),[Zr II/H],d(Zr II),f(Zr II),[Nb/H],d(Nb),f(Nb),[Nb I/H],d(Nb I),f(Nb I),[Nb II/H],d(Nb II),f(Nb II),[Mo/H],d(Mo),f(Mo),[Mo I/H],d(Mo I),f(Mo I),[Mo II/H],d(Mo II),f(Mo II),[Tc I/H],d(Tc I),f(Tc I),[Ru/H],d(Ru),f(Ru),[Ru I/H],d(Ru I),f(Ru I),[Rh/H],d(Rh),f(Rh),[Rh I/H],d(Rh I),f(Rh I),[Pd/H],d(Pd),f(Pd),[Pd I/H],d(Pd I),f(Pd I),[Ag/H],d(Ag),f(Ag),[Ag I/H],d(Ag I),f(Ag I),[Cd I/H],d(Cd I),f(Cd I),[Cd II/H],d(Cd II),f(Cd II),[Sn I/H],d(Sn I),f(Sn I),[Te/H],d(Te),f(Te),[Te I/H],d(Te I),f(Te I),[Ba/H],d(Ba),f(Ba),[Ba II/H],d(Ba II),f(Ba II),[La/H],d(La),f(La),[La I/H],d(La I),f(La I),[La II/H],d(La II),f(La II),[Ce/H],d(Ce),f(Ce),[Ce II/H],d(Ce II),f(Ce II),[Pr/H],d(Pr),f(Pr),[Pr II/H],d(Pr II),f(Pr II),[Nd/H],d(Nd),f(Nd),[Nd II/H],d(Nd II),f(Nd II),[Sm/H],d(Sm),f(Sm),[Sm II/H],d(Sm II),f(Sm II),[Eu/H],d(Eu),f(Eu),[Eu I/H],d(Eu I),f(Eu I),[Eu II/H],d(Eu II),f(Eu II),[Gd/H],d(Gd),f(Gd),[Gd I/H],d(Gd I),f(Gd I),[Gd II/H],d(Gd II),f(Gd II),[Tb/H],d(Tb),f(Tb),[Tb II/H],d(Tb II),f(Tb II),[Dy/H],d(Dy),f(Dy),[Dy II/H],d(Dy II),f(Dy II),[Ho/H],d(Ho),f(Ho),[Ho II/H],d(Ho II),f(Ho II),[Er/H],d(Er),f(Er),[Er II/H],d(Er II),f(Er II),[Tm/H],d(Tm),f(Tm),[Tm II/H],d(Tm II),f(Tm II),[Yb/H],d(Yb),f(Yb),[Yb II/H],d(Yb II),f(Yb II),[Lu II/H],d(Lu II),f(Lu II),[Hf/H],d(Hf),f(Hf),[Hf II/H],d(Hf II),f(Hf II),[W I/H],d(W I),f(W I),[W II/H],d(W II),f(W II),[Os/H],d(Os),f(Os),[Os I/H],d(Os I),f(Os I),[Os II/H],d(Os II),f(Os II),[Ir/H],d(Ir),f(Ir),[Ir I/H],d(Ir I),f(Ir I),[Pt/H],d(Pt),f(Pt),[Pt I/H],d(Pt I),f(Pt I),[Au I/H],d(Au I),f(Au I),[Hg II/H],d(Hg II),f(Hg II),[Pb/H],d(Pb),f(Pb),[Pb I/H],d(Pb I),f(Pb I),[Pb II/H],d(Pb II),f(Pb II),[Bi I/H],d(Bi I),f(Bi I),[Th/H],d(Th),f(Th),[Th II/H],d(Th II),f(Th II),[U/H],d(U),f(U),[U II/H],d(U II),f(U II),12C/13C,d(12C/13C),f(12C/13C),142Nd/144Nd,d(142Nd/144Nd),f(142Nd/144Nd),151Eu/153Eu,d(151Eu/153Eu),f(151Eu/153Eu),152Sm/154Sm,d(152Sm/154Sm),f(152Sm/154Sm),6Li/7Li,d(6Li/7Li),f(6Li/7Li),V,d(V)_1,B,d(B),B-V,d(B-V),U-B,d(U-B),V-R,d(V-R),J-H,d(J-H),H-K,d(H-K),J-K,d(J-K),V-I,d(V-I),V-J,d(V-J),V-

Convert coordinates to SkyCoord object:

In [923]:
idx_hascoords = ~t_saga_vmp['RA (2000)'].mask.copy()
print(f'{np.mean(~idx_hascoords):.2%} of stars are missing coordinates.')

5.43% of stars are missing coordinates.


Detect nonsensical (RA, Dec) entries:

In [924]:
idx = np.where(idx_hascoords)[0]
for i,(s_ra,s_dec) in enumerate(zip(t_saga_vmp['RA (2000)'][idx_hascoords],
                                    t_saga_vmp['Decl (2000)'][idx_hascoords])):
    sec = float(s_ra.split()[2])
    if sec >= 60:
        print(f'Nonsensical RA: {s_ra}')
        idx_hascoords[idx[i]] = False

    try:
        sec = float(s_dec.split()[2])
    except:
        print(f"Can't read Dec: {s_dec}")
        print(t_saga_vmp['Object'][idx[i]])
        idx_hascoords[idx[i]] = False
        continue
    
    if sec >= 60:
        print(f'Nonsensical Dec: {s_dec}')
        idx_hascoords[idx[i]] = False

Nonsensical Dec: -71 22 60.0
Nonsensical RA: 19 21 736.55
Can't read Dec: ++  35006.49
HD30443
Nonsensical RA: 14 52 310.03
Nonsensical Dec: -30 31 60


In [925]:
from astropy.coordinates import SkyCoord
from astropy.coordinates.name_resolve import NameResolveError

Make an attempt to look up missing coordinates, by querying their names:

In [926]:
from astropy.coordinates import SkyCoord
from astropy.coordinates.name_resolve import NameResolveError

missing_coords = []
for s in tqdm(t_saga_vmp['Object'][~idx_hascoords]):
    try:
        missing_coords.append(SkyCoord.from_name(s))
    except NameResolveError:
        missing_coords.append(None)

  0%|          | 0/1123 [00:00<?, ?it/s]

In [927]:
idx_nocoords = np.where(~idx_hascoords)[0]
idx_foundcoords = []
ra_found,dec_found = [], []

for i,c in enumerate(missing_coords):
    if c is not None:
        idx_foundcoords.append(idx_nocoords[i])
        ra_found.append(c.ra)
        dec_found.append(c.dec)

idx_foundcoords = np.array(idx_foundcoords)

In [928]:
import astropy.units as units

ra = np.full(len(t_saga_vmp), np.nan) * units.deg
dec = np.full(len(t_saga_vmp), np.nan) * units.deg

c_hascoords = SkyCoord(
    t_saga_vmp['RA (2000)'][idx_hascoords],
    t_saga_vmp['Decl (2000)'][idx_hascoords],
    unit='deg',
    frame='icrs'
)
ra[idx_hascoords] = c_hascoords.ra
dec[idx_hascoords] = c_hascoords.dec

ra[idx_foundcoords] = ra_found
dec[idx_foundcoords] = dec_found

coords_saga_vmp = SkyCoord(ra, dec, frame='icrs')
coords_saga_vmp

<SkyCoord (ICRS): (ra, dec) in deg
    [( 16.72056389,  -1.92511111), (  1.01472222,  -2.47222222),
     ( 22.28379444, -16.65752778), ..., (168.63817198,  25.71038621),
     ( 24.19933977,  41.40545871), (170.26050434,   7.06443629)]>

In [929]:
t_saga_vmp['coordinates'] = coords_saga_vmp

Cross-match with Gaia DR3:

In [951]:
Gaia.login()

INFO:astroquery:Login to gaia TAP server


INFO: Login to gaia TAP server [astroquery.gaia.core]


User:  ggreen
Password:  ········


INFO:astroquery:OK
INFO:astroquery:Login to gaia data server


INFO: OK [astroquery.utils.tap.core]
INFO: Login to gaia data server [astroquery.gaia.core]


INFO:astroquery:OK


INFO: OK [astroquery.utils.tap.core]


In [952]:
crossmatch_info = gaia_crossmatch(coords_saga_vmp.ra, coords_saga_vmp.dec)

INFO:astroquery:Sending pytable.


INFO: Sending pytable. [astroquery.utils.tap.core]
500 Error 500:
User must be logged in to perform this action.


HTTPError: Error 500:
User must be logged in to perform this action.

In [930]:
t_saga_vmp

F Author,Reference,Object,RA (2000),Decl (2000),Teff,log g,[M/H],xsi,Binarity,Period,Vrad,Facility,Resolution,SN,Solar Abundance,A(Li),d(Li),f(Li),A(Li I),d(Li I),f(Li I),[Be/H],d(Be),f(Be),[Be I/H],d(Be I),f(Be I),[Be II/H],d(Be II),f(Be II),[B I/H],d(B I),f(B I),[C/H],d(C),f(C),[C2/H],d(C2),f(C2),[CH/H],d(CH),f(CH),[C I/H],d(C I),f(C I),[N/H],d(N),f(N),[CN/H],d(CN),f(CN),[NH/H],d(NH),f(NH),[O/H],d(O),f(O),[O I/H],d(O I),f(O I),[[O I]/H],d([O I]),f([O I]),[OH/H],d(OH),f(OH),[HF/H],d(HF),f(HF),[Na/H],d(Na),f(Na),[Na I/H],d(Na I),f(Na I),[Mg/H],d(Mg),f(Mg),[Mg I/H],d(Mg I),f(Mg I),[Mg II/H],d(Mg II),f(Mg II),[Al/H],d(Al),f(Al),[Al I/H],d(Al I),f(Al I),[Si/H],d(Si),f(Si),[Si I/H],d(Si I),f(Si I),[Si II/H],d(Si II),f(Si II),[P I/H],d(P I),f(P I),[S/H],d(S),f(S),[S I/H],d(S I),f(S I),[[S I]/H],d([S I]),f([S I]),[K/H],d(K),f(K),[K I/H],d(K I),f(K I),[Ca/H],d(Ca),f(Ca),[Ca I/H],d(Ca I),f(Ca I),[Ca II/H],d(Ca II),f(Ca II),[Sc/H],d(Sc),f(Sc),[Sc I/H],d(Sc I),f(Sc I),[Sc II/H],d(Sc II),f(Sc II),[Ti/H],d(Ti),f(Ti),[Ti I/H],d(Ti I),f(Ti I),[Ti II/H],d(Ti II),f(Ti II),[V/H],d(V),f(V),[V I/H],d(V I),f(V I),[V II/H],d(V II),f(V II),[Cr/H],d(Cr),f(Cr),[Cr I/H],d(Cr I),f(Cr I),[Cr II/H],d(Cr II),f(Cr II),[Mn/H],d(Mn),f(Mn),[Mn I/H],d(Mn I),f(Mn I),[Mn II/H],d(Mn II),f(Mn II),[Fe/H],d(Fe),f(Fe),[Fe I/H],d(Fe I),f(Fe I),[Fe II/H],d(Fe II),f(Fe II),[Co/H],d(Co),f(Co),[Co I/H],d(Co I),f(Co I),[Co II/H],d(Co II),f(Co II),[Ni/H],d(Ni),f(Ni),[Ni I/H],d(Ni I),f(Ni I),[Ni II/H],d(Ni II),f(Ni II),[Cu/H],d(Cu),f(Cu),[Cu I/H],d(Cu I),f(Cu I),[Cu II/H],d(Cu II),f(Cu II),[Zn/H],d(Zn),f(Zn),[Zn I/H],d(Zn I),f(Zn I),[Zn II/H],d(Zn II),f(Zn II),[Ga/H],d(Ga),f(Ga),[Ga I/H],d(Ga I),f(Ga I),[Ge/H],d(Ge),f(Ge),[Ge I/H],d(Ge I),f(Ge I),[As I/H],d(As I),f(As I),[Se I/H],d(Se I),f(Se I),[Rb I/H],d(Rb I),f(Rb I),[Sr/H],d(Sr),f(Sr),[Sr I/H],d(Sr I),f(Sr I),[Sr II/H],d(Sr II),f(Sr II),[Y/H],d(Y),f(Y),[Y II/H],d(Y II),f(Y II),[Zr/H],d(Zr),f(Zr),[Zr I/H],d(Zr I),f(Zr I),[Zr II/H],d(Zr II),f(Zr II),[Nb/H],d(Nb),f(Nb),[Nb I/H],d(Nb I),f(Nb I),[Nb II/H],d(Nb II),f(Nb II),[Mo/H],d(Mo),f(Mo),[Mo I/H],d(Mo I),f(Mo I),[Mo II/H],d(Mo II),f(Mo II),[Tc I/H],d(Tc I),f(Tc I),[Ru/H],d(Ru),f(Ru),[Ru I/H],d(Ru I),f(Ru I),[Rh/H],d(Rh),f(Rh),[Rh I/H],d(Rh I),f(Rh I),[Pd/H],d(Pd),f(Pd),[Pd I/H],d(Pd I),f(Pd I),[Ag/H],d(Ag),f(Ag),[Ag I/H],d(Ag I),f(Ag I),[Cd I/H],d(Cd I),f(Cd I),[Cd II/H],d(Cd II),f(Cd II),[Sn I/H],d(Sn I),f(Sn I),[Te/H],d(Te),f(Te),[Te I/H],d(Te I),f(Te I),[Ba/H],d(Ba),f(Ba),[Ba II/H],d(Ba II),f(Ba II),[La/H],d(La),f(La),[La I/H],d(La I),f(La I),[La II/H],d(La II),f(La II),[Ce/H],d(Ce),f(Ce),[Ce II/H],d(Ce II),f(Ce II),[Pr/H],d(Pr),f(Pr),[Pr II/H],d(Pr II),f(Pr II),[Nd/H],d(Nd),f(Nd),[Nd II/H],d(Nd II),f(Nd II),[Sm/H],d(Sm),f(Sm),[Sm II/H],d(Sm II),f(Sm II),[Eu/H],d(Eu),f(Eu),[Eu I/H],d(Eu I),f(Eu I),[Eu II/H],d(Eu II),f(Eu II),[Gd/H],d(Gd),f(Gd),[Gd I/H],d(Gd I),f(Gd I),[Gd II/H],d(Gd II),f(Gd II),[Tb/H],d(Tb),f(Tb),[Tb II/H],d(Tb II),f(Tb II),[Dy/H],d(Dy),f(Dy),[Dy II/H],d(Dy II),f(Dy II),[Ho/H],d(Ho),f(Ho),[Ho II/H],d(Ho II),f(Ho II),[Er/H],d(Er),f(Er),[Er II/H],d(Er II),f(Er II),[Tm/H],d(Tm),f(Tm),[Tm II/H],d(Tm II),f(Tm II),[Yb/H],d(Yb),f(Yb),[Yb II/H],d(Yb II),f(Yb II),[Lu II/H],d(Lu II),f(Lu II),[Hf/H],d(Hf),f(Hf),[Hf II/H],d(Hf II),f(Hf II),[W I/H],d(W I),f(W I),[W II/H],d(W II),f(W II),[Os/H],d(Os),f(Os),[Os I/H],d(Os I),f(Os I),[Os II/H],d(Os II),f(Os II),[Ir/H],d(Ir),f(Ir),[Ir I/H],d(Ir I),f(Ir I),[Pt/H],d(Pt),f(Pt),[Pt I/H],d(Pt I),f(Pt I),[Au I/H],d(Au I),f(Au I),[Hg II/H],d(Hg II),f(Hg II),[Pb/H],d(Pb),f(Pb),[Pb I/H],d(Pb I),f(Pb I),[Pb II/H],d(Pb II),f(Pb II),[Bi I/H],d(Bi I),f(Bi I),[Th/H],d(Th),f(Th),[Th II/H],d(Th II),f(Th II),[U/H],d(U),f(U),[U II/H],d(U II),f(U II),12C/13C,d(12C/13C),f(12C/13C),142Nd/144Nd,d(142Nd/144Nd),f(142Nd/144Nd),151Eu/153Eu,d(151Eu/153Eu),f(151Eu/153Eu),152Sm/154Sm,d(152Sm/154Sm),f(152Sm/154Sm),6Li/7Li,d(6Li/7Li),f(6Li/7Li),V,d(V)_1,B,d(B),B-V,d(B-V),U-B,d(U-B),V-R,d(V-R),J-H,d(J-H),H-K,d(H-K),J-K,d(J-K),V-I,d(V-I),V-J,d(V-J),V-

In [863]:
print(c_hascoords)

<SkyCoord (ICRS): (ra, dec) in deg
    [(0., 0.), (0., 0.), (0., 0.), ..., (0., 0.), (0., 0.), (0., 0.)]>


In [842]:
print(np.count_nonzero(t_saga_vmp['RA (2000)'].mask))

1118


### LAMOST M-dwarfs (Cycle-StarNet, Zhang *et al* 2025)

References:
* [Zhang Shuo *et al* (2025)](https://ui.adsabs.harvard.edu/abs/2025ApJS..277...47Z/abstract).
* Zenodo: [doi:10.5281/zenodo.14030249](https://doi.org/10.5281/zenodo.14030249)

In [522]:
fn = 'DR10_Cycle-SN_M_dwarfs.csv'
url = rf'https://zenodo.org/records/14030249/files/{fn}?download=1'

! wget -O {data_dir}/{fn} {url}

--2025-06-03 14:10:46--  https://zenodo.org/records/14030249/files/DR10_Cycle-SN_M_dwarfs.csv?download=1
Resolving zenodo.org (zenodo.org)... 188.185.43.25, 188.185.48.194, 188.185.45.92, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.25|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 81491036 (78M) [text/plain]
Saving to: ‘/data/beegfs/astro-storage/groups/green/green/projects/gaia_Bp_Rp_reddening_variation/data//DR10_Cycle-SN_M_dwarfs.csv’

/data/beegfs/astro- 100%[===================>]  77.72M  26.8MB/s    in 2.9s    

2025-06-03 14:10:49 (26.8 MB/s) - ‘/data/beegfs/astro-storage/groups/green/green/projects/gaia_Bp_Rp_reddening_variation/data//DR10_Cycle-SN_M_dwarfs.csv’ saved [81491036/81491036]



In [64]:
fn = os.path.join(data_dir, 'DR10_Cycle-SN_M_dwarfs.csv')
t_csn_mdwarf = Table.read(fn)
t_csn_mdwarf.colnames

['ID',
 'obsid',
 'RA',
 'Dec',
 'SNRr',
 'SNRi',
 'Subclass',
 'SpT',
 'Gmag',
 'BPmag',
 'RPmag',
 'Plx',
 'e_Plx',
 'RUWE',
 'RV',
 'e_RV',
 'Teff',
 'e_Teff',
 'Logg',
 'e_Logg',
 '[M/H]',
 'e_[M/H]',
 'Activity',
 'VBFlag']

In [65]:
# fn = os.path.join(data_dir, 'ZhangShuo_Mdwarfs_GaiaDR3_2024.h5')
# t_csn_mdwarf = Table.read(fn)
# t_csn_mdwarf.colnames

In [66]:
print(f'Loaded {len(t_csn_mdwarf):,} catalog entries.')

Loaded 507,513 catalog entries.


In [60]:
kw = dict(bins=np.arange(0,3,0.25), histtype='step')
plt.hist(np.log10(t_csn_mdwarf['SNRr']), label='r', **kw)
plt.hist(np.log10(t_csn_mdwarf['SNRi']), label='i', **kw)

(array([0.00000e+00, 0.00000e+00, 2.00000e+01, 1.42600e+03, 2.86280e+04,
        1.04114e+05, 1.76438e+05, 1.35464e+05, 4.88110e+04, 1.26120e+04,
        0.00000e+00]),
 array([0.  , 0.25, 0.5 , 0.75, 1.  , 1.25, 1.5 , 1.75, 2.  , 2.25, 2.5 ,
        2.75]),
 [<matplotlib.patches.Polygon at 0x7f8283b212b0>])

<Figure size 640x480 with 1 Axes>

In [54]:
# Logging in may speed up the queries, but is not necessary
Gaia.login()

INFO: Login to gaia TAP server [astroquery.gaia.core]


User:  ggreen
Password:  ········


INFO: OK [astroquery.utils.tap.core]
INFO: Login to gaia data server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]


In [61]:
crossmatch_info = gaia_crossmatch_batched(
    t_csn_mdwarf['RA'],
    t_csn_mdwarf['Dec'],
    max_batch_size=1024*64,
    max_separation=1.0*units.arcsec,
    verbose=True
)

  0%|          | 0/8 [00:00<?, ?it/s]

Launched query: 'SELECT
            t.index,
            g.source_id, g.ra, g.dec,
            DISTANCE(
              POINT(t.ra, t.dec),
              POINT(g.ra, g.dec)
            ) * 3600. AS angsep_arcsec
          FROM tap_upload.crossmatch_table_0 AS t
          JOIN gaiadr3.gaia_source AS g
          ON DISTANCE(POINT(t.ra, t.dec), POINT(g.ra, g.dec)) < 0.0002777777777777778'
------>https
host = gea.esac.esa.int:443
context = /tap-server/tap/async
Content-type = multipart/form-data; boundary====1765905425592===
303 303
[('Date', 'Tue, 16 Dec 2025 17:17:05 GMT'), ('Server', 'Apache/2.4.6 (SLES Expanded Support platform 7) OpenSSL/1.0.2k-fips mod_jk/1.2.43'), ('X-VO-Authenticated', 'ggreen'), ('Location', 'https://gea.esac.esa.int/tap-server/tap/async/0b97742e-daa3-11f0-9f4e-bc97e148b76b-O'), ('X-Content-Type-Options', 'nosniff'), ('X-XSS-Protection', '0'), ('Cache-Control', 'no-cache, no-store, max-age=0, must-revalidate'), ('Pragma', 'no-cache'), ('Expires', '0'), ('X-Frame-Op

  0%|          | 0/8 [00:00<?, ?it/s]

Downloaded results for job 1 of 8.
Downloaded results for job 2 of 8.
Downloaded results for job 3 of 8.
Downloaded results for job 4 of 8.
Downloaded results for job 5 of 8.
Downloaded results for job 6 of 8.
Downloaded results for job 7 of 8.
Downloaded results for job 8 of 8.
500 Error 500:
java.lang.NullPointerException: Cannot invoke "String.equalsIgnoreCase(String)" because "originalSchema" is null
Failed to delete crossmatch_table_0: Error 500:
java.lang.NullPointerException: Cannot invoke "String.equalsIgnoreCase(String)" because "originalSchema" is null
500 Error 500:
java.lang.NullPointerException: Cannot invoke "String.equalsIgnoreCase(String)" because "originalSchema" is null
Failed to delete crossmatch_table_65536: Error 500:
java.lang.NullPointerException: Cannot invoke "String.equalsIgnoreCase(String)" because "originalSchema" is null
500 Error 500:
java.lang.NullPointerException: Cannot invoke "String.equalsIgnoreCase(String)" because "originalSchema" is null
Failed to 

<Figure size 640x480 with 1 Axes>

482,832 (100.0%) under 1 arcsec
Done.


In [84]:
t_csn_mdwarf = apply_crossmatch(crossmatch_info, t_csn_mdwarf, max_separation=0.5*units.arcsec)

84.26% of sources have crossmatch.


In [ ]:
# fn = os.path.join(data_dir, 'DR10_Cycle-SN_M_dwarfs_GDR3_match.h5')
# t_csn_mdwarf.write(fn, serialize_metadata=True, overwrite=True)

In [123]:
# idx = (t_index['angsep_arcsec'] < 0.5)
# print(f'{np.count_nonzero(idx):,} ({np.mean(idx):.1%})')
# t_index = t_index[idx]

436,744 (88.5%)


In [85]:
# Save cross-matched data
fn = os.path.join(data_dir, 'DR10_Cycle-SN_M_dwarfs_gdr3_crossmatch.h5')
t_csn_mdwarf.write(
    fn,
    serialize_metadata=True, overwrite=True,
    compression='gzip', compression_opts=3, chunks=True
)

/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


Start here if cross-match has already been run before:

In [1044]:
fn = os.path.join(data_dir, 'DR10_Cycle-SN_M_dwarfs_gdr3_crossmatch.h5')
t_csn_mdwarf = Table.read(fn)

In [86]:
idx = (
    (t_csn_mdwarf['SNRr']>15)
  & (t_csn_mdwarf['SNRi']>20)
  & (t_csn_mdwarf['Logg']>4.4)
  & (t_csn_mdwarf['Logg']<5.2)
)
print(f'{np.count_nonzero(idx):,} pass cuts.')

272,776 pass cuts.


In [87]:
np.unique(t_csn_mdwarf['Teff'], return_counts=True)

(<Column name='Teff' dtype='int64' length=1131>
 2928
 2942
 2943
 2966
 2976
 2978
 2983
 2993
 2997
 3009
 3010
 3013
  ...
 4207
 4209
 4210
 4215
 4228
 4229
 4235
 4246
 4255
 4270
 4276,
 array([1, 1, 1, ..., 1, 1, 1]))

In [88]:
plot_teff_logg_feh(
    t_csn_mdwarf['Teff'][idx],
    t_csn_mdwarf['Logg'][idx],
    t_csn_mdwarf['[M/H]'][idx]
)
plt.show()

<Figure size 800x400 with 2 Axes>

In [89]:
t_csn_mdwarf.colnames

['ID',
 'obsid',
 'RA',
 'Dec',
 'SNRr',
 'SNRi',
 'Subclass',
 'SpT',
 'Gmag',
 'BPmag',
 'RPmag',
 'Plx',
 'e_Plx',
 'RUWE',
 'RV',
 'e_RV',
 'Teff',
 'e_Teff',
 'Logg',
 'e_Logg',
 '[M/H]',
 'e_[M/H]',
 'Activity',
 'VBFlag',
 'gdr3_source_id',
 'ra_gdr3',
 'dec_gdr3']

In [91]:
id_col = 'gdr3_source_id'
param_cols = ('Teff', '[M/H]', 'Logg')
param_unc_cols = ('e_Teff', 'e_[M/H]', 'e_Logg')
extra_cols = ('ra_gdr3', 'dec_gdr3')

t_csnmd = extract_table(
    t_csn_mdwarf[idx],
    id_col, param_cols, param_unc_cols, extra_cols,
    param_transf=param_transf,
    param_unc_transf=apply_err_floors(param_unc_transf,err_floor_default),
    param_errs=(0.1,0.1,0.1)
)
t_csnmd

id,x,sigma_x,x_err,ra_gdr3,dec_gdr3
,,,,deg,deg
int64,float64[3],float64[3],float32[3],float64,float64
844742414114613248,-1.2039924327646523 .. 4.78,0.0694262534041194 .. 0.13601470508735444,0.1 .. 0.1,173.33780876724552,55.57078627640396
844709020745112064,-1.7378681484619338 .. 4.87,0.06168795419699806 .. 0.08944271909999159,0.1 .. 0.1,172.62363580268763,55.26219663658922
843980147614804352,-1.2847944990447018 .. 4.74,0.07181420830537101 .. 0.14560219778561037,0.1 .. 0.1,170.18821766275556,55.533490548611816
843280205384516352,-1.7962484215560068 .. 4.83,0.06522261084820623 .. 0.0806225774829855,0.1 .. 0.1,172.17624502936698,55.59750414548393
844034161123568128,-1.7733847063802965 .. 4.81,0.06352372480151161 .. 0.08944271909999159,0.1 .. 0.1,170.9320536873287,55.6779571946803
844142291220136064,-1.7096354365208835 .. 4.83,0.05718741233750299 .. 0.0806225774829855,0.1 .. 0.1,171.97709311321833,56.57029380875092
844416035255950848,-1.3629821751687006 .. 4.74,0.05864712996416838 .. 0.1077032961426901,0.1 .. 0.1,171.35783592679644,57.191295613957045
844400126697092736,-1.687163488581687 .. 4.81,0.0596999820450601 .. 0.08944271909999159,0.1 .. 0.1,171.510452661056,56.94895588696732


In [92]:
plot_catalog_uncertainties(t_csnmd)
plot_catalog_params(t_csnmd)

<Figure size 800x280 with 4 Axes>

<Figure size 500x300 with 1 Axes>

<Figure size 500x400 with 1 Axes>

In [120]:
# Save catalog
fn = 'training_data/lamost_csn_dr10.h5'
t_csnmd.write(fn, serialize_metadata=True, overwrite=True)
! du -sh {fn}

/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


22M	training_data/lamost_csn_dr10.h5


### RAVE (DR6, BDASP+GAUGUIN)

References:

* [Steinmetz *et al.* (2020)](https://ui.adsabs.harvard.edu/abs/2020AJ....160...83S/abstract)
* BDASP table documentation: https://doi.org/10.17876/rave/dr.6/008
* GAUGUIN table documentation: https://doi.org/10.17876/rave/dr.6/010

In [318]:
url_base = 'https://www.rave-survey.org/files/hdf5'

fname_list = [
    'ravedr6_dr6_bdasp.h5',
    'ravedr6_dr6_gauguin_bdasp.h5',
    'ravedr6_dr6_x_gaiadr2.h5',
    'ravedr6_dr6_obsdata.h5'
]

In [ ]:
for fn in fname_list:
    ! wget -O {data_dir}/{fn} {url_base}/{fn}

In [319]:
import pandas as pd

rave_dr6_df = None

for fn in fname_list:
    print(f'Loading {fn} ...')
    df = pd.read_hdf(os.path.join(data_dir, fn))
    if rave_dr6_df is None:
        rave_dr6_df = df
    else:
        rave_dr6_df = pd.merge(rave_dr6_df, df,
                               on='rave_obs_id', how='inner')

Loading ravedr6_dr6_bdasp.h5 ...
Loading ravedr6_dr6_gauguin_bdasp.h5 ...
Loading ravedr6_dr6_x_gaiadr2.h5 ...
Loading ravedr6_dr6_obsdata.h5 ...


In [320]:
t_ravedr6 = Table.from_pandas(rave_dr6_df)

In [321]:
for k in t_ravedr6.colnames:
    if not t_ravedr6[k].dtype.name.startswith('float'):
        print(k, t_ravedr6[k].dtype)

rave_obs_id <U21
age_bdasp int64
n_elements_gauguin int64
source_id int64
solution_id int64
designation <U28
random_index int64
astrometric_n_obs_al int64
astrometric_n_obs_ac int64
astrometric_n_good_obs_al int64
astrometric_n_bad_obs_al int64
astrometric_params_solved int64
astrometric_primary_flag bool
astrometric_matched_observations int64
visibility_periods_used int64
frame_rotator_object_type int64
matched_observations int64
duplicated_source bool
phot_g_n_obs int64
phot_bp_n_obs int64
phot_rp_n_obs int64
phot_proc_mode int64
rv_nb_transits int64
phot_variable_flag <U13
field <U16
obsdate int64
fieldname <U7
fibernumber int64
ut_start <U10
ut_end <U10
lst_start <U10
lst_end <U10
platenumber int64
healpix4096 int64
cluster_flag object
n_repeats int64
footprint_flag int64
pos <U44


Cross-match with Gaia DR3:

In [1080]:
Gaia.login()

INFO:astroquery:Login to gaia TAP server


INFO: Login to gaia TAP server [astroquery.gaia.core]


User:  ggreen
Password:  ········


INFO:astroquery:OK
INFO:astroquery:Login to gaia data server


INFO: OK [astroquery.utils.tap.core]
INFO: Login to gaia data server [astroquery.gaia.core]


INFO:astroquery:OK


INFO: OK [astroquery.utils.tap.core]


In [99]:
crossmatch_info = gaia_crossmatch_batched(
    t_ravedr6['ra'], t_ravedr6['dec'],
    max_batch_size=1024*64,
    max_separation=1.0*units.arcsec,
    verbose=True
)

Launching jobs...


  0%|          | 0/7 [00:00<?, ?it/s]

Launched query: 'SELECT
            t.index,
            g.source_id, g.ra, g.dec,
            DISTANCE(
              POINT(t.ra, t.dec),
              POINT(g.ra, g.dec)
            ) * 3600. AS angsep_arcsec
          FROM tap_upload.crossmatch_table_0 AS t
          JOIN gaiadr3.gaia_source AS g
          ON DISTANCE(POINT(t.ra, t.dec), POINT(g.ra, g.dec)) < 0.0002777777777777778'
------>https
host = gea.esac.esa.int:443
context = /tap-server/tap/async
Content-type = multipart/form-data; boundary====1765906862988===
303 303
[('Date', 'Tue, 16 Dec 2025 17:41:03 GMT'), ('Server', 'Apache/2.4.6 (SLES Expanded Support platform 7) OpenSSL/1.0.2k-fips mod_jk/1.2.43'), ('X-VO-Authenticated', 'ggreen'), ('Location', 'https://gea.esac.esa.int/tap-server/tap/async/64590157-daa6-11f0-9f4e-bc97e148b76b-O'), ('X-Content-Type-Options', 'nosniff'), ('X-XSS-Protection', '0'), ('Cache-Control', 'no-cache, no-store, max-age=0, must-revalidate'), ('Pragma', 'no-cache'), ('Expires', '0'), ('X-Frame-Op

  0%|          | 0/7 [00:00<?, ?it/s]

Downloaded results for job 1 of 7.
Downloaded results for job 2 of 7.
Downloaded results for job 3 of 7.
Downloaded results for job 4 of 7.
Downloaded results for job 5 of 7.
Downloaded results for job 6 of 7.
Downloaded results for job 7 of 7.
500 Error 500:
java.lang.NullPointerException: Cannot invoke "String.equalsIgnoreCase(String)" because "originalSchema" is null
Failed to delete crossmatch_table_0: Error 500:
java.lang.NullPointerException: Cannot invoke "String.equalsIgnoreCase(String)" because "originalSchema" is null
500 Error 500:
java.lang.NullPointerException: Cannot invoke "String.equalsIgnoreCase(String)" because "originalSchema" is null
Failed to delete crossmatch_table_65536: Error 500:
java.lang.NullPointerException: Cannot invoke "String.equalsIgnoreCase(String)" because "originalSchema" is null
500 Error 500:
java.lang.NullPointerException: Cannot invoke "String.equalsIgnoreCase(String)" because "originalSchema" is null
Failed to delete crossmatch_table_131072: Err

<Figure size 640x480 with 1 Axes>

440,435 (100.0%) under 1 arcsec
Done.


In [100]:
t_ravedr6 = apply_crossmatch(crossmatch_info, t_ravedr6, max_separation=0.3*units.arcsec)

99.99% of sources have crossmatch.


In [116]:
idx = (t_ravedr6['source_id'] != t_ravedr6['gdr3_source_id'])
print(t_ravedr6[['source_id','gdr3_source_id']][idx])

     source_id         gdr3_source_id  
------------------- -------------------
5470732838301821440 5470732842597619968
5458242046908752896 5458242046909094656
5472146677112483328 5472146677112967552
5472888705725609088 5472888710022390016
5473993375610282752 5473993375610648960
5474624525348309120 5474624529645710464
5468464893707553536 5468464893708011648
6157793881854625536 6157793881856452224
6157874352362044544 6157874352364836480
3468060167375389312 3468060167376212480
                ...                 ...
5781459604615662464 5781459604610795776
5806400307900702592 5806400307907553280
5807644096073426432 5807644096077958272
5804540690145391616 5804540690146748672
5805873779268070784 5805873779273808640
5805397622019532416 5805397622015029248
5820777139691626240 5820777144018479744
5855484053168355840 5855484053186165504
5845001687283211136 5845001687291329920
5843854965341684096 5843854961046669824
Length = 31007 rows


In [110]:
# Save Gaia crossmatch
# Need to first remove column with `Object` datatype
t_ravedr6.remove_column('cluster_flag')
fn = os.path.join(data_dir, 'rave_dr6_gdr3_crossmatch.h5')
t_ravedr6.write(fn, serialize_metadata=True, overwrite=True)

Start here if Gaia DR3 crossmatch has already been generated:

In [244]:
fn = os.path.join(data_dir, 'rave_dr6_gdr3_crossmatch.h5')
t_ravedr6 = Table.read(fn)

In [245]:
print(f'{len(t_ravedr6):,} RAVE DR6 sources have Gaia DR3 crossmatch.')

440,411 RAVE DR6 sources have Gaia DR3 crossmatch.


In [246]:
t_ravedr6.colnames

['rave_obs_id',
 'distance_bdasp',
 'distance_error_bdasp',
 'age_bdasp',
 'age_error_bdasp',
 'teff_bdasp',
 'teff_error_bdasp',
 'logg_bdasp',
 'logg_error_bdasp',
 'mass_bdasp',
 'mass_error_bdasp',
 'log_a_v_bdasp',
 'log_a_v_error_bdasp',
 'm_h_bdasp',
 'm_h_error_bdasp',
 'parallax_bdasp',
 'parallax_error_bdasp',
 'dist_mod_bdasp',
 'dist_mod_error_bdasp',
 'a_v_inf_prior_bdasp',
 'fe_h_gauguin',
 'fe_h_gauguin.mask',
 'fe_h_error_gauguin',
 'fe_h_error_gauguin.mask',
 'fe_h_chisq_gauguin',
 'fe_h_chisq_gauguin.mask',
 'fe_h_nl_gauguin',
 'fe_h_nl_gauguin.mask',
 'al_h_gauguin',
 'al_h_gauguin.mask',
 'al_h_error_gauguin',
 'al_h_error_gauguin.mask',
 'al_h_chisq_gauguin',
 'al_h_chisq_gauguin.mask',
 'al_h_nl_gauguin',
 'al_h_nl_gauguin.mask',
 'ni_h_gauguin',
 'ni_h_gauguin.mask',
 'ni_h_error_gauguin',
 'ni_h_error_gauguin.mask',
 'ni_h_chisq_gauguin',
 'ni_h_chisq_gauguin.mask',
 'ni_h_nl_gauguin',
 'ni_h_nl_gauguin.mask',
 'alpha_fe_gauguin',
 'alpha_fe_gauguin.mask',
 'alp

In [247]:
plt.hist(np.log10(t_ravedr6['fe_h_chisq_gauguin']), bins=100, range=(-5,2))
plt.show()

<Figure size 640x480 with 1 Axes>

In [248]:
plt.hist(np.log10(t_ravedr6['alpha_fe_error_gauguin']), bins=100, range=(-2,0.5))
plt.show()

<Figure size 640x480 with 1 Axes>

In [250]:
idx = (
    np.isfinite(t_ravedr6['fe_h_gauguin'])
  & (t_ravedr6['fe_h_chisq_gauguin'] < 1e-2)
  & (t_ravedr6['fe_h_nl_gauguin'] > 2)
  & (t_ravedr6['teff_error_bdasp'] < 300.)
  & (t_ravedr6['logg_error_bdasp'] < 0.3)
  & (t_ravedr6['fe_h_error_gauguin'] < 0.3)
  & (t_ravedr6['alpha_fe_error_gauguin'] < 0.3)
)
print(f'{np.count_nonzero(idx):,} of sources ({np.mean(idx):.3%}) pass quality cuts.')

plot_teff_logg_feh_abundance(
    t_ravedr6['teff_bdasp'][idx],
    t_ravedr6['logg_bdasp'][idx],
    t_ravedr6['fe_h_gauguin'][idx],
    t_ravedr6['alpha_fe_gauguin'][idx]
)
plt.show()

174,871 of sources (39.706%) pass quality cuts.


<Figure size 800x400 with 2 Axes>

In [251]:
id_col = 'gdr3_source_id'
param_cols = ('teff_bdasp', 'fe_h_gauguin', 'logg_bdasp', 'alpha_fe_gauguin')
param_unc_cols = ('teff_error_bdasp', 'fe_h_error_gauguin',
                  'logg_error_bdasp', 'alpha_fe_error_gauguin')
extra_cols = ('ra_gdr3', 'ra_gdr3')

t_rave = extract_table(
    t_ravedr6[idx],
    id_col, param_cols, param_unc_cols, extra_cols,
    param_transf=param_transf,
    param_unc_transf=apply_err_floors(param_unc_transf,err_floor_default),
    param_errs=(0.1,0.1,0.1,0.5)
)
t_rave

id,x,sigma_x,x_err,ra_gdr3
int64,float64[4],float64[4],float32[4],float64
5740356767587386752,0.12825393975950483 .. -0.182095,0.1372098252406619 .. 0.13716947118072592,0.1 .. 0.5,143.07670258087896
5740203793737454080,0.783684786480261 .. -0.401936,0.07687953133648953 .. 0.17207446789108485,0.1 .. 0.5,142.50327523598438
5740210012850100480,0.5359266304741022 .. -0.07725800000000001,0.10023860276516157 .. 0.22359807712947805,0.1 .. 0.5,142.33835026532216
5741138309900458880,-0.2111471029478121 .. -0.320768,0.07408841364884551 .. 0.1595224192895782,0.1 .. 0.5,142.786392226172
3823678507385428352,-0.18328582332259008 .. -0.279356,0.06245359418092998 .. 0.11690230493878212,0.1 .. 0.5,143.5517326976399
3820418043092146176,0.1285993275234579 .. -0.40991700000000003,0.06286467926249617 .. 0.16851557592400768,0.1 .. 0.5,145.1972598802625
5740420264383860352,-0.1626776867402825 .. -0.20211099999999999,0.06358518329773231 .. 0.1835859725469242,0.1 .. 0.5,144.64049556801848
5740775917739842688,-0.23566963418819603 .. -0.257286,0.08967945973380767 .. 0.13369652211258154,0.1 .. 0.5,143.46408314359985
5469700293803398656,-0.3484963037449123 .. -0.20153,0.09207447929959804 .. 0.12671297392532463,0.1 .. 0.5,159.90771914565653


In [252]:
plot_catalog_uncertainties(t_rave)
plot_catalog_params(t_rave)

<Figure size 1050x280 with 5 Axes>

<Figure size 500x300 with 1 Axes>

<Figure size 500x400 with 1 Axes>

In [253]:
# Save catalog
fn = 'training_data/rave_dr6.h5'
t_rave.write(fn, serialize_metadata=True, overwrite=True)
! du -sh {fn}

/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


17M	training_data/rave_dr6.h5


### PRISTINE PIGS

The "The Pristine Inner Galaxy Survey" (PIGS) of very metal-poor (VMP) stars. We use the FERRE parameter estimates.

References:

* [Ardern-Arentsen *et al.* (2024)](https://ui.adsabs.harvard.edu/abs/2024MNRAS.530.3391A/abstract)
* https://cdsarc.cds.unistra.fr/viz-bin/cat/J/MNRAS/530/3391#/browse

In [123]:
url = 'https://cdsarc.cds.unistra.fr/viz-bin/nph-Cat/fits.gz?J/MNRAS/530/3391/tabled2.dat.gz'
fname = os.path.join(data_dir, 'Pristine_PIGS_tabled2.fits.gz')

readme_url = 'https://cdsarc.cds.unistra.fr/ftp/J/MNRAS/530/3391/ReadMe'
readme_fname = os.path.join(data_dir, 'Pristine_PIGS_ReadMe.txt')

In [ ]:
! wget -O {fname} {url}
! wget -O {readme_fname} {readme_url}

In [124]:
with fits.open(fname, mode='readonly') as f:
    f.info()
    t_pristine_pigs = Table(f['tabled2.dat'].data)

t_pristine_pigs

Filename: data/Pristine_PIGS_tabled2.fits.gz
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      81   ()      
  1  tabled2.dat    1 TableHDU       227   13235R x 37C   [A27, A19, I19, F15.11, F15.11, F24.20, F20.16, F19.16, F21.16, F21.16, F21.16, F22.16, F20.16, F18.13, F19.14, F18.16, F18.16, F19.16, F19.16, F19.16, F20.16, F22.19, F22.19, F19.16, F19.13, F19.14, F19.16, F19.16, F19.16, F19.16, F21.16, F21.15, F19.16, F19.16, A5, A5, A5]   


Star,Field,GaiaDR3,RAdeg,DEdeg,GLON,GLAT,E(B-V),SNR4000-4100,SNR5000-5100,SNRCaT,RV,e_RV,TeffF,e_TeffFtot,loggF,e_loggFtot,[Fe/H]F,e_[Fe/H]Ftot,[C/Fe]F,e_[C/Fe]Ftot,mccor100,s_nccor100,chi2F,TeffU,e_TeffUtot,loggU,e_loggUtot,[Fe/H]U,e_[Fe/H]Utot,srrU,sigU,fbpC2aU,fbpC2bU,double,goodF,goodU
str27,str19,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str5,str5,str5
Pristine_164229.13-285103.6,Field251.2-29.7,6032221476369350272,250.62136681191,-28.85099947763,-8.00127513665916,11.29140906720208,0.39,26.59279112884964,73.63919067382812,73.55672376391618,78.5522,0.894,4614.153,159.72771792099832,1.236,0.4087723204914932,-2.0980000000000003,0.1871287257478124,-0.216,0.2481739714393917,0.5044000000000001,0.1787516758415473,-3.464,4685.99850485,113.08666908328344,1.5294613,0.257221418947742,-1.9208217,0.1380654429373691,54.670733,59.798307,0.0106951871658,0.0231213872832,false,true,true
Pristine_164823.57-295252.7,Field251.2-29.7,6031122136559047936,252.09822340966,-29.8813251526,-7.977364226493478,9.642953543217873,0.42,25.6689453125,68.07379214638158,57.3252948609941,10.7175,0.8490000000000001,5395.576,137.99002435774116,2.922,0.3584165732775202,-1.418,0.1516562230836572,-0.153,0.2267104091125945,0.0,0.0,-3.197,5161.394634699999,112.51281004985483,2.3364642000000004,0.2545692474139157,-1.416228,0.1370239463318013,49.949423,59.828027,0.0106951871658,0.0173410404624,false,true,true
Pristine_164638.05-290642.8,Field251.2-29.7,6031416801339494656,251.65854763847,-29.11189425181,-7.617551155353055,10.42583802603906,0.34,21.96149040670956,62.69646951426631,47.34598297141797,3.4648,0.687,5164.768,133.58105036871808,1.984,0.4120170020763706,-1.958,0.238956586015117,-0.732,0.4600784977588064,0.183,0.209773807957644,-3.0,5332.71303579,113.84053295276664,2.7340175,0.2588945177498674,-1.7306582,0.138914250191501,45.573051,57.089302,0.0374331550802,0.0116279069767,false,true,true
Pristine_170656.15-265404.5,Field256.57-26.42,4108810842107564800,256.73396897196,-26.9012649999,-3.068736434258483,8.231712054409522,0.39,11.012308502197266,46.63520050048828,58.36841546304163,45.9338,2.641,4902.463,175.6782629950559,2.166,0.4009606464480024,-1.433,0.1581708253756046,-0.235,0.2304342170772388,0.0,0.0,-2.864,5030.87045095,122.05877928983818,2.6130383,0.2740203420029987,-1.2572828,0.1463678713567667,32.265074,59.18356800000001,0.0,0.0,false,true,true
Pristine_170939.83-265354.0,Field256.57-26.42,4108846129557459712,257.41597241153,-26.89835649656,-2.705531807323325,7.7412498067821085,0.45,11.686810980902774,44.77943338373656,54.55293525821089,-216.9634,2.57,4794.762,181.33956802201223,1.507,0.4507130461834892,-1.722,0.1616044553841261,-0.423,0.2646930677218427,0.3627,0.191215437228126,-2.985,4644.57615729,123.71959192133497,1.4622819,0.2844055159956283,-1.8511287,0.1549246392483471,34.229022,72.491298,0.0106951871658,0.00581395348837,false,true,true
Pristine_171839.65-284738.6,Field259.0-29.0,4107464593223837952,259.66521703062,-28.79406400858,-3.1057755670700544,5.022772495832103,0.88,4.300454860780297,27.88470623944257,37.124771500696376,-71.0236,1.419,5036.188,235.48825542935256,2.594,0.5438235007794349,-1.338,0.1887898567190515,-0.265,0.2702017053980229,0.0,0.0,-2.656,4665.01626557,144.65925101956586,1.7691307,0.3463637999194912,-1.7140825000000002,0.1843795848868761,18.579,51.462798,0.0213903743316,0.0462427745665,false,false,false
Pristine_183211.06-254717.4,Field278.6-26.4,4076105765614950016,278.04607847962,-25.78818543384,7.7617054979549955,-7.534975203408869,0.45999998,11.470773092130337,56.1065687242445,61.04342517921148,175.658,0.727,5374.937,152.34384029428298,2.901,0.4606584852143722,-1.711,0.1625,0.243,0.2154852431142328,0.0135,0.0176598042703957,-2.766,5222.09165077,118.3773701485454,3.0080566,0.2697832770824379,-1.7771707,0.1445620696

In [125]:
t_pristine_pigs.colnames

['Star',
 'Field',
 'GaiaDR3',
 'RAdeg',
 'DEdeg',
 'GLON',
 'GLAT',
 'E(B-V)',
 'SNR4000-4100',
 'SNR5000-5100',
 'SNRCaT',
 'RV',
 'e_RV',
 'TeffF',
 'e_TeffFtot',
 'loggF',
 'e_loggFtot',
 '[Fe/H]F',
 'e_[Fe/H]Ftot',
 '[C/Fe]F',
 'e_[C/Fe]Ftot',
 'mccor100',
 's_nccor100',
 'chi2F',
 'TeffU',
 'e_TeffUtot',
 'loggU',
 'e_loggUtot',
 '[Fe/H]U',
 'e_[Fe/H]Utot',
 'srrU',
 'sigU',
 'fbpC2aU',
 'fbpC2bU',
 'double',
 'goodF',
 'goodU']

In [126]:
idx = (
    (t_pristine_pigs['goodF'] == 'true ')
  & (t_pristine_pigs['TeffF'] > 4501.)
  & (t_pristine_pigs['TeffF'] < 6999.)
  & (t_pristine_pigs['loggF'] > 1.01)
  & (t_pristine_pigs['loggF'] < 4.99)
)
print(f'{np.count_nonzero(idx):,} of sources ({np.mean(idx):.3%}) pass quality cuts.')

plot_teff_logg_feh(
    t_pristine_pigs['TeffF'][idx],
    t_pristine_pigs['loggF'][idx],
    t_pristine_pigs['[Fe/H]F'][idx]
)

plt.show()

11,425 of sources (86.324%) pass quality cuts.


<Figure size 800x400 with 2 Axes>

In [127]:
id_col = 'GaiaDR3'
param_cols = ('TeffF', '[Fe/H]F', 'loggF')
param_unc_cols = ('e_TeffFtot', 'e_[Fe/H]Ftot', 'e_loggFtot')
extra_cols = ('[C/Fe]F', 'e_[C/Fe]Ftot', 'RAdeg', 'DEdeg')

t_pigs = extract_table(
    t_pristine_pigs[idx],
    id_col, param_cols, param_unc_cols, extra_cols,
    param_transf=param_transf,
    param_unc_transf=apply_err_floors(param_unc_transf,err_floor_default),
    param_errs=(0.1,0.1,0.1)
)
t_pigs

id,x,sigma_x,x_err,[C/Fe]F,e_[C/Fe]Ftot,RAdeg,DEdeg
int64,float64[3],float64[3],float32[3],float64,float64,float64,float64
6032221476369350272,-0.40154796642166457 .. 1.236,0.17337318419013056 .. 0.4107247375067638,0.1 .. 0.1,-0.216,0.2481739714393917,250.62136681191,-28.85099947763
6031122136559047936,0.3807072305029884 .. 2.922,0.12826371849349932 .. 0.36064170585222116,0.1 .. 0.1,-0.153,0.2267104091125945,252.09822340966,-29.8813251526
6031416801339494656,0.16211135706639285 .. 1.984,0.1297055700672031 .. 0.4139541158147845,0.1 .. 0.1,-0.732,0.4600784977588064,251.65854763847,-29.11189425181
4108810842107564800,-0.09850090272013749 .. 2.166,0.17945231376619045 .. 0.40295091512490694,0.1 .. 0.1,-0.235,0.2304342170772388,256.73396897196,-26.9012649999
4108846129557459712,-0.20956920183527222 .. 1.507,0.18936596396810698 .. 0.45248453012230155,0.1 .. 0.1,-0.423,0.2646930677218427,257.41597241153,-26.89835649656
4076105765614950016,0.36154470290351426 .. 2.901,0.14206923940259447 .. 0.4623918684406117,0.1 .. 0.1,0.243,0.2154852431142328,278.04607847962,-25.78818543384
6031371588251573888,-0.017677211587861282 .. 2.375,0.13330134666397556 .. 0.3158511674824077,0.1 .. 0.1,-0.047,0.2053591001148963,252.01928960544,-29.17350188136
6034483068713287040,0.15581572831041263 .. 3.017,0.18024731914844178 .. 0.45518577526104664,0.1 .. 0.1,-0.061,0.2436540336214445,253.73591569044,-25.97826881281
6032669213803826560,-0.22049198869406675 .. 1.776,0.16030731398857584 .. 0.37970021069259363,0.1 .. 0.1,-0.473,0.2181464739572932,252.83602034563,-29.22963296443


In [128]:
# Save catalog
fn = 'training_data/pristine_pigs.h5'
t_pigs.write(fn, serialize_metadata=True, overwrite=True)
! du -sh {fn}

1.1M	training_data/pristine_pigs.h5


/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


### SDSS-V (SLAM, M-Dwarfs)

Reference: [Qiu *et al* (2025)](https://ui.adsabs.harvard.edu/abs/2025arXiv251120005Q/abstract), [Zhang *et al* (2020)](https://ui.adsabs.harvard.edu/abs/2020ApJS..246....9Z/abstract), [Li *et al* (2021)](https://ui.adsabs.harvard.edu/abs/2021ApJS..253...45L/abstract)

More information: https://sdss.org/dr19/mwm/astra/pipelines-in-astra/slam/

In [228]:
# url_base = 'https://data.sdss5.org/sas/ipl-3/spectro/astra/0.6.0/summary'
url_base = 'https://data.sdss5.org/sas/dr19/spectro/astra/0.6.0/summary'
fn = 'astraAllStarSlam-0.6.0.fits.gz'

fname_sdss_slam = os.path.join(data_dir, 'sdssv', fn)

In [115]:
os.makedirs(os.path.join(data_dir, 'sdssv'), exist_ok=True)

username = getpass('Username: ')
password = getpass('Password: ')

print(f'Downloading {fn} ...')

url = f'{url_base}/{fn}'

download_with_auth(url, fname_sdss_slam, username, password)

print(f'Downloaded to {fname_sdss_slam}.')

Username:  ········
Password:  ········


astraAllStarSlam-0.6.0.fits.gz:   0%|          | 0.00/71.5M [00:00<?, ?B/s]

Downloaded to /data/beegfs/astro-storage/groups/green/green/projects/gaia_Bp_Rp_reddening_variation/data/sdssv/astraAllStarSlam-0.6.0.fits.gz.


In [136]:
! ls -l {fname_sdss_slam}

-rw-r--r-- 1 green storage-astro-nodes-green 74975034 Jul  1 15:37 data/sdssv/astraAllStarSlam-0.6.0.fits.gz


In [229]:
t_sdssv_slam = Table.read(fname_sdss_slam)
print(f'{len(t_sdssv_slam):,} sources in catalog.')
t_sdssv_slam

160,438 sources in catalog.


sdss_id,sdss4_apogee_id,gaia_dr2_source_id,gaia_dr3_source_id,tic_v8_id,healpix,lead,version_id,catalogid,catalogid21,catalogid25,catalogid31,n_associated,n_neighborhood,sdss5_target_flags,sdss4_apogee_target1_flags,sdss4_apogee_target2_flags,sdss4_apogee2_target1_flags,sdss4_apogee2_target2_flags,sdss4_apogee2_target3_flags,sdss4_apogee_member_flags,sdss4_apogee_extra_target_flags,ra,dec,l,b,plx,e_plx,pmra,e_pmra,pmde,e_pmde,gaia_v_rad,gaia_e_v_rad,g_mag,bp_mag,rp_mag,j_mag,e_j_mag,h_mag,e_h_mag,k_mag,e_k_mag,ph_qual,bl_flg,cc_flg,w1_mag,e_w1_mag,w1_flux,w1_dflux,w1_frac,w2_mag,e_w2_mag,w2_flux,w2_dflux,w2_frac,w1uflags,w2uflags,w1aflags,w2aflags,mag4_5,d4_5m,rms_f4_5,sqf_4_5,mf4_5,csf,zgr_teff,zgr_e_teff,zgr_logg,zgr_e_logg,zgr_fe_h,zgr_e_fe_h,zgr_e,zgr_e_e,zgr_plx,zgr_e_plx,zgr_teff_confidence,zgr_logg_confidence,zgr_fe_h_confidence,zgr_ln_prior,zgr_chi2,zgr_quality_flags,r_med_geo,r_lo_geo,r_hi_geo,r_med_photogeo,r_lo_photogeo,r_hi_photogeo,bailer_jones_flags,ebv,e_ebv,ebv_flags,ebv_zhang_2023,e_ebv_zhang_2023,ebv_sfd,e_ebv_sfd,ebv_rjce_glimpse,e_ebv_rjce_glimpse,ebv_rjce_allwise,e_ebv_rjce_allwise,ebv_bayestar_2019,e_ebv_bayestar_2019,ebv_edenhofer_2023,e_ebv_edenhofer_2023,c_star,u_jkc_mag,u_jkc_mag_flag,b_jkc_mag,b_jkc_mag_flag,v_jkc_mag,v_jkc_mag_flag,r_jkc_mag,r_jkc_mag_flag,i_jkc_mag,i_jkc_mag_flag,u_sdss_mag,u_sdss_mag_flag,g_sdss_mag,g_sdss_mag_flag,r_sdss_mag,r_sdss_mag_flag,i_sdss_mag,i_sdss_mag_flag,z_sdss_mag,z_sdss_mag_flag,y_ps1_mag,y_ps1_mag_flag,n_boss_visits,boss_min_mjd,boss_max_mjd,n_apogee_visits,apogee_min_mjd,apogee_max_mjd,spectrum_pk,source,release,filetype,v_astra,run2d,telescope,min_mjd,max_mjd,n_visits,n_good_visits,n_good_rvs,v_rad,e_v_rad,std_v_rad,median_e_v_rad,xcsao_teff,xcsao_e_teff,xcsao_logg,xcsao_e_logg,xcsao_fe_h,xcsao_e_fe_h,xcsao_meanrxc,snr,gri_gaia_transform_flags,zwarning_flags,nmf_rchi2,nmf_flags,task_pk,source_pk,created,t_elapsed,t_overhead,tag,teff,e_teff,logg,e_logg,fe_h,e_fe_h,fe_h_niu,e_fe_h_niu,alpha_fe,e_alpha_fe,initial_teff,initial_logg,initial_fe_h,initial_alpha_fe,initial_fe_h_niu,success,status,optimality,result_flags,flag_warn,flag_bad,chi2,rchi2
int64,bytes18,int64,int64,int64,int32,bytes17,int32,int64,int64,int64,int64,int32,int32,uint8[57],int64,int64,int64,int64,int64,int64,int64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,bytes3,bytes3,bytes3,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int64,int64,int64,int64,float32,float32,float32,int64,int64,int64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int64,float32,float32,float32,float32,float32,float32,bytes5,float32,float32,int64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int32,float32,int32,float32,int32,float32,int32,float32,int32,float32,int32,float32,int32,float32,int32,float32,int32,float32,int32,float32,int32,int32,int32,int32,int32,int32,int32,int64,int64,bytes5,bytes7,bytes5,bytes6,bytes6,int32,int32,int32,int32,int32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int64,int64,float32,int64,int64,int64,bytes26,float32,float32,bytes1,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,bool,int32,bool,int64,bool,bool,float32,float32
75252981,--,3147307164988425984,3147307164988425984,468955618,81321,tic_v8,25,27021598102054571,4544487347,27021598102054571,63050395121360456,1,0,0 .. 0,0,0,0,0,0,0,0,118.87514,10.048503,211.03108,18.744108,0.59778243,0.08692945,-6.2564464,0.09981683,-4.1921825,0.06401512,--,--,17.65054,18.543514,16.732792,15.593,0.058,14.912,0.07,14.787,0.08,AAA,111,000,14.692992,0.008310634,1321.914,10.118428,0.96572363,14.82123,0.031595174,1144.7435,33.3123,0.96524537,0,0,0,0,--,--,

In [230]:
t_sdssv_slam.colnames

['sdss_id',
 'sdss4_apogee_id',
 'gaia_dr2_source_id',
 'gaia_dr3_source_id',
 'tic_v8_id',
 'healpix',
 'lead',
 'version_id',
 'catalogid',
 'catalogid21',
 'catalogid25',
 'catalogid31',
 'n_associated',
 'n_neighborhood',
 'sdss5_target_flags',
 'sdss4_apogee_target1_flags',
 'sdss4_apogee_target2_flags',
 'sdss4_apogee2_target1_flags',
 'sdss4_apogee2_target2_flags',
 'sdss4_apogee2_target3_flags',
 'sdss4_apogee_member_flags',
 'sdss4_apogee_extra_target_flags',
 'ra',
 'dec',
 'l',
 'b',
 'plx',
 'e_plx',
 'pmra',
 'e_pmra',
 'pmde',
 'e_pmde',
 'gaia_v_rad',
 'gaia_e_v_rad',
 'g_mag',
 'bp_mag',
 'rp_mag',
 'j_mag',
 'e_j_mag',
 'h_mag',
 'e_h_mag',
 'k_mag',
 'e_k_mag',
 'ph_qual',
 'bl_flg',
 'cc_flg',
 'w1_mag',
 'e_w1_mag',
 'w1_flux',
 'w1_dflux',
 'w1_frac',
 'w2_mag',
 'e_w2_mag',
 'w2_flux',
 'w2_dflux',
 'w2_frac',
 'w1uflags',
 'w2uflags',
 'w1aflags',
 'w2aflags',
 'mag4_5',
 'd4_5m',
 'rms_f4_5',
 'sqf_4_5',
 'mf4_5',
 'csf',
 'zgr_teff',
 'zgr_e_teff',
 'zgr_logg',

In [231]:
[s for s in t_sdssv_slam.colnames if 'chi' in s]

['zgr_chi2', 'nmf_rchi2', 'chi2', 'rchi2']

In [232]:
np.nanpercentile(t_sdssv_slam['nmf_rchi2'], [0.,1.,10.,50.,90.,99.,100.])

0.294543981552124
0.6932550650835038
0.895572555065155
1.9950460195541382
8.531040573120128
55.066277008056645
4139214.0


In [233]:
np.unique(t_sdssv_slam['result_flags'], return_counts=True)

(<Column name='result_flags' dtype='int64' length=5>
 0
 1
 2
 4
 6,
 array([159281,    953,     81,    119,      4]))

In [234]:
plt.hist(t_sdssv_slam['snr'], bins=np.logspace(0,3,50))
plt.xscale('log')
plt.xlabel('SNR')
plt.show()

<Figure size 640x480 with 1 Axes>

In [235]:
plt.hist(t_sdssv_slam['e_teff'], bins=np.logspace(2,5,50), log=True)
plt.xscale('log')
plt.xlabel(r'$\sigma_{T_{\mathrm{eff}}}\ (\mathrm{K})$')
plt.show()

<Figure size 640x480 with 1 Axes>

In [236]:
idx = (
    (t_sdssv_slam['teff'] > 3150) &
    (t_sdssv_slam['teff'] < 3750) &
    (t_sdssv_slam['e_teff'] < 600.) &
    # (t_sdssv_slam['result_flags'] == 0) &
    (t_sdssv_slam['fe_h'] > -0.6) &
    (t_sdssv_slam['fe_h'] < 0.3) &
    (t_sdssv_slam['alpha_fe'] > -0.2) &
    (t_sdssv_slam['alpha_fe'] < 0.2) &
    (t_sdssv_slam['snr'] > 15)
)

print(f'{np.count_nonzero(idx):,} of sources ({np.mean(idx):.3%}) pass quality cuts.')

plot_teff_logg_feh_abundance(
    t_sdssv_slam['teff'][idx],
    t_sdssv_slam['logg'][idx],
    t_sdssv_slam['fe_h'][idx],
    t_sdssv_slam['alpha_fe'][idx]
)

plt.show()

21,162 of sources (13.190%) pass quality cuts.


<Figure size 800x400 with 2 Axes>

In [237]:
plt.hist(t_sdssv_slam['teff'][idx], bins=100)
plt.xlabel('teff')
plt.show()

<Figure size 640x480 with 1 Axes>

In [241]:
id_col = 'gaia_dr3_source_id'
param_cols = ('teff', 'fe_h', 'logg', 'alpha_fe')
param_unc_cols = ('e_teff', 'e_fe_h', 'e_logg', 'e_alpha_fe')
extra_cols = ('ra', 'dec')
    
t_slam = extract_table(
    t_sdssv_slam[idx],
    id_col, param_cols, param_unc_cols, extra_cols,
    param_transf=param_transf,
    # param_unc_transf=transf_slam_errs(),
    # param_unc_transf=apply_err_floors(param_unc_transf,err_floor_default),
    param_unc_transf=apply_err_floors(
        [
            lambda x,xs,f=f,s=s: f(x,s*xs)
            for f,s in zip(param_unc_transf, [0.2,1,1,1])
        ], # Scale T_eff errors by 0.5
        err_floor_default
    ),
    param_errs=(0.2,0.2,0.2,0.2)
)
t_slam

id,x,sigma_x,x_err,ra,dec
int64,float32[4],float32[4],float32[4],float32,float32
1665724695486359296,-1.8470072 .. -0.008825563,0.07926335 .. 0.05000148,0.2 .. 0.2,200.76175,64.33813
3214038690340674048,-1.9264791 .. -0.06827703,0.1411399 .. 0.050000355,0.2 .. 0.2,81.02303,-2.6094685
471822193494098304,-1.5985436 .. -0.0035693184,0.12783168 .. 0.050001055,0.2 .. 0.2,65.51161,60.011745
851120513564828288,-1.9578762 .. -0.03635864,0.09423706 .. 0.050000727,0.2 .. 0.2,162.70999,56.579967
1677742907333698432,-2.101936 .. -0.05309443,0.15263239 .. 0.050000455,0.2 .. 0.2,200.72906,64.53084
2247952523718927872,-1.5468079 .. -0.020175753,0.13277839 .. 0.05000111,0.2 .. 0.2,300.27805,65.72026
1677757888179663872,-1.533236 .. -0.03626078,0.14706975 .. 0.05000137,0.2 .. 0.2,199.75137,64.743614
4372700603424872832,-2.059727 .. -0.050966304,0.12993224 .. 0.05000067,0.2 .. 0.2,267.628,1.4368179
932171180709544960,-1.8173549 .. -0.047842417,0.082983285 .. 0.050001502,0.2 .. 0.2,125.11475,50.652565


In [242]:
plot_catalog_params(t_slam)
plot_catalog_uncertainties(t_slam)

<Figure size 500x400 with 1 Axes>

<Figure size 1050x280 with 5 Axes>

<Figure size 500x300 with 1 Axes>

In [243]:
# Save catalog
fn = 'training_data/sdssv_slam.h5'
t_slam.write(fn, serialize_metadata=True, overwrite=True)
! du -sh {fn}

/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


1.3M	training_data/sdssv_slam.h5


### SDSS-V (Cannon, M-Dwarfs)

Reference: [Behmard *et al* (2025)](https://ui.adsabs.harvard.edu/abs/2025ApJ...982...13B/abstract)

In [382]:
url_base = 'https://data.sdss.org/sas/dr19/vac/mwm/m-dwarf/elemental_abundances'
fname = 'mdwarf_abundances-1.0.2.fits'

full_fname = os.path.join(data_dir, 'sdssv', fname)

In [383]:
os.makedirs(os.path.join(data_dir, 'sdssv'), exist_ok=True)

! wget {url_base}/{fname} -O {full_fname}
! du -sh {fname}

--2025-12-20 14:22:02--  https://data.sdss.org/sas/dr19/vac/mwm/m-dwarf/elemental_abundances/mdwarf_abundances-1.0.2.fits
Resolving data.sdss.org (data.sdss.org)... 155.101.19.133
Connecting to data.sdss.org (data.sdss.org)|155.101.19.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3605760 (3.4M) [application/octet-stream]
Saving to: ‘data/sdssv/mdwarf_abundances-1.0.2.fits’

data/sdssv/mdwarf_a 100%[===================>]   3.44M  32.6KB/s    in 1m 41s  

2025-12-20 14:23:44 (34.9 KB/s) - ‘data/sdssv/mdwarf_abundances-1.0.2.fits’ saved [3605760/3605760]

2.0K	mdwarf_abundances-1.0.2.fits


In [390]:
with fits.open(full_fname, mode='readonly') as f:
    f.info()
    t_sdssv_mdwarf_cannon = Table(f[1].data[:])

Filename: data/sdssv/mdwarf_abundances-1.0.2.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1                1 BinTableHDU     64   16573R x 28C   [K, K, D, D, D, D, D, D, D, D, D, D, D, D, D, L, D, D, D, D, D, D, D, D, D, D, D, D]   


In [391]:
t_sdssv_mdwarf_cannon

gaia_dr3_source_id,name,Teff_cannon,fe_h_cannon,mg_h_cannon,al_h_cannon,si_h_cannon,c_h_cannon,o_h_cannon,ca_h_cannon,ti_h_cannon,cr_h_cannon,n_h_cannon,ni_h_cannon,spec_chisq,temp_agree,Teff_err,fe_h_err,mg_h_err,al_h_err,si_h_err,c_h_err,o_h_err,ca_h_err,ti_h_err,cr_h_err,n_h_err,ni_h_err
int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,bool,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
421503353090268416,54381371,3814.648561547542,-0.0026486318042882,0.02564204788069,0.054333113229706,0.0222007382525857,0.0183511551374926,-0.0185591173210488,0.0154553429777801,0.1362495059477906,0.072953816710778,0.0489106198904564,0.0535792690646078,8753.929507709254,True,13.20000269924078,0.0203115604090667,0.024966344579773,0.0247698212148589,0.021991647123353,0.0293774904199275,0.0246942094417536,0.0222677689050608,0.0285459160175209,0.0275924980359803,0.0318014034346874,0.0207874986940785
421535342004177920,54381716,3830.414302151241,-0.0621527905611096,-0.045996493744664,-0.0995353903467919,-0.0709169750058694,-0.0493116077124443,-0.0758250861642127,-0.033820878358197,0.0486834025742208,0.0522873108454577,-0.0779223642792822,-0.0595735438752935,12584.248649436911,True,13.200001985396051,0.0191801385841903,0.0235887940745609,0.0236705272947288,0.0209378311511172,0.0280941682215525,0.0233747374397651,0.021150429982295,0.027872737155828,0.0266052700511725,0.030781533176615,0.01975173385262
421917594095590272,54386478,3382.3367039294553,0.0357585264523618,0.1385049156100295,0.1669841924850513,0.0551491701194992,0.1194555258146349,0.0622115416265288,0.0342384493665602,0.2400097332554127,0.1581739979714938,0.1982887602897148,0.0659358767912941,16319.58735223872,True,13.20000014850269,0.0157654739289914,0.0192981936453863,0.0197709775152192,0.017259347075325,0.0247331397156639,0.0202252460495248,0.0177667766188163,0.0246900719799358,0.0236971501343219,0.0277000660479351,0.0162651190335944
422026445746556928,54387723,3602.57956126951,0.1584922147036102,0.1582577578486272,0.2149661362671516,0.1569614781169834,0.1813294569612283,0.0976414917695456,0.1746043423155464,0.3403653084873614,0.324508469939931,0.2735279480817246,0.2140569763910171,19317.588030455867,True,13.200000751832084,0.0169518548576504,0.0207108152261884,0.0209984923607121,0.0184502257709006,0.0258639349608833,0.0213352207452541,0.0189295570834638,0.0256200419736223,0.0246955819025294,0.0286857259508143,0.0174416371329197
422148079220822912,54389138,3792.402877026217,-0.0586424360356241,-0.0354007989798167,-0.0511984910884459,-0.042037527955931,-0.1360781709230892,-0.0614660438031734,-0.0403919065303462,-0.0727290297171287,-0.0648313215409417,-0.122396406810141,-0.0269379833024788,59377.20173869174,True,13.20000062738348,0.0167189784074385,0.0204605849055865,0.0208083605984958,0.0182598410427469,0.0256368193285686,0.0210883721108217,0.0187209416737007,0.0255678025899917,0.0245758447237757,0.0284896822314785,0.0172418011065762
422204253090333824,54389783,3279.5747666589727,-0.0265281670753715,0.0324382995797319,0.0301043516477926,-0.0287951408684242,0.0284828704745827,-0.0137426971256779,-0.041340063941561,0.208509460961046,0.1380512647533235,0.0747834943504776,0.0060017558772241,23258.05954063882,True,13.200000504321626,0.0164281753890708,0.0201118749530382,0.020493836335231,0.0179459163908754,0.025327741206456,0.0208372430944233,0.0184315895857226,0.0252208592113604,0.0242084465259586,0.0282265642917973,0.0169199374110213
422228171769143552,54390069,3578.4035420662035,0.0385436818797131,0.0977197297269187,0.1111379754935628,0.0649030086382138,0.1026163436587952,0.0418139559834217,0.037897413162142,0.1272496695683377,0.1968395089354065,0.1417022354620959,0.0982545416814594,8690.80902071162,True,13.200002998088786,0.0207842630427054,0.0252302831787315,0.0252592400845314,0.0224390946894165,0.0296126990886831,0.0247586407719269,0.0227937082741677,0.0290892286359107,0.

In [398]:
idx = t_sdssv_mdwarf_cannon['temp_agree']
plt.hist(np.log10(t_sdssv_mdwarf_cannon['spec_chisq'][idx]), bins=100, log=True)
plt.hist(np.log10(t_sdssv_mdwarf_cannon['spec_chisq'][~idx]), bins=100, log=True)
plt.show()

<Figure size 640x480 with 1 Axes>

In [408]:
plt.hist(np.log10(t_sdssv_mdwarf_cannon['fe_h_err']), bins=100, range=(-2,0))
plt.show()

<Figure size 640x480 with 1 Axes>

In [417]:
plt.scatter(abundances['Mg'], t_sdssv_mdwarf_cannon['alpha_fe'])

<Figure size 640x480 with 1 Axes>

In [419]:
a_Fe,a_Fe_err,abundances = calc_alpha_Fe_from_abundances(
    t_sdssv_mdwarf_cannon,
    '{x.lower()}_h_cannon',
    '{x.lower()}_h_err'
)
plot_alpha_abundances(a_Fe, abundances)
plt.show()

# Convert from [alpha/H] to [alpha/Fe]
a_Fe -= t_sdssv_mdwarf_cannon['fe_h_cannon']
a_Fe_err = np.sqrt(a_Fe_err**2 + t_sdssv_mdwarf_cannon['fe_h_err']**2)

t_sdssv_mdwarf_cannon['alpha_fe'] = a_Fe
t_sdssv_mdwarf_cannon['alpha_fe_err'] = a_Fe_err

<Figure size 750x500 with 6 Axes>

In [423]:
idx = (
    (t_sdssv_mdwarf_cannon['fe_h_err'] < 0.3) &
    t_sdssv_mdwarf_cannon['temp_agree']
)

print(f'{np.count_nonzero(idx):,} of sources ({np.mean(idx):.3%}) pass quality cuts.')

plot_teff_logg_feh_abundance(
    t_sdssv_mdwarf_cannon['Teff_cannon'][idx],
    t_sdssv_mdwarf_cannon['Teff_cannon'][idx], # Dummy column: there is no log(g) in this catalog.
    t_sdssv_mdwarf_cannon['fe_h_cannon'][idx],
    t_sdssv_mdwarf_cannon['alpha_fe'][idx]
)

plt.show()

15,994 of sources (96.506%) pass quality cuts.


<Figure size 800x400 with 2 Axes>

In [432]:
id_col = 'gaia_dr3_source_id'
param_cols = ('Teff_cannon', 'fe_h_cannon', 'fe_h_cannon', 'alpha_fe')
param_unc_cols = ('Teff_err', 'fe_h_err', 'fe_h_err', 'alpha_fe_err')
extra_cols = ()

t_sdssv_mdc = extract_table(
    t_sdssv_mdwarf_cannon[idx],
    id_col, param_cols, param_unc_cols, extra_cols,
    param_transf=param_transf,
    param_unc_transf=apply_err_floors(param_unc_transf,err_floor_default),
    param_errs=(0.2,0.2,1e9,0.2)
)
t_sdssv_mdc['x'][:,2] = 0.
t_sdssv_mdc['sigma_x'][:,2] = 1e9
t_sdssv_mdc

id,x,sigma_x,x_err
int64,float64[4],float64[4],float32[4]
421503353090268416,-1.3529468605601394 .. 0.033188237831251875,0.01998373856222247 .. 0.05522676991375346,0.2 .. 0.2
421535342004177920,-1.3323247110318446 .. 0.020941877975794027,0.019922114262974647 .. 0.05472235355651004,0.2 .. 0.2
421917594095590272,-1.9543555463795645 .. 0.057867334316338906,0.021926297683442387 .. 0.053334942541755546,0.2 .. 0.2
422026445746556928,-1.6389388938530278 .. 0.017122026169282017,0.020871750986180625 .. 0.053786647300094324,0.2 .. 0.2
422148079220822912,-1.3821904487164995 .. 0.010372948944971878,0.02007166901227071 .. 0.05369821172675734,0.2 .. 0.2
422204253090333824,-2.1086207137694353 .. 0.042720881588691265,0.022472159584126485 .. 0.05358491898682468,0.2 .. 0.2
422228171769143552,-1.6726057472586602 .. 0.031222993413310397,0.02098047945387004 .. 0.0554352511508617,0.2 .. 0.2
422491023770086528,-1.4312830260868843 .. 0.028334011177106856,0.020220735959398235 .. 0.05339381336163916,0.2 .. 0.2
422673886296537600,-1.5413834317919561 .. 0.04204646350936392,0.020561744089299405 .. 0.05529679984870474,0.2 .. 0.2


In [433]:
plot_catalog_params(t_sdssv_mdc)
plot_catalog_uncertainties(t_sdssv_mdc)

<Figure size 500x400 with 1 Axes>

/data/beegfs/astro-storage/groups/green/green/projects/merge_coordinates/plot_utils.py:360: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  im = ax.imshow(img.T, **kw)
/data/beegfs/astro-storage/groups/green/green/projects/merge_coordinates/plot_utils.py:360: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  im = ax.imshow(img.T, **kw)
/data/beegfs/astro-storage/groups/green/green/projects/merge_coordinates/plot_utils.py:360: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  im = ax.imshow(img.T, **kw)


<Figure size 1050x280 with 5 Axes>

<Figure size 500x300 with 1 Axes>

In [434]:
# Save catalog
fn = 'training_data/sdssv_mdc.h5'
t_sdssv_mdc.write(fn, serialize_metadata=True, overwrite=True)
! du -sh {fn}

/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


1.4M	training_data/sdssv_mdc.h5


### Load all catalogs (optional)

If the training data has already been prepared, load it using the following code:

In [254]:
# SDSS-V ASPCAP
fn = 'training_data/sdssv_aspcap.h5'
! du -sh {fn}
t_sdssv_asp = Table.read(fn)
print(f'{len(t_sdssv_asp):,}')

# SDSS-V BOSSNet
fn = 'training_data/sdssv_bossnet.h5'
! du -sh {fn}
t_sdssv_boss = Table.read(fn)
print(f'{len(t_sdssv_boss):,}')

# DESI Data-Driven Payne
fn = 'training_data/desi_ddp.h5'
! du -sh {fn}
t_desi_ddp = Table.read(fn)
print(f'{len(t_desi_ddp):,}')

# DESI DR1
fn = 'training_data/desi_dr1.h5'
! du -sh {fn}
t_desi = Table.read(fn)
print(f'{len(t_desi):,}')

# LAMOST MRS
fn = 'training_data/lamost_mrs.h5'
! du -sh {fn}
t_lmrs = Table.read(fn)
print(f'{len(t_lmrs):,}')

# APOGEE DR17
fn = 'training_data/apogee_dr17.h5'
! du -sh {fn}
t_apo = Table.read(fn)
print(f'{len(t_apo):,}')

# LAMOST HotPayne
fn = 'training_data/lamost_hotpayne.h5'
! du -sh {fn}
t_lhp = Table.read(fn)
print(f'{len(t_lhp):,}')

# GALAH DR4
fn = 'training_data/galah_dr4.h5'
! du -sh {fn}
t_galah = Table.read(fn)
print(f'{len(t_galah):,}')

# LAMOST DD-Payne (DR9)
fn = 'training_data/lamost_ddp_dr9.h5'
! du -sh {fn}
t_ddp = Table.read(fn)
print(f'{len(t_ddp):,}')

# LAMOST M-Stars (Ding+2022)
fn = 'training_data/lamost_mstar_ding2022.h5'
! du -sh {fn}
t_lmd = Table.read(fn)
print(f'{len(t_lmd):,}')

# VMP stars (Li+2022)
fn = 'training_data/vmp_stars_li2022.h5'
! du -sh {fn}
t_vmp = Table.read(fn)
print(f'{len(t_vmp):,}')

# LAMOST M-dwarfs Cycle-StarNet
fn = 'training_data/lamost_csn_dr10.h5'
! du -sh {fn}
t_csnmd = Table.read(fn)
print(f'{len(t_csnmd):,}')

# RAVE (DR6)
fn = 'training_data/rave_dr6.h5'
! du -sh {fn}
t_rave = Table.read(fn)
print(f'{len(t_rave):,}')

# PRISTINE PIGS VMP catalog
fn = 'training_data/pristine_pigs.h5'
! du -sh {fn}
t_pigs = Table.read(fn)
print(f'{len(t_pigs):,}')

# SDSS-V SLAM (M-dwarfs)
fn = 'training_data/sdssv_slam.h5'
! du -sh {fn}
t_slam = Table.read(fn)
print(f'{len(t_slam):,}')

# SDSS-V M-Dwarf Cannon
fn = 'training_data/sdssv_mdc.h5'
! du -sh {fn}
t_sdssv_mdc = Table.read(fn)
print(f'{len(t_sdssv_mdc):,}')

29M	training_data/sdssv_aspcap.h5
467,719
14M	training_data/sdssv_bossnet.h5
268,589
17M	training_data/desi_ddp.h5
141,766
368M	training_data/desi_dr1.h5
3,214,877
98M	training_data/lamost_mrs.h5
854,512
40M	training_data/apogee_dr17.h5
572,775
36M	training_data/lamost_hotpayne.h5
361,825
37M	training_data/galah_dr4.h5
530,521
283M	training_data/lamost_ddp_dr9.h5
4,115,061
21M	training_data/lamost_mstar_ding2022.h5
308,548
27K	training_data/vmp_stars_li2022.h5
298
22M	training_data/lamost_csn_dr10.h5
272,776
17M	training_data/rave_dr6.h5
174,871
1.1M	training_data/pristine_pigs.h5
11,425
1.3M	training_data/sdssv_slam.h5
21,162
1.4M	training_data/sdssv_mdc.h5
15,994


### Join catalogs

In [255]:
def extract_pairs(t0, t1, c=(0,1), rng=None):
    if rng is None:
        rng = np.random.default_rng()

    # Join the two catalogs
    t_joined = astropy.table.join(
        t0, t1,
        keys='id',
        join_type='inner',
        table_names=('0','1')
    )
    print(f'{len(t_joined)} pairs.')

    # Determine how many parameters (3 if [alpha/Fe] is missing)
    # does each catalog have?
    n,d0 = t_joined['x_0'].shape
    _,d1 = t_joined['x_1'].shape

    # Choose which catalog to include on the left/right side of
    # the training pair
    k = rng.integers(0, 2, size=n)

    # Create empty training pairs
    x0 = np.full((n,4), 0., dtype='f4') # Unknown params default to 0
    x1 = np.full((n,4), 0., dtype='f4')
    xe0 = np.full((n,4), 1e9, dtype='f4') # Default for unknown params
    xe1 = np.full((n,4), 1e9, dtype='f4')
    xs0 = np.full((n,4), 1e9, dtype='f4') # Default for unknown params
    xs1 = np.full((n,4), 1e9, dtype='f4')
    c0 = np.empty((n,), dtype='i4')
    c1 = np.empty((n,), dtype='i4')

    # Order: (x0,x1) = (t0,t1)
    idx = (k == 0)
    x0[idx,:d0] = t_joined['x_0'][idx]
    xe0[idx,:d0] = t_joined['x_err_0'][idx]
    xs0[idx,:d0] = t_joined['sigma_x_0'][idx]
    c0[idx] = c[0]
    x1[idx,:d1] = t_joined['x_1'][idx]
    xe1[idx,:d1] = t_joined['x_err_1'][idx]
    xs1[idx,:d1] = t_joined['sigma_x_1'][idx]
    c1[idx] = c[1]

    # Order: (x0,x1) = (t1,t0)
    idx = (k == 1)
    x0[idx,:d1] = t_joined['x_1'][idx]
    xe0[idx,:d1] = t_joined['x_err_1'][idx]
    xs0[idx,:d1] = t_joined['sigma_x_1'][idx]
    c0[idx] = c[1]
    x1[idx,:d0] = t_joined['x_0'][idx]
    xe1[idx,:d0] = t_joined['x_err_0'][idx]
    xs1[idx,:d0] = t_joined['sigma_x_0'][idx]
    c1[idx] = c[0]
    
    # c0.shape = (-1,1)
    # c1.shape = (-1,1)
    
    return {'x0':x0, 'x1':x1,
            'xe0':xe0, 'xe1':xe1,
            'xs0':xs0, 'xs1':xs1,
            'c0':c0, 'c1':c1}

Extract pairs of stars observed in multiple surveys:

In [256]:
table_pairs = [
    (t_apo, 'APOGEE (DR17)'),
    (t_lmrs, 'LAMOST MRS'),
    (t_lhp, 'LAMOST HotPayne'),
    (t_galah, 'GALAH (DR4)'),
    (t_ddp, 'LAMOST DD-Payne (DR9)'),
    (t_lmd, 'LAMOST M-Stars (Ding)'),
    (t_vmp, 'VMP (Li)'),
    (t_csnmd, 'LAMOST M-Dwarfs (Zhang)'),
    (t_desi, 'DESI (DR1)'),
    (t_desi_ddp, 'DESI (Zhang)'),
    (t_sdssv_asp, 'SDSS-V (ASPCAP)'),
    (t_sdssv_boss, 'SDSS-V (BOSSNet)'),
    (t_rave, 'RAVE (DR6)'),
    (t_pigs, 'PIGS'),
    (t_slam, 'SDSS-V (SLAM)'),
    (t_sdssv_mdc, 'SDSS-V (MDw Cannon)')
]

Check table columns for NaNs:

In [257]:
for t,key in table_pairs:
    for col in t.colnames:
        if np.any(np.isnan(t[col])):
            print(f'{key}[{col}] contains NaNs!')

In [258]:
tables = [t for t,_ in table_pairs]
cat_labels = [l for _,l in table_pairs]

cat_len = [len(t) for t in tables]

n_tot = sum(cat_len)
for i in np.argsort(cat_len)[::-1]:
    print(f'{cat_labels[i]: >25s} : {cat_len[i]: >9,} : {cat_len[i]/n_tot: >6.2%}')

    LAMOST DD-Payne (DR9) : 4,115,061 : 36.31%
               DESI (DR1) : 3,214,877 : 28.37%
               LAMOST MRS :   854,512 :  7.54%
            APOGEE (DR17) :   572,775 :  5.05%
              GALAH (DR4) :   530,521 :  4.68%
          SDSS-V (ASPCAP) :   467,719 :  4.13%
          LAMOST HotPayne :   361,825 :  3.19%
    LAMOST M-Stars (Ding) :   308,548 :  2.72%
  LAMOST M-Dwarfs (Zhang) :   272,776 :  2.41%
         SDSS-V (BOSSNet) :   268,589 :  2.37%
               RAVE (DR6) :   174,871 :  1.54%
             DESI (Zhang) :   141,766 :  1.25%
            SDSS-V (SLAM) :    21,162 :  0.19%
      SDSS-V (MDw Cannon) :    15,994 :  0.14%
                     PIGS :    11,425 :  0.10%
                 VMP (Li) :       298 :  0.00%


In [259]:
# tables = [t_apo, t_lmrs, t_lhp, t_galah, t_ddp, t_lmd, t_vmp, t_csnmd,
#           t_desi, t_desi_ddp, t_sdssv_asp, t_sdssv_boss, t_rave, t_pigs]

# cat_labels = [
#     'APOGEE (DR17)',
#     'LAMOST MRS',
#     'LAMOST HotPayne',
#     'GALAH (DR4)',
#     'LAMOST DD-Payne (DR9)',
#     'LAMOST M-Stars (Ding)',
#     'VMP (Li)',
#     'LAMOST M-Dwarfs (Zhang)',
#     'DESI (DR1)',
#     'DESI (Zhang)',
#     'SDSS-V (ASPCAP)',
#     'SDSS-V (BOSSNet)',
#     'RAVE (DR6)',
#     'PIGS'
# ]

rng = np.random.default_rng(3)

pair_dict = {}

# Extract matches for each pair of tables
for j in range(len(tables)):
    for k in range(j+1,len(tables)):
        print(f'Extracting pairs for tables {j} & {k} ...')
        res = extract_pairs(tables[j], tables[k], c=(j,k), rng=rng)
        for key in res:
            if key not in pair_dict:
                pair_dict[key] = []
            pair_dict[key].append(res[key])

# Concatenate arrays and randomize their order
n = sum([len(x) for x in pair_dict['x0']])
idx = np.arange(n)
rng.shuffle(idx)
for key in pair_dict:
    pair_dict[key] = np.concatenate(pair_dict[key], axis=0)[idx]

Extracting pairs for tables 0 & 1 ...
94187 pairs.
Extracting pairs for tables 0 & 2 ...
999 pairs.
Extracting pairs for tables 0 & 3 ...
37892 pairs.
Extracting pairs for tables 0 & 4 ...
129045 pairs.
Extracting pairs for tables 0 & 5 ...
9071 pairs.
Extracting pairs for tables 0 & 6 ...
5 pairs.
Extracting pairs for tables 0 & 7 ...
2148 pairs.
Extracting pairs for tables 0 & 8 ...
15141 pairs.
Extracting pairs for tables 0 & 9 ...
302 pairs.
Extracting pairs for tables 0 & 10 ...
388840 pairs.
Extracting pairs for tables 0 & 11 ...
10913 pairs.
Extracting pairs for tables 0 & 12 ...
7336 pairs.
Extracting pairs for tables 0 & 13 ...
95 pairs.
Extracting pairs for tables 0 & 14 ...
89 pairs.
Extracting pairs for tables 0 & 15 ...
2707 pairs.
Extracting pairs for tables 1 & 2 ...
27608 pairs.
Extracting pairs for tables 1 & 3 ...
24500 pairs.
Extracting pairs for tables 1 & 4 ...
440344 pairs.
Extracting pairs for tables 1 & 5 ...
7133 pairs.
Extracting pairs for tables 1 & 6 ...
6 p

Count number of stars from each catalog that end up in the pairs:

In [260]:
_,n_cat_0 = np.unique(pair_dict['c0'], return_counts=True)
_,n_cat_1 = np.unique(pair_dict['c1'], return_counts=True)
n_cat = n_cat_0 + n_cat_1
for n,label in zip(n_cat, cat_labels):
    print(f'{label: >25s}: {n: >7,}')

            APOGEE (DR17): 698,770
               LAMOST MRS: 685,693
          LAMOST HotPayne:  77,977
              GALAH (DR4): 138,044
    LAMOST DD-Payne (DR9): 966,178
    LAMOST M-Stars (Ding): 325,822
                 VMP (Li):     193
  LAMOST M-Dwarfs (Zhang): 299,257
               DESI (DR1): 338,312
             DESI (Zhang):  50,789
          SDSS-V (ASPCAP): 586,344
         SDSS-V (BOSSNet):  82,115
               RAVE (DR6):  34,290
                     PIGS:     181
            SDSS-V (SLAM):   5,854
      SDSS-V (MDw Cannon):  13,409


Extract control points from each catalog. For each catalog, the number of control points we extract is equal to the lesser of:

  * Total # of pairs / (# of tables - 1)
  * Twice the # of paired stars from this catalog

The first condition sets the number of control points to be roughly equal to the number of pairs. The second condition prevents the number of control points for any given survey from being much larger than the number of paired stars from that survey.

In [261]:
# Extract control points from each table
ctrl_dict = {
    'x_control':[],
    'xe_control':[],
    'c_control':[],
    'from_catalog':[]
}

n_max = pair_dict['x0'].shape[0] // (len(tables)-1)

for i,t in enumerate(tables):
    n_sel = min(n_max, max(2*n_cat[i],1024), len(t))
    print(f'Extracting {n_sel} control points from table {i} ...')
    idx = np.arange(len(t))
    rng.shuffle(idx)
    idx = idx[:n_sel]
    # print(f'  --> {len(idx)} control points.')
    # Pad missing [alpha/Fe] values
    x_i = t['x'][idx].astype('f4')
    xe_i = t['x_err'][idx].astype('f4')
    n,d = x_i.shape
    if d < 4:
        x_i = np.concatenate([x_i, np.zeros((n,1), dtype='f4')], axis=1)
        xe_i = np.concatenate([xe_i, np.full((n,1), 1e9, dtype='f4')], axis=1)
    if np.any(np.isnan(x_i)):
        print(f'NaNs detected in table {i}!')
    ctrl_dict['x_control'].append(x_i)
    ctrl_dict['xe_control'].append(xe_i)
    ctrl_dict['c_control'].append(np.full(len(idx), i, dtype='i4'))
    ctrl_dict['from_catalog'].append(np.ones(len(idx), dtype='bool'))

# Add in additional control points, drawn from a wide distribution
# covering the entire parameter space
n_draw = n_max // 16
x_pct = []
for t in tables:
    x_pct_i = np.percentile(t['x'].data, [1., 99.], axis=0)
    # Pad missing [alpha/Fe] columns
    if x_pct_i.shape[1] < 4:
        x_pct_i = np.concatenate([
            x_pct_i,
            np.reshape(np.array([np.nan,np.nan]), (2,1)).astype('f4')
        ], axis=1)
    x_pct.append(x_pct_i)
x_pct = np.array(x_pct)
# x_pct = np.array([
#     np.percentile(t['x'].data, [1., 99.], axis=0)
#     for t in tables
# ])
x_low = np.nanmin(x_pct[:,0], axis=0)
x_high = np.nanmax(x_pct[:,1], axis=0)
x_mean = 0.5 * (x_high + x_low)
x_width = 0.5 * (x_high - x_low)
n_params = len(x_mean)
for i in range(len(tables)):
    print(f'Generating {n_draw} dispersed points for table {i} ...')
    x_draw = x_mean[None,:]+x_width[None,:]*rng.normal(size=(n_draw,n_params))
    x_draw = x_draw.astype('f4')
    ctrl_dict['x_control'].append(x_draw)
    ctrl_dict['xe_control'].append(np.full_like(x_draw, 0.2))
    ctrl_dict['c_control'].append(np.full(len(x_draw), i, dtype='i4'))
    ctrl_dict['from_catalog'].append(np.zeros(len(x_draw), dtype='bool'))

# Concatenate arrays and randomize their order
n = sum([len(x) for x in ctrl_dict['x_control']])
idx = np.arange(n)
rng.shuffle(idx)
for key in ctrl_dict:
    ctrl_dict[key] = np.concatenate(ctrl_dict[key], axis=0)[idx]

Extracting 143440 control points from table 0 ...
Extracting 143440 control points from table 1 ...
Extracting 143440 control points from table 2 ...
Extracting 143440 control points from table 3 ...
Extracting 143440 control points from table 4 ...
Extracting 143440 control points from table 5 ...
Extracting 298 control points from table 6 ...
Extracting 143440 control points from table 7 ...
Extracting 143440 control points from table 8 ...
Extracting 101578 control points from table 9 ...
Extracting 143440 control points from table 10 ...
Extracting 143440 control points from table 11 ...
Extracting 68580 control points from table 12 ...
Extracting 1024 control points from table 13 ...
Extracting 11708 control points from table 14 ...
Extracting 15994 control points from table 15 ...
Generating 8965 dispersed points for table 0 ...
Generating 8965 dispersed points for table 1 ...
Generating 8965 dispersed points for table 2 ...
Generating 8965 dispersed points for table 3 ...
Genera

In [262]:
# Save pairs and control points
import h5py

fn = 'training_data/pairs_and_ctrl_points.h5'
kw = dict(
    compression='gzip',
    compression_opts=3,
    chunks=True
)

with h5py.File(fn, 'w') as f:
    print('Saving pairs ...')
    for key in tqdm(pair_dict):
        f.create_dataset(f'pair_dict/{key}', data=pair_dict[key], **kw)

    print('Saving control points ...')
    for key in tqdm(ctrl_dict):
        f.create_dataset(f'ctrl_dict/{key}', data=ctrl_dict[key], **kw)

! h5ls -rv {fn}
! du -sh {fn}

Saving pairs ...


  0%|          | 0/8 [00:00<?, ?it/s]

Saving control points ...


  0%|          | 0/4 [00:00<?, ?it/s]

Opened "training_data/pairs_and_ctrl_points.h5" with sec2 driver.
/                        Group
    Location:  1:96
    Links:     1
/ctrl_dict               Group
    Location:  1:116700776
    Links:     1
/ctrl_dict/c_control     Dataset {1777022/1777022}
    Location:  1:144958954
    Links:     1
    Chunks:    {6942} 27768 bytes
    Storage:   7108088 logical bytes, 1469429 allocated bytes, 483.73% utilization
    Filter-0:  deflate-1 OPT {3}
    Type:      native int
/ctrl_dict/from_catalog  Dataset {1777022/1777022}
    Location:  1:146441231
    Links:     1
    Chunks:    {27766} 27766 bytes
    Storage:   1777022 logical bytes, 190709 allocated bytes, 931.80% utilization
    Filter-0:  deflate-1 OPT {3}
    Type:      enum native signed char {
                   FALSE            = 0
                   TRUE             = 1
               }
/ctrl_dict/x_control     Dataset {1777022/1777022, 4/4}
    Location:  1:116701480
    Links:     1
    Chunks:    {13883, 1} 55532 bytes

In [263]:
# Load training pairs and control points
import h5py

fn = 'training_data/pairs_and_ctrl_points.h5'

with h5py.File(fn, 'r') as f:
    print('Loading pairs ...')
    pair_dict = {k:f['pair_dict'][k][:] for k in tqdm(f['pair_dict'].keys())}
    print('Loading control points ...')
    ctrl_dict = {k:f['ctrl_dict'][k][:] for k in tqdm(f['ctrl_dict'].keys())}

print(f'Pair keys: {pair_dict.keys()}')
print(f'Number of pairs: {len(pair_dict["x0"]):,}')
print(f'Control keys: {ctrl_dict.keys()}')
print(f'Number of control points: {len(ctrl_dict["x_control"]):,}')

Loading pairs ...


  0%|          | 0/8 [00:00<?, ?it/s]

Loading control points ...


  0%|          | 0/4 [00:00<?, ?it/s]

Pair keys: dict_keys(['c0', 'c1', 'x0', 'x1', 'xe0', 'xe1', 'xs0', 'xs1'])
Number of pairs: 2,151,614
Control keys: dict_keys(['c_control', 'from_catalog', 'x_control', 'xe_control'])
Number of control points: 1,777,022


In [264]:
for key in ('xe0','xe1'):
    idx = (pair_dict[key] > 1e6)
    pair_dict[key][idx] = 0.05

idx = (ctrl_dict['xe_control'] > 1e6)
ctrl_dict['xe_control'][idx] = 0.05

In [265]:
print('Pair dict (min, median, max):')
for key in pair_dict:
    print(f'  {key} : {np.percentile(pair_dict[key],[0,50,100])}')

print('Control dict (min, median, max):')
for key in ctrl_dict:
    print(f'  {key} : {np.percentile(ctrl_dict[key].astype('f4'),[0,50,100])}')

Pair dict (min, median, max):
  c0 : [ 0.  4. 15.]
  c1 : [ 0.  4. 15.]
  x0 : [-4.12076283  0.05216469 10.82912827]
  x1 : [-4.36999989  0.05221309  9.95493698]
  xe0 : [0.05       0.1        0.99897337]
  xe1 : [0.05       0.1        0.91818517]
  xs0 : [9.99999978e-03 5.11957034e-02 1.00000000e+09]
  xs1 : [9.99999978e-03 5.11957034e-02 1.00000000e+09]
Control dict (min, median, max):
  c_control : [ 0.  5. 15.]
  from_catalog : [0. 1. 1.]
  x_control : [-17.51118088   0.05378003  23.43317032]
  xe_control : [0.05       0.1        0.94235247]


### Plot paired stars and control points

In [266]:
# plt.hist(5000*np.exp(ctrl_dict['x_control'][:,0]/5), bins=100, log=True)

kw = dict(bins=100, log=True, histtype='step')

for from_cat in (True,False):
    idx = ctrl_dict['from_catalog'] == from_cat
    plt.hist(ctrl_dict['x_control'][idx,0], **kw)
plt.gca().xaxis.set_minor_locator(AutoMinorLocator())
plt.grid(True, alpha=0.3)
plt.grid(True, which='minor', alpha=0.1)
plt.xlabel(r'$5\ln\left(T_{\rm eff}/5000\,\mathrm{K}\right)$')
plt.show()

for from_cat in (True,False):
    idx = ctrl_dict['from_catalog'] == from_cat
    plt.hist(ctrl_dict['x_control'][idx,1], **kw)
plt.gca().xaxis.set_minor_locator(AutoMinorLocator())
plt.grid(True, alpha=0.3)
plt.grid(True, which='minor', alpha=0.1)
plt.xlabel(r'$\left[\mathrm{Fe/H}\right]$')
plt.show()

for from_cat in (True,False):
    idx = ctrl_dict['from_catalog'] == from_cat
    plt.hist(ctrl_dict['x_control'][idx,2], **kw)
plt.gca().xaxis.set_minor_locator(AutoMinorLocator())
plt.grid(True, alpha=0.3)
plt.grid(True, which='minor', alpha=0.1)
plt.xlabel(r'$\log g$')
plt.show()

for from_cat in (True,False):
    idx = ctrl_dict['from_catalog'] == from_cat
    plt.hist(ctrl_dict['x_control'][idx,3], **kw)
plt.gca().xaxis.set_minor_locator(AutoMinorLocator())
plt.grid(True, alpha=0.3)
plt.grid(True, which='minor', alpha=0.1)
plt.xlabel(r'$\left[\alpha/\mathrm{Fe}\right]$')
plt.show()

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

In [267]:
idx = ctrl_dict['from_catalog']
x_min,x_max = np.percentile(ctrl_dict['x_control'][idx], [0.05,99.95], axis=0)
w = x_max - x_min
x_min -= 0.4*w
x_max += 0.4*w
print(x_min, x_max)

[-6.53833771 -4.3405915  -2.07642924 -2.77030193] [11.33739257  2.23339874  7.26750233  2.69810597]


In [268]:
for key in pair_dict:
    print(key, pair_dict[key].dtype)

for key in ctrl_dict:
    print(key, ctrl_dict[key].dtype)

c0 int32
c1 int32
x0 float32
x1 float32
xe0 float32
xe1 float32
xs0 float32
xs1 float32
c_control int32
from_catalog bool
x_control float32
xe_control float32


In [269]:
for i,l in enumerate(cat_labels):
    print(f'{i: >2d} {l}')

 0 APOGEE (DR17)
 1 LAMOST MRS
 2 LAMOST HotPayne
 3 GALAH (DR4)
 4 LAMOST DD-Payne (DR9)
 5 LAMOST M-Stars (Ding)
 6 VMP (Li)
 7 LAMOST M-Dwarfs (Zhang)
 8 DESI (DR1)
 9 DESI (Zhang)
10 SDSS-V (ASPCAP)
11 SDSS-V (BOSSNet)
12 RAVE (DR6)
13 PIGS
14 SDSS-V (SLAM)
15 SDSS-V (MDw Cannon)


In [270]:
# Which catalogs to plot
c = (7,14)

fig,ax_arr = plt.subplots(
    1,4, figsize=(9,2.5),
    subplot_kw={'aspect':'equal'},
    layout='constrained'
)

idx0 = (
    (pair_dict['c0']==c[0])
  & (pair_dict['c1']==c[1])
)
idx1 = (
    (pair_dict['c1']==c[0])
  & (pair_dict['c0']==c[1])
)
x = np.concatenate([pair_dict['x0'][idx0], pair_dict['x1'][idx1]], axis=0)
y = np.concatenate([pair_dict['x1'][idx0], pair_dict['x0'][idx1]], axis=0)
x_e = np.concatenate([pair_dict['xe0'][idx0], pair_dict['xe1'][idx1]], axis=0)
y_e = np.concatenate([pair_dict['xe1'][idx0], pair_dict['xe0'][idx1]], axis=0)
x_s = np.concatenate([pair_dict['xs0'][idx0], pair_dict['xs1'][idx1]], axis=0)
y_s = np.concatenate([pair_dict['xs1'][idx0], pair_dict['xs0'][idx1]], axis=0)

param_labels = ['5 ln(teff/5000K)', '[fe/h]', 'logg', '[alpha/fe]']

for i,(ax,p) in enumerate(zip(ax_arr.flat,param_labels)):
    xlim = (x_min[i],x_max[i])
    # idx = (x_e[:,i] < 1e5) & (y_e[:,i] < 1e5)
    idx = (x_s[:,i] < 1e5) & (y_s[:,i] < 1e5)
    ax.hist2d(
        x[idx,i],
        y[idx,i],
        bins=100,
        range=[xlim,xlim],
        # norm=Normalize()
        norm=LogNorm()
    )
    # xlim = ax.get_xlim()
    # ylim = ax.get_ylim()
    # xlim = (min(xlim[0],ylim[0]), max(xlim[1],ylim[1]))
    ax.plot(xlim, xlim, alpha=0.5)
    ax.set_xlim(xlim)
    ax.set_ylim(xlim)
    ax.set_title(p)
    
    ax.set_xlabel(cat_labels[c[0]])
    if i == 0:
        ax.set_ylabel(cat_labels[c[1]])
        
    ax.grid(True, alpha=0.2)

plt.show()

# Kiel diagrams of pairs and control points
fig,ax_arr = plt.subplots(1,3, figsize=(9,3), layout='constrained')
xlabel = r'$5 \ln \left( T_{\rm eff} / 5000\,\mathrm{K} \right)$'
# xlim = list(np.log(np.array([2.5, 100.])/5))
# ylim = [-1., 7.]
xlim = (x_min[0], x_max[0])
ylim = (x_min[2], x_max[2])
kw = dict(range=[xlim,ylim], bins=100)

ax = ax_arr[0]
ax.hist2d(x[:,0], y[:,2], norm=LogNorm(), **kw)
ax.set_xlim(xlim[::-1])
ax.set_ylim(ylim[::-1])
ax.set_xlabel(xlabel)
ax.set_ylabel(r'$\log g$')
ax.set_title(f'{cat_labels[c[0]]} - {cat_labels[c[1]]} pairs')
ax.grid(True, alpha=0.2)

for i in range(2):
    ax = ax_arr[i+1]
    idx = (ctrl_dict['c_control']==c[i])
    ax.hist2d(
        ctrl_dict['x_control'][idx,0],
        ctrl_dict['x_control'][idx,2],
        norm=LogNorm(),
        **kw
    )
    ax.set_xlim(xlim[::-1])
    ax.set_ylim(ylim[::-1])
    ax.set_xlabel(r'$5\ln\left( T_{\rm eff}/5000\,\mathrm{K} \right)$')
    ax.set_title(f'{cat_labels[c[i]]} control points')
    ax.grid(True, alpha=0.2)

plt.show()

<Figure size 900x250 with 4 Axes>

<Figure size 900x300 with 3 Axes>

In [271]:
list(set([s[:-1] for s in pair_dict.keys()]))

['x', 'xe', 'xs', 'c']

In [272]:
def extract_pairs_for_survey(survey_idx):
    idx0 = (pair_dict['c0'] == survey_idx)
    idx1 = (pair_dict['c1'] == survey_idx)
    
    keys = list(set([s[:-1] for s in pair_dict.keys()]))

    survey_data = {}
    paired_data = {}
    
    for k in keys:
        survey_data[k] = np.concatenate(
            [pair_dict[f'{k}0'][idx0],
             pair_dict[f'{k}1'][idx1]],
            axis=0
        )
        paired_data[k] = np.concatenate(
            [pair_dict[f'{k}1'][idx0],
             pair_dict[f'{k}0'][idx1]],
            axis=0
        )

    return survey_data, paired_data

In [273]:
d,dp = extract_pairs_for_survey(14)

In [274]:
kw = dict(bins=100, histtype='step', range=(-1,1))

for i in range(4):
    idx = (dp['xs'][:,i] < 1e6)
    plt.hist(dp['x'][idx,i]-d['x'][idx,i], label=f'param {i}', **kw)

plt.legend()
plt.show()

<Figure size 640x480 with 1 Axes>

In [275]:
kw = dict(bins=100, histtype='step', range=(-5,5))

for i in range(4):
    idx = (dp['xs'][:,i] < 1e6)
    delta = dp['x'][idx,i] - d['x'][idx,i]
    sigma = np.sqrt(dp['xs'][idx,i]**2 + d['xs'][idx,i]**2)
    pct = np.percentile(delta/sigma, [16., 50., 84.])
    print(i, pct[1], pct[2] - pct[0])
    plt.hist(delta/sigma, label=f'param {i}', **kw)

plt.legend()
plt.show()

0 0.6163188517093658 2.313267273902893
1 -0.7592656016349792 7.4000691223144495
2 0.20995377004146576 3.177050166130066
3 0.8745773136615753 1.4758226722478864


<Figure size 640x480 with 1 Axes>

In [276]:
fig,ax = plt.subplots(1, 1, figsize=(5,5))
x, y = d['x'][:,0], dp['x'][:,0]
ax.scatter(
    x, y,
    c=embeddings[dp['c']][:,0],
    edgecolors='none',
    s=4,
    alpha=0.7
)
ax.set_aspect('equal', adjustable='datalim')
xlim = np.percentile(np.hstack([x,y]), [1., 99.])
w = xlim[1] - xlim[0]
xlim = (xlim[0]-0.2*w, xlim[1]+0.2*w)
ax.set_xlim(xlim)
ax.set_ylim(xlim)
# ax.axis('tight')
# xlim = ax.get_xlim()
# xlim
# ax.set_ylim(ax.get_xlim())
plt.show()

<Figure size 500x500 with 1 Axes>

## Map stellar parameters onto common scale

We now learn a transformation from each catalog to a "harmonized" parameter space in which the catalogs all agree as much as possible.

In [277]:
batch_size = 1024
n_epochs = 32
val_frac = 1/8

n_val = int(len(pair_dict['x0']) * val_frac)
val_epochs = int(np.ceil(n_epochs / val_frac))

data_pairs = tf.data.Dataset.from_tensor_slices(tuple([
    pair_dict[k][n_val:] for k in ('x0','x1','xe0','xe1','xs0','xs1','c0','c1')
]))
data_pairs = data_pairs.shuffle(buffer_size=batch_size*64)
data_pairs = data_pairs.repeat(n_epochs)
data_pairs = data_pairs.batch(batch_size, drop_remainder=True)

val_pairs = tf.data.Dataset.from_tensor_slices(tuple([
    pair_dict[k][:n_val] for k in ('x0','x1','xe0','xe1','xs0','xs1','c0','c1')
]))
val_pairs = val_pairs.shuffle(buffer_size=batch_size*64)
val_pairs = val_pairs.repeat(val_epochs)
val_pairs = val_pairs.batch(batch_size, drop_remainder=True)

data_pairs

<_BatchDataset element_spec=(TensorSpec(shape=(1024, 4), dtype=tf.float32, name=None), TensorSpec(shape=(1024, 4), dtype=tf.float32, name=None), TensorSpec(shape=(1024, 4), dtype=tf.float32, name=None), TensorSpec(shape=(1024, 4), dtype=tf.float32, name=None), TensorSpec(shape=(1024, 4), dtype=tf.float32, name=None), TensorSpec(shape=(1024, 4), dtype=tf.float32, name=None), TensorSpec(shape=(1024,), dtype=tf.int32, name=None), TensorSpec(shape=(1024,), dtype=tf.int32, name=None))>

In [278]:
n_val = int(len(ctrl_dict['x_control']) * val_frac)

data_ctrl = tf.data.Dataset.from_tensor_slices(tuple([
    ctrl_dict[k][n_val:] for k in ('x_control','xe_control','c_control')
]))
data_ctrl = data_ctrl.shuffle(buffer_size=batch_size*64)
data_ctrl = data_ctrl.repeat(n_epochs)
data_ctrl = data_ctrl.batch(batch_size, drop_remainder=True)

val_ctrl = tf.data.Dataset.from_tensor_slices(tuple([
    ctrl_dict[k][:n_val] for k in ('x_control','xe_control','c_control')
]))
val_ctrl = val_ctrl.shuffle(buffer_size=batch_size*64)
val_ctrl = val_ctrl.repeat(val_epochs)
val_ctrl = val_ctrl.batch(batch_size, drop_remainder=True)

data_ctrl

<_BatchDataset element_spec=(TensorSpec(shape=(1024, 4), dtype=tf.float32, name=None), TensorSpec(shape=(1024, 4), dtype=tf.float32, name=None), TensorSpec(shape=(1024,), dtype=tf.int32, name=None))>

In [279]:
n_steps_est = min([len(x) for x in (data_pairs,data_ctrl,val_pairs,val_ctrl)])
print(f'{n_steps_est} steps expected.')

48590 steps expected.


In [296]:
def plot_loss_hist(metrics_hist):
    # Smooth time series with a Kaiser window
    def smooth_time_series(x, n_smooth=None):
        if n_smooth is None:
            n_smooth = np.clip(len(x)//16, 4, 128)
        w = np.kaiser(2*n_smooth,5)
        w /= np.sum(w)
        x_conv = scipy.ndimage.convolve(x, w, mode='reflect')
        return x_conv
    
    step_full = np.arange(len(metrics_hist['loss'])) / 1024
    step_half = step_full[len(step_full)//2:]
    
    # Create subplots
    fig,(ax0,ax1,ax2) = plt.subplots(1,3, figsize=(8,2.5), layout='constrained')
    
    # All penalties and loss on one semilogy plot
    ax0.semilogy(step_full, metrics_hist['loss'], label='loss')
    ax0.semilogy(step_full, metrics_hist['penalty_pair'], label='penalty (pairs)', alpha=0.7)
    ax0.semilogy(step_full, metrics_hist['penalty_ctrl'], label='penalty (control)', alpha=0.7)
    ax0.semilogy(step_full, metrics_hist['penalty_div'], label='penalty (div)', alpha=0.7)
    ax0.semilogy(step_full, metrics_hist['penalty_l2'], label='penalty (l2)', alpha=0.7)
    leg = ax0.legend(fontsize=8)
    # Set opacity of legend lines to 1
    for legline in leg.get_lines():
        legline.set_alpha(1)

    
    # Smoothed training and validation loss in the 2nd half of training
    loss_smooth = smooth_time_series(metrics_hist['loss'])
    loss_val_smooth = smooth_time_series(metrics_hist['loss_val'])
    
    n = len(loss_smooth)
    l, = ax1.plot(step_half, metrics_hist['loss'][n//2:], alpha=0.25)
    ax1.plot(
        step_half,
        loss_smooth[n//2:],
        c=l.get_color(),
        label='training loss'
    )
    ax1.plot(
        step_half,
        loss_val_smooth[n//2:],
        label='validation loss'
    )
    ax1.legend(fontsize=8)

    # Smoothed penalties in the 2nd half of training
    penalty_pair_smooth = smooth_time_series(metrics_hist['penalty_pair'])
    penalty_ctrl_smooth = smooth_time_series(metrics_hist['penalty_ctrl'])
    penalty_div_smooth = smooth_time_series(metrics_hist['penalty_div'])
    penalty_l2_smooth = smooth_time_series(metrics_hist['penalty_l2'])
    ax2.plot([],[])  # dummy plot to get colors right
    for y,l in [(penalty_pair_smooth, 'penalty (pairs)'),
                (penalty_ctrl_smooth, 'penalty (control)'),
                (penalty_div_smooth, 'penalty (div)'),
                (penalty_l2_smooth, 'penalty (l2)')]:
        ax2.plot(
            step_half,
            y[n//2:],
            label=l
        )
    
    for ax in (ax0,ax1,ax2):
        ax.set_xlabel('step / 1024')
        ax.grid(True, alpha=0.2)
        ax.xaxis.set_minor_locator(AutoMinorLocator())
    
    plt.show()

Choose a consistent color scheme for the different surveys / reductions, which we will use in all of the following plots:

In [281]:
from distinctipy import distinctipy, color_swatch
class_color = distinctipy.get_colors(
    len(tables),
    exclude_colors=[(1,1,1),(0,0,0)],
    pastel_factor=0.,
    n_attempts=1000,
    rng=1
)
print('Using colors:', class_color)
color_swatch(class_color)

Using colors: [(0.0, 1.0, 0.0), (1.0, 0.0, 1.0), (0.0, 0.5, 1.0), (1.0, 0.5, 0.0), (0.5, 0.75, 0.5), (0.381945586915037, 0.10787169822799336, 0.6031504697402067), (0.8838709662651899, 0.00754724122007755, 0.0985531196862357), (0.7611183745927239, 0.4639600404602737, 0.9936303837820597), (0.0, 1.0, 1.0), (1.0, 1.0, 0.0), (0.30964414196302903, 0.4702414145890923, 0.0008697560813262228), (0.0, 0.0, 1.0), (0.0, 1.0, 0.5), (0.0, 0.5, 0.5), (0.727878574406246, 0.346098195032302, 0.40072078015163903), (0.5529202646814797, 0.8823263245721562, 0.029363404730843312)]


<Figure size 800x800 with 1 Axes>

In [282]:
import seaborn as sns

In [283]:
def plot_transformed_kiel(bij, kde=False):
    n_plot = 1024*2
    xt_pred = []
    for k,t in enumerate(tables):
        x = tf.constant(t['x'][:n_plot].astype('f4'))
        # Pad missing [alpha/Fe] values
        if x.shape[1] < 4:
            x = tf.concat([x, tf.zeros([x.shape[0],1])], axis=1)
        c = k * tf.ones(x.shape[0], dtype=tf.int32)
        xt_pred.append(bij.return_path(x, c, 2).numpy())

    n_classes = len(xt_pred)
    
    fig,ax_arr = plt.subplots(1,2, figsize=(8,4), dpi=120)
    
    kw = dict(edgecolors='none', s=4, alpha=0.2)
    # prop_cycle = plt.rcParams['axes.prop_cycle']
    # colors = prop_cycle.by_key()['color']
    # class_color = colors[:n_classes]
    
    x = np.concatenate([x[0] for x in xt_pred], axis=0)
    xylim = np.percentile(x[:,::2], [1., 99.], axis=0)
    w = xylim[1] - xylim[0]
    xylim[0] -= 0.3*w
    xylim[1] += 0.3*w
    xlim,ylim = xylim.T
    xlim = xlim[::-1]
    ylim = ylim[::-1]
    
    for j,ax in enumerate(ax_arr.flat):
        for k,label in enumerate(cat_labels):
            if kde:
                x = xt_pred[k][j,:,:]
                sns.kdeplot(
                    x=x[:,0],
                    y=x[:,2],
                    color=class_color[k],
                    alpha=0.3,
                    label=label,
                    ax=ax
                )
            else:
                x = xt_pred[k][j,:n_plot,:]
                sc = ax.scatter(
                    x[:,0], x[:,2],
                    color=class_color[k],
                    label=label,
                    **kw
                )
        ax.set_xlabel(r'$5\ln\left(T_{\rm eff}/5000\,\mathrm{K}\right)$')
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.grid(True, alpha=0.2)
        ax.grid(True, which='minor', alpha=0.08)
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
    
    ax_arr[0].legend(loc='upper left', markerscale=3, fontsize=5)
    ax_arr[0].set_ylabel(r'$\log g$')
    ax_arr[0].set_title('Original')
    ax_arr[1].set_title('Transformed')
    
    plt.show()

In [284]:
def plot_transformed_params(bij):
    param_labels = [
        r'$5\ln\left(T_{\rm eff}/5000\,\mathrm{K}\right)$',
        r'$\left[\mathrm{Fe/H}\right]$',
        r'$\log g$',
        r'$\left[\alpha/\mathrm{Fe}\right]$'
    ]
    
    n_plot = 1024*2
    xt_pred = []
    for k,t in enumerate(tables):
        x = tf.constant(t['x'][:n_plot].astype('f4'))
        # Pad missing [alpha/Fe] values
        if x.shape[1] < 4:
            x = tf.concat([x, tf.zeros([x.shape[0],1])], axis=1)
        c = k * tf.ones(x.shape[0], dtype=tf.int32)
        xt_pred.append(bij.return_path(x, c, 2).numpy())

    n_classes = len(xt_pred)
    
    fig,ax_arr = plt.subplots(2,2, figsize=(8,8), dpi=120, layout='constrained')
    
    kw = dict(edgecolors='none', s=4, alpha=0.2)
    # prop_cycle = plt.rcParams['axes.prop_cycle']
    # colors = prop_cycle.by_key()['color']
    # class_color = colors[:n_classes]
    
    x = np.concatenate([x[0] for x in xt_pred], axis=0)
    xylim = np.percentile(x, [1., 99.], axis=0)
    w = xylim[1] - xylim[0]
    xylim[0] -= 0.3*w
    xylim[1] += 0.3*w
    xylim[:,0] = xylim[::-1,0]
    xylim[:,2] = xylim[::-1,2]

    param_idx = [(0,2),(1,3)]

    for i,ax_row in enumerate(ax_arr):
        p0,p1 = param_idx[i]
        for j,ax in enumerate(ax_row):
            for k,label in enumerate(cat_labels):
                    x = xt_pred[k][j,:n_plot,:]
                    sc = ax.scatter(
                        x[:,p0], x[:,p1],
                        color=class_color[k],
                        label=label,
                        **kw
                    )
            ax.set_xlabel(param_labels[p0])
            if j == 0:
                ax.set_ylabel(param_labels[p1])
            else:
                ax.set_yticklabels([])
            ax.xaxis.set_minor_locator(AutoMinorLocator())
            ax.yaxis.set_minor_locator(AutoMinorLocator())
            ax.grid(True, alpha=0.2)
            ax.grid(True, which='minor', alpha=0.08)
            ax.set_xlim(xylim[:,p0])
            ax.set_ylim(xylim[:,p1])
    
    ax_arr.flat[0].legend(loc='upper left', markerscale=3, fontsize=5)
    ax_arr.flat[0].set_title('Original')
    ax_arr.flat[1].set_title('Transformed')
    
    plt.show()

In [586]:
plot_transformed_params(bij)

<Figure size 960x960 with 4 Axes>

In [140]:
plot_transformed_kiel(bij, kde=True)

/tmp/ipykernel_953174/300025392.py:34: UserWarning: KDE cannot be estimated (0 variance or perfect covariance). Pass `warn_singular=False` to disable this warning.
  sns.kdeplot(
/tmp/ipykernel_953174/300025392.py:58: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax_arr[0].legend(loc='upper left', markerscale=3, fontsize=5)


<Figure size 960x480 with 2 Axes>

In [106]:
! tar -cf models_1.tar models/

In [54]:
for i,(d_pair,d_ctrl,v_pair,v_ctrl) in enumerate(zip(*datasets)):
    # Training points
    (x0,x1,xe0,xe1,xs0,xs1,c0,c1) = d_pair
    (x_ctrl,xe_ctrl,c_ctrl) = d_ctrl
    print(tf.reduce_max(xe0,axis=0))
    print(len(x0))
    break

tf.Tensor([5.65536e-01 3.00000e-01 1.00000e+09 2.00000e-01], shape=(4,), dtype=float32)
2048


In [71]:
[v for v in bij.trainable_variables if 'embedding' in v.name]

[<tf.Variable 'vector_network/embedding/embeddings:0' shape=(16, 4) dtype=float32, numpy=
 array([[-4.1685459e-01,  6.7802989e-01, -2.8970617e-01, -8.0235612e-01],
        [-3.0028531e-01, -4.9597326e-01, -8.1823421e-01, -1.2513949e-03],
        [ 3.9886463e-02, -1.0125303e+00, -8.4422612e-01,  8.5123068e-01],
        [-6.9563520e-01, -1.4420858e-01,  8.8056380e-01,  7.6420760e-01],
        [-8.8387632e-01,  1.0000446e+00, -8.0862939e-01, -2.6085842e-01],
        [-8.4467687e-02, -3.7119403e-01,  1.4652002e+00,  1.4230562e+00],
        [-2.5269991e-01,  5.1945727e-02,  1.7034410e-01, -1.0961838e-01],
        [-1.0326761e-01,  9.5199510e-02,  1.6381899e+00,  3.2629421e-01],
        [ 4.3814886e-01,  8.9114761e-01, -3.8248596e-01,  1.4243183e+00],
        [-8.9613020e-01, -4.9058276e-01,  5.5206990e-01,  7.0913577e-01],
        [-7.1831250e-01,  4.8465163e-01, -4.5832667e-01, -7.7032429e-01],
        [-3.0471221e-01, -1.8718007e-01,  3.3722848e-01,  1.3865800e+00],
        [ 5.9999634e-0

In [70]:
[v.name for v in bij.trainable_variables]

['vector_network/embedding/embeddings:0',
 'vector_network/film_0/b:0',
 'vector_network/film_0/w:0',
 'vector_network/film_1/b:0',
 'vector_network/film_1/w:0',
 'vector_network/film_2/b:0',
 'vector_network/film_2/w:0',
 'vector_network/film_3/b:0',
 'vector_network/film_3/w:0',
 'vector_network/film_4/b:0',
 'vector_network/film_4/w:0',
 'vector_network/film_5/b:0',
 'vector_network/film_5/w:0',
 'vector_network/hidden_0/b:0',
 'vector_network/hidden_0/w:0',
 'vector_network/hidden_1/b:0',
 'vector_network/hidden_1/w:0',
 'vector_network/hidden_2/b:0',
 'vector_network/hidden_2/w:0',
 'vector_network/hidden_3/b:0',
 'vector_network/hidden_3/w:0',
 'vector_network/hidden_4/b:0',
 'vector_network/hidden_4/w:0',
 'vector_network/hidden_5/b:0',
 'vector_network/hidden_5/w:0',
 'vector_network/norm_0/offset:0',
 'vector_network/norm_0/scale:0',
 'vector_network/norm_1/offset:0',
 'vector_network/norm_1/scale:0',
 'vector_network/norm_2/offset:0',
 'vector_network/norm_2/scale:0',
 'vecto

In [74]:
# Filter trainable variables
vars_filtered = variables
for s in ['hidden']:
    vars_filtered = [v for v in vars_filtered if s not in v.name]
print('Training variables:')
print('\n'.join([v.name for v in vars_filtered]))

Training variables:
vector_network/embedding/embeddings:0
vector_network/film_0/b:0
vector_network/film_0/w:0
vector_network/film_1/b:0
vector_network/film_1/w:0
vector_network/film_2/b:0
vector_network/film_2/w:0
vector_network/film_3/b:0
vector_network/film_3/w:0
vector_network/film_4/b:0
vector_network/film_4/w:0
vector_network/film_5/b:0
vector_network/film_5/w:0
vector_network/norm_0/offset:0
vector_network/norm_0/scale:0
vector_network/norm_1/offset:0
vector_network/norm_1/scale:0
vector_network/norm_2/offset:0
vector_network/norm_2/scale:0
vector_network/norm_3/offset:0
vector_network/norm_3/scale:0
vector_network/norm_4/offset:0
vector_network/norm_4/scale:0
vector_network/norm_5/offset:0
vector_network/norm_5/scale:0
vector_network/output/b:0
vector_network/output/w:0


In [546]:
[v.name for v in bij.trainable_variables if ('w:' in v.name or 'kernel' in v.name)]

['vector_network/film_0/w:0',
 'vector_network/film_1/w:0',
 'vector_network/film_2/w:0',
 'vector_network/film_3/w:0',
 'vector_network/film_4/w:0',
 'vector_network/film_5/w:0',
 'vector_network/hidden_0/w:0',
 'vector_network/hidden_1/w:0',
 'vector_network/hidden_2/w:0',
 'vector_network/hidden_3/w:0',
 'vector_network/hidden_4/w:0',
 'vector_network/hidden_5/w:0',
 'vector_network/output/w:0']

In [67]:
tf.add_n([
    tf.nn.l2_loss(v) for v in bij.trainable_variables
    if 'w:' in v.name
]) * 1e-4

<tf.Tensor: shape=(), dtype=float32, numpy=0.0901213139295578>

In [68]:
(metrics_history['penalty_l2'][-1] - metrics_history['penalty_l2'][0]) / np.std(metrics_history['penalty_l2'])

np.float64(3.0956743968094362)

In [291]:
n_steps

48590

In [445]:
ln_err_model = VectorNetwork(
    n_cat, n_dim, 2, 16,
    use_residual=True, use_film=True, use_layer_norm=True,
    output_bias=np.log(0.01),
    output_scale=0.01,
    name='error_model'
)

Using 4-dim embedding for 16 categories.


In [446]:
ln_err_model(tf.zeros([1,n_dim]), tf.ones([1], dtype=tf.int32))

<tf.Tensor: shape=(1, 4), dtype=float32, numpy=array([[-4.579503 , -4.6067185, -4.5925417, -4.599251 ]], dtype=float32)>

In [427]:
tf.reduce_sum([tf.size(x) for x in err_model.trainable_variables])

<tf.Tensor: shape=(), dtype=int32, numpy=888>

In [433]:
tf.reduce_sum([tf.size(x) for x in bij.trainable_variables])

<tf.Tensor: shape=(), dtype=int32, numpy=11352>

### Training loop

In [458]:
lnx_plot = np.linspace(-500., 500., 10)
lnx_plot + np.log(1 + np.sqrt(1 + np.exp(-2*lnx_plot)))

/tmp/ipykernel_2429448/1128804503.py:2: RuntimeWarning: overflow encountered in exp
  lnx_plot + np.log(1 + np.sqrt(1 + np.exp(-2*lnx_plot)))


array([         inf,          inf,   0.        ,   0.        ,
         0.        ,  56.24870274, 167.35981385, 278.47092496,
       389.58203607, 500.69314718])

In [461]:
tf.math.softplus(lnx_plot)

<tf.Tensor: shape=(128,), dtype=float64, numpy=
array([0.04858735, 0.05087909, 0.05327607, 0.05578285, 0.05840418,
       0.06114496, 0.0640103 , 0.06700546, 0.07013592, 0.07340734,
       0.07682557, 0.08039666, 0.08412688, 0.08802267, 0.0920907 ,
       0.09633783, 0.10077115, 0.10539794, 0.11022569, 0.11526209,
       0.12051504, 0.12599265, 0.13170323, 0.13765526, 0.14385742,
       0.1503186 , 0.15704782, 0.16405431, 0.17134742, 0.17893667,
       0.18683171, 0.19504231, 0.20357834, 0.21244977, 0.22166665,
       0.23123906, 0.24117714, 0.25149103, 0.26219087, 0.27328675,
       0.28478873, 0.29670677, 0.30905071, 0.32183027, 0.33505501,
       0.34873426, 0.36287717, 0.37749259, 0.39258912, 0.40817502,
       0.42425823, 0.4408463 , 0.45794638, 0.47556519, 0.49370899,
       0.51238356, 0.53159419, 0.55134562, 0.57164205, 0.59248712,
       0.61388389, 0.6358348 , 0.65834173, 0.68140591, 0.70502795,
       0.72920787, 0.75394504, 0.77923822, 0.80508555, 0.83148457,
       0.85843

In [484]:
chi2_plot = np.linspace(0., 2., 100)
plt.axhline(1, c='k', alpha=0.5, ls='--')
plt.plot(chi2_plot, 1/np.sqrt(1+chi2_plot**2), label=r'arcsinh(x)')
plt.plot(chi2_plot, 1/(1+chi2_plot), label='softplus(ln(x))')
plt.xlabel(r'$x$')
plt.ylabel(r'$df/dx$')
plt.legend()
# plt.plot(chi2_plot, 40*tf.math.softplus(np.log(chi2_plot/40)))
plt.show()

<Figure size 640x480 with 1 Axes>

In [491]:
15*tf.keras.ops.logaddexp(tf.math.log(chi2_plot/15), 0.5 * tf.nn.softplus(2.0 * tf.math.log(chi2_plot/15)))

<tf.Tensor: shape=(50,), dtype=float64, numpy=
array([ 0.        ,  2.03457201,  4.03287097,  5.96406017,  7.80611572,
        9.54644954, 11.18049643, 12.70946791, 14.13817922, 15.47335801,
       16.72249294, 17.893125  , 18.99245133, 20.02713071, 21.00321047,
       21.92612234, 22.80071442, 23.63130004, 24.42171212, 25.17535735,
       25.89526682, 26.58414204, 27.24439578, 27.87818794, 28.48745689,
       29.0739466 , 29.63923019, 30.18473029, 30.71173671, 31.22142173,
       31.71485335, 32.19300683, 32.65677482, 33.10697606, 33.54436313,
       33.96962917, 34.38341383, 34.78630852, 35.17886099, 35.56157948,
       35.93493634, 36.29937124, 36.65529412, 37.00308772, 37.34310991,
       37.67569577, 38.00115945, 38.31979587, 38.63188222, 38.93767934])>

In [489]:
chi2_plot = np.linspace(0., 100.)
plt.axline((0,0), slope=1, c='k', alpha=0.5, ls='--')
plt.plot(chi2_plot, 15*np.arcsinh(chi2_plot/15))
plt.plot(chi2_plot, 15*tf.keras.ops.logaddexp(np.log(chi2_plot/15), 0.5 * tf.nn.softplus(2.0 * np.log(chi2_plot/15))))
# plt.plot(chi2_plot, 40*tf.math.softplus(np.log(chi2_plot/40)))
plt.show()

/tmp/ipykernel_2429448/4072726706.py:4: RuntimeWarning: divide by zero encountered in log
  plt.plot(chi2_plot, 15*tf.keras.ops.logaddexp(np.log(chi2_plot/15), 0.5 * tf.nn.softplus(2.0 * np.log(chi2_plot/15))))


<Figure size 640x480 with 1 Axes>

In [470]:
lnx_plot = np.linspace(-5., 1., 128)
plt.plot(np.exp(lnx_plot), np.arcsinh(np.exp(lnx_plot)), label='arcsinh(x)')
# plt.plot(np.exp(lnx_plot), lnx_plot + np.log(1 + np.sqrt(1 + np.exp(-2*lnx_plot))))
plt.plot(np.exp(lnx_plot), tf.math.softplus(lnx_plot), label='softplus(lnx)')
plt.axline((0,0), slope=1, c='k', alpha=0.5, ls='--')
plt.legend()
# y_plot = lnx_plot + np.log(1 + np.sqrt(1 + np.exp(-2*lnx_plot)))
# plt.plot(np.exp(lnx_plot), y_plot)

<Figure size 640x480 with 1 Axes>

In [492]:
def arcsinh_of_exp(lnx):
    """
    Calculates arcsinh(x), given input ln(x), in a numerically stable
    manner, using Tensorflow functions.
    """
    return tf.keras.ops.logaddexp(lnx, 0.5*tf.math.softplus(2*lnx))

In [510]:
lnx_plot = np.linspace(-15., 15., 128).astype('f4')
z_samp = np.random.normal(size=(128,1024)).astype('f4')
y_plot = tf.reduce_mean(
    tf.math.softplus(2*(lnx_plot[:,None]-tf.math.log(1.)))
    + tf.math.exp(tf.math.log(z_samp**2) - 2*lnx_plot[:,None]),
    axis=1
)
plt.semilogy(lnx_plot, y_plot)
plt.grid(True, alpha=0.1)
plt.show()

<Figure size 640x480 with 1 Axes>

In [539]:
delta_x_plot = np.linspace(-5., 5., 128)
delta_x_plot = np.stack([delta_x_plot]+[np.zeros(128)]*2, axis=1)
delta_x_plot = delta_x_plot.astype('f4')
ln_delta_err_plot = np.linspace(-5., 5., 15).astype('f4')[:,None,None]

ln_x1_diffsq = tf.math.log(delta_x_plot**2)[None,:,:]
ln_ivar1_infl = (
    tf.math.softplus(2*(1.-ln_delta_err_plot))
    - 2*1.
)
ln_chi2_plot = tf.math.reduce_logsumexp((
        ln_x1_diffsq + ln_ivar1_infl # chi^2 term
        + tf.math.softplus(2*(ln_delta_err_plot-1.)) # lndet term
    ),
    axis=2
)

cmap = plt.get_cmap('turbo').copy()
norm = plt.Normalize(vmin=-5,vmax=5.)
for yy,vv in zip(ln_chi2_plot,ln_delta_err_plot.flat):
    plt.plot(delta_x_plot[:,0], yy, color=cmap(norm(vv)), alpha=0.5)
import matplotlib.cm as cm
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
plt.gcf().colorbar(sm, ax=plt.gca())
plt.show()

<Figure size 640x480 with 2 Axes>

In [531]:
ln_chi2_plot

<tf.Tensor: shape=(5, 128), dtype=float32, numpy=
array([[13.218967  , 13.187221  , 13.154963  , 13.122174  , 13.0888405 ,
        13.054941  , 13.020457  , 12.985369  , 12.949654  , 12.913289  ,
        12.876251  , 12.838514  , 12.800051  , 12.760834  , 12.720833  ,
        12.680016  , 12.638347  , 12.595792  , 12.552312  , 12.507866  ,
        12.46241   , 12.415895  , 12.368275  , 12.319491  , 12.269489  ,
        12.2182045 , 12.165569  , 12.111512  , 12.055953  , 11.998806  ,
        11.939979  , 11.879368  , 11.816863  , 11.752341  , 11.685668  ,
        11.616695  , 11.545259  , 11.471177  , 11.394244  , 11.314234  ,
        11.230888  , 11.143918  , 11.052994  , 10.957737  , 10.857717  ,
        10.752429  , 10.64129   , 10.523608  , 10.398567  , 10.265185  ,
        10.122267  ,  9.968345  ,  9.801581  ,  9.619638  ,  9.419471  ,
         9.19702   ,  8.946693  ,  8.660492  ,  8.326384  ,  7.9250417 ,
         7.422413  ,  6.7494683 ,  5.7278175 ,  3.530593  ,  3.530593  ,
 

In [521]:
print(ln_x1_diffsq.shape)
print(ln_ivar1_infl.shape)
print(ln_delta_err_plot.shape)
print(ln_chi2_plot.shape)
print(delta_x_plot.shape)

(1, 128, 3)
(5, 1, 1)
(5, 1, 1)
(5, 128)
(128, 3)


In [288]:
# Neural network structure
hidden_size = 64
n_hidden = 3
vector_nn_kwargs = dict(
    use_residual=True,
    use_film=True,
    use_layer_norm=True
)

# Error model neural network structure
hidden_size_err = 16
n_hidden_err = 2
err_nn_kwargs = dict(
    use_residual=True,
    use_film=True,
    use_layer_norm=True,
    output_bias=np.log(0.01),
    output_scale=0.01
)

plot_every = 5*60.0 # seconds

# Model weights l2 loss
l2_weights = 1e-4

# Probability cloud scale (isotropic)
prob_cloud_scale = 0.05

# Value at which to start dampening the effect of chi^2
chi2_turnover = 15.

# Regularization constants for path length
l1_path = 1e-1
l2_path = 1e-1
# sigma_x = 0.1

# Regularization constants for final distance of points
l1_total = 1e-1
l2_total = 1e-1
sigma_ctrl = 0.3

# Regularization constants for stretching/contraction of space
eps_div = 0.1
weight_div = 0.3e1

n_dim = pair_dict['x0'].shape[1]
n_cat = len(tables)

# Set up bijector model
bij = ODEBijector(n_dim, n_cat, hidden_size, n_hidden,
                  vector_network_kwargs=vector_nn_kwargs)
n_bij_vars = sum([np.prod(tf.shape(v)) for v in bij.trainable_variables])
print(f'{n_bij_variables} variables in bijector model.')

# Set up error model
ln_err_model = VectorNetwork(n_cat, n_dim,
                             n_hidden_err, hidden_size_err,
                             **err_nn_kwargs)
ln_err_model(tf.zeros([1,n_dim]), tf.ones([1], dtype=tf.int32)) # Init
n_err_vars = sum([np.prod(tf.shape(v))
                  for v in ln_err_model.trainable_variables])
print(f'{n_err_variables} variables in error model.')

# Define trainable variables
variables = bij.trainable_variables + ln_err_model.trainable_variables
n_variables = sum([np.prod(tf.shape(v)) for v in variables])
print(f'{n_variables} variables in model.')

# Set up optimizer
n_steps = min([len(data_pairs), len(data_ctrl)])
# learning_rate = keras.optimizers.schedules.PiecewiseConstantDecay(
#     [n_steps//3, 2*n_steps//3], [1e-3, 1e-4, 1e-5]
# )
learning_rate = keras.optimizers.schedules.PiecewiseConstantDecay(
    [i*n_steps//8 for i in range(1,8)],
    [1e-3 / 2**i for i in range(8)]
)
# learning_rate = keras.optimizers.schedules.PiecewiseConstantDecay(
#     [n_steps//3, 2*n_steps//3], [5e-4, 5e-5, 5e-6]
# )
opt = keras.optimizers.Adam(learning_rate=learning_rate)

# Function to take one training step
@tf.function
def grad_step(x0, x1, xe0, xe1, xs0, xs1, c0, c1,
              x_ctrl, xe_ctrl, c_ctrl,
              apply_grads=True, freeze_list=()):
    print('Tracing <grad_step> ...')

    # Filter trainable variables
    vars_filtered = variables
    for s in freeze_list:
        vars_filtered = [v for v in vars_filtered if s not in v.name]
    print('Training variables:')
    print('\n'.join([v.name for v in vars_filtered]))
    
    # x0 = x0 + prob_cloud_scale * tf.random.normal(x0.shape)
    
    # Add in arbitrary scatter to control points
    x_ctrl = x_ctrl + sigma_ctrl * tf.random.normal(x_ctrl.shape)

    # Create a paired control point, to measure stretching/contraction
    x1_ctrl = x_ctrl + eps_div * tf.random.normal(x_ctrl.shape)

    # When xs0 ~ infinity, then set xs1 to ~infinity. This occurs when
    # [alpha/Fe] is not set for x0. In that case, xs1 for [alpha/Fe] must
    # also be set to ~infinity, so that this parameter does not affect the
    # loss.
    xs1 = tf.where(xs0 > 1e6, xs0, xs1)
    
    with tf.GradientTape(watch_accessed_variables=False) as g:
        g.watch(vars_filtered)

        # Additional (learned) errors for x0, x1
        ln_delta_err_0 = tf.math.clip(ln_err_model(x0, c0), -15, 15)
        ln_delta_err_1 = tf.math.clip(ln_err_model(x1, c1), -15, 15)

        # Add in Gaussian uncertainties to x0
        xs0_clipped = tf.clip_by_value(xs0, 1e-5, 1.)
        delta_x0 = xs0_clipped * tf.random.normal(x0.shape)
        x0 = x0 + delta_x0
        
        # Transform x0 to coordinate system of x1
        x0_pred,s0 = bij.forward_with_s(x0, c=c0)
        x1_pred,s1 = bij.inverse_with_s(x0_pred, c=c1)
        
        # Transform control points to common coordinate system
        x_ctrl_pred,s_ctrl = bij.forward_with_s(x_ctrl, c=c_ctrl)

        # Transform offset control points to common coordinate system
        x1_ctrl_pred,s_ctrl1 = bij.forward_with_s(x1_ctrl, c=c_ctrl)
        
        # Compare transformed x0 to x1
        ln_x1_diffsq = tf.math.log((x1_pred-x1)**2)
        ln_xs1 = tf.math.log(xs1)
        ln_ivar1_infl = (
            tf.math.softplus(2*(ln_xs1-ln_delta_err_1))
            - 2*ln_xs1
        )
        ln_chi2 = tf.math.logsumexp((
                ln_x1_diffsq + ln_ivar1_infl # chi^2 term
                + tf.math.softplus(2*(ln_delta_err_1-ln_xs1)) # lndet term
            ),
            axis=1
        )
        # TODO: apply turnover in log space
        # chi2 = tf.math.exp(ln_x1_diff2 + ln_ivar1_infl)
        # chi2 = tf.reduce_sum(((x1_pred-x1)/xs1)**2, axis=1)
        # chi2 = tf.reduce_sum(((x1_pred-x1)/prob_cloud_scale)**2, axis=1)

        # Soften chi^2
        # chi2 = chi2_turnover * tf.math.asinh(chi2/chi2_turnover)
        chi2 = chi2_turnover*arcsinh_of_exp(ln_chi2-tf.math.log(chi2_tunover))

        # Importance sampling term
        chi2 = chi2 + tf.reduce_sum(
            tf.math.softplus(2*(ln_delta_err_0-tf.math.log(xs0_clipped)))
            + tf.math.exp(tf.math.log(delta_x0**2) - 2*ln_delta_err_0),
            axis=1
        )

        # Take the average chi2 across all parameters of all pairs
        chi2 = tf.reduce_mean(chi2)
        
        # Penalties of pair transformations
        length2 = s0**2 + s1**2
        penalty_pair = (
            l1_path * tf.reduce_mean(tf.math.sqrt(length2))
          + l2_path * tf.reduce_mean(length2)
        )
        
        # Penalties on control-point transformations
        length1_ctrl = tf.reduce_mean(tf.abs(x_ctrl_pred-x_ctrl)/xe_ctrl)
        length2_ctrl = tf.reduce_mean(((x_ctrl_pred-x_ctrl)/xe_ctrl)**2)
        penalty_ctrl = (
            l1_total * length1_ctrl
          + l2_total * length2_ctrl
          + l1_path * tf.reduce_mean(tf.math.abs(s_ctrl))
          + l2_path * tf.reduce_mean(s_ctrl**2)
        )

        # Penalties on stretching/contraction of space
        length_div = tf.reduce_sum((x1_ctrl-x_ctrl)**2, axis=1)
        length_div_p = tf.reduce_sum((x1_ctrl_pred-x_ctrl_pred)**2, axis=1)
        eps = 1e-2 * eps_div
        penalty_div = weight_div * tf.reduce_mean(
            tf.math.log((length_div+eps)/(length_div_p+eps))**2
        )

        # Penalties on model weights
        penalty_l2 = l2_weights * tf.add_n([
            tf.nn.l2_loss(v) for v in vars_filtered
            if 'w:' in v.name
        ])
        
        # loss = chi2 + penalty_pair + penalty_ctrl + penalty_div
        loss = chi2 + penalty_ctrl + penalty_div + penalty_l2
    
    # res = (loss, penalty, penalty_ctrl, penalty_div)
    res = {
        'loss': loss,
        'penalty_pair': penalty_pair,
        'penalty_ctrl': penalty_ctrl,
        'penalty_div': penalty_div,
        'penalty_l2': penalty_l2
    }
    
    # Calculate and apply gradients
    if apply_grads:
        grads = g.gradient(loss, vars_filtered)
        grads,norm = tf.clip_by_global_norm(grads, 100.)
        opt.apply_gradients(zip(grads, vars_filtered))
        # res = res + (norm,)
        res['norm'] = norm
    
    return res

# Loop over batches of training data
data_pairs_iter = tqdm(data_pairs, total=n_steps)

t_last_plot = time.monotonic()

datasets = (data_pairs_iter, data_ctrl, val_pairs, val_ctrl)

metrics_history = {}

var_freeze_list = () # Initially, train all variables

for i,(d_pair,d_ctrl,v_pair,v_ctrl) in enumerate(zip(*datasets)):
    # Training points
    (x0,x1,xe0,xe1,xs0,xs1,c0,c1) = d_pair
    (x_ctrl,xe_ctrl,c_ctrl) = d_ctrl
    
    res = grad_step(
        x0, x1, xe0, xe1, xs0, xs1, c0, c1, # Pairs
        x_ctrl, xe_ctrl, c_ctrl,  # Control points
        apply_grads=True
    )
    for k in res:
        if k not in metrics_history:
            metrics_history[k] = []
        metrics_history[k].append(float(res[k].numpy()))

    # Validation points
    (x0,x1,xe0,xe1,xs0,xs1,c0,c1) = v_pair
    (x_ctrl,xe_ctrl,c_ctrl) = v_ctrl

    res = grad_step(
        x0, x1, xe0, xe1, xs0, xs1, c0, c1, # Pairs
        x_ctrl, xe_ctrl, c_ctrl,  # Control points
        apply_grads=False,
        freeze_list=var_freeze_list
    )
    for k in res:
        key = k + '_val'
        if key not in metrics_history:
            metrics_history[key] = []
        metrics_history[key].append(float(res[k].numpy()))

    # Calculate running average of metrics
    metrics_avg = {}
    for k in metrics_history:
        metrics_avg[k] = np.mean(metrics_history[k][-16:])

    # Update progress bar
    chi2_avg = (
        metrics_avg['loss']
      - metrics_avg['penalty_pair']
      - metrics_avg['penalty_ctrl']
      - metrics_avg['penalty_l2']
    )
    data_pairs_iter.set_postfix({
        'loss': metrics_avg['loss'],
        'chi2': chi2_avg,
        'plty_p': metrics_avg['penalty_pair'],
        'plty_c': metrics_avg['penalty_ctrl'],
        'plty_d': metrics_avg['penalty_div'],
        'plty_l2': metrics_avg['penalty_l2'],
        'norm': metrics_avg['norm'],
        'lr': float(learning_rate(i).numpy())
    })

    # Make plots at regular intervals
    # if i % plot_every == plot_every-1:
    if time.monotonic() > t_last_plot+plot_every:
        t_last_plot = time.monotonic()
        plot_loss_hist(metrics_history)
        plot_transformed_params(bij)

    # Freeze embedding and FiLM parameters late in training
    if i == 3 * n_steps // 4:
        freeze_list = ('embedding', 'film')

# Save the results
checkpoint = tf.train.Checkpoint(bij=bij)
checkpoint.save('models/bij_checkpoint')

Using 4-dim embedding for 16 categories.
11352 variables in model.


  0%|          | 0/58833 [00:00<?, ?it/s]

Tracing <grad_step> ...
Training variables:
vector_network/embedding/embeddings:0
vector_network/film_0/b:0
vector_network/film_0/w:0
vector_network/film_1/b:0
vector_network/film_1/w:0
vector_network/film_2/b:0
vector_network/film_2/w:0
vector_network/film_output/b:0
vector_network/film_output/w:0
vector_network/hidden_0/b:0
vector_network/hidden_0/w:0
vector_network/hidden_1/b:0
vector_network/hidden_1/w:0
vector_network/hidden_2/b:0
vector_network/hidden_2/w:0
vector_network/norm_0/offset:0
vector_network/norm_0/scale:0
vector_network/norm_1/offset:0
vector_network/norm_1/scale:0
vector_network/norm_2/offset:0
vector_network/norm_2/scale:0
vector_network/output/b:0
vector_network/output/w:0
Tracing <grad_step> ...
Training variables:
vector_network/embedding/embeddings:0
vector_network/film_0/b:0
vector_network/film_0/w:0
vector_network/film_1/b:0
vector_network/film_1/w:0
vector_network/film_2/b:0
vector_network/film_2/w:0
vector_network/film_output/b:0
vector_network/film_output/w

<Figure size 800x400 with 2 Axes>

<Figure size 960x960 with 4 Axes>

<Figure size 800x400 with 2 Axes>

... (177 outputs left out)

<Figure size 800x400 with 2 Axes>

<Figure size 960x960 with 4 Axes>

2025-12-31 02:02:17.637516: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


'models/bij_checkpoint-1'

### Save bijections

In [292]:
bij.to_harmonized = tf.function(
    func=bij._forward,
    input_signature=(
        tf.TensorSpec([None,n_dim], tf.float32), # x
        tf.TensorSpec([None], tf.int32),         # c
    )
)
bij.from_harmonized = tf.function(
    func=bij._inverse,
    input_signature=(
        tf.TensorSpec([None,n_dim], tf.float32), # x
        tf.TensorSpec([None], tf.int32),         # c
    )
)
tf.saved_model.save(bij, 'models/bijector')

INFO:tensorflow:Assets written to: models/bijector/assets


INFO:tensorflow:Assets written to: models/bijector/assets


In [293]:
with open('models/metrics_history.json', 'w') as f:
    json.dump(metrics_history, f)

In [295]:
! mv bijector.tar.gz bijector.tar.gz.bak3

In [294]:
! tar -cf bijector.tar models/
! rm bijector.tar.gz
! gzip -5 bijector.tar
! du -sh bijector.tar.gz

rm: cannot remove 'bijector.tar.gz': No such file or directory
5.5M	bijector.tar.gz


### Plot results

In [297]:
plot_loss_hist(metrics_history)
plot_transformed_params(bij)

<Figure size 800x250 with 3 Axes>

<Figure size 960x960 with 4 Axes>

In [299]:
embeddings = [
    x for x in bij.trainable_variables
    if x.name == 'vector_network/embedding/embeddings:0'
][0].numpy()
embeddings.shape

(16, 4)

In [300]:
embeddings

array([[-0.95610386, -1.4405923 ,  0.1235268 , -0.13658877],
       [-0.93353546,  0.37425435, -0.3734836 , -0.03066967],
       [ 0.2523609 , -1.9636464 ,  0.63755816, -0.06768921],
       [-0.14707525, -0.25292453,  1.2768099 , -0.99976456],
       [-0.7680583 ,  0.5151564 ,  0.55746174, -0.78769845],
       [ 0.69620913,  0.68749475,  0.5705336 , -1.9456164 ],
       [ 0.26197723, -0.09552938, -0.6717463 ,  0.01328741],
       [ 0.14262347, -1.4266646 , -1.2187986 , -0.22474866],
       [-1.5525306 , -0.71587265,  1.0572178 , -0.08286954],
       [ 0.64759165, -0.09519006, -0.9455236 ,  0.15777342],
       [ 0.06033377, -0.5324089 ,  1.7564648 ,  0.20275737],
       [ 0.2568626 , -0.20388468, -0.6652965 , -1.385193  ],
       [ 0.22502862,  0.67059857, -0.8037377 , -1.1454301 ],
       [-0.05936691, -0.23312113,  0.3480484 , -0.18351087],
       [-0.02366838, -0.74217236, -0.7625318 , -0.68395966],
       [-1.5354288 , -0.5662971 ,  0.33730164, -1.0101192 ]],
      dtype=float32)

In [301]:
embeddings_norm = np.linalg.norm(embeddings, axis=1)
embeddings_inner_prod = np.einsum('ni,mi->nm', embeddings, embeddings)
embeddings_inner_prod /= (embeddings_norm[:,None]*embeddings_norm[None,:])
embeddings_dist = scipy.spatial.distance_matrix(embeddings, embeddings)
embeddings_dist[np.diag_indices_from(embeddings_dist)] = np.nan

fig,ax = plt.subplots(1,1, figsize=(5,4))
im = ax.matshow(embeddings_inner_prod, cmap='coolwarm_r', vmin=-1, vmax=1)
fig.colorbar(im, ax=ax)
plt.show()

fig,ax = plt.subplots(1,1, figsize=(5,4))
im = ax.matshow(embeddings_dist, cmap='viridis_r')
fig.colorbar(im, ax=ax)
ax.set_xticks(np.arange(len(embeddings)))
ax.set_yticks(np.arange(len(embeddings)))
ax.set_xticklabels(cat_labels)
ax.set_yticklabels(cat_labels)
ax.tick_params(axis='x', labelrotation=90)
plt.show()

print(np.nanmin(embeddings_dist, axis=0))

<Figure size 500x400 with 2 Axes>

<Figure size 500x400 with 2 Axes>

[1.32499365 1.21942112 1.41525549 1.23973909 1.21287234 1.65867904
 0.49449855 0.95668007 1.18352717 0.49449855 1.34061846 0.91781619
 0.91781619 1.09585781 0.93254177 1.18352717]


In [302]:
list(enumerate(cat_labels))

[(0, 'APOGEE (DR17)'),
 (1, 'LAMOST MRS'),
 (2, 'LAMOST HotPayne'),
 (3, 'GALAH (DR4)'),
 (4, 'LAMOST DD-Payne (DR9)'),
 (5, 'LAMOST M-Stars (Ding)'),
 (6, 'VMP (Li)'),
 (7, 'LAMOST M-Dwarfs (Zhang)'),
 (8, 'DESI (DR1)'),
 (9, 'DESI (Zhang)'),
 (10, 'SDSS-V (ASPCAP)'),
 (11, 'SDSS-V (BOSSNet)'),
 (12, 'RAVE (DR6)'),
 (13, 'PIGS'),
 (14, 'SDSS-V (SLAM)'),
 (15, 'SDSS-V (MDw Cannon)')]

Generate trajectories of catalog points during transformation:

In [319]:
tables[0]['sigma_x'][:10].data

array([[0.0113895 , 0.01231265, 0.04542294, 0.05022367],
       [0.02528456, 0.0120665 , 0.04693545, 0.05054066],
       [0.03888883, 0.01221064, 0.05039937, 0.05136332],
       [0.0353249 , 0.0114969 , 0.04767228, 0.05096207],
       [0.02536606, 0.02218738, 0.0918074 , 0.05631456],
       [0.01420025, 0.01166846, 0.04641976, 0.05036623],
       [0.0385291 , 0.0128989 , 0.0520938 , 0.0514968 ],
       [0.01835272, 0.01402636, 0.05349277, 0.05112278],
       [0.01369899, 0.01219023, 0.04682058, 0.05036069],
       [0.01724893, 0.01282174, 0.049559  , 0.05063606]], dtype=float32)

In [324]:
n_frames = 120
n_plot = 1024*2

# Catalog points
xt_pred = []
xt_pred_mask = []

for k,t in enumerate(tqdm(tables)):
    x = tf.constant(t['x'][:n_plot].astype('f4'))
    xt_pred_mask.append(t['sigma_x'][:n_plot] < 1e6) # Mask points without measurements
    # Pad missing [alpha/Fe] values
    if x.shape[1] < 4:
        x = tf.concat([x, tf.zeros([x.shape[0],1])], axis=1)
        xt_pred_mask[-1] = np.concatenate([
            xt_pred_mask[-1],
            np.zeros([x.shape[0],1], dtype=xt_pred_mask[-1].dtype)
        ], axis=1)
    c = k * tf.ones(x.shape[0], dtype=tf.int32)
    xt_pred.append(bij.return_path(x, c, n_frames).numpy())

xt_pred[0].shape

  0%|          | 0/16 [00:00<?, ?it/s]

(120, 2048, 4)

Generate trajectories of paired points during transformation:

In [304]:
# Pairs
xt_pair = {}
n_dim = pair_dict['x0'].shape[1]

def get_paired_trajectories(i0, i1, n_max=None):
    """
    Returns the trajectories of the first `n_max` pairs of classes
    `(i0, i1)`. If there are no pairs of these classes, then returns
    `None`.
    """
    idx = np.where((pair_dict['c0']==i0) & (pair_dict['c1']==i1))[0][:n_max]
    
    n = len(idx)
    if n == 0:
        return [np.empty([n_frames,0,n_dim], dtype='f4') for j in range(2)]
    
    c = i0 * tf.ones([n], dtype=tf.int32)
    xt0 = bij.return_path(pair_dict['x0'][idx], c, n_frames).numpy()
    
    c = i1 * tf.ones([n], dtype=tf.int32)
    xt1 = bij.return_path(pair_dict['x1'][idx], c, n_frames).numpy()

    return (xt0, xt1)

for j in range(len(tables)):
    for k in range(j+1,len(tables)):
        # idx = np.where((pair_dict['c0']==j) & (pair_dict['c1']==k))[0][:n_plot]
        # n = len(idx)
        # print(f'({j},{k}) : {n} pairs')
        # if n == 0:
        #     continue
        # c = j * tf.ones([n], dtype=tf.int32)
        # xt0 = bij.return_path(pair_dict['x0'][idx], c, n_frames).numpy()
        # c = k * tf.ones([n], dtype=tf.int32)
        # xt1 = bij.return_path(pair_dict['x1'][idx], c, n_frames).numpy()
        xt0,xt1 = get_paired_trajectories(j, k, n_max=n_plot//2)
        xt1_r,xt0_r = get_paired_trajectories(k, j, n_max=n_plot//2)
        # print(xt0.shape)
        # print(xt0_r.shape)
        xt0 = np.concat([xt0,xt0_r], axis=1)
        xt1 = np.concat([xt1,xt1_r], axis=1)
        n = xt0.shape[1]
        print(f'({j},{k}) : {n} pairs')
        if n == 0:
            continue
        xt_pair[(j,k)] = (xt0,xt1)

(0,1) : 2048 pairs
(0,2) : 999 pairs
(0,3) : 2048 pairs
(0,4) : 2048 pairs
(0,5) : 2048 pairs
(0,6) : 5 pairs
(0,7) : 2048 pairs
(0,8) : 2048 pairs
(0,9) : 302 pairs
(0,10) : 2048 pairs
(0,11) : 2048 pairs
(0,12) : 2048 pairs
(0,13) : 95 pairs
(0,14) : 89 pairs
(0,15) : 2048 pairs
(1,2) : 2048 pairs
(1,3) : 2048 pairs
(1,4) : 2048 pairs
(1,5) : 2048 pairs
(1,6) : 6 pairs
(1,7) : 1962 pairs
(1,8) : 2048 pairs
(1,9) : 77 pairs
(1,10) : 2048 pairs
(1,11) : 2048 pairs
(1,12) : 810 pairs
(1,13) : 0 pairs
(1,14) : 7 pairs
(1,15) : 159 pairs
(2,3) : 152 pairs
(2,4) : 2048 pairs
(2,5) : 140 pairs
(2,6) : 0 pairs
(2,7) : 11 pairs
(2,8) : 2048 pairs
(2,9) : 83 pairs
(2,10) : 325 pairs
(2,11) : 2048 pairs
(2,12) : 1 pairs
(2,13) : 0 pairs
(2,14) : 1 pairs
(2,15) : 3 pairs
(3,4) : 2048 pairs
(3,5) : 310 pairs
(3,6) : 3 pairs
(3,7) : 13 pairs
(3,8) : 2048 pairs
(3,9) : 1 pairs
(3,10) : 2048 pairs
(3,11) : 366 pairs
(3,12) : 2048 pairs
(3,13) : 7 pairs
(3,14) : 0 pairs
(3,15) : 11 pairs
(4,5) : 2048

In [305]:
n_pairs = {k: xt_pair[k][0].shape[1] for k in xt_pair}
print(n_pairs)

{(0, 1): 2048, (0, 2): 999, (0, 3): 2048, (0, 4): 2048, (0, 5): 2048, (0, 6): 5, (0, 7): 2048, (0, 8): 2048, (0, 9): 302, (0, 10): 2048, (0, 11): 2048, (0, 12): 2048, (0, 13): 95, (0, 14): 89, (0, 15): 2048, (1, 2): 2048, (1, 3): 2048, (1, 4): 2048, (1, 5): 2048, (1, 6): 6, (1, 7): 1962, (1, 8): 2048, (1, 9): 77, (1, 10): 2048, (1, 11): 2048, (1, 12): 810, (1, 14): 7, (1, 15): 159, (2, 3): 152, (2, 4): 2048, (2, 5): 140, (2, 7): 11, (2, 8): 2048, (2, 9): 83, (2, 10): 325, (2, 11): 2048, (2, 12): 1, (2, 14): 1, (2, 15): 3, (3, 4): 2048, (3, 5): 310, (3, 6): 3, (3, 7): 13, (3, 8): 2048, (3, 9): 1, (3, 10): 2048, (3, 11): 366, (3, 12): 2048, (3, 13): 7, (3, 15): 11, (4, 5): 2048, (4, 6): 159, (4, 7): 128, (4, 8): 2048, (4, 9): 2048, (4, 10): 2048, (4, 11): 2048, (4, 12): 2048, (4, 14): 5, (5, 7): 2048, (5, 8): 2048, (5, 10): 2048, (5, 11): 733, (5, 12): 31, (5, 14): 1936, (5, 15): 2048, (6, 8): 15, (6, 10): 2, (6, 11): 3, (7, 8): 2048, (7, 9): 1, (7, 10): 123, (7, 11): 495, (7, 14): 1532,

Plot frames for a movie of the transformation in ($T_{\rm eff}, \log g$)-space, highlighting either control points (of a subset of surveys) or a particular set of paired points:

In [306]:
# %env PATH=/home/green/storage/green/bins/ffmpeg-7.0.2-amd64-static:/usr/local/autofs.home/home/green/envs/py3.10.9/bin:/opt/apps/software/anaconda/3/2023-03/condabin:/usr/share/Modules/bin:/usr/local/bin:/usr/bin:/usr/local/sbin:/usr/sbin:/opt/puppetlabs/bin:/home/green/.local/bin:/home/green/bin

In [307]:
! rm frames/*.png

In [308]:
from matplotlib.ticker import FixedLocator, FuncFormatter

def format_teff_axis(ax, label_minor=False):
    # Teff values in units of 1000 K
    teff_major_ticks = np.array([5, 10, 20, 30])
    teff_minor_ticks = np.arange(3, 30, 1)

    # Axis coordinate = 5 ln(Teff / 5)
    major_locs = 5 * np.log(teff_major_ticks / 5)
    minor_locs = 5 * np.log(teff_minor_ticks / 5)

    ax.xaxis.set_major_locator(FixedLocator(major_locs))
    ax.xaxis.set_minor_locator(FixedLocator(minor_locs))

    # Major tick labels
    ax.set_xticklabels([f"${x:.0f}$" for x in teff_major_ticks])

    if label_minor:
        # Formatter: invert x = 5 ln(Teff/5) → Teff = 5 exp(x/5)
        def minor_fmt(x, pos):
            teff = 5 * np.exp(x / 5)
            return f"{teff:.0f}"

        ax.xaxis.set_minor_formatter(FuncFormatter(minor_fmt))
        ax.tick_params(axis="x", which="minor", labelbottom=True, labelsize=8)
    else:
        ax.xaxis.set_minor_formatter(FuncFormatter(lambda x, pos: ""))
        ax.tick_params(axis="x", which="minor", labelbottom=False)

    ax.invert_xaxis()

In [309]:
def interpolate_at_time(arr, t):
    """
    Linearly interpolates trajectories at a fractional time t ∈ [0, 1].

    Parameters:
        arr (np.ndarray): Trajectories with shape (n_frames, n_points, n_dim)
        t (float): Time between 0 and 1

    Returns:
        np.ndarray: Interpolated positions with shape (n_points, n_dim)
    """
    # print(f't = {t}')
    n_frames = arr.shape[0]
    t_idx = t * (n_frames - 1)
    i0 = int(np.floor(t_idx))
    i1 = min(i0 + 1, n_frames - 1)
    alpha = t_idx - i0
    return (1 - alpha) * arr[i0] + alpha * arr[i1]

def get_frame_times(n_video_frames, ease):
    if ease == 'linear':
        t_frames = np.linspace(0.,1.,n_video_frames)
    elif ease == 'sigmoid':
        t_frames = 0.5*(np.tanh(np.linspace(-2, 2, n_video_frames))+1)
        t_frames -= t_frames[0]
        t_frames /= t_frames[-1]
    elif ease == 'sine':
        t_frames = 0.5* (1 + np.sin(np.pi*np.linspace(-0.5,0.5,n_video_frames)))
    else:
        raise ValueError('"ease" must be "linear", "sigmoid" or "sine".')
    return t_frames

def gen_kiel_video(c_highlight, p_highlight,
                   n_video_frames=90, ease='sigmoid'):
    n_classes = len(xt_pred)
    
    base_fname = 'kiel'
    if len(c_highlight):
        base_fname += '_control_' + '_'.join([str(s) for s in c_highlight])
    if (p_highlight is not None) and len(p_highlight):
        base_fname += '_pair_' + '_'.join([str(s) for s in p_highlight])
    fname = f'frames/{base_fname}_{{:03d}}.png'
    print(fname)
    
    fig,ax = plt.subplots(figsize=(8,8), dpi=100)
    
    kw = dict(edgecolors='none', s=4)
    # prop_cycle = plt.rcParams['axes.prop_cycle']
    # colors = prop_cycle.by_key()['color']
    # class_color = colors[:n_classes]
    
    x = np.concatenate([x[0] for x in xt_pred], axis=0)
    xylim = np.percentile(x[:,:3:2], [1., 99.], axis=0)
    w = xylim[1] - xylim[0]
    xylim[0] -= 0.3*w
    xylim[1] += 0.3*w
    xlim,ylim = xylim.T
    xlim = xlim[::-1]
    ylim = ylim[::-1]
    
    scatterplots = []
    
    for k,label in enumerate(cat_labels):
        highlight = (k in c_highlight)
        x = xt_pred[k][0,:n_plot,:]
        sc = ax.scatter(
            x[:,0], x[:,2],
            color=class_color[k] if highlight else 'gray',
            alpha=0.5 if highlight else 0.2,
            zorder=100 if highlight else 0,
            label=label,
            **kw
        )
        scatterplots.append(sc)
    
    if (p_highlight is not None) and len(p_highlight):
        xtp = xt_pair[p_highlight]
        scatterplots_p = []
        for i,x in enumerate(xtp):
            sc = ax.scatter(
                x[0,:,0], x[0,:,2],
                color=class_color[p_highlight[i]],
                alpha=0.4,
                zorder=110,
                label=cat_labels[p_highlight[i]]+' (pairs)',
                **kw
            )
            scatterplots_p.append(sc)
    
    ax.legend(loc='upper left', markerscale=4)
    ax.set_xlabel(r'$5\ln\left(T_{\rm eff}/5000\,\mathrm{K}\right)$')
    ax.set_ylabel(r'$\log g$')
    ax.grid(True, alpha=0.2)
    
    # n_frames = xt_pred[0].shape[0]

    t_frames = get_frame_times(n_video_frames, ease)
    
    for j,t in enumerate(tqdm(t_frames)):
        for k in range(n_classes):
            # x = xt_pred[k][j,:n_plot,:]
            x = interpolate_at_time(xt_pred[k][:,:n_plot,:], t)
            sc = scatterplots[k]
            sc.set_offsets(np.c_[x[:,0], x[:,2]])
        if (p_highlight is not None) and len(p_highlight):
            xtp = xt_pair[p_highlight]
            for i,x in enumerate(xtp):
                xp = interpolate_at_time(x, t)
                sc = scatterplots_p[i]
                # sc.set_offsets(np.c_[x[j,:,0], x[j,:,2]])
                sc.set_offsets(np.c_[xp[:,0], xp[:,2]])
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        fig.savefig(fname.format(j), dpi=200)
    
    plt.show()
    
    ! ffmpeg -y -f image2 -framerate 30 \
             -i "frames/{base_fname}_%03d.png" \
             -frames:v {n_video_frames} \
             -profile:v main -movflags +faststart -pix_fmt yuv420p \
             -c:v libx264 -crf 20 \
             videos/{base_fname}.mp4

In [310]:
len(xt_pred[0])

120

In [311]:
cat_labels

['APOGEE (DR17)',
 'LAMOST MRS',
 'LAMOST HotPayne',
 'GALAH (DR4)',
 'LAMOST DD-Payne (DR9)',
 'LAMOST M-Stars (Ding)',
 'VMP (Li)',
 'LAMOST M-Dwarfs (Zhang)',
 'DESI (DR1)',
 'DESI (Zhang)',
 'SDSS-V (ASPCAP)',
 'SDSS-V (BOSSNet)',
 'RAVE (DR6)',
 'PIGS',
 'SDSS-V (SLAM)',
 'SDSS-V (MDw Cannon)']

In [333]:
xt_pred_mask[15][:,0] & xt_pred_mask[15][:,2]

array([False, False, False, ..., False, False, False])

In [363]:
def gen_params_video(c_highlight, p_highlight,
                     n_video_frames=90, ease='sine',
                     optimize_colors=False):
    n_classes = len(xt_pred)
    
    base_fname = 'params'
    if len(c_highlight):
        base_fname += '_control_' + '_'.join([str(s) for s in c_highlight])
    if (p_highlight is not None) and len(p_highlight):
        base_fname += '_pair_' + '_'.join([str(s) for s in p_highlight])
    fname = f'frames/{base_fname}_{{:03d}}.png'
    print(fname)

    # Optimize colors for plot?
    class_color_local = class_color
    if optimize_colors:
        new_color_idx = [
            k for k in range(n_classes)
            if k in c_highlight or k in p_highlight
        ]
        if len(new_color_idx):
            new_colors = distinctipy.get_colors(
                len(new_color_idx),
                exclude_colors=[(0,0,0),(1,1,1)]
            )
            class_color_local = {k:c for k,c in zip(new_color_idx,new_colors)}
            print(f'Using colors: {class_color_local}')
    
    fig,ax_arr = plt.subplots(1, 2, figsize=(8,4),
                              dpi=200, layout='constrained')
    
    kw = dict(edgecolors='none', s=4)
    # prop_cycle = plt.rcParams['axes.prop_cycle']
    # colors = prop_cycle.by_key()['color']
    # class_color = colors[:n_classes]

    # Determine axis limits for each parameter
    x = []
    for k,x_k in enumerate(xt_pred):
        idx = xt_pred_mask[k]
        x.append(x_k[0] + 0.) # Force copy
        x[-1][~idx] = np.nan
    x = np.concatenate(x, axis=0)
    # x = np.concatenate([x[0] for x in xt_pred], axis=0)
    xlim = np.nanpercentile(x, [1., 99.], axis=0)
    w = xlim[1] - xlim[0]
    xlim[0] -= 0.3*w
    xlim[1] += 0.3*w

    # Shift teff and logg axes slightly
    xlim[:,0] += 0.2*w[0]
    xlim[:,2] -= 0.1*w[2]

    # Invert teff and logg axes
    xlim[:,0] = xlim[::-1,0]
    xlim[:,2] = xlim[::-1,2]

    print(xlim)

    # Which parameters to show in each plot
    param_idx = [(0,2),(1,3)]
    
    scatterplots = []
    
    for k,label in enumerate(cat_labels):
        highlight = (k in c_highlight)
        if (not highlight) or ((p_highlight is not None) and len(p_highlight)):
            label = None
        x = xt_pred[k][0,:n_plot,:]
        sc = []
        for (i0,i1),ax in zip(param_idx,ax_arr):
            idx = xt_pred_mask[k][:,i0] & xt_pred_mask[k][:,i1]
            sc.append(ax.scatter(
                x[idx,i0], x[idx,i1],
                color=class_color_local[k] if highlight else 'gray',
                alpha=0.2 if highlight else 0.1,
                zorder=100 if highlight else 0,
                label=label,
                **kw
            ))
        scatterplots.append(sc)
    
    if (p_highlight is not None) and len(p_highlight):
        xtp = xt_pair[p_highlight]
        scatterplots_p = []
        for i,x in enumerate(xtp):
            sc = []
            for (j0,j1),ax in zip(param_idx,ax_arr):
                sc.append(ax.scatter(
                    x[0,:,j0], x[0,:,j1],
                    color=class_color_local[p_highlight[i]],
                    alpha=0.2,
                    zorder=110,
                    label=cat_labels[p_highlight[i]]+' (pairs)',
                    **kw
                ))
            scatterplots_p.append(sc)

    param_labels = [
        r'$5\ln\left(T_{\rm eff}/5000\,\mathrm{K}\right)$',
        r'$\left[\mathrm{Fe/H}\right]$',
        r'$\log g$',
        r'$\left[\alpha/\mathrm{Fe}\right]$'
    ]

    for (i0,i1),ax in zip(param_idx,ax_arr):
        ax.set_xlabel(param_labels[i0])
        ax.set_ylabel(param_labels[i1])
        ax.grid(True, alpha=0.2)
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
    
    ax_arr[0].legend(loc='upper left', markerscale=3, fontsize=7)
    
    # n_frames = xt_pred[0].shape[0]

    t_frames = get_frame_times(n_video_frames, ease)
    
    for j,t in enumerate(tqdm(t_frames)):
        for k in range(n_classes):
            # x = xt_pred[k][j,:n_plot,:]
            x = interpolate_at_time(xt_pred[k][:,:n_plot,:], t)
            for sc,(i0,i1) in zip(scatterplots[k],param_idx):
                idx = xt_pred_mask[k][:,i0] & xt_pred_mask[k][:,i1]
                sc.set_offsets(np.c_[x[idx,i0], x[idx,i1]])
        if (p_highlight is not None) and len(p_highlight):
            xtp = xt_pair[p_highlight]
            for i,x in enumerate(xtp):
                xp = interpolate_at_time(x, t)
                for sc,(i0,i1) in zip(scatterplots_p[i],param_idx):
                    sc.set_offsets(np.c_[xp[:,i0], xp[:,i1]])
        for (i0,i1),ax in zip(param_idx,ax_arr):
            ax.set_xlim(xlim[:,i0])
            ax.set_ylim(xlim[:,i1])
            if i0 == 0:
                format_teff_axis(ax)
                ax.set_xlim(xlim[:,i0])
        fig.savefig(fname.format(j), dpi=200)
    
    plt.show()
    
    ! ffmpeg -y -f image2 -framerate 30 \
             -i "frames/{base_fname}_%03d.png" \
             -frames:v {n_video_frames} \
             -profile:v main -movflags +faststart -pix_fmt yuv420p \
             -c:v libx264 -crf 20 \
             videos/{base_fname}.mp4

In [366]:
gen_params_video([], (8,10), n_video_frames=120, optimize_colors=True)

frames/params_pair_8_10_{:03d}.png
Using colors: {8: (0.0, 1.0, 0.0), 10: (1.0, 0.0, 1.0)}
[[ 5.93463089 -3.30495316  5.70459048 -1.06734006]
 [-2.64374405  1.2375503  -0.57213276  0.8631493 ]]


  0%|          | 0/120 [00:00<?, ?it/s]

<Figure size 1600x800 with 2 Axes>

ffmpeg version 7.0.2-static https://johnvansickle.com/ffmpeg/  Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 8 (Debian 8.3.0-6)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-debug --disable-ffplay --disable-indev=sndio --disable-outdev=sndio --cc=gcc --enable-fontconfig --enable-frei0r --enable-gnutls --enable-gmp --enable-libgme --enable-gray --enable-libaom --enable-libfribidi --enable-libass --enable-libvmaf --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-librubberband --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libvorbis --enable-libopus --enable-libtheora --enable-libvidstab --enable-libvo-amrwbenc --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libdav1d --enable-libxvid --enable-libzvbi --enable-libzimg
  libavutil      59.  8.100 / 59.  8.100
  libavcodec     61.  3.100 / 61.  3.100
  libavformat   

In [341]:
gen_params_video([i], (), n_video_frames=120)

frames/params_control_10_{:03d}.png
[[ 4.86233402 -3.30495316  6.39075233 -0.92885208]
 [-3.71604091  1.2375503  -1.474789    0.78738265]]
parameters (0,2), table 0: 100.00%
parameters (1,3), table 0: 100.00%
parameters (0,2), table 1: 100.00%
parameters (1,3), table 1: 100.00%
parameters (0,2), table 2: 100.00%
parameters (1,3), table 2: 100.00%
parameters (0,2), table 3: 100.00%
parameters (1,3), table 3: 100.00%
parameters (0,2), table 4: 100.00%
parameters (1,3), table 4: 100.00%
parameters (0,2), table 5: 100.00%
parameters (1,3), table 5: 0.00%
parameters (0,2), table 6: 100.00%
parameters (1,3), table 6: 0.00%
parameters (0,2), table 7: 100.00%
parameters (1,3), table 7: 0.00%
parameters (0,2), table 8: 100.00%
parameters (1,3), table 8: 100.00%
parameters (0,2), table 9: 100.00%
parameters (1,3), table 9: 100.00%
parameters (0,2), table 10: 100.00%
parameters (1,3), table 10: 100.00%
parameters (0,2), table 11: 100.00%
parameters (1,3), table 11: 0.00%
parameters (0,2), table 1

  0%|          | 0/120 [00:00<?, ?it/s]

<Figure size 1600x800 with 2 Axes>

ffmpeg version 7.0.2-static https://johnvansickle.com/ffmpeg/  Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 8 (Debian 8.3.0-6)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-debug --disable-ffplay --disable-indev=sndio --disable-outdev=sndio --cc=gcc --enable-fontconfig --enable-frei0r --enable-gnutls --enable-gmp --enable-libgme --enable-gray --enable-libaom --enable-libfribidi --enable-libass --enable-libvmaf --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-librubberband --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libvorbis --enable-libopus --enable-libtheora --enable-libvidstab --enable-libvo-amrwbenc --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libdav1d --enable-libxvid --enable-libzvbi --enable-libzimg
  libavutil      59.  8.100 / 59.  8.100
  libavcodec     61.  3.100 / 61.  3.100
  libavformat   

In [360]:
class_color

[(0.0, 1.0, 0.0),
 (1.0, 0.0, 1.0),
 (0.0, 0.5, 1.0),
 (1.0, 0.5, 0.0),
 (0.5, 0.75, 0.5),
 (0.381945586915037, 0.10787169822799336, 0.6031504697402067),
 (0.8838709662651899, 0.00754724122007755, 0.0985531196862357),
 (0.7611183745927239, 0.4639600404602737, 0.9936303837820597),
 (0.0, 1.0, 1.0),
 (1.0, 1.0, 0.0),
 (0.30964414196302903, 0.4702414145890923, 0.0008697560813262228),
 (0.0, 0.0, 1.0),
 (0.0, 1.0, 0.5),
 (0.0, 0.5, 0.5),
 (0.727878574406246, 0.346098195032302, 0.40072078015163903),
 (0.5529202646814797, 0.8823263245721562, 0.029363404730843312)]

In [364]:
gen_params_video([], (5,14), n_video_frames=5)

frames/params_pair_5_14_{:03d}.png
[[ 5.93463089 -3.30495316  5.70459048 -1.06734006]
 [-2.64374405  1.2375503  -0.57213276  0.8631493 ]]


  0%|          | 0/5 [00:00<?, ?it/s]

<Figure size 1600x800 with 2 Axes>

ffmpeg version 7.0.2-static https://johnvansickle.com/ffmpeg/  Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 8 (Debian 8.3.0-6)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-debug --disable-ffplay --disable-indev=sndio --disable-outdev=sndio --cc=gcc --enable-fontconfig --enable-frei0r --enable-gnutls --enable-gmp --enable-libgme --enable-gray --enable-libaom --enable-libfribidi --enable-libass --enable-libvmaf --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-librubberband --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libvorbis --enable-libopus --enable-libtheora --enable-libvidstab --enable-libvo-amrwbenc --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libdav1d --enable-libxvid --enable-libzvbi --enable-libzimg
  libavutil      59.  8.100 / 59.  8.100
  libavcodec     61.  3.100 / 61.  3.100
  libavformat   

In [131]:
# ! rm frames/*.png

In [127]:
gen_kiel_video([], (5,7), n_video_frames=15)

frames/kiel_pair_5_7_{:03d}.png


  0%|          | 0/15 [00:00<?, ?it/s]

<Figure size 800x800 with 1 Axes>

ffmpeg version 7.0.2-static https://johnvansickle.com/ffmpeg/  Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 8 (Debian 8.3.0-6)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-debug --disable-ffplay --disable-indev=sndio --disable-outdev=sndio --cc=gcc --enable-fontconfig --enable-frei0r --enable-gnutls --enable-gmp --enable-libgme --enable-gray --enable-libaom --enable-libfribidi --enable-libass --enable-libvmaf --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-librubberband --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libvorbis --enable-libopus --enable-libtheora --enable-libvidstab --enable-libvo-amrwbenc --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libdav1d --enable-libxvid --enable-libzvbi --enable-libzimg
  libavutil      59.  8.100 / 59.  8.100
  libavcodec     61.  3.100 / 61.  3.100
  libavformat   

In [ ]:
# base_fname = 'kiel_pair_0_8'

# ! ffmpeg -y -f image2 -framerate 30 -i "frames/{base_fname}_%03d.png" \
#          -i deepnote.mp3 \
#          -map 0:v:0 -map 1:a:0 \
#          -profile:v main -movflags +faststart -pix_fmt yuv420p \
#          -c:v libx264 -crf 20 \
#          -c:a copy \
#          {base_fname}_soundeffect.mp4

Generate Kiel-diagram videos for all catalogs, and for every pair of catalogs that have more than some minimum number of overlapping stars:

In [132]:
for i in tqdm(range(len(xt_pred))):
    gen_params_video([i], (), n_video_frames=120)

for k in tqdm(n_pairs):
    if n_pairs[k] >= 50:
        gen_params_video([], k, n_video_frames=120)

  0%|          | 0/16 [00:00<?, ?it/s]

frames/params_control_0_{:03d}.png
[[ 4.86233402 -3.30495316  6.39075233 -0.92885208]
 [-3.71604091  1.2375503  -1.474789    0.78738265]]


  0%|          | 0/120 [00:00<?, ?it/s]

<Figure size 1600x800 with 2 Axes>

ffmpeg version 7.0.2-static https://johnvansickle.com/ffmpeg/  Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 8 (Debian 8.3.0-6)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-debug --disable-ffplay --disable-indev=sndio --disable-outdev=sndio --cc=gcc --enable-fontconfig --enable-frei0r --enable-gnutls --enable-gmp --enable-libgme --enable-gray --enable-libaom --enable-libfribidi --enable-libass --enable-libvmaf --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-librubberband --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libvorbis --enable-libopus --enable-libtheora --enable-libvidstab --enable-libvo-amrwbenc --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libdav1d --enable-libxvid --enable-libzvbi --enable-libzimg
  libavutil      59.  8.100 / 59.  8.100
  libavcodec     61.  3.100 / 61.  3.100
  libavformat   

  0%|          | 0/120 [00:00<?, ?it/s]

... (255 outputs left out)

ffmpeg version 7.0.2-static https://johnvansickle.com/ffmpeg/  Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 8 (Debian 8.3.0-6)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-debug --disable-ffplay --disable-indev=sndio --disable-outdev=sndio --cc=gcc --enable-fontconfig --enable-frei0r --enable-gnutls --enable-gmp --enable-libgme --enable-gray --enable-libaom --enable-libfribidi --enable-libass --enable-libvmaf --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-librubberband --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libvorbis --enable-libopus --enable-libtheora --enable-libvidstab --enable-libvo-amrwbenc --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libdav1d --enable-libxvid --enable-libzvbi --enable-libzimg
  libavutil      59.  8.100 / 59.  8.100
  libavcodec     61.  3.100 / 61.  3.100
  libavformat   

  0%|          | 0/120 [00:00<?, ?it/s]

<Figure size 1600x800 with 2 Axes>

ffmpeg version 7.0.2-static https://johnvansickle.com/ffmpeg/  Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 8 (Debian 8.3.0-6)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-debug --disable-ffplay --disable-indev=sndio --disable-outdev=sndio --cc=gcc --enable-fontconfig --enable-frei0r --enable-gnutls --enable-gmp --enable-libgme --enable-gray --enable-libaom --enable-libfribidi --enable-libass --enable-libvmaf --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-librubberband --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libvorbis --enable-libopus --enable-libtheora --enable-libvidstab --enable-libvo-amrwbenc --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libdav1d --enable-libxvid --enable-libzvbi --enable-libzimg
  libavutil      59.  8.100 / 59.  8.100
  libavcodec     61.  3.100 / 61.  3.100
  libavformat   

In [133]:
# gen_kiel_video([], ())
# gen_kiel_video([0], ())   # APOGEE
# gen_kiel_video([1], ())   # LAMOST MRS
# gen_kiel_video([2], ())   # LAMOST HotPayne
# gen_kiel_video([3], ())   # GALAH
# gen_kiel_video([4], ())   # LAMOST DD-Payne
# gen_kiel_video([5], ())   # LAMOST M-Stars
# gen_kiel_video([6], ())   # VMP
# gen_kiel_video([], (0,1)) # (APOGEE, LAMOST MRS)
# gen_kiel_video([], (0,2)) # (APOGEE, LAMOST HotPayne)
# gen_kiel_video([], (1,2)) # (LAMOST MRS, LAMOST HotPayne)
# gen_kiel_video([], (2,3)) # (LAMOST HotPayne, GALAH)
# gen_kiel_video([], (4,6)) # (VMP, LAMOST DD-Payne)
# gen_kiel_video([], (0,3)) # (APOGEE, GALAH)
# gen_kiel_video([], (0,4)) # (APOGEE, LAMOST DD-Payne)
# gen_kiel_video([], (0,5)) # (APOGEE, LAMOST M-Stars)

# # (..., LAMOST M-Dwarfs)
# gen_kiel_video([], (0,7)) # (APOGEE, LAMOST M-Dwarfs)
# gen_kiel_video([], (1,7)) # (LAMOST MRS, LAMOST M-Dwarfs)
# gen_kiel_video([], (4,7)) # (LAMOST DD-Payne, LAMOST M-Dwarfs)
# # gen_kiel_video([], (5,7)) # (LAMOST M-Stars, LAMOST M-Dwarfs)

In [367]:
param_tex_labels = [
    r'$T_\mathrm{eff} / 1000\,\mathrm{K}$',
    r'$\left[\mathrm{Fe/H}\right]$',
    r'$\log g$',
    r'$\left[\alpha/\mathrm{Fe}\right]$'
]

In [368]:
def gen_param1v1_video(pair0, pair1,
                       n_video_frames=120, ease='sine',
                       plot_diff=False):
    suffix = '_diff' if plot_diff else ''
    base_fname = f'param1v1{suffix}_pair_{pair0}_{pair1}'
    fname = f'frames/{base_fname}_{{:03d}}.png'
    print(fname)
    
    fig,ax_arr = plt.subplots(
        2, 2,
        figsize=(9,6),
        # subplot_kw={'aspect':'equal'},
        layout='tight',
        dpi=100
    )

    x0,x1 = xt_pair[(pair0,pair1)]

    scatterplots = []
    
    for i,(ax,p) in enumerate(zip(ax_arr.flat,param_tex_labels)):
        x = x0[0,:,i]
        y = x1[0,:,i]
        if plot_diff:
            y = y - x
        
        sc = ax.scatter(x, y, s=2, alpha=0.1)
        scatterplots.append(sc)
        
        if plot_diff:
            ax.set_xlim(np.min(x0[:,:,i]), np.max(x0[:,:,i]))
            ax.axhline(0, c='k', alpha=0.3)
        else:
            xlim = ax.get_xlim()
            ylim = ax.get_ylim()
            xlim = (min(xlim[0],ylim[0]), max(xlim[1],ylim[1]))
            ax.plot(xlim, xlim, c='k', alpha=0.3)
            ax.set_xlim(xlim)
            ax.set_ylim(xlim)

        if i == 0:
            # Special formatting for T_eff axis
            format_teff_axis(ax, label_minor=True)
        
        ax.set_xlabel(p)
        # ax.set_xlabel(cat_labels[pair0])
        # if i == 0:
        #     if plot_diff:
        #         ax.set_ylabel(f'{cat_labels[pair1]} - {cat_labels[pair0]}')
        #     else:
        #         ax.set_ylabel(cat_labels[pair1])

    fig.supxlabel(cat_labels[pair0])
    
    if plot_diff:
        fig.supylabel(f'{cat_labels[pair1]} - {cat_labels[pair0]}')
    else:
        fig.supylabel(cat_labels[pair1])
    
    # n_frames = x0.shape[0]

    t_frames = get_frame_times(n_video_frames, ease)
    
    for j,t in enumerate(tqdm(t_frames)):
        x = interpolate_at_time(x0, t)
        y = interpolate_at_time(x1, t)
        if plot_diff:
            y = y - x
        
        for i,ax in enumerate(ax_arr.flat):
            # x = x0[j,:,i]
            # y = x1[j,:,i]
            sc = scatterplots[i]
            xlim = ax.get_xlim()
            ylim = ax.get_ylim()
            sc.set_offsets(np.c_[x[:,i], y[:,i]])
            ax.set_xlim(xlim)
            ax.set_ylim(ylim)
        
        fig.savefig(fname.format(j), dpi=200)
        if j == 0:
            fig.set_layout_engine(None)

    plt.show()
    
    ! ffmpeg -y -f image2 -framerate 30 \
             -i "frames/{base_fname}_%03d.png" \
             -frames:v {n_video_frames} \
             -profile:v main -movflags +faststart -pix_fmt yuv420p \
             -c:v libx264 -crf 20 \
             videos/{base_fname}.mp4

In [369]:
list(enumerate(cat_labels))

[(0, 'APOGEE (DR17)'),
 (1, 'LAMOST MRS'),
 (2, 'LAMOST HotPayne'),
 (3, 'GALAH (DR4)'),
 (4, 'LAMOST DD-Payne (DR9)'),
 (5, 'LAMOST M-Stars (Ding)'),
 (6, 'VMP (Li)'),
 (7, 'LAMOST M-Dwarfs (Zhang)'),
 (8, 'DESI (DR1)'),
 (9, 'DESI (Zhang)'),
 (10, 'SDSS-V (ASPCAP)'),
 (11, 'SDSS-V (BOSSNet)'),
 (12, 'RAVE (DR6)'),
 (13, 'PIGS'),
 (14, 'SDSS-V (SLAM)'),
 (15, 'SDSS-V (MDw Cannon)')]

In [370]:
gen_param1v1_video(7, 14, n_video_frames=120, plot_diff=True)

frames/param1v1_diff_pair_7_14_{:03d}.png


  0%|          | 0/120 [00:00<?, ?it/s]

<Figure size 900x600 with 4 Axes>

ffmpeg version 7.0.2-static https://johnvansickle.com/ffmpeg/  Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 8 (Debian 8.3.0-6)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-debug --disable-ffplay --disable-indev=sndio --disable-outdev=sndio --cc=gcc --enable-fontconfig --enable-frei0r --enable-gnutls --enable-gmp --enable-libgme --enable-gray --enable-libaom --enable-libfribidi --enable-libass --enable-libvmaf --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-librubberband --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libvorbis --enable-libopus --enable-libtheora --enable-libvidstab --enable-libvo-amrwbenc --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libdav1d --enable-libxvid --enable-libzvbi --enable-libzimg
  libavutil      59.  8.100 / 59.  8.100
  libavcodec     61.  3.100 / 61.  3.100
  libavformat   

In [371]:
for k in tqdm(n_pairs):
    # if 14 not in k:
    #     continue
    if n_pairs[k] >= 50:
        print(k)
        gen_param1v1_video(*k, plot_diff=True, n_video_frames=120)

# gen_param1v1_video(0,1) # (APOGEE, LAMOST MRS)
# gen_param1v1_video(0,2) # (APOGEE, LAMOST HotPayne)
# gen_param1v1_video(1,2) # (LAMOST MRS, LAMOST HotPayne)
# gen_param1v1_video(2,3) # (LAMOST HotPayne, GALAH)
# gen_param1v1_video(0,3) # (APOGEE, GALAH)
# gen_param1v1_video(0,4) # (APOGEE, LAMOST DD-Payne)
# gen_param1v1_video(0,5) # (APOGEE, LAMOST M-Stars)
# gen_param1v1_video(4,6) # (LAMOST DD-Payne, VMP)

  0%|          | 0/93 [00:00<?, ?it/s]

(0, 1)
frames/param1v1_diff_pair_0_1_{:03d}.png


  0%|          | 0/120 [00:00<?, ?it/s]

<Figure size 900x600 with 4 Axes>

ffmpeg version 7.0.2-static https://johnvansickle.com/ffmpeg/  Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 8 (Debian 8.3.0-6)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-debug --disable-ffplay --disable-indev=sndio --disable-outdev=sndio --cc=gcc --enable-fontconfig --enable-frei0r --enable-gnutls --enable-gmp --enable-libgme --enable-gray --enable-libaom --enable-libfribidi --enable-libass --enable-libvmaf --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-librubberband --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libvorbis --enable-libopus --enable-libtheora --enable-libvidstab --enable-libvo-amrwbenc --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libdav1d --enable-libxvid --enable-libzvbi --enable-libzimg
  libavutil      59.  8.100 / 59.  8.100
  libavcodec     61.  3.100 / 61.  3.100
  libavformat   

  0%|          | 0/120 [00:00<?, ?it/s]

... (205 outputs left out)

ffmpeg version 7.0.2-static https://johnvansickle.com/ffmpeg/  Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 8 (Debian 8.3.0-6)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-debug --disable-ffplay --disable-indev=sndio --disable-outdev=sndio --cc=gcc --enable-fontconfig --enable-frei0r --enable-gnutls --enable-gmp --enable-libgme --enable-gray --enable-libaom --enable-libfribidi --enable-libass --enable-libvmaf --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-librubberband --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libvorbis --enable-libopus --enable-libtheora --enable-libvidstab --enable-libvo-amrwbenc --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libdav1d --enable-libxvid --enable-libzvbi --enable-libzimg
  libavutil      59.  8.100 / 59.  8.100
  libavcodec     61.  3.100 / 61.  3.100
  libavformat   

  0%|          | 0/120 [00:00<?, ?it/s]

<Figure size 900x600 with 4 Axes>

ffmpeg version 7.0.2-static https://johnvansickle.com/ffmpeg/  Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 8 (Debian 8.3.0-6)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-debug --disable-ffplay --disable-indev=sndio --disable-outdev=sndio --cc=gcc --enable-fontconfig --enable-frei0r --enable-gnutls --enable-gmp --enable-libgme --enable-gray --enable-libaom --enable-libfribidi --enable-libass --enable-libvmaf --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-librubberband --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libvorbis --enable-libopus --enable-libtheora --enable-libvidstab --enable-libvo-amrwbenc --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libdav1d --enable-libxvid --enable-libzvbi --enable-libzimg
  libavutil      59.  8.100 / 59.  8.100
  libavcodec     61.  3.100 / 61.  3.100
  libavformat   

In [479]:
! tar -cf videos.tar videos/param1v1_diff_pair_*_*.mp4 videos/kiel_pair_*_*.mp4 videos/kiel_control_*.mp4
! du -sh videos.tar

183M	videos.tar


### Calculate pair loss before/after transformation

In [420]:
def get_transformed_pairs(i0, i1, n_max=None):
    """
    Returns the trajectories of the first `n_max` pairs of classes
    `(i0, i1)`. If there are no pairs of these classes, then returns
    `None`.
    """

    # Select x0, x1 and their associated uncertainties
    idx = np.where((pair_dict['c0']==i0) & (pair_dict['c1']==i1))[0][:n_max]
    
    x0 = tf.constant(pair_dict['x0'][idx])
    x1 = tf.constant(pair_dict['x1'][idx])
    xs0 = tf.constant(pair_dict['xs0'][idx])
    xs1 = tf.constant(pair_dict['xs1'][idx])
    c0 = tf.constant(pair_dict['c0'][idx])
    c1 = tf.constant(pair_dict['c1'][idx])
    
    # Add in noise to x0
    x0 = x0 + tf.clip_by_value(xs0, 1e-5, 1.) * tf.random.normal(x0.shape)

    # Calculate chi^2 before transformation
    chi2_orig = tf.reduce_sum(((x0-x1)/xs1)**2, axis=1)
    chi2_orig_mean = chi2_turnover * tf.math.asinh(chi2_orig/chi2_turnover)
    chi2_orig_mean = tf.reduce_mean(chi2_orig_mean)

    # When xs0 ~ infinity, then set xs1 to ~infinity. This occurs when
    # [alpha/Fe] is not set for x0. In that case, xs1 for [alpha/Fe] must
    # also be set to ~infinity, so that this parameter does not affect the
    # loss.
    xs1 = tf.where(xs0 > 1e6, xs0, xs1)

    v = bij.trainable_variables[:1]
    
    with tf.GradientTape() as g:
        # Transform x0 to coordinate system of x1
        x0_pred,s0 = bij.forward_with_s(x0, c=c0)
        x1_pred,s1 = bij.inverse_with_s(x0_pred, c=c1)
        
        # Transform control points to common coordinate system
        x_ctrl_pred,s_ctrl = bij.forward_with_s(x_ctrl, c=c_ctrl)
        
        # Compare transformed x0 to x1
        chi2 = tf.reduce_sum(((x1_pred-x1)/xs1)**2, axis=1)
        chi2_mean = chi2_turnover * tf.math.asinh(chi2/chi2_turnover)
        chi2_mean = tf.reduce_mean(chi2_mean)

    grads = g.gradient(chi2_mean, v)
    
    return {
        'chi2_mean': chi2_mean,
        'chi2_orig_mean': chi2_orig_mean,
        'grads': grads,
        'x0': x0,
        'x1': x1,
        'x1_pred': x1_pred,
        'xs1': xs1,
        'chi2': chi2,
        'chi2_orig': chi2_orig
    }

In [421]:
for key in n_pairs:
    if (14 in key) and (n_pairs[key] > 50):
        print(key, n_pairs[key])

(0, 14) 67
(5, 14) 1613
(7, 14) 1264
(8, 14) 1233
(10, 14) 90
(11, 14) 129
(14, 15) 584


In [451]:
import gc
gc.collect()

11325

In [452]:
embedding_grads = {}
for key in n_pairs:
    if (14 in key) and (n_pairs[key] > 50):
        print(key, n_pairs[key])
        embedding_grads[key] = get_transformed_pairs(
            key[0], key[1], n_max=256
        )['grads']
print(embedding_grads)

(0, 14) 67
(5, 14) 1613
(7, 14) 1264
(8, 14) 1233
(10, 14) 90
(11, 14) 129
(14, 15) 584
{(0, 14): (<tf.Tensor: shape=(16, 4), dtype=float32, numpy=
array([[-4.1814427,  3.5511417, -3.5997696, -5.8192816],
       [ 0.       ,  0.       ,  0.       ,  0.       ],
       [ 0.       ,  0.       ,  0.       ,  0.       ],
       [ 0.       ,  0.       ,  0.       ,  0.       ],
       [ 0.       ,  0.       ,  0.       ,  0.       ],
       [ 0.       ,  0.       ,  0.       ,  0.       ],
       [ 0.       ,  0.       ,  0.       ,  0.       ],
       [ 0.       ,  0.       ,  0.       ,  0.       ],
       [ 0.       ,  0.       ,  0.       ,  0.       ],
       [ 0.       ,  0.       ,  0.       ,  0.       ],
       [ 0.       ,  0.       ,  0.       ,  0.       ],
       [ 0.       ,  0.       ,  0.       ,  0.       ],
       [ 0.       ,  0.       ,  0.       ,  0.       ],
       [ 0.       ,  0.       ,  0.       ,  0.       ],
       [ 1.6564422, 14.490601 ,  3.574746 ,  4.7722807

In [450]:
kw = dict(bins=20, range=(-20,20), histtype='step')
chi_before = ((d['x0']-d['x1'])/d['xs1']).numpy()
chi_after = ((d['x1_pred']-d['x1'])/d['xs1']).numpy()
plt.hist(chi_before[:,2], label='before', **kw)
plt.hist(chi_after[:,2], label='after', **kw)
plt.legend()
plt.show()

<Figure size 640x480 with 1 Axes>

## Save unified catalog

### Transform individual catalogs

In [497]:
# Give an explicit input signature, so that different batch sizes
# do not trigger retracing
@tf.function(input_signature=[
    tf.TensorSpec([None,n_params]),
    tf.TensorSpec([None,n_params]),
    tf.TensorSpec([None],dtype=tf.int32)
])
def transform_estimates(xx, sigma_xx, cc):
    print('Tracing <transform_estimates> ...')
    with tf.GradientTape(watch_accessed_variables=False, persistent=True) as g:
        g.watch(xx)
        yy = bij._forward(xx, c=cc)
    dyy_dxx = g.batch_jacobian(yy, xx, experimental_use_pfor=False)
    # cov_yy = tf.linalg.matmul(dyy_dxx, dyy_dxx, transpose_b=True)
    cov_yy = tf.einsum(
        'bik,bk,bjk->bij',
        dyy_dxx, sigma_xx**2, dyy_dxx,
        optimize='optimal'
    )
    return yy, cov_yy

In [498]:
fname_out = 'transformed_data.h5'
! rm {fname_out}

for i,t in enumerate(tables):
    print(f'Catalog {i+1} of {len(tables)} ...')
    
    x = t['x'].astype('f4')
    sx = t['sigma_x'].astype('f4')
    # Pad missing [alpha/Fe] values
    if x.shape[1] < 4:
        no_alpha = True
        x = np.concatenate([
            x, np.zeros([x.shape[0],1],dtype='f4')
        ], axis=1)
        # Set [alpha/Fe] uncertainty to zero so that it doesn't affect
        # covariance matrix transformation
        sx = np.concatenate([
            sx, np.zeros([sx.shape[0],1], dtype='f4')
        ], axis=1)
    else:
        no_alpha = False
    c = i * np.ones(x.shape[0], dtype='i4')
    
    d = tf.data.Dataset.from_tensor_slices((x,sx,c))
    d = d.batch(4096)
    
    y = []
    cov_y = []
    
    for xx,sxx,cc in tqdm(d):
        yy, cov_yy = transform_estimates(xx, sxx, cc)
        y.append(yy.numpy())
        cov_y.append(cov_yy.numpy())
    
    y = np.concatenate(y, axis=0)
    cov_y = np.concatenate(cov_y, axis=0)

    if no_alpha:
        # If [alpha/Fe] is not present, set its value to zero and
        # uncertainty to ~infinity
        y[:,3] = 0.
        cov_y[:,3,3] = 1e9
        cov_y[:,:3,3] = 0.
        cov_y[:,3,:3] = 0.

    kw = dict(append=True, chunks=True, compression='lzf')
    Table({
        'params_orig': x,
        'params_err_orig': sx,
        'params_harm': y,
        'params_cov_harm': cov_y,
        'gdr3_source_id': t['id']
    }).write(fname_out, path=f'catalog_{i}', **kw)

Catalog 1 of 12 ...


  0%|          | 0/141 [00:00<?, ?it/s]

Tracing <transform_estimates> ...
Catalog 2 of 12 ...


  0%|          | 0/210 [00:00<?, ?it/s]

Catalog 3 of 12 ...


  0%|          | 0/83 [00:00<?, ?it/s]

... (3 outputs left out)

  0%|          | 0/738 [00:00<?, ?it/s]

Catalog 6 of 12 ...


  0%|          | 0/48 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [499]:
! h5ls {fname_out}

catalog_0                Dataset {575192}
catalog_1                Dataset {857366}
catalog_10               Dataset {1177679}
catalog_11               Dataset {219534}
catalog_2                Dataset {337102}
catalog_3                Dataset {325511}
catalog_4                Dataset {3022667}
catalog_5                Dataset {193374}
catalog_6                Dataset {355}
catalog_7                Dataset {275849}
catalog_8                Dataset {3182332}
catalog_9                Dataset {139018}


### Merge duplicated sources

For sources that are contained in multiple catalogs, we will combine their different "harmonized" parameter estimates into a single estimate.

In [375]:
import h5py

fname = 'transformed_data.h5'

with h5py.File(fname, 'r') as f:
    n_tables = len(f.keys())

harmonized_tables = [Table.read(fname, path=f'catalog_{i}')
                     for i in range(n_tables)]

In [376]:
harmonized_tables[0].colnames

['params_orig',
 'params_err_orig',
 'params_harm',
 'params_cov_harm',
 'gdr3_source_id']

In [377]:
for j in range(3):
    for i,t in enumerate(harmonized_tables):
        plt.hist(
            np.log10(t['params_err_orig'][:,j]),
            histtype='step',
            label=cat_labels[i],
            range=(-2,1),
            bins=100,
            density=True,
            log=True
        )
        plt.xlabel(f'parameter {j} uncertainty')
    plt.legend()
    plt.show()

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/numpy/lib/_histograms_impl.py:895: RuntimeWarning: invalid value encountered in divide
  return n/db/n.sum(), bin_edges


<Figure size 640x480 with 1 Axes>

In [378]:
sid_all = np.hstack([t['gdr3_source_id'] for t in harmonized_tables])
sid_unique,sid_counts = np.unique(sid_all, return_counts=True)

In [380]:
len(sid_unique)

9257354

Combining multivariate Gaussians - see: https://math.stackexchange.com/a/4415122/65184.

In [381]:
def combine_parameters(ignore_idx=None):
    # (n,d) = (# of unique sources, dimensionality of parameter space)
    n = len(sid_unique)
    d = harmonized_tables[0]['params_orig'].shape[1]

    # Add a small amount along the diagonals of the covariance matrices
    eps_err = 0.03
    eps_cov = eps_err**2 * np.eye(d)

    # Sums of C^{-1} and C^{-1} \mu, used to calculate combined
    # means and covariances
    sum_Cinv = np.zeros((n,d,d))
    sum_Cinv_mu = np.zeros((n,d))

    # Number of catalog entries per source
    n_entries = np.zeros(n, dtype='i4')

    # Combined parameter estimates and covariances
    params_comb = np.zeros((n,d))
    params_cov_comb = np.zeros((n,d,d))

    # Bitmask indicating which catalogs each source is found in
    params_source = np.zeros(n, dtype='u2')

    # Add in individual estimates from each catalog
    for i,t in enumerate(tqdm(harmonized_tables)):
        if ignore_idx is not None:
            t = t[~ignore_idx[i]]
        
        idx_insert = np.searchsorted(sid_unique, t['gdr3_source_id'])
        assert np.all(t['gdr3_source_id'] == sid_unique[idx_insert])

        params_source[idx_insert] |= 2**i
    
        np.add.at(n_entries, idx_insert, 1)
        
        idx = (sid_counts[idx_insert] == 1)
    
        # Copy over means and covariances of stars with only 1 catalog entry
        params_comb[idx_insert[idx]] = t['params_harm'][idx]
        params_cov_comb[idx_insert[idx]] = t['params_cov_harm'][idx]
    
        # Sum C^{-1} and C^{-1}\mu across multiple observations of each star
        Cinv = np.linalg.inv(t['params_cov_harm'][~idx]+eps_cov[None,:,:])
        np.add.at(sum_Cinv, idx_insert[~idx], Cinv)
        np.add.at(
            sum_Cinv_mu,
            idx_insert[~idx],
            np.einsum('nij,nj->ni', Cinv, t['params_harm'][~idx])
        )

    # Convert sums over C^{-1} and C^{-1} \mu to combined estimates
    idx = (n_entries > 1)
    params_cov_comb[idx] = np.linalg.inv(sum_Cinv[idx])
    params_comb[idx] = np.einsum(
        'nij,nj->ni',
        params_cov_comb[idx],
        sum_Cinv_mu[idx]
    )

    # params_cov_comb += eps_cov[None,:,:]

    # Sources with no good catalog entries
    idx = (n_entries == 0)
    params_comb[idx] = np.nan
    params_cov_comb[idx] = np.nan

    # Extract diagonals of covariance matrices
    params_err_comb = np.stack(
        [np.sqrt(params_cov_comb[:,i,i]**2+eps_err**2) for i in range(d)],
        axis=1
    )

    # ln(det(cov)) is a measure of the total uncertainty for each source
    _,lndet_cov = np.linalg.slogdet(params_cov_comb)

    res = dict(
        params_comb=params_comb,
        params_cov_comb=params_cov_comb,
        params_err_comb=params_err_comb,
        lndet_cov=lndet_cov,
        params_source=params_source
    )
    return res

In [382]:
def calc_chi2_vs_combined(combined_results):
    p = combined_results['params_comb']
    d = p.shape[1]
    
    chi2_tables = []
    
    eps_err = 0.03
    eps_cov = eps_err**2 * np.eye(d)
    
    for t in tqdm(harmonized_tables):
        idx_insert = np.searchsorted(sid_unique, t['gdr3_source_id'])
        
        Cinv = np.linalg.inv(t['params_cov_harm'] + eps_cov[None,:,:])
        dparams = t['params_harm'] - p[idx_insert]
        chi2 = np.einsum('ni,nij,nj->n', dparams, Cinv, dparams, optimize=True)
    
        chi2_tables.append(chi2)

    return chi2_tables

In [396]:
comb_dict = combine_parameters()

for i in range(3):
    chi2_tables = calc_chi2_vs_combined(comb_dict)
    ignore_idx = [(chi2 > 25) for chi2 in chi2_tables]
    comb_dict = combine_parameters(ignore_idx=ignore_idx)
    print('\n'.join([f'{label: >24s}: {np.mean(idx):.2%}' for label,idx in zip(cat_labels, ignore_idx)]))
    print('')

  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

           APOGEE (DR17): 0.42%
              LAMOST MRS: 1.10%
         LAMOST HotPayne: 1.08%
             GALAH (DR4): 0.08%
   LAMOST DD-Payne (DR9): 0.20%
   LAMOST M-Stars (Ding): 0.25%
                VMP (Li): 3.36%
 LAMOST M-Dwarfs (Zhang): 0.18%
              DESI (DR1): 0.13%
            DESI (Zhang): 0.28%
         SDSS-V (ASPCAP): 0.12%
        SDSS-V (BOSSNet): 0.52%
              RAVE (DR6): 0.09%
                    PIGS: 0.03%
           SDSS-V (SLAM): 0.56%
     SDSS-V (MDw Cannon): 0.00%



  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

           APOGEE (DR17): 0.34%
              LAMOST MRS: 0.85%
         LAMOST HotPayne: 0.86%
             GALAH (DR4): 0.09%
   LAMOST DD-Payne (DR9): 0.17%
   LAMOST M-Stars (Ding): 0.09%
                VMP (Li): 2.68%
 LAMOST M-Dwarfs (Zhang): 0.10%
              DESI (DR1): 0.13%
            DESI (Zhang): 0.23%
         SDSS-V (ASPCAP): 0.11%
        SDSS-V (BOSSNet): 0.43%
              RAVE (DR6): 0.08%
                    PIGS: 0.03%
           SDSS-V (SLAM): 0.41%
     SDSS-V (MDw Cannon): 0.00%



  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

           APOGEE (DR17): 0.34%
              LAMOST MRS: 0.81%
         LAMOST HotPayne: 0.67%
             GALAH (DR4): 0.04%
   LAMOST DD-Payne (DR9): 0.15%
   LAMOST M-Stars (Ding): 0.20%
                VMP (Li): 1.01%
 LAMOST M-Dwarfs (Zhang): 0.14%
              DESI (DR1): 0.10%
            DESI (Zhang): 0.19%
         SDSS-V (ASPCAP): 0.10%
        SDSS-V (BOSSNet): 0.30%
              RAVE (DR6): 0.04%
                    PIGS: 0.01%
           SDSS-V (SLAM): 0.47%
     SDSS-V (MDw Cannon): 0.00%



In [384]:
comb_dict.keys()

dict_keys(['params_comb', 'params_cov_comb', 'params_err_comb', 'lndet_cov', 'params_source'])

In [385]:
plt.hist(comb_dict['lndet_cov'], range=(-35,0), bins=100)
plt.show()

<Figure size 640x480 with 1 Axes>

In [386]:
np.mean(comb_dict['lndet_cov'] < 6*np.log(0.25))

np.float64(0.9399250584994373)

In [387]:
idx = (
    np.all(np.isfinite(comb_dict['params_comb']), axis=1)
  & (comb_dict['lndet_cov'] < 6*np.log(0.2))
)
print(f'Merged harmonized catalog: {np.count_nonzero(idx):,} ({np.mean(idx):.2%}) good sources.')

fig,ax_arr = plt.subplots(1,2, figsize=(8,4), layout='tight')

ax = ax_arr[0]
ax.hist2d(comb_dict['params_comb'][idx,0], # T_eff
          comb_dict['params_comb'][idx,2], # log(g)
          bins=100, norm=LogNorm())
ax.invert_xaxis()
ax.invert_yaxis()
ax.set_xlabel(r'$5\ln\left(T_\mathrm{eff}/5000\,\mathrm{K}\right)$')
ax.set_ylabel(r'$\log g$')
ax.grid(True, alpha=0.2)

ax = ax_arr[1]
ax.hist2d(comb_dict['params_comb'][idx,0], # T_eff
          comb_dict['params_comb'][idx,1], # [Fe/H]
          bins=100, norm=LogNorm())
ax.invert_xaxis()
ax.set_xlabel(r'$5\ln\left(T_\mathrm{eff}/5000\,\mathrm{K}\right)$')
ax.set_ylabel(r'$\left[\mathrm{Fe/H}\right]$')
ax.grid(True, alpha=0.2)

plt.show()

Merged harmonized catalog: 8,693,761 (93.91%) good sources.


<Figure size 800x400 with 2 Axes>

In [394]:
np.nanpercentile(comb_dict['params_err_comb'][:,3], [1., 5., 50., 60., 70., 80., 90., 95., 99., 99.9])

array([3.00172495e-02, 3.00478409e-02, 3.01701618e-02, 3.02056072e-02,
       3.02619056e-02, 3.04046169e-02, 3.17597953e-02, 5.00000000e+08,
       1.00000000e+09, 1.00000000e+09])

In [395]:
idx = (
    np.all(np.isfinite(comb_dict['params_comb']), axis=1)
  & (comb_dict['lndet_cov'] < 6*np.log(0.2))
)
print(f'Merged harmonized catalog: {np.count_nonzero(idx):,} ({np.mean(idx):.2%}) good sources.')

fig,ax_arr = plt.subplots(1,2, figsize=(8,4), layout='tight')

ax = ax_arr[0]
ax.hist2d(comb_dict['params_comb'][idx,0], # T_eff
          comb_dict['params_comb'][idx,2], # log(g)
          bins=100, norm=LogNorm())
ax.invert_xaxis()
ax.invert_yaxis()
ax.set_xlabel(r'$5\ln\left(T_\mathrm{eff}/5000\,\mathrm{K}\right)$')
ax.set_ylabel(r'$\log g$')
ax.grid(True, alpha=0.2)

ax = ax_arr[1]
idx = idx & (
    np.isfinite(comb_dict['params_err_comb'][:,3])
  & (comb_dict['params_err_comb'][:,3] < 1e5)
)
ax.hist2d(comb_dict['params_comb'][idx,1], # [Fe/H]
          comb_dict['params_comb'][idx,3], # [alpha/Fe]
          bins=100, norm=LogNorm())
# ax.invert_xaxis()
ax.set_xlabel(r'$\left[\mathrm{Fe/H}\right]$')
ax.set_ylabel(r'$\left[\alpha/\mathrm{Fe}\right]$')
ax.grid(True, alpha=0.2)

plt.show()

Merged harmonized catalog: 8,693,761 (93.91%) good sources.


<Figure size 800x400 with 2 Axes>

In [510]:
np.count_nonzero(np.any(np.isnan(comb_dict['params_comb']), axis=1))

9068

In [397]:
print('\n'.join([f'{label: >24s}: {np.mean(idx):.2%}' for label,idx in zip(cat_labels, ignore_idx)]))

           APOGEE (DR17): 0.34%
              LAMOST MRS: 0.81%
         LAMOST HotPayne: 0.67%
             GALAH (DR4): 0.04%
   LAMOST DD-Payne (DR9): 0.15%
   LAMOST M-Stars (Ding): 0.20%
                VMP (Li): 1.01%
 LAMOST M-Dwarfs (Zhang): 0.14%
              DESI (DR1): 0.10%
            DESI (Zhang): 0.19%
         SDSS-V (ASPCAP): 0.10%
        SDSS-V (BOSSNet): 0.30%
              RAVE (DR6): 0.04%
                    PIGS: 0.01%
           SDSS-V (SLAM): 0.47%
     SDSS-V (MDw Cannon): 0.00%


In [398]:
for i,label in enumerate(cat_labels):
    n = np.count_nonzero(comb_dict['params_source'] & 2**i) / 1e6
    print(f'{label: >24s}: {n: >5.3f} M')

print()
print(f'Total: {len(comb_dict['params_comb'])/1e6:.3f} M')

           APOGEE (DR17): 0.518 M
              LAMOST MRS: 0.462 M
         LAMOST HotPayne: 0.267 M
             GALAH (DR4): 0.530 M
   LAMOST DD-Payne (DR9): 4.109 M
   LAMOST M-Stars (Ding): 0.254 M
                VMP (Li): 0.000 M
 LAMOST M-Dwarfs (Zhang): 0.233 M
              DESI (DR1): 3.078 M
            DESI (Zhang): 0.128 M
         SDSS-V (ASPCAP): 0.431 M
        SDSS-V (BOSSNet): 0.268 M
              RAVE (DR6): 0.157 M
                    PIGS: 0.011 M
           SDSS-V (SLAM): 0.021 M
     SDSS-V (MDw Cannon): 0.016 M

Total: 9.257 M


In [399]:
param_max = np.max(np.vstack([np.max(t['x'], axis=0) for t in tables]), axis=0)
param_min = np.min(np.vstack([np.min(t['x'], axis=0) for t in tables]), axis=0)
print(param_min)
print(param_max)

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 4 and the array at index 5 has size 3

In [401]:
# These limits are set by hand
param_lims = [(-2.5,11.5), (-4.5,1.9), (-2.5,6.), (-2.,2.)]

In [402]:
comb_dict.keys()

dict_keys(['params_comb', 'params_cov_comb', 'params_err_comb', 'lndet_cov', 'params_source'])

In [403]:
fname = 'merged_params_catalog_clean.h5'

idx_finite = (
    np.all(np.isfinite(comb_dict['params_comb']), axis=1)
  & np.all(np.isfinite(comb_dict['params_err_comb']), axis=1)
)
print(f'Removing {np.count_nonzero(~idx_finite)} stars with NaNs.')

idx_bounds = np.ones_like(idx_finite)
for p,(p0,p1) in zip(comb_dict['params_comb'].T,param_lims):
    idx_bounds &= (p > p0) & (p < p1)
print(f'Removing {np.count_nonzero(~idx_bounds)} out-of-bounds stars.')

idx = idx_finite & idx_bounds

t_merged_clean = Table({
    'gdr3_source_id': sid_unique[idx],
    'params_est': comb_dict['params_comb'][idx].astype('f4'),
    'params_err': comb_dict['params_err_comb'][idx].astype('f4'),
    'params_cov': comb_dict['params_cov_comb'][idx].astype('f4'),
    'lndet_cov': comb_dict['lndet_cov'][idx].astype('f4'),
    'params_source': comb_dict['params_source'][idx]
})

t_merged_clean.write(fname, overwrite=True, chunks=True, compression='lzf')

! h5ls {fname}
! du -sh {fname}

Removing 5401 stars with NaNs.
Removing 10298 out-of-bounds stars.


/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:282: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


__astropy_table__        Dataset {9247056}
904M	merged_params_catalog_clean.h5


In [404]:
t_merged_clean.keys()

['gdr3_source_id',
 'params_est',
 'params_err',
 'params_cov',
 'lndet_cov',
 'params_source']

In [405]:
! realpath {fname}

/data/beegfs/astro-storage/groups/green/green/projects/merge_coordinates/merged_params_catalog_clean.h5


In [410]:
def plot_param_distributions(catalog_idx=None):
    fig,ax = plt.subplots(1,1, figsize=(5,4))

    xlim = (
        np.min(t_merged_clean['params_est'].flat),
        np.max(t_merged_clean['params_est'].flat)
    )
    w = xlim[1] - xlim[0]
    xlim = (xlim[0]-0.05*w, xlim[1]+0.05*w)
    
    # xlim = plot_utils.choose_lim(
    #     t_merged_clean['params_est'].flat,
    #     pct=[0.05,99.95], expand=1.0
    # )

    params = t_merged_clean['params_est'].T
    errs = t_merged_clean['params_err'].T
    
    if catalog_idx is None:
        title = 'Entire Catalog'
    else:
        title = cat_labels[catalog_idx]
        idx = (t_merged_clean['params_source'] & 2**catalog_idx).astype('bool')
        params = params[:,idx]
        errs = errs[:,idx]
    
    kw = dict(log=True, range=xlim, bins=100, histtype='step')
    for i,(p,s) in enumerate(zip(params,errs)):
        idx = (s < 10)
        ax.hist(p[idx], label=f'param {i}', **kw)

    ax.set_xlabel('parameter value')
    ax.set_title(title)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.grid(True, alpha=0.2)
    ax.grid(True, which='minor', alpha=0.05)
    ax.legend()
    
    plt.show()

In [411]:
plot_param_distributions()
for i in range(len(tables)):
    plot_param_distributions(catalog_idx=i)

<Figure size 500x400 with 1 Axes>

<Figure size 500x400 with 1 Axes>

<Figure size 500x400 with 1 Axes>

<Figure size 500x400 with 1 Axes>

<Figure size 500x400 with 1 Axes>

<Figure size 500x400 with 1 Axes>

... (7 outputs left out)

<Figure size 500x400 with 1 Axes>

<Figure size 500x400 with 1 Axes>

<Figure size 500x400 with 1 Axes>

<Figure size 500x400 with 1 Axes>

In [520]:
comb_dict.keys()

dict_keys(['params_comb', 'params_cov_comb', 'params_err_comb', 'lndet_cov', 'params_source'])

In [298]:
np.count_nonzero(comb_dict['params_source'] & 2**0)

520094

In [299]:
for i in range(3):
    plt.hist(
        np.log10(comb_dict['params_err_comb'][:,i]),
        bins=100, range=(-2,1), log=True,
        histtype='step',
        label=f'param {i}'
    )
plt.legend()
plt.xlabel(r'$\log_{10} \sigma$')
plt.show()

<Figure size 640x480 with 1 Axes>

In [444]:
def plot_uncertainties_kiel(catalog_idx=None, statistic='median'):
    xlim = (12., -5.)
    ylim = (6, -1.)
    
    p = comb_dict['params_comb'].T
    s = comb_dict['params_err_comb'].T
    if catalog_idx is not None:
        idx = (comb_dict['params_source'] & 2**catalog_idx).astype('bool')
        p = p[:,idx]
        s = s[:,idx]

    d = len(p)
    fig,ax_arr = plt.subplots(
        1, d+1,
        figsize=(3*d+0.5,3),
        width_ratios=[1,1,1,0.1],
        layout='constrained'
    )
    
    norm = LogNorm(vmin=3e-2, vmax=5e-1)
    for i,(s_i,ax) in enumerate(zip(s,ax_arr)):
        im = plot_utils.hist2d_reduce(
            p[0], p[2], s_i,
            xlim=xlim, ylim=ylim, ax=ax,
            hist_kw=dict(bins=50, statistic='median'),
            imshow_kw=dict(norm=norm)
        )
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_title(f'$\sigma_{{{i}}}$')
        if i != 0:
            ax.set_yticklabels([])
        ax.grid(True, which='major', alpha=0.2)

    title = 'All Catalogs' if catalog_idx is None else cat_labels[catalog_idx]
    fig.suptitle(title)
    fig.colorbar(im, cax=ax_arr[-1], label=rf'$\mathrm{{{statistic}}}\ \sigma$')
    plt.show()

<>:30: SyntaxWarning: invalid escape sequence '\s'
<>:30: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2991738/4122538164.py:30: SyntaxWarning: invalid escape sequence '\s'
  ax.set_title(f'$\sigma_{{{i}}}$')


In [445]:
cat_labels

['APOGEE',
 'LAMOST MRS',
 'LAMOST HotPayne',
 'GALAH',
 'LAMOST DD-Payne',
 'LAMOST M-Stars (Ding)',
 'VMP',
 'LAMOST M-Dwarfs (Zhang)',
 'DESI (DR1)',
 'DESI (Zhang)',
 'SDSS-V']

In [446]:
plot_uncertainties_kiel()
for i in range(len(cat_labels)):
    plot_uncertainties_kiel(catalog_idx=i)

<Figure size 950x300 with 4 Axes>

<Figure size 950x300 with 4 Axes>

<Figure size 950x300 with 4 Axes>

<Figure size 950x300 with 4 Axes>

<Figure size 950x300 with 4 Axes>

<Figure size 950x300 with 4 Axes>

... (2 outputs left out)

<Figure size 950x300 with 4 Axes>

<Figure size 950x300 with 4 Axes>

<Figure size 950x300 with 4 Axes>

<Figure size 950x300 with 4 Axes>

In [262]:
for t in harmonized_tables:
    print(np.percentile(t['params_harm'][:,0].data, [0., 1., 50., 99., 100.]))
    print(np.percentile(t['params_harm'][:,1].data, [0., 1., 50., 99., 100.]))
    print(np.percentile(t['params_harm'][:,2].data, [0., 1., 50., 99., 100.]))
    print()

[-2.26248455 -1.63462646 -0.19534649  1.40377249  2.00474763]
[-2.47795892 -1.4337754  -0.18983289  0.33045185  0.89079255]
[-0.17306474  0.54922396  2.7632134   4.7061256   4.89779472]

[-1.85345197 -1.10621329  0.62047315  1.98794411  4.19520473]
[-2.23489118 -0.75820254 -0.13919981  0.28408977  0.5791896 ]
[-0.40530816  1.27344837  4.07241106  4.6879771   5.1813612 ]

[ 1.54074967  1.68960947  2.46955264  6.66418386 11.62974548]
[-4.06062603 -2.01563375 -0.34466846  0.57917757  1.91740882]
[-0.45845303  2.61346954  4.01326776  4.68358402  5.00668287]

[-1.04505646 -0.8702403   0.48849684  1.99450425  2.55299759]
[-3.79099441 -1.42079409 -0.15366474  0.33214923  0.98507613]
[0.29427952 1.46747561 3.93526649 4.66208591 4.95010471]

[-2.66066194 -0.77908323  0.51672292  1.8175244   2.2214818 ]
[-6.918715   -1.61852163 -0.23925085  0.28542386  0.89457667]
[-3.71560311  1.59613415  4.13765764  4.8015455   8.62495518]

[-2.49740839 -2.13460774 -1.39343053 -1.02708059 -0.85660034]
[-2.6650

In [342]:
idx = np.all(comb_dict['params_err_comb'] < 1., axis=1)
plt.hist(comb_dict['params_comb'][idx,2], log=True)#, range=(0,5))#, range=(-1,6), bins=25)
plt.gca().grid(True, alpha=0.2)
plt.show()

<Figure size 640x480 with 1 Axes>

In [412]:
pix_idx_nside16 = sid_unique // 549755813888 // 4**6 # nside=16

In [413]:
np.max(pix_idx_nside16)

np.int64(3071)

In [414]:
12 * 2**8

3072

In [319]:
np.where(comb_dict['params_source'] & 2**0)[0].size

519731

In [329]:
from importlib import reload

In [332]:
reload(plot_utils)

<module 'plot_utils' from '/data/beegfs/astro-storage/groups/green/green/projects/merge_coordinates/plot_utils.py'>

In [415]:
pix_idx_nside8 = sid_unique // 549755813888 // 4**6 # nside=8

def skyplot_subset(idx):
    pix_idx,counts = np.unique(
        pix_idx_nside8[idx],
        return_counts=True
    )
    m = np.zeros(12 * 2**8)
    m[pix_idx] = counts
    fig,ax,im = plot_utils.plot_mollweide(
        m,
        norm=LogNorm(),
        interpolation_stage='rgba',
        input_frame='icrs',
        plot_frame='galactic'
    )
    return fig,ax,im

fig,ax,im = skyplot_subset(slice(None))
ax.set_title('Combined')
fig.colorbar(im, ax=ax, label=r'$\mathrm{stars}\,\mathrm{pix}^{-1}$')
plt.show()

for i,label in enumerate(cat_labels):
    idx_src = np.where(comb_dict['params_source'] & 2**i)[0]
    fig,ax,im = skyplot_subset(idx_src)
    ax.set_title(label)
    fig.colorbar(im, ax=ax, label=r'$\mathrm{stars}\,\mathrm{pix}^{-1}$')
    plt.show()

<Figure size 600x260 with 2 Axes>

<Figure size 600x260 with 2 Axes>

<Figure size 600x260 with 2 Axes>

<Figure size 600x260 with 2 Axes>

<Figure size 600x260 with 2 Axes>

<Figure size 600x260 with 2 Axes>

... (7 outputs left out)

<Figure size 600x260 with 2 Axes>

<Figure size 600x260 with 2 Axes>

<Figure size 600x260 with 2 Axes>

<Figure size 600x260 with 2 Axes>

In [305]:
chi2_tables = []

for t in tqdm(harmonized_tables):
    idx_insert = np.searchsorted(sid_unique, t['gdr3_source_id'])
    
    Cinv = np.linalg.inv(t['params_cov_harm'])
    dparams = t['params_harm'] - params_comb[idx_insert]
    chi2 = np.einsum('ni,nij,nj->n', dparams, Cinv, dparams, optimize=True)

    chi2_tables.append(chi2)

del Cinv
del dparams

  0%|          | 0/9 [00:00<?, ?it/s]

In [339]:
for i,chi2 in enumerate(chi2_tables):
    res = plt.hist(
        np.log10(chi2[chi2>0]),
        bins=100, log=False, histtype='step',
        label=cat_labels[i]
    )
    c = res[2][0].get_edgecolor()
    chi2_pct = np.percentile(chi2[chi2>0],90.)
    print(chi2_pct)
    # plt.axvline(np.log10(chi2_pct), c=c, alpha=0.5)
# print(np.mean(chi2[chi2>0] > chi2_threshold))
# plt.axvline(np.log10(chi2_threshold), c='orange', alpha=0.5)
plt.legend()
plt.show()

11.696773052326687
6.637430451317924
10.040871140391005
3.078410027773796
9.458001834334846
1.5172553844905603
12.791582363855438
4.242579768459757
9.046141075246313


<Figure size 640x480 with 1 Axes>

In [337]:
[np.percentile(chi2[chi2>0], 90.) for chi2 in chi2_tables]

[np.float64(11.696773052326687),
 np.float64(6.637430451317924),
 np.float64(10.040871140391005),
 np.float64(3.078410027773796),
 np.float64(9.458001834334846),
 np.float64(1.5172553844905603),
 np.float64(12.791582363855438),
 np.float64(4.242579768459757),
 np.float64(9.046141075246313)]

In [338]:
ignore_idx = [
    chi2>9.#max(np.percentile(chi2[chi2>0], 90.), 3**2)
    for chi2 in chi2_tables
]
params_comb, params_cov_comb = combine_parameters(ignore_idx)

  0%|          | 0/9 [00:00<?, ?it/s]

/usr/local/autofs.home/home/green/envs/py3/lib/python3.12/site-packages/numpy/linalg/_linalg.py:2311: RuntimeWarning: invalid value encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)


ValueError: too many values to unpack (expected 2)

In [316]:
import scipy.integrate as integrate
import scipy.optimize as optimize

In [325]:
int_den,_ = integrate.quad(lambda r: r**2 * np.exp(-0.5*r**2), 0., np.inf)
# int_num,_ = integrate.quad(lambda r: r**2 * np.exp(-0.5*r**2), 0., 3.)
def gaussian_percentile(r1):
    int_num,_ = integrate.quad(lambda r: r**2 * np.exp(-0.5*r**2), 0., r1)
    return int_num/int_den

chi2_threshold = optimize.newton(lambda r1: gaussian_percentile(r1)-0.999, 3.)
print(chi2_threshold)

4.033142223656423


In [297]:
5*np.log(4/5)

-1.1157177565710485

In [299]:
5*np.exp(-2.2/5)

3.2201821054157067

In [267]:
xlim = (-5., 10.)
ylim = (-1., 5.5)

params_comb = comb_dict['params_comb']

plt.hist2d(
    params_comb[:,0],
    params_comb[:,2],
    range=[xlim,ylim],
    bins=100,
    norm=LogNorm()
)
plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.gca().set_title('Combined')
plt.colorbar()
plt.show()

idx = (sid_counts == 1)
plt.hist2d(
    params_comb[idx,0],
    params_comb[idx,2],
    range=[xlim,ylim],
    bins=100,
    norm=LogNorm()
)
plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.gca().set_title('Combined (unique)')
plt.colorbar()
plt.show()

plt.hist2d(
    params_comb[~idx,0],
    params_comb[~idx,2],
    range=[xlim,ylim],
    bins=100,
    norm=LogNorm()
)
plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.gca().set_title('Combined (multiple)')
plt.colorbar()
plt.show()

idx = (comb_dict['params_err_comb'][:,0] > 0.5)
plt.hist2d(
    params_comb[idx,0],
    params_comb[idx,2],
    range=[xlim,ylim],
    bins=100,
    norm=LogNorm()
)
plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.gca().set_title('Combined (high uncertainties)')
plt.colorbar()
plt.show()

for i,t in enumerate(harmonized_tables):
    plt.hist2d(
        t['params_harm'][:,0],
        t['params_harm'][:,2],
        range=[xlim,ylim],
        bins=100,
        norm=LogNorm()
    )
    plt.gca().invert_xaxis()
    plt.gca().invert_yaxis()
    plt.gca().set_title(cat_labels[i])
    plt.colorbar()
    plt.show()

<Figure size 640x480 with 2 Axes>

<Figure size 640x480 with 2 Axes>

<Figure size 640x480 with 2 Axes>

<Figure size 640x480 with 2 Axes>

<Figure size 640x480 with 2 Axes>

<Figure size 640x480 with 2 Axes>

... (3 outputs left out)

<Figure size 640x480 with 2 Axes>

<Figure size 640x480 with 2 Axes>

<Figure size 640x480 with 2 Axes>

<Figure size 640x480 with 2 Axes>

In [79]:
np.mean(np.exp(params_comb[:,0]/5)*params_err_comb[:,0] > 0.5)

0.03621860231255264

In [94]:
plt.hist(lndet_cov[np.isfinite(lndet_cov)], bins=100)

(array([4.00000e+00, 2.50000e+01, 5.80000e+01, 9.00000e+01, 2.66000e+02,
        5.43000e+02, 1.16600e+03, 2.34100e+03, 4.58100e+03, 8.58100e+03,
        1.51530e+04, 2.23830e+04, 3.01920e+04, 4.21090e+04, 5.29500e+04,
        5.34610e+04, 5.30080e+04, 6.91570e+04, 1.46722e+05, 1.68393e+05,
        1.05285e+05, 2.52467e+05, 6.26800e+05, 6.67720e+05, 5.86614e+05,
        3.78245e+05, 1.26249e+05, 2.93150e+04, 2.54800e+04, 3.08100e+03,
        1.16800e+03, 6.74230e+04, 6.44000e+04, 1.23000e+02, 4.30000e+01,
        1.80000e+01, 1.00000e+01, 7.00000e+00, 2.00000e+00, 0.00000e+00,
        1.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00,
        0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00,
        0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00,
        0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00,
        0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00,
        0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00

<Figure size 640x480 with 1 Axes>

In [322]:
import importlib
import plot_utils
importlib.reload(plot_utils)

<module 'plot_utils' from '/data/beegfs/astro-storage/groups/green/green/projects/merge_coordinates/plot_utils.py'>

In [129]:
teff_err = np.exp(params_comb[:,0]/5)*params_err_comb[:,0] # in kiloKelvin
idx = (
    (teff_err > 0.9)
  #   (
  #       (params_err_comb[:,0] > 0.5)
  #     & (teff_err > 0.5)
  #   )
  # | (params_err_comb[:,1] > 0.5)
  # | (params_err_comb[:,2] > 0.5)
  # | (lndet_cov > 0.)
    # (params_err_comb[:,0] > 0.5)
)
print(np.mean(idx))

fig,ax = plt.subplots(1, 1, figsize=(7,6))
im = plot_utils.hist2d_reduce(
    params_comb[:,0],
    params_comb[:,1],
    params_err_comb[:,0],
    xlim=xlim,
    ylim=ylim,
    ax=ax,
    hist_kw={'bins':100},
    imshow_kw={'norm':LogNorm()}
)
ax.invert_xaxis()
ax.invert_yaxis()
fig.colorbar(im, ax=ax)
plt.show()

# plt.hist2d(
#     params_comb[idx,0],
#     params_comb[idx,1],
#     range=[xlim,ylim],
#     bins=100,
#     norm=LogNorm()
# )
# plt.gca().invert_xaxis()
# plt.gca().invert_yaxis()
# plt.gca().set_title('Combined (high uncertainties)')
# plt.colorbar()
# plt.show()

# plt.hist2d(
#     params_comb[~idx,0],
#     params_comb[~idx,1],
#     range=[xlim,ylim],
#     bins=100,
#     norm=LogNorm()
# )
# plt.gca().invert_xaxis()
# plt.gca().invert_yaxis()
# plt.gca().set_title('Combined (low uncertainties)')
# plt.colorbar()
# plt.show()

0.0022072777767146753


<Figure size 700x600 with 2 Axes>

In [214]:
np.unique(sid_counts, return_counts=True)

(array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
        35, 36, 37, 38, 39, 43, 45]),
 array([3121946,  331186,   78119,   26855,   13681,    8859,    6028,
           4496,    3253,    2202,    1612,    1279,    1034,     976,
            936,     696,     549,     451,     336,     291,     187,
            161,     127,      72,      72,      73,      84,      89,
             70,      32,      13,       7,       2,       2,       7,
              6,       2,       5,       1,       1,       1]))

In [160]:
xlim = (-2, 3.5)
ylim = (-0.5, 5.5)

plt.hist2d(
    t['x'][:,0],
    t['x'][:,1],
    range=[xlim,ylim],
    bins=100,
    norm=LogNorm()
)
plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.show()

plt.hist2d(y[:,0], y[:,1], range=[xlim,ylim], bins=100, norm=LogNorm())
plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.show()

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

In [544]:
harmonized_tables[0].colnames

['params_orig',
 'params_err_orig',
 'params_harm',
 'params_cov_harm',
 'gdr3_source_id']

In [545]:
harmonized_tables[0]['params_orig'].shape

(575192, 3)

# Paper plots

In [420]:
# PASP dimensions
text_width = 7.10    # in inches
column_width = 3.35 # in inches

In [412]:
# cat_labels = [
#     'APOGEE (DR17)',
#     'LAMOST MRS',
#     'LAMOST HotPayne',
#     'GALAH (DR4)',
#     'LAMOST DD-Payne (DR9)',
#     'LAMOST M-Stars (Ding)',
#     'VMP (Li)',
#     'LAMOST M-Dwarfs (Zhang)',
#     'DESI (DR1)',
#     'DESI (Zhang)',
#     'SDSS-V (ASPCAP)',
#     'SDSS-V (BOSSNet)',
#     # 'RAVE (DR6)',
#     # 'PIGS'
# ]

In [ ]:
idx = (
    np.all(np.isfinite(comb_dict['params_comb']), axis=1)
  & (comb_dict['lndet_cov'] < 6*np.log(0.2))
)
print(f'Merged harmonized catalog: {np.count_nonzero(idx):,} ({np.mean(idx):.2%}) good sources.')

fig,ax_arr = plt.subplots(1,2, figsize=(8,4), layout='tight')

ax = ax_arr[0]
ax.hist2d(comb_dict['params_comb'][idx,0], # T_eff
          comb_dict['params_comb'][idx,2], # log(g)
          bins=100, norm=LogNorm())
ax.invert_xaxis()
ax.invert_yaxis()
ax.set_xlabel(r'$5\ln\left(T_\mathrm{eff}/5000\,\mathrm{K}\right)$')
ax.set_ylabel(r'$\log g$')
ax.grid(True, alpha=0.2)

ax = ax_arr[1]
ax.hist2d(comb_dict['params_comb'][idx,0], # T_eff
          comb_dict['params_comb'][idx,1], # [Fe/H]
          bins=100, norm=LogNorm())
ax.invert_xaxis()
ax.set_xlabel(r'$5\ln\left(T_\mathrm{eff}/5000\,\mathrm{K}\right)$')
ax.set_ylabel(r'$\left[\mathrm{Fe/H}\right]$')
ax.grid(True, alpha=0.2)

plt.show()

In [441]:
from matplotlib.ticker import FixedLocator, MultipleLocator

highlight_colors = distinctipy.get_colors(
    len(harmonized_tables),
    exclude_colors=[(0,0,0)],#,(1,1,1)],
    rng=5
)

n_plot = 1024#*16

teff_lim = (-2.8, 5.0)
logg_lim = (-0.8, 6.0)
feh_lim = (-4.0, 0.5)

teff_major_ticks = np.array([5, 10, 20, 30])
teff_minor_ticks = np.arange(3, 30, 1)

with plt.style.context('dark_background'):
    for key in ('orig','harm'):
        fig,ax_arr = plt.subplots(
            1,2, figsize=(text_width,0.5*text_width),
            layout='constrained'
        )
        
        for i,t in enumerate(harmonized_tables):
            p = t[f'params_{key}'][:n_plot]
            kw = dict(
                color=highlight_colors[i],
                alpha=0.2,
                s=4,
                edgecolors='none',
                rasterized=True,
                label=cat_labels[i]
            )
            ax_arr[0].scatter(p[:,0], p[:,2], **kw)
            ax_arr[1].scatter(p[:,0], p[:,1], **kw)
        
        for ax in ax_arr:
            ax.set_aspect('auto')
            
            ax.xaxis.set_major_locator(FixedLocator(5*np.log(teff_major_ticks/5)))
            ax.xaxis.set_minor_locator(FixedLocator(5*np.log(teff_minor_ticks/5)))
            ax.set_xticklabels([f'${x:.0f}$' for x in teff_major_ticks])
            ax.set_xlim(teff_lim)
            ax.invert_xaxis()
        
            ax.yaxis.set_major_locator(MultipleLocator(2))
            ax.yaxis.set_minor_locator(MultipleLocator(0.5))
        
            ax.set_xlabel(r'$T_{\mathrm{eff}} / 1000\,\mathrm{K}$')
        
        ax_arr[0].set_ylim(logg_lim)
        ax_arr[1].set_ylim(feh_lim)
        
        ax_arr[0].invert_yaxis()
        ax_arr[0].set_ylabel(r'$\log g$')
        ax_arr[1].set_ylabel(r'$\left[\mathrm{Fe/H}\right]$')
        
        leg = ax_arr[0].legend(
            loc='upper left',
            markerscale=2.5,
            handletextpad=0.,
            fontsize=6
        )
        for handle in leg.legend_handles:
            handle.set_alpha(1.0)
        
        fig.savefig(f'catalogs_overview_{key}.svg', dpi=200)
        
        plt.show()

<Figure size 710x355 with 2 Axes>

<Figure size 710x355 with 2 Axes>

In [621]:
from matplotlib.ticker import FixedLocator, MultipleLocator

highlight_idx = [0, 8]
highlight_colors = distinctipy.get_colors(
    len(highlight_idx),
    exclude_colors=[(0,0,0),(1,1,1)],
    rng=5
)

fig,ax_arr = plt.subplots(
    1,2, figsize=(text_width,0.4*text_width),
    sharex=True, sharey=True,
    layout='constrained'
)

n_plot = None#1024*16

teff_bins = np.linspace(-3.3, 8.7, 100)
logg_bins = np.linspace(-0.8, 6.0, 100)

cmap = plt.get_cmap('Grays').copy()
cmap.set_bad('w')

teff_major_ticks = np.array([5, 10, 20, 30])
teff_minor_ticks = np.arange(3, 30, 1)

for ax,key in zip(ax_arr, ['params_orig', 'params_harm']):
    n_tot = None
    
    for i,t in enumerate(harmonized_tables):
        # if i == 8:
        #     continue
        
        p = t[key][:n_plot]
        n = np.histogram2d(
            p[:,0], p[:,2],
            bins=[teff_bins,logg_bins]
        )[0]
        if n_tot is None:
            n_tot = n
        else:
            n_tot += n
    
    ax.imshow(
        n_tot.T,
        origin='lower',
        extent=[teff_bins[0],teff_bins[-1],logg_bins[0],logg_bins[-1]],
        cmap=cmap,
        norm=LogNorm()
    )

    # Highlight this catalog?
    for i,j in enumerate(highlight_idx):
        p = harmonized_tables[j][key][:256]
        label = cat_labels[j].replace(' ',r'\ ')
        ax.scatter(
            p[:,0], p[:,2],
            edgecolors='none', alpha=0.8, s=2,
            color=highlight_colors[i],
            label=fr'$\mathrm{{{label}}}$'
        )

    ax.set_aspect('auto')
    ax.invert_xaxis()
    ax.invert_yaxis()
    
    ax.xaxis.set_major_locator(FixedLocator(5*np.log(teff_major_ticks/5)))
    ax.xaxis.set_minor_locator(FixedLocator(5*np.log(teff_minor_ticks/5)))
    ax.set_xticklabels([f'${x:.0f}$' for x in teff_major_ticks])

    ax.yaxis.set_major_locator(MultipleLocator(2))
    ax.yaxis.set_minor_locator(MultipleLocator(0.5))

    ax.set_xlabel(r'$T_{\mathrm{eff}} / 1000\,\mathrm{K}$')

if len(highlight_idx):
    leg = ax_arr[0].legend(
        loc='upper left',
        markerscale=2.5,
        handletextpad=0.,
        fontsize=9
    )
    for handle in leg.legend_handles:
        handle.set_alpha(1.0)

ax_arr[0].set_ylabel(r'$\log g$')

ax_arr[0].set_title(r'$\mathrm{input\ parameters}$')
ax_arr[1].set_title(r'$\mathrm{harmonized\ parameters}$')

fig.savefig('kiel_density_apogee_desi.pdf', dpi=300)

plt.show()

<Figure size 600x300 with 2 Axes>

In [648]:
print(cat_labels)

['APOGEE', 'LAMOST MRS', 'LAMOST HotPayne', 'GALAH', 'LAMOST DD-Payne', 'LAMOST M-Stars (Ding)', 'VMP', 'LAMOST M-Dwarfs (Zhang)', 'DESI (DR1)', 'DESI (Zhang)', 'SDSS-V (ASPCAP)', 'SDSS-V (BOSSNet)']


In [678]:
pair_idx = (0, 8)

n_plot = None#1024*16

idx0 = (
    (pair_dict['c0']==pair_idx[0])
  & (pair_dict['c1']==pair_idx[1])
)
idx1 = (
    (pair_dict['c1']==pair_idx[0])
  & (pair_dict['c0']==pair_idx[1])
)
x = np.concatenate([pair_dict['x0'][idx0], pair_dict['x1'][idx1]], axis=0)
y = np.concatenate([pair_dict['x1'][idx0], pair_dict['x0'][idx1]], axis=0)

highlight_colors = distinctipy.get_colors(
    2,
    exclude_colors=[(0,0,0),(1,1,1)],
    rng=5
)

fig,ax_arr = plt.subplots(
    1,3, figsize=(text_width,0.4*text_width),
    width_ratios=[1,1,0.05],
    # sharex=True, sharey=True,
    layout='constrained'
)

teff_bins = np.linspace(-3.3, 8.7, 100)
logg_bins = np.linspace(-0.8, 6.0, 100)

cmap = plt.get_cmap('Grays').copy()
cmap.set_bad('w')

teff_major_ticks = np.array([5, 10, 20, 30])
teff_minor_ticks = np.arange(3, 30, 1)

norm = LogNorm()

for ax,key in zip(ax_arr, ['params_orig', 'params_harm']):
    n_tot = None
    
    for i,t in enumerate(harmonized_tables):
        # if i == 8:
        #     continue
        
        p = t[key][:n_plot]
        n = np.histogram2d(
            p[:,0], p[:,2],
            bins=[teff_bins,logg_bins]
        )[0]
        if n_tot is None:
            n_tot = n
        else:
            n_tot += n
    
    im = ax.imshow(
        n_tot.T,
        origin='lower',
        extent=[teff_bins[0],teff_bins[-1],
                logg_bins[0],logg_bins[-1]],
        cmap=cmap,
        norm=norm
    )
    
    # Plot pairs
    for i,(j,p) in enumerate(zip(pair_idx,[x,y])):
        p = p[:256]
        if key == 'params_harm':
            p = bij(p, c=j*np.ones(len(p))).numpy()
        label = cat_labels[j].replace(' ',r'\ ')
        ax.scatter(
            p[:,0], p[:,2],
            edgecolors='none', alpha=0.8, s=2,
            color=highlight_colors[i],
            label=fr'$\mathrm{{{label}}}$'
        )

    ax.set_xlim(teff_bins[0], teff_bins[-1])
    ax.set_ylim(logg_bins[0], logg_bins[-1])

    ax.set_aspect('auto')
    ax.invert_xaxis()
    ax.invert_yaxis()
    
    ax.xaxis.set_major_locator(FixedLocator(5*np.log(teff_major_ticks/5)))
    ax.xaxis.set_minor_locator(FixedLocator(5*np.log(teff_minor_ticks/5)))
    ax.set_xticklabels([f'${x:.0f}$' for x in teff_major_ticks])

    ax.yaxis.set_major_locator(MultipleLocator(2))
    ax.yaxis.set_minor_locator(MultipleLocator(0.5))

    ax.set_xlabel(r'$T_{\mathrm{eff}} / 1000\,\mathrm{K}$')

leg = ax_arr[0].legend(
    loc='upper left',
    markerscale=2.5,
    handletextpad=0.,
    fontsize=9
)
for handle in leg.legend_handles:
    handle.set_alpha(1.0)

ax_arr[0].set_ylabel(r'$\log g$')

ax_arr[0].set_title(r'$\mathrm{input\ parameters}$')
ax_arr[1].set_title(r'$\mathrm{harmonized\ parameters}$')

ax_arr[1].set_yticklabels([])

cb = fig.colorbar(
    im, cax=ax_arr[2],
    label=r'$\mathrm{counts}$',
    extend='min'
)

fig.savefig(
    f'kiel_density_pairs_{pair_idx[0]}_{pair_idx[1]}.pdf',
    dpi=300
)

plt.show()

<Figure size 710x284 with 3 Axes>

In [680]:
pair_idx = (0, 8)

# Get paired stars
idx0 = (
    (pair_dict['c0']==pair_idx[0])
  & (pair_dict['c1']==pair_idx[1])
)
idx1 = (
    (pair_dict['c1']==pair_idx[0])
  & (pair_dict['c0']==pair_idx[1])
)
x0 = np.concatenate([pair_dict['x0'][idx0], pair_dict['x1'][idx1]], axis=0)
y0 = np.concatenate([pair_dict['x1'][idx0], pair_dict['x0'][idx1]], axis=0)

# Transform to harmonized space
xp = bij(x0, c=pair_idx[0]*np.ones(len(x0))).numpy()
yp = bij(y0, c=pair_idx[1]*np.ones(len(y0))).numpy()

fig,ax_arr = plt.subplots(
    1, 2, figsize=(text_width,0.4*text_width),
    layout='constrained'
)

labels = [
    cat_labels[i].replace(' ',r'\ ').replace('(','').replace(')','')
    for i in pair_idx
]

for ax,(x,y) in zip(ax_arr, [(x0,y0),(xp,yp)]):
    plot_utils.plot_corr(
        ax, x[:,1], y[:,1],
        x_lim=(-2.5,0.6),
        d_max=0.6,
        diff=True,
        bins=(50,31),
        pct=(16,50,84),
        cmap='binary',
        envelope_kw={}
    )

    ax.xaxis.set_major_locator(MultipleLocator(1.0))
    ax.yaxis.set_major_locator(MultipleLocator(0.3))
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())

    ax.axhline(0, color='r', alpha=0.3)

    ax.set_xlabel(rf'$\left[\mathrm{{Fe/H}}\right]\ (\mathrm{{{labels[0]}}})$')

ax_arr[1].set_yticklabels([])

ax_arr[0].set_ylabel(
    rf'$\Delta\left[\mathrm{{Fe/H}}\right]\ (\mathrm{{{labels[1]}-{labels[0]}}})$'
)

ax_arr[0].set_title(r'$\mathrm{input\ parameters}$')
ax_arr[1].set_title(r'$\mathrm{harmonized\ parameters}$')

fig.savefig(
    f'feh_correlation_{pair_idx[0]}_{pair_idx[1]}.pdf',
    dpi=300
)

plt.show()

<Figure size 710x284 with 2 Axes>

In [688]:
bij.conditional_velocity?

Signature: bij.conditional_velocity(t, x, c)
Docstring: <no docstring>
File:      /tmp/ipykernel_2991738/386281407.py
Type:      method

In [710]:
v[...,0].ndim

2

In [798]:
plt.rcParams['figure.dpi'] = 150

In [810]:
cat_idx = 0

teff_bins = np.linspace(-2.5, 3, 65).astype('f4')
logg_bins = np.linspace(-0.5, 5.3, 65).astype('f4')

# teff_bins = np.linspace(-5, 10, 33).astype('f4')
# logg_bins = np.linspace(-2, 8.0, 33).astype('f4')

teff_centers = 0.5 * (teff_bins[1:] + teff_bins[:-1])
logg_centers = 0.5 * (logg_bins[1:] + logg_bins[:-1])
teff_centers = teff_centers[::4]
logg_centers = logg_centers[::4]

teff_grid, logg_grid = np.meshgrid(teff_centers, logg_centers, indexing='ij')
feh_grid = 0*np.ones_like(teff_grid)

params_grid = np.stack([teff_grid, feh_grid, logg_grid], axis=2)
shape = params_grid.shape
params_grid.shape = (-1,3)

c = cat_idx * np.ones(params_grid.shape[0], dtype='f4')

v = bij.conditional_velocity(1., params_grid, c).numpy()
v.shape = shape

# Example trajectories
x0 = np.array([
    [1.5, 0., 3.8],
    [1.6, 0., 3.],
    [-1.5, 0., 1.]
], dtype='f4')
c = cat_idx * np.ones(len(x0), dtype='f4')
xt = bij.return_path(x0, c, 100).numpy()
# print(xt)

fig,ax = plt.subplots(
    1, 1,
    figsize=(column_width,column_width),
    layout='constrained'
)

# n_tot = None
# for i,t in enumerate(harmonized_tables):
p = tables[cat_idx]['x']
n_tot = np.histogram2d(
    p[:,0], p[:,2],
    bins=[teff_bins,logg_bins]
)[0]
    # if n_tot is None:
    #     n_tot = n
    # else:
    #     n_tot += n

# Background density
cmap = plt.get_cmap('Grays').copy()
cmap.set_bad('w')
im = ax.imshow(
    n_tot.T,
    origin='lower',
    extent=[teff_bins[0],teff_bins[-1],
            logg_bins[0],logg_bins[-1]],
    cmap=cmap,
    norm=LogNorm(),
    alpha=0.5
)

# Arrow plot
ax.quiver(
    teff_grid, logg_grid,
    -v[...,0], -v[...,2], # Have to take negative, because of inverted axes
    scale=10,
    color='b'
)

# print(xt.shape)
for i,x in enumerate(x0):
    ax.scatter([x[0]], [x[2]], c='m', s=9)
    ax.plot(xt[:,i,0], xt[:,i,2], c='m', alpha=1)
    ax.arrow(
        xt[-1,i,0],
        xt[-1,i,2],
        np.diff(xt[-2:,i,0])[0],
        np.diff(xt[-2:,i,2])[0],
        shape='full',
        color='m',
        lw=0,
        length_includes_head=False,
        head_width=0.15,
        head_length=0.1
    )

ax.set_aspect('auto')
ax.invert_xaxis()
ax.invert_yaxis()

teff_major_ticks = np.array([4, 6, 8, 12])
teff_minor_ticks = np.arange(1, 15, 1)

ax.xaxis.set_major_locator(FixedLocator(5*np.log(teff_major_ticks/5)))
ax.xaxis.set_minor_locator(FixedLocator(5*np.log(teff_minor_ticks/5)))
ax.set_xticklabels([f'${x:.0f}$' for x in teff_major_ticks])

ax.yaxis.set_major_locator(MultipleLocator(2))
ax.yaxis.set_minor_locator(MultipleLocator(0.5))

ax.set_xlabel(r'$T_{\mathrm{eff}} / 1000\,\mathrm{K}$')
ax.set_ylabel(r'$\log g$')

fig.savefig(
    f'parameter_trajectories_{cat_idx}.pdf',
    dpi=300
)

plt.show()

<Figure size 502.5x502.5 with 1 Axes>

In [725]:
cat_labels

['APOGEE (DR17)',
 'LAMOST MRS',
 'LAMOST HotPayne',
 'GALAH',
 'LAMOST DD-Payne',
 'LAMOST M-Stars (Ding)',
 'VMP',
 'LAMOST M-Dwarfs (Zhang)',
 'DESI (DR1)',
 'DESI (Zhang)',
 'SDSS-V (ASPCAP)',
 'SDSS-V (BOSSNet)']

In [802]:
tables[0]

id,x,sigma_x,x_err,ALPHA_M,RA,DEC
int64,float32[3],float32[3],float32[3],float32,float64,float64
538028216707715712,-1.4733173 .. 0.9045983,0.011389498 .. 0.023732744,0.1 .. 0.1,0.036922,0.000103,74.285408
2413929812587459072,0.4781623 .. 4.304115,0.02528456 .. 0.026512949,0.1 .. 0.1,0.0909783,0.000832,-19.413851
422596679964513792,0.9940746 .. 3.6739705,0.03888883 .. 0.03225054,0.1399123 .. 0.1,0.0291593,0.001335,57.61953
422596679964513792,1.0448419 .. 3.7155607,0.0353249 .. 0.027796507,0.16481212 .. 0.1,0.0480663,0.001335,57.61953
4689447878791422208,-0.46561825 .. 1.4988511,0.025366064 .. 0.083238214,0.1 .. 0.1,0.008442001,0.001467,-73.394287
421077597267551104,0.031166488 .. 3.456132,0.014200249 .. 0.025588945,0.1 .. 0.1,0.055704,0.00285,57.173164
2772097619417608704,0.8661947 .. 4.1231923,0.038529098 .. 0.03483912,0.1 .. 0.1,0.0124087995,0.004322,15.430942
421086363305436800,-0.02546376 .. 2.3882506,0.018352715 .. 0.036898185,0.1 .. 0.1,0.10715,0.005558,57.354549
431594980053422720,-0.32913595 .. 2.245527,0.013698991 .. 0.026309058,0.1 .. 0.1,0.0076940004,0.008802,63.463078


In [323]:
! ls frames | head -n 10

kiel_control_0_000.png
kiel_control_0_001.png
kiel_control_0_002.png
kiel_control_0_003.png
kiel_control_0_004.png
kiel_control_0_005.png
kiel_control_0_006.png
kiel_control_0_007.png
kiel_control_0_008.png
kiel_control_0_009.png
ls: write error: Broken pipe


In [387]:
! tar -cf figures.tar `ls -rt *.pdf | tail -n 9`

In [390]:
fname = 'merged_params_catalog_clean.h5'
cat_merged = Table.read(fname)
cat_merged

gdr3_source_id,params_est,params_err,params_cov,lndet_cov,params_source
int64,float32[3],float32[3],"float32[3,3]",float32,uint16
4295806720,0.39563847 .. 4.625024,0.030082395 .. 0.0321739,0.0022249725 .. 0.011625833,-16.328497,256
34361129088,-0.24699377 .. 4.6130257,0.030058533 .. 0.030389503,0.0018749549 .. 0.004849935,-17.423239,16
828929527040,-0.46216515 .. 4.5656376,0.030000903 .. 0.03002311,0.00023280768 .. 0.0011777342,-23.008606,256
1275606125952,0.36135522 .. 4.4285436,0.030139435 .. 0.031211158,0.0028957785 .. 0.00861025,-16.076918,16
1653563247744,1.1117179 .. 4.0936456,0.030022355 .. 0.035725147,0.001158367 .. 0.019398095,-15.483305,16
2851858288640,0.7922202 .. 4.223821,0.03000514 .. 0.03000704,0.00055533706 .. 0.00064996135,-22.577585,17
2920577765120,0.6959468 .. 4.1762505,0.030036407 .. 0.0318526,0.0014784056 .. 0.010704584,-16.83661,16
3062312520192,0.72492605 .. 4.2306337,0.030006213 .. 0.030273197,0.00061062706 .. 0.0040578865,-19.010712,16
3332894779520,-0.06486145 .. 3.6420054,0.030001482 .. 0.03000305,0.00029812797 .. 0.0004278464,-24.070944,17


In [404]:
idx = (np.sum(cat_merged['params_est']**2,axis=1) > 1e-5)

with plt.style.context('dark_background'):
    plot_teff_logg_feh(
        5000*np.exp(cat_merged['params_est'][idx,0]/5.),
        cat_merged['params_est'][idx,2],
        cat_merged['params_est'][idx,1]
    )
    plt.savefig('harmonized_catalog_overview.svg', dpi=200)
    plt.show()

<Figure size 800x400 with 2 Axes>

# Videos (alternate)

In [372]:
from PIL import Image
import math

def detect_crop_box(fname):
    """
    Detects the bounding box of non-white pixels in an image
    and returns an ffmpeg crop string: 'crop=w:h:x:y', ensuring
    width and height are even by rounding up.
    
    Args:
        fname (str): Path to the input image.
        
    Returns:
        str: Crop filter string for ffmpeg (e.g., 'crop=1202:600:28:50')
    """
    img = Image.open(fname).convert("RGB")
    arr = np.asarray(img)

    non_white = np.any(arr != [255, 255, 255], axis=-1)
    rows = np.any(non_white, axis=1)
    cols = np.any(non_white, axis=0)

    if not np.any(rows) or not np.any(cols):
        # Entire image is white
        h, w = arr.shape[:2]
        w_even = math.ceil(w / 2) * 2
        h_even = math.ceil(h / 2) * 2
        return f"crop={w_even}:{h_even}:0:0"

    top = np.argmax(rows)
    bottom = len(rows) - np.argmax(rows[::-1])
    left = np.argmax(cols)
    right = len(cols) - np.argmax(cols[::-1])

    crop_w = right - left
    crop_h = bottom - top

    # Round width and height up to the next even number
    crop_w_even = math.ceil(crop_w / 2) * 2
    crop_h_even = math.ceil(crop_h / 2) * 2

    return f"crop={crop_w_even}:{crop_h_even}:{left}:{top},pad=ceil(iw/2)*2:ceil(ih/2)*2"

In [374]:
from glob import glob

def gen_video(base_fname, out_suffix='', crf=20, n_video_frames=None):
    if n_video_frames is None:
        n_video_frames = len(glob(f'frames/{base_fname}_???.png'))
        print(f'Detected {n_video_frames} frames.')

    crop_params = detect_crop_box(f'frames/{base_fname}_000.png')
    print(crop_params)
    
    !ffmpeg -y -f image2 -framerate 30 \
            -i frames/{base_fname}_%03d.png \
            -frames:v {n_video_frames} \
            -vf "{crop_params}" \
            -profile:v main -movflags +faststart -pix_fmt yuv420p \
            -c:v libx264 -crf {crf} \
            videos/{base_fname}{out_suffix}.mp4

    # ! ffmpeg -y -f image2 -framerate 30 \
    #          -i "frames/{base_fname}_%03d.png" \
    #          -frames:v {n_video_frames} \
    #          -profile:v main -movflags +faststart -pix_fmt yuv420p \
    #          -c:v libx264 -crf {crf} \
    #          videos/{base_fname}{out_suffix}.mp4

In [388]:
for j in range(14):
    base_fname = f'kiel_control_{j}'
    n = len(glob(f'frames/{base_fname}_???.png'))
    if n > 0:
        print('')
        print(j, n)
        print('')
        gen_video(base_fname, out_suffix='_lq', crf=25, n_video_frames=n)


0 120

crop=1312:1318:131:190,pad=ceil(iw/2)*2:ceil(ih/2)*2
ffmpeg version 7.0.2-static https://johnvansickle.com/ffmpeg/  Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 8 (Debian 8.3.0-6)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-debug --disable-ffplay --disable-indev=sndio --disable-outdev=sndio --cc=gcc --enable-fontconfig --enable-frei0r --enable-gnutls --enable-gmp --enable-libgme --enable-gray --enable-libaom --enable-libfribidi --enable-libass --enable-libvmaf --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-librubberband --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libvorbis --enable-libopus --enable-libtheora --enable-libvidstab --enable-libvo-amrwbenc --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libdav1d --enable-libxvid --enable-libzvbi --enable-libzimg
  libavutil      59.  8.100 / 59.  8.

In [389]:
! tar -cf videos_lq.tar videos/kiel_control_*_lq.mp4

In [381]:
for j in range(14):
    for k in range(j+1,14):
        base_fname = f'param1v1_diff_pair_{j}_{k}'
        n = len(glob(f'frames/{base_fname}_???.png'))
        if n > 0:
            print('')
            print(j, k, n)
            print('')
            gen_video(base_fname, out_suffix='_lq', crf=25, n_video_frames=n)


0 1 120

crop=1786:494:8:74,pad=ceil(iw/2)*2:ceil(ih/2)*2
ffmpeg version 7.0.2-static https://johnvansickle.com/ffmpeg/  Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 8 (Debian 8.3.0-6)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-debug --disable-ffplay --disable-indev=sndio --disable-outdev=sndio --cc=gcc --enable-fontconfig --enable-frei0r --enable-gnutls --enable-gmp --enable-libgme --enable-gray --enable-libaom --enable-libfribidi --enable-libass --enable-libvmaf --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-librubberband --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libvorbis --enable-libopus --enable-libtheora --enable-libvidstab --enable-libvo-amrwbenc --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libdav1d --enable-libxvid --enable-libzvbi --enable-libzimg
  libavutil      59.  8.100 / 59.  8.10

In [382]:
! tar -cf videos_lq.tar videos/param1v1_diff_pair_*_*_lq.mp4